### ERAA Workflow Automation
The topic of the workflow is the European Resource Adequacy Assessment (ERAA) 2025, and how it can be carried out using the full PLEXOS® Platform Ecosystem (Desktop, Cloud, CLI, Python SDK and Python API) as well as all Phases (LT, PASA, MT and ST) 


#### 0. Configuration and SDK Initiation



In [ ]:
# ------------------------------------------------------------
# PLEXOS ERAA Import Framework
# Version: stable
# Uses:
#   - PropertyName2EnumId dynamic resolution
#   - Scenario-specific AddProperty
#   - Membership-safe writes
#   - Enum numeric writes for enum properties
# ------------------------------------------------------------

import importlib
import os
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable


def ensure(pkg):
    try:
        importlib.import_module(pkg.split("==")[0].replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

# Core importer dependencies
ensure("pandas")
ensure("openpyxl")
ensure("dataclasses-json")
ensure("requests")
ensure("python-dateutil")
ensure("plexosdb")

from eecloud.cloudsdk import CloudSDK  # same SDK you used before
# from eecloud.models import *  # optional (only when you need Contracts_* types)

# ------------------------------------------------------------
# Central configuration
# By default this resolves relative to the current importer workspace.
# You can override with ERAA_MVP_BASE_DIR or ERAA_MVP_DATABASE_DIR.
# ------------------------------------------------------------
IMPORTER_DIR = Path.cwd().resolve()
BASE_DIR = Path(os.environ["ERAA_MVP_BASE_DIR"]).resolve() if os.environ.get("ERAA_MVP_BASE_DIR") else IMPORTER_DIR.parent
DATABASE_DIR = Path(os.environ["ERAA_MVP_DATABASE_DIR"]).resolve() if os.environ.get("ERAA_MVP_DATABASE_DIR") else (BASE_DIR / "12R02")
if not DATABASE_DIR.exists():
    DATABASE_DIR = BASE_DIR / "Database"
DATA_FILES_ROOT = DATABASE_DIR / "Data Files"
XML_CANDIDATES = [
    DATABASE_DIR / "ERAA_MVP_Github.xml",
]
XML_PATH = next((p for p in XML_CANDIDATES if p.exists()), XML_CANDIDATES[0])
STUDY_ID = "2972237e-4c06-491c-86cb-2a81bfd0d77e"


def require_study_id() -> str:
    study_id = globals().get("STUDY_ID")
    if not isinstance(study_id, str) or not study_id.strip():
        raise RuntimeError("STUDY_ID must be set in the configuration cell before running this step.")
    return study_id.strip()


BIDDING_ZONE_XLSX = BASE_DIR / "Bidding_Zone_List.xlsx"

COMMON_DATA_DIR = BASE_DIR / "ERAA_2025_CommonData"
DASHBOARD_RAWDATA_DIR = BASE_DIR / "ERAA_2025_Dashboard_RawData"
DEMAND_DATA_DIR = BASE_DIR / "ERAA_2025_DemandData"
PECD_RES_DIR = BASE_DIR / "ERAA_2025_PECDRES"
OTHER_DATA_DIR = BASE_DIR / "ERAA_2025_OtherData"
NTC_DIR_ROOT = BASE_DIR / "ERAA_2025_NTCs"
ECONOMIC_TECHNICAL_INVESTMENT_DIR = BASE_DIR / "ERAA_2025_EconomicTechnicalInvestmentParameters"

API_DIR = Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0 API")
BIN_DIR = Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0")
DEFAULT_WINDOWS_CLI = r"C:\Users\jonathan.kromann\AppData\Local\Programs\PLEXOS.Cloud\plexos-cloud.exe"
CLI_PATH = os.environ.get("CLOUD_CLI_PATH") or os.environ.get("cloud_cli_path") or DEFAULT_WINDOWS_CLI

pxc = CloudSDK(CLI_PATH)  # prefer explicit CLI_PATH
print("IMPORTER_DIR:", IMPORTER_DIR)
print("BASE_DIR:", BASE_DIR)
print("DATABASE_DIR:", DATABASE_DIR)
print("XML_PATH:", XML_PATH)
print("CLI_PATH:", CLI_PATH)

# Optional local helper from older workflow variants. This notebook does not call it directly.
if str(IMPORTER_DIR) not in sys.path:
    sys.path.insert(0, str(IMPORTER_DIR))

try:
    import utilities
except ModuleNotFoundError:
    utilities = None
    print("[INFO] Optional local helper utilities.py was not found; continuing without it.")

#### 1. Step Decorator, Read Bidding Zone File, Create Regions and Nodes

In [ ]:
import time
from pathlib import Path
from typing import Callable
import sqlite3
import pandas as pd

from plexosdb import PlexosDB
from plexosdb.enums import ClassEnum, CollectionEnum


# --------------------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------------------

BZONE_XLSX = BASE_DIR / "Bidding_Zone_List.xlsx"


# --------------------------------------------------------------------------------------
# Step Decorator
# --------------------------------------------------------------------------------------

def step(name: str) -> Callable:
    def _decorator(fn: Callable) -> Callable:
        def _wrapped(*args, **kwargs):
            print("\n" + "=" * 90)
            print(f"STEP: {name}")
            print("=" * 90)
            t0 = time.time()
            out = fn(*args, **kwargs)
            print(f"[OK] Done: {name} ({time.time() - t0:.2f}s)")
            return out
        return _wrapped
    return _decorator


# --------------------------------------------------------------------------------------
# Read bidding zones
# --------------------------------------------------------------------------------------

@step("Read bidding zones (A2 downward)")
def read_bidding_zones(xlsx_path: Path) -> list[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)

    zones = (
        df.iloc[1:, 0]  # A2 onward
        .dropna()
        .astype(str)
        .map(lambda s: s.strip())
    )

    seen, out = set(), []
    for z in zones:
        if z and z not in seen:
            seen.add(z)
            out.append(z)

    print(f"Found {len(out)} zones. Sample: {out[:10]}")
    return out


bidding_zones = read_bidding_zones(BZONE_XLSX)


# --------------------------------------------------------------------------------------
# SQLite helpers
# --------------------------------------------------------------------------------------

def _get_sqlite_conn(db: PlexosDB) -> sqlite3.Connection:
    candidates = [
        "conn", "_conn", "connection", "_connection", "db", "_db",
        "sqlite", "_sqlite", "sqlite_db", "_sqlite_db",
        "manager", "_manager", "db_manager", "_db_manager",
    ]

    def walk(obj, depth=0):
        if depth > 3 or obj is None:
            return None

        if isinstance(obj, sqlite3.Connection):
            return obj

        for name in candidates:
            if hasattr(obj, name):
                val = getattr(obj, name)
                if isinstance(val, sqlite3.Connection):
                    return val
                found = walk(val, depth + 1)
                if found is not None:
                    return found

        for name in dir(obj):
            if name.startswith("__"):
                continue
            try:
                val = getattr(obj, name)
            except Exception:
                continue
            if isinstance(val, sqlite3.Connection):
                return val

        return None

    conn = walk(db)
    if conn is None:
        raise RuntimeError("Could not locate sqlite3.Connection inside PlexosDB.")

    return conn


def _build_existing_index(db: PlexosDB):
    """
    Build fast lookup sets of existing Nodes, Regions, and Node->Region membership pairs by name.
    This avoids brittle collection_id assumptions and prevents add_membership assertions.
    """
    conn = _get_sqlite_conn(db)
    cur = conn.cursor()

    # Existing Nodes and Regions by name
    cur.execute("SELECT name FROM t_object WHERE class_id=?", (ClassEnum.Node.value,))
    existing_nodes = {r[0] for r in cur.fetchall()}

    cur.execute("SELECT name FROM t_object WHERE class_id=?", (ClassEnum.Region.value,))
    existing_regions = {r[0] for r in cur.fetchall()}

    # Existing memberships: join to names so we can compare by (node_name, region_name)
    cur.execute(
        """
        SELECT p.name AS parent_name, c.name AS child_name
        FROM t_membership m
        JOIN t_object p ON p.object_id = m.parent_object_id
        JOIN t_object c ON c.object_id = m.child_object_id
        WHERE p.class_id = ? AND c.class_id = ?
        """,
        (ClassEnum.Node.value, ClassEnum.Region.value),
    )
    existing_pairs = {(r[0], r[1]) for r in cur.fetchall()}

    return existing_nodes, existing_regions, existing_pairs


@step("STEP - Create Regions + Nodes + Memberships (Node -> Region) [IN-PLACE]")
def step_nodes_regions_memberships_inplace(xml_path: Path, zones: list[str]) -> Path:
    db = PlexosDB.from_xml(str(xml_path))

    # Build an index of what already exists BEFORE we start modifying
    existing_nodes, existing_regions, existing_pairs = _build_existing_index(db)

    created_regions = created_nodes = created_memberships = 0
    skipped_regions = skipped_nodes = skipped_memberships = 0

    for zone in zones:
        # --- Region ---
        if zone in existing_regions:
            skipped_regions += 1
        else:
            db.add_object(ClassEnum.Region, name=zone)
            existing_regions.add(zone)
            created_regions += 1

        # --- Node ---
        if zone in existing_nodes:
            skipped_nodes += 1
        else:
            db.add_object(ClassEnum.Node, name=zone)
            existing_nodes.add(zone)
            created_nodes += 1

        # --- Membership Node(zone) -> Region(zone) ---
        pair = (zone, zone)
        if pair in existing_pairs:
            skipped_memberships += 1
        else:
            try:
                db.add_membership(
                    parent_class_enum=ClassEnum.Node,
                    child_class_enum=ClassEnum.Region,
                    parent_object_name=zone,
                    child_object_name=zone,
                    collection_enum=CollectionEnum.Region,
                )
                existing_pairs.add(pair)
                created_memberships += 1
            except AssertionError:
                existing_pairs.add(pair)
                skipped_memberships += 1

    print(f"Created Regions:        {created_regions}")
    print(f"Created Nodes:          {created_nodes}")
    print(f"Created Memberships:    {created_memberships}")
    print(f"Skipped Regions:        {skipped_regions} (already existed)")
    print(f"Skipped Nodes:          {skipped_nodes} (already existed)")
    print(f"Skipped Memberships:    {skipped_memberships} (already existed)")

    ok = db.to_xml(Path(xml_path))
    if not ok:
        raise RuntimeError(f"Export failed: {xml_path}")

    return xml_path


updated_xml = step_nodes_regions_memberships_inplace(XML_PATH, bidding_zones)
print("Updated in-place:", updated_xml)

#### 2. Create Fuels and Emission Object

In [ ]:
from __future__ import annotations

import time
from pathlib import Path
from typing import Callable
from datetime import datetime
import inspect

import pandas as pd

from plexosdb import PlexosDB
from plexosdb.enums import ClassEnum, CollectionEnum


# --------------------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------------------


COMMON_DATA_XLSX = BASE_DIR / r"ERAA_2025_CommonData\Common Data.xlsx"
COMMODITY_PRICES_XLSX = BASE_DIR / r"ERAA_2025_Dashboard_RawData\Commodity Prices.xlsx"


# --------------------------------------------------------------------------------------
# Step Decorator
# --------------------------------------------------------------------------------------

def step(name: str) -> Callable:
    def _decorator(fn: Callable) -> Callable:
        def _wrapped(*args, **kwargs):
            print("\n" + "=" * 90)
            print(f"STEP: {name}")
            print("=" * 90)
            t0 = time.time()
            out = fn(*args, **kwargs)
            print(f"âœ“ Done: {name} ({time.time() - t0:.2f}s)")
            return out
        return _wrapped
    return _decorator


# --------------------------------------------------------------------------------------
# STEP helpers
# --------------------------------------------------------------------------------------

DENY_FUELS = {"fuel", "fuels"}  # never create these


def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())


# --- Fuel name canonicalization / aliases (DB names vs mapping names) ---
FUEL_NAME_ALIASES_LC = {
    "shale oil": "Oil Shale",
    "oil shale": "Oil Shale",
}


def _is_denied(name: str) -> bool:
    return _clean_cell(name).lower() in DENY_FUELS


# --------------------------------------------------------------------------------------
# STEP â€” Read fuels
# --------------------------------------------------------------------------------------

@step("STEP â€” Read Fuels from C15:C41 + add '<C> - <D>' when 'CCS' in D (D15:D41)")
def read_fuels_with_ccs(xlsx_path: Path) -> set[str]:
    sheets = pd.read_excel(xlsx_path, sheet_name=None, header=None)

    best_name = None
    best_df = None
    best_count = -1

    for name, df in sheets.items():
        c = df.iloc[14:41, 2]  # C15:C41
        count = sum(bool(_clean_cell(v)) and not _is_denied(v) for v in c)
        if count > best_count:
            best_count = count
            best_name = name
            best_df = df

    if best_df is None or best_count <= 0:
        raise RuntimeError("Could not find any non-empty (non-denied) values in C15:C41 in any sheet.")

    print(f"Using sheet: {best_name!r} (non-empty non-denied C15:C41 count={best_count})")

    df = best_df
    c = df.iloc[14:41, 2]  # C15:C41
    d = df.iloc[14:41, 3]  # D15:D41

    base_fuels: set[str] = set()
    fuels_to_create: set[str] = set()

    for v in c:
        fuel = _clean_cell(v)
        if not fuel or _is_denied(fuel):
            continue
        base_fuels.add(fuel)

    fuels_to_create.update(base_fuels)

    for fuel_raw, d_raw in zip(c, d):
        fuel = _clean_cell(fuel_raw)
        if not fuel or _is_denied(fuel):
            continue

        d_str = _clean_cell(d_raw)
        if "CCS" in d_str.upper():
            fuels_to_create.add(f"{fuel} - {d_str}")

    fuels_to_create = {f for f in fuels_to_create if not _is_denied(f)}

    print(f"Base unique fuels (C15:C41): {len(base_fuels)}")
    print("Base sample:", sorted(base_fuels)[:25])

    print(f"Total fuels to create (base + CCS variants): {len(fuels_to_create)}")
    print("Total sample:", sorted(fuels_to_create)[:40])

    return fuels_to_create


# --------------------------------------------------------------------------------------
# STEP â€” Create fuels
# --------------------------------------------------------------------------------------

@step("STEP â€” Create Fuel objects in XML [IN-PLACE] (NO BACKUP) (NEVER create 'Fuel')")
def step_create_fuels_inplace(xml_path: Path, fuels: set[str]) -> Path:
    db = PlexosDB.from_xml(str(xml_path))

    created = skipped = blocked = 0

    for raw_name in sorted(fuels):
        name = _clean_cell(raw_name)
        if not name:
            continue
        if _is_denied(name):
            blocked += 1
            continue

        if db.check_object_exists(ClassEnum.Fuel, name):
            skipped += 1
            continue

        db.add_object(ClassEnum.Fuel, name=name)
        created += 1

    print(f"Created Fuels: {created}")
    print(f"Skipped Fuels: {skipped} (already existed)")
    print(f"Blocked Fuels: {blocked} (refused to create 'Fuel')")

    ok = db.to_xml(xml_path)
    if not ok:
        raise RuntimeError(f"Export failed: {xml_path}")

    PlexosDB.from_xml(str(xml_path))
    return xml_path


# --------------------------------------------------------------------------------------
# STEP â€” CO2 memberships + production rate helpers
# --------------------------------------------------------------------------------------

def _read_co2_production_rates(xlsx_path: Path) -> dict[str, float]:
    """
    1) Base fuel rate: first match of fuel name in C15:C41, value from G of same row.
    2) CCS fuel rate: base fuel rate / 10
    """
    sheets = pd.read_excel(xlsx_path, sheet_name=None, header=None)

    best_name = None
    best_df = None
    best_count = -1
    for name, df in sheets.items():
        c = df.iloc[14:41, 2]  # C15:C41
        count = sum(bool(_clean_cell(v)) and not _is_denied(v) for v in c)
        if count > best_count:
            best_count = count
            best_name = name
            best_df = df

    if best_df is None or best_count <= 0:
        raise RuntimeError("Could not find any non-empty (non-denied) values in C15:C41 in any sheet for CO2 rates.")

    print(f"Using sheet for CO2 rates: {best_name!r}")

    df = best_df
    c = df.iloc[14:41, 2]  # C15:C41 fuel names
    g = df.iloc[14:41, 6]  # G15:G41 production rates

    base_rate: dict[str, float] = {}
    for fuel_raw, rate_raw in zip(c, g):
        fuel = _clean_cell(fuel_raw)
        if not fuel or _is_denied(fuel):
            continue
        if pd.isna(rate_raw):
            continue
        if fuel in base_rate:
            continue
        try:
            val = float(rate_raw)
        except Exception:
            continue
        base_rate[fuel] = val

    print("Base CO2 rates found for fuels:", base_rate)
    return base_rate


@step("STEP â€” Create CO2 + Emission->Fuel memberships + set Production Rate (NO BACKUP)")
def step_co2_memberships_and_rates_addproperty(
    xml_path: Path,
    xlsx_path: Path,
    fuels: set[str],
) -> Path:
    db = PlexosDB.from_xml(str(xml_path))

    if not db.check_object_exists(ClassEnum.Emission, "CO2"):
        db.add_object(ClassEnum.Emission, name="CO2")
        print("Created Emission object: CO2")
    else:
        print("Emission object CO2 already exists")

    created_memberships = skipped_memberships = 0
    for raw_name in sorted(fuels):
        fuel = _clean_cell(raw_name)
        if not fuel or _is_denied(fuel):
            continue
        if not db.check_object_exists(ClassEnum.Fuel, fuel):
            print(f"WARNING: Fuel '{fuel}' not found in DB, skipping membership.")
            continue

        try:
            db.add_membership(
                parent_class_enum=ClassEnum.Emission,
                child_class_enum=ClassEnum.Fuel,
                parent_object_name="CO2",
                child_object_name=fuel,
                collection_enum=CollectionEnum.Fuels,
            )
            created_memberships += 1
        except AssertionError:
            skipped_memberships += 1

    print(
        f"Emission->Fuel memberships: created={created_memberships}, "
        f"skipped(existing)={skipped_memberships}"
    )

    base_rate = _read_co2_production_rates(xlsx_path)

    valid_props = db.list_valid_properties(
        CollectionEnum.Fuels,
        parent_class_enum=ClassEnum.Emission,
        child_class_enum=ClassEnum.Fuel,
    )
    if "Production Rate" not in valid_props:
        raise RuntimeError(
            f"'Production Rate' not in valid properties for "
            f"(Collection=Fuels, Parent=Emission, Child=Fuel).\n"
            f"Valid props: {valid_props}"
        )
    print("Confirmed 'Production Rate' is a valid Emission-Fuels property.")

    written = skipped_no_base = 0
    for raw_name in sorted(fuels):
        fuel = _clean_cell(raw_name)
        if not fuel or _is_denied(fuel):
            continue

        if "CCS" not in fuel.upper():
            base_name = fuel
            val = base_rate.get(base_name)
        else:
            base_name = fuel.split(" - ")[0].strip()
            base_val = base_rate.get(base_name)
            val = base_val / 10.0 if base_val is not None else None

        if val is None:
            print(f"Skipping Production Rate for '{fuel}': no base value found.")
            skipped_no_base += 1
            continue

        try:
            db.add_property(
                ClassEnum.Fuel,
                fuel,
                "Production Rate",
                float(val),
                collection_enum=CollectionEnum.Fuels,
                parent_class_enum=ClassEnum.Emission,
                parent_object_name="CO2",
            )
            written += 1
        except Exception as ex:
            print(f"Failed to write Production Rate for '{fuel}': {ex}")

    print(f"Production Rate entries written: {written}")
    print(f"Skipped (no base value): {skipped_no_base}")

    ok = db.to_xml(xml_path)
    if not ok:
        raise RuntimeError(f"Export failed: {xml_path}")
    return xml_path


# --------------------------------------------------------------------------------------
# STEP helpers
# --------------------------------------------------------------------------------------

try:
    from loguru import logger as _loguru_logger
except Exception:
    _loguru_logger = None

PRICE_SHEET = "Commodity Prices"
ERAA_VERSION = "ERAA 2025"
PRICE_YEARS = [2028, 2030, 2033, 2035]

NAME_MAP = {
    "Oil shale": "Shale oil",
    "Oil Shale": "Shale oil",
}

DATE_RULES = {
    2028: ("2028-01-01 00:00:00", "2029-12-31 00:00:00"),
    2030: ("2030-01-01 00:00:00", "2032-12-31 00:00:00"),
    2033: ("2033-01-01 00:00:00", "2034-12-31 00:00:00"),
    2035: ("2035-01-01 00:00:00", None),
}

def _parse_dt(s: str | None) -> datetime | None:
    if s is None:
        return None
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def _dates_for_year(y: int):
    if y not in DATE_RULES:
        raise RuntimeError(f"No date rule configured for year {y}")
    dfrom_s, dto_s = DATE_RULES[y]
    return _parse_dt(dfrom_s), _parse_dt(dto_s)

def _as_float(x):
    if pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _read_commodity_prices_table(xlsx_path: Path) -> pd.DataFrame:
    df = pd.read_excel(xlsx_path, sheet_name=PRICE_SHEET)
    if df is None or df.shape[1] < 5:
        raise RuntimeError(f"Sheet {PRICE_SHEET!r} not found or has < 5 columns in {xlsx_path}")

    out = pd.DataFrame({
        "Version": df.iloc[:, 0],
        "Year":    df.iloc[:, 1],
        "Name":    df.iloc[:, 2],
        "Unit":    df.iloc[:, 3],
        "Price":   df.iloc[:, 4],
    })

    out["Version"] = out["Version"].apply(_clean_cell)
    out["Name"]    = out["Name"].apply(_clean_cell)
    out["Unit"]    = out["Unit"].apply(_clean_cell)
    out["Year"]    = pd.to_numeric(out["Year"], errors="coerce")
    out["Price"]   = out["Price"].apply(_as_float)
    return out

def _pick_price(df: pd.DataFrame, name: str, year: int):
    sub = df[(df["Name"].str.lower() == name.lower()) & (df["Year"] == year) & df["Price"].notna()]
    if sub.empty:
        return None, None
    return float(sub["Price"].mean()), next((u for u in sub["Unit"].tolist() if u), "")

def _avg_lignite_price(df: pd.DataFrame, year: int):
    sub = df[(df["Year"] == year) & (df["Name"].str.lower().str.startswith("lignite")) & df["Price"].notna()]
    if sub.empty:
        return None, None
    return float(sub["Price"].mean()), next((u for u in sub["Unit"].tolist() if u), "")

def _find_price_collection_and_property(db: PlexosDB, *, kind: str):
    system_name = "System"
    if not db.check_object_exists(ClassEnum.System, system_name):
        systems = db.get_objects(ClassEnum.System)
        if not systems:
            raise RuntimeError("No System object found in DB; cannot attach prices.")
        system_name = systems[0]

    parent_class = ClassEnum.System
    parent_name = system_name

    if kind == "fuel":
        child_class = ClassEnum.Fuel
        candidate_collections = [CollectionEnum.Fuels]
        prop_candidates = ["Fuel Price", "Price", "Commodity Price", "Variable Cost"]
    else:
        child_class = ClassEnum.Emission
        candidate_collections = [CollectionEnum.Emissions]
        prop_candidates = ["Emission Price", "Price", "CO2 Price", "Carbon Price"]

    for col in candidate_collections:
        try:
            props = db.list_valid_properties(col, parent_class_enum=parent_class, child_class_enum=child_class)
        except Exception:
            continue

        props_lower = {p.lower(): p for p in props}
        for pc in prop_candidates:
            if pc.lower() in props_lower:
                return col, props_lower[pc.lower()], parent_class, parent_name

        for p in props:
            if "price" in p.lower():
                return col, p, parent_class, parent_name

    raise RuntimeError(f"Could not auto-detect {kind} price property/collection.")

def _supported_kw(db_method, *names: str) -> set[str]:
    try:
        sig = inspect.signature(db_method)
        return set(sig.parameters.keys()).intersection(names)
    except Exception:
        return set()

def _safe_add_membership(db: PlexosDB, **kwargs) -> bool:
    try:
        db.add_membership(**kwargs)
        return True
    except Exception:
        return False

def _add_property_with_dates(
    db: PlexosDB, *,
    class_enum: ClassEnum,
    object_name: str,
    property_name: str,
    value: float,
    collection_enum: CollectionEnum,
    parent_class_enum: ClassEnum,
    parent_object_name: str,
    date_from: datetime | None,
    date_to: datetime | None,
) -> bool:
    kw = {}
    accepted = _supported_kw(db.add_property, "date_from", "date_to")
    if "date_from" in accepted and date_from is not None:
        kw["date_from"] = date_from

    try:
        db.add_property(
            class_enum,
            object_name,
            property_name,
            float(value),
            collection_enum=collection_enum,
            parent_class_enum=parent_class_enum,
            parent_object_name=parent_object_name,
            **kw
        )
        return True
    except Exception as ex:
        print(f"Failed add_property for {class_enum.name}.{object_name} '{property_name}' with dates: {ex}")
        return False


@step("STEP â€” Import commodity prices with membership dates (Fuel Price + CO2 Emission Price) (NO BACKUP)")
def step_import_prices_inplace(xml_path: Path, prices_xlsx: Path, fuels: set[str]) -> Path:
    price_df = _read_commodity_prices_table(prices_xlsx)
    df = price_df[
        (price_df["Version"].str.lower() == ERAA_VERSION.lower()) &
        (price_df["Year"].isin(PRICE_YEARS)) &
        price_df["Name"].astype(bool) &
        price_df["Price"].notna()
    ].copy()

    if df.empty:
        raise RuntimeError(
            f"No rows found on sheet {PRICE_SHEET!r} for Version='{ERAA_VERSION}' and Years={PRICE_YEARS}."
        )

    print(f"Price rows kept: {len(df)} (Sheet={PRICE_SHEET}, Version={ERAA_VERSION}, Years={PRICE_YEARS})")

    db = PlexosDB.from_xml(str(xml_path))

    if not db.check_object_exists(ClassEnum.Emission, "CO2"):
        db.add_object(ClassEnum.Emission, name="CO2")
        print("Created Emission: CO2")

    fuel_col, fuel_prop, fuel_parent_class, fuel_parent_name = _find_price_collection_and_property(db, kind="fuel")
    emis_col, emis_prop, emis_parent_class, emis_parent_name = _find_price_collection_and_property(db, kind="emission")

    print(f"Fuel price target: Parent={fuel_parent_class.name}.{fuel_parent_name} Collection={fuel_col.name} Property={fuel_prop!r}")
    print(f"Emission price target: Parent={emis_parent_class.name}.{emis_parent_name} Collection={emis_col.name} Property={emis_prop!r}")

    fuels_clean = sorted({_clean_cell(f) for f in fuels if _clean_cell(f) and not _is_denied(f)})

    # --- SUPPRESS duplicate-membership spam during membership creation only ---
    loguru_token = None
    if _loguru_logger is not None:
        loguru_token = _loguru_logger.add(
            lambda msg: None,
            filter=lambda record: (
                record.get("name", "").startswith("plexosdb") and
                "UNIQUE constraint failed: t_membership" in record.get("message", "")
            )
        )

    try:
        for f in fuels_clean:
            if not db.check_object_exists(ClassEnum.Fuel, f):
                continue
            _safe_add_membership(
                db,
                parent_class_enum=fuel_parent_class,
                child_class_enum=ClassEnum.Fuel,
                parent_object_name=fuel_parent_name,
                child_object_name=f,
                collection_enum=fuel_col,
            )

        _safe_add_membership(
            db,
            parent_class_enum=emis_parent_class,
            child_class_enum=ClassEnum.Emission,
            parent_object_name=emis_parent_name,
            child_object_name="CO2",
            collection_enum=emis_col,
        )
    finally:
        if _loguru_logger is not None and loguru_token is not None:
            try:
                _loguru_logger.remove(loguru_token)
            except Exception:
                pass

    fuel_written = 0
    fuel_skipped = 0
    co2_written = 0
    co2_skipped = 0

    for y in PRICE_YEARS:
        p, unit = _pick_price(df, "CO2 Price", y)
        if p is None:
            co2_skipped += 1
            continue

        p_kg = float(p) / 1000.0
        dfrom, dto = _dates_for_year(y)

        ok = _add_property_with_dates(
            db,
            class_enum=ClassEnum.Emission,
            object_name="CO2",
            property_name=emis_prop,
            value=p_kg,
            collection_enum=emis_col,
            parent_class_enum=emis_parent_class,
            parent_object_name=emis_parent_name,
            date_from=dfrom,
            date_to=dto,
        )
        if ok:
            co2_written += 1

    lignite_targets = [n for n in ["Lignite", "Lignite - CCS"] if db.check_object_exists(ClassEnum.Fuel, n)]
    for y in PRICE_YEARS:
        avg_p, unit = _avg_lignite_price(df, y)
        if avg_p is None:
            continue

        dfrom, dto = _dates_for_year(y)
        for target in lignite_targets:
            ok = _add_property_with_dates(
                db,
                class_enum=ClassEnum.Fuel,
                object_name=target,
                property_name=fuel_prop,
                value=float(avg_p),
                collection_enum=fuel_col,
                parent_class_enum=fuel_parent_class,
                parent_object_name=fuel_parent_name,
                date_from=dfrom,
                date_to=dto,
            )
            if ok:
                fuel_written += 1

    for fuel in fuels_clean:
        if fuel.lower().startswith("lignite"):
            continue
        if not db.check_object_exists(ClassEnum.Fuel, fuel):
            fuel_skipped += 1
            continue

        is_ccs = (" - " in fuel and "CCS" in fuel.upper())
        base_name = fuel.split(" - ", 1)[0].strip() if is_ccs else fuel

        excel_name_exact = NAME_MAP.get(fuel, fuel)
        excel_name_base  = NAME_MAP.get(base_name, base_name)

        for y in PRICE_YEARS:
            p, unit = _pick_price(df, excel_name_exact, y)
            if p is None:
                p, unit = _pick_price(df, excel_name_base, y)
            if p is None:
                fuel_skipped += 1
                continue

            dfrom, dto = _dates_for_year(y)

            ok = _add_property_with_dates(
                db,
                class_enum=ClassEnum.Fuel,
                object_name=fuel,
                property_name=fuel_prop,
                value=float(p),
                collection_enum=fuel_col,
                parent_class_enum=fuel_parent_class,
                parent_object_name=fuel_parent_name,
                date_from=dfrom,
                date_to=dto,
            )
            if ok:
                fuel_written += 1

    print(f"Fuel price entries written: {fuel_written}")
    print(f"Fuel price entries skipped: {fuel_skipped}")
    print(f"CO2 emission price entries written: {co2_written}")
    print(f"CO2 emission price entries skipped: {co2_skipped}")

    ok = db.to_xml(xml_path)
    if not ok:
        raise RuntimeError(f"Export failed: {xml_path}")
    return xml_path


# --------------------------------------------------------------------------------------
# RUN STEPS
# --------------------------------------------------------------------------------------

fuels_to_create = read_fuels_with_ccs(COMMON_DATA_XLSX)
updated_xml = step_create_fuels_inplace(XML_PATH, fuels_to_create)
print("Updated in-place after STEP:", updated_xml)

updated_xml = step_co2_memberships_and_rates_addproperty(
    XML_PATH, COMMON_DATA_XLSX, fuels_to_create
)
print("Updated in-place after STEP:", updated_xml)

updated_xml = step_import_prices_inplace(XML_PATH, COMMODITY_PRICES_XLSX, fuels_to_create)
print("Updated in-place after STEP:", updated_xml)


#### 3. Create Demand .CSVs and Load-Region Coupling

In [ ]:
import os, csv, shutil, re
from pathlib import Path
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
DEMAND_SRC_DIR = DEMAND_DATA_DIR / "Demand timeseries"

FINAL_DIR = DATA_FILES_ROOT / "Final Demand Time Series"
PLEXOS_REL_PREFIX = r"Data Files\Final Demand Time Series"

# PLEXOS API dll folders
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

SCENARIO_NAME = "36_WS"

# Optional: if True, overwrite in FINAL_DIR even if already exists
OVERWRITE_FINAL_FILES = True


# -----------------------------
# CSV HELPERS
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _parse_region_year(name: str):
    parts = name.split("_")
    if len(parts) < 3:
        return None, None
    region = parts[0].strip()
    m = re.search(r"(\d{4})", name)
    year = int(m.group(1)) if m else None
    return region, year


def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:4096]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _rewrite_demand_csv_inplace(path: Path, valid_year: int):
    delim = _detect_csv_delimiter(path)
    df = pd.read_csv(path, sep=delim, engine="python")

    # ---- Ensure we have Year/Month/Day columns in the output (needed for 2028-02-29 insertion) ----
    # Accept either:
    #   A) Columns already have Year/Month/Day
    #   B) A single "Date" column (parse it and derive Month/Day)
    if "Month" not in df.columns or "Day" not in df.columns:
        if "Date" not in df.columns:
            raise RuntimeError(f"{path.name}: expected either columns Month+Day or column 'Date' not found")

        # Parse Date -> Month/Day (keep robust to formats)
        dts = pd.to_datetime(df["Date"], errors="coerce", infer_datetime_format=True, dayfirst=False)
        if dts.isna().all():
            # fallback: try dayfirst
            dts = pd.to_datetime(df["Date"], errors="coerce", infer_datetime_format=True, dayfirst=True)

        if dts.isna().all():
            raise RuntimeError(f"{path.name}: could not parse 'Date' to derive Month/Day")

        df["Month"] = dts.dt.month.astype(int)
        df["Day"] = dts.dt.day.astype(int)

        # Replace Date with Year (constant filename year)
        df = df.drop(columns=["Date"])
        df.insert(0, "Year", int(valid_year))
    else:
        # If there is a Date column, we ignore it and force Year anyway (keep behavior consistent)
        if "Year" not in df.columns:
            df.insert(0, "Year", int(valid_year))
        else:
            df["Year"] = int(valid_year)
        if "Date" in df.columns:
            df = df.drop(columns=["Date"])

    # Hour -> Period and shift values by +1
    if "Hour" not in df.columns and "Period" not in df.columns:
        raise RuntimeError(f"{path.name}: expected column 'Hour' (or 'Period') not found")

    if "Hour" in df.columns:
        df = df.rename(columns={"Hour": "Period"})

    df["Period"] = df["Period"].astype(int) + 1

    # WS01..WS36 -> 1..36
    rename_map = {}
    for i in range(1, 37):
        ws = f"WS{i:02d}"
        if ws not in df.columns:
            raise RuntimeError(f"{path.name}: missing expected column {ws}")
        rename_map[ws] = str(i)
    df = df.rename(columns=rename_map)

    # ---- Insert Feb 29th for target year 2028 by duplicating Feb 28th ----
    if int(valid_year) == 2028:
        mask_feb28 = (
            (df["Year"].astype(int) == 2028)
            & (df["Month"].astype(int) == 2)
            & (df["Day"].astype(int) == 28)
        )
        feb28 = df.loc[mask_feb28].copy()

        if feb28.empty:
            raise RuntimeError(f"{path.name}: could not find Year=2028, Month=2, Day=28 rows to duplicate for Feb 29")

        feb28["Period"] = feb28["Period"].astype(int)
        feb28 = feb28.sort_values(["Period"])

        feb29 = feb28.copy()
        feb29["Day"] = 29

        idx_feb28 = df.index[mask_feb28]
        last_idx = idx_feb28.max()

        part1 = df.loc[df.index <= last_idx].copy()
        part2 = df.loc[df.index > last_idx].copy()

        part1_wo = part1.loc[~mask_feb28].copy()
        part1_new = pd.concat([part1_wo, feb28], ignore_index=True)

        df = pd.concat([part1_new, feb29, part2], ignore_index=True)
        df["Period"] = df["Period"].astype(int)

    df.to_csv(path, index=False, sep=delim)


# -----------------------------
# PLEXOS_NET LOADER
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    # load core + net (prefer API dir)
    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(str(name), class_enum_value, bool(add_to_system), str(category), str(description))


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(str(a)), SystemNS.String(str(b)), SystemNS.String(str(c)), SystemNS.String(str(d))))


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _find_system_regions_collection_enum(CollectionEnum):
    if hasattr(CollectionEnum, "SystemRegions"):
        return getattr(CollectionEnum, "SystemRegions")
    names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    candidates = [n for n in names if ("system" in n.lower() and "region" in n.lower())]
    if not candidates:
        raise RuntimeError("Could not find System->Regions collection enum")
    return getattr(CollectionEnum, candidates[0])


def _resolve_variable_profile_enum(db, SystemNS):
    prop_candidates = [
        "Profile",
        "Profiles",
        "Profile Data",
        "Profile File",
        "Profile Filename",
        "Profile File Name",
        "Profile Data File",
        "Profile DataFile",
        "Data File",
        "DataFile",
        "Filename",
        "File Name",
        "File",
        "Path",
        "Data",
        "Time Series",
    ]

    collection_candidates = [
        "Variables",
        "SystemVariables",
        "Variable",
    ]

    last = None
    for col in collection_candidates:
        for prop in prop_candidates:
            try:
                enum_id = int(db.PropertyName2EnumId(
                    SystemNS.String("System"),
                    SystemNS.String("Variable"),
                    SystemNS.String(str(col)),
                    SystemNS.String(str(prop)),
                ))
                return enum_id, col, prop
            except Exception as e:
                last = (col, prop, str(e))
                continue

    raise RuntimeError(
        "Could not resolve a Variable profile/file property.\n"
        f"Last attempt: {last}\n"
        "Please confirm the exact property name shown in PLEXOS UI for Variable profile."
    )


# -----------------------------
# MAIN STEP
# -----------------------------
def step_full_demand_variables_and_region_load():
    # A) Copy + rewrite CSVs into FINAL_DIR
    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    allowed = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No regions found in Bidding_Zone_List.xlsx")

    picked = []
    for p in sorted(DEMAND_SRC_DIR.glob("*.csv")):
        region, year = _parse_region_year(p.name)
        if region and year and region in allowed:
            picked.append((p, region, year))
    if not picked:
        raise RuntimeError(f"No demand CSVs matched bidding zones in {DEMAND_SRC_DIR}")

    copied = []
    for src, region, year in picked:
        dst = FINAL_DIR / src.name
        if dst.exists() and not OVERWRITE_FINAL_FILES:
            _rewrite_demand_csv_inplace(dst, valid_year=year)
        else:
            shutil.copy2(src, dst)
            _rewrite_demand_csv_inplace(dst, valid_year=year)
        copied.append((dst, region, year))
    print(f"Copied+rewrote {len(copied)} CSVs into: {FINAL_DIR}")

    # B) Open PLEXOS XML
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        # C) Ensure Scenario exists
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)

        # D) Resolve enum ids
        region_load_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Region", "Regions", "Load"),
            ("System", "Region", "SystemRegions", "Load"),
            ("Region", "System", "Regions", "Load"),
            ("System", "Regions", "Region", "Load"),
            ("System", "SystemRegions", "Region", "Load"),
        ])
        print("Resolved EnumId for Region.Load:", region_load_enum)

        sampling_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Variable", "Variables", "Sampling Method"),
            ("System", "Variable", "Variables", "SamplingMethod"),
            ("System", "Variable", "SystemVariables", "Sampling Method"),
            ("System", "Variable", "SystemVariables", "SamplingMethod"),
            ("Variable", "System", "Variables", "Sampling Method"),
            ("Variable", "System", "Variables", "SamplingMethod"),
        ])
        print("Resolved EnumId for Variable.Sampling Method:", sampling_enum)

        profile_enum, profile_col, profile_prop = _resolve_variable_profile_enum(db, SystemNS)
        print(f"Resolved Variable profile property: collection={profile_col!r} property={profile_prop!r} enum_id={profile_enum}")

        # E) Collections
        sys_regions_enum = _find_system_regions_collection_enum(CollectionEnum)

        if hasattr(CollectionEnum, "SystemVariables"):
            var_mem_collection = getattr(CollectionEnum, "SystemVariables")
        elif hasattr(CollectionEnum, "Variables"):
            var_mem_collection = getattr(CollectionEnum, "Variables")
        else:
            raise RuntimeError("Could not find CollectionEnum.SystemVariables or CollectionEnum.Variables")

        # F) Existing objects
        regions_in_db = set(_get_objects_safe(db, ClassEnum.Region))
        variables_in_db = set(_get_objects_safe(db, ClassEnum.Variable))

        written_profiles = 0
        written_sampling = 0
        written_load = 0
        created_vars = 0

        # G) Create Variables + Sampling Method + Profiles + Region.Load expressions
        for dst, region, year in copied:
            var_name = f"{region}_Demand_{year}"
            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

            if var_name not in variables_in_db:
                _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category="", description="")
                variables_in_db.add(var_name)
                created_vars += 1

            try:
                db.GetMembershipID(var_mem_collection, "System", var_name)
            except Exception:
                try:
                    db.AddMembership(var_mem_collection, "System", var_name)
                except Exception:
                    pass

            var_mem_id = int(db.GetMembershipID(var_mem_collection, "System", var_name))

            # Sampling Method = User (2)
            db.AddProperty(
                var_mem_id,
                sampling_enum,
                1,
                2.0,
                dt_from,
                None,
                None,
                None,
                None,
                SystemNS.String(str(SCENARIO_NAME)),
                None
            )
            written_sampling += 1

            # Profile bands 1..36
            plexos_rel = str(Path(PLEXOS_REL_PREFIX) / dst.name)
            for band in range(1, 37):
                db.AddProperty(
                    var_mem_id,
                    profile_enum,
                    int(band),
                    0.0,
                    dt_from,
                    None,
                    None,
                    SystemNS.String(str(plexos_rel)),
                    None,
                    SystemNS.String(str(SCENARIO_NAME)),
                    None
                )
                written_profiles += 1

            if region in regions_in_db:
                region_mem = int(db.GetMembershipID(sys_regions_enum, "System", region))
                db.AddProperty(
                    region_mem,
                    region_load_enum,
                    1,
                    0.0,
                    dt_from,
                    None,
                    SystemNS.String(str(var_name)),
                    None,
                    None,
                    None,
                    None
                )
                written_load += 1

        print(f"Variables created: {created_vars}")
        print(f"Sampling Method rows written: {written_sampling}")
        print(f"Variable profile rows written (bands): {written_profiles}")
        print(f"Region.Load expression rows written: {written_load}")
        print("Done.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_full_demand_variables_and_region_load()

#### 4. Generator Creation and Common Data Population

In [ ]:
import os
from pathlib import Path
from typing import Optional, Tuple
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
GEN_CAP_CSV = DASHBOARD_RAWDATA_DIR / "GenerationCapacities.csv"
TECH_FUEL_MAP_XLSX = BASE_DIR / "Generator_Technology_Fuel_Mapping.xlsx"


OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_NAME = "Generator_Max_Capacities.csv"
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PLEXOS_REL_CSV_PATH = r"Data Files\Generator Data\Generator_Max_Capacities.csv"

DATAFILE_OBJECT_NAME = "Generator_Max_Capacities"

# -----------------------------
# CHANGEABLE FILTERS
# -----------------------------
FILTER_DATA_VERSION = "ERAA 2025 final"
FILTER_OPERATION_STATUS = "Available on market"

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# PROPERTY NAMES (UI text)
# -----------------------------
PROP_GENERATOR_MAXCAP = "Max Capacity"
PROP_UNITS = "Units"
PROP_POWER2X_MAXLOAD_UI = "Max Load"

# Fuels to create when needed
NEW_FUELS_ZEROED = {"Biomass", "Waste", "Geothermal"}
PTX_TOKEN = "PtX"

# -----------------------------
# NEW: Technologies to exclude (Technology column)
# -----------------------------
EXCLUDED_TECHNOLOGIES = {
    "Battery residential",
    "Battery utility scale",
    "EV - iDSR",
    "HP - iDSR",
}

# -----------------------------
# FIX 1: Fuel name canonicalization / aliases (mapping name -> DB fuel object name)
# -----------------------------
FUEL_NAME_ALIASES_LC = {
    "shale oil": "Oil Shale",
    "oil shale": "Oil Shale",
}

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# COLLECTION ENUM FALLBACK IDS (only used if enum names can't be resolved)
# -----------------------------
# You said: Generator collection ID = 1, Fuel collection ID = 40
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1
FALLBACK_SYSTEM_FUELS_COLLECTION_ID = 40
# DataFiles fallback unknown in your note; leave as None unless you know it.
FALLBACK_SYSTEM_DATAFILES_COLLECTION_ID = None


# -----------------------------
# HELPERS (earlier methodology)
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    # Your build: AddObject(string, ClassEnum, bool, string, string)
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _resolve_enum_id_optional(db, SystemNS, trials, label: str) -> Optional[int]:
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    print(f"[WARN] Could not resolve EnumId for {label}. Last attempt: {last}")
    return None


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _resolve_fuel_property_enum_optional(db, SystemNS, desired_prop: str) -> Tuple[Optional[int], Optional[str], Optional[str], Optional[str]]:
    child_candidates = ["Fuel", "Fuels", "Emission Fuel", "EmissionFuel"]
    collection_candidates = ["Fuels", "SystemFuels", "System Fuels", "Fuel", "SystemFuel"]

    if desired_prop.lower() == "fuel price":
        prop_candidates = ["Fuel Price", "Price", "Fuel Cost", "Cost", "Commodity Price", "Commodity Cost", "Unit Cost"]
    elif desired_prop.lower() == "emission fuels production rate":
        prop_candidates = [
            "Emission Fuels Production Rate",
            "Emissions Fuels Production Rate",
            "Emission Fuel Production Rate",
            "Emission Production Rate",
            "Production Rate",
            "Emissions Production Rate",
            "CO2 Production Rate",
        ]
    else:
        prop_candidates = [desired_prop]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, child, col, prop
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue

    print(f"[WARN] Could not resolve Fuel enum for '{desired_prop}'. Last attempt: {last}")
    return None, None, None, None


def _read_tech_fuel_mapping(xlsx: Path) -> dict[str, str]:
    df = pd.read_excel(xlsx, sheet_name=0, header=None)
    mapping = {}
    for _, row in df.iterrows():
        if len(row) < 2:
            continue
        tech = "" if pd.isna(row[0]) else str(row[0]).strip()
        fuel = "" if pd.isna(row[1]) else str(row[1]).strip()
        if tech:
            mapping[tech] = fuel
    return mapping


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc in ("data_version", "data version"):
            rename[c] = "data_version"
        elif lc in ("target_year", "target year", "year"):
            rename[c] = "Target_Year"
        elif lc in ("market_node", "market node"):
            rename[c] = "Bidding_Zone"
        elif lc == "technology":
            rename[c] = "Technology"
        elif lc in ("technology_simplified", "technology simplified"):
            rename[c] = "Technology_Simplified"
        elif lc in ("operational_status", "operational status"):
            rename[c] = "Operation Status"
        elif lc == "value":
            rename[c] = "Value"

    df.rename(columns=rename, inplace=True)
    return df


def _sanitize_name(s: str) -> str:
    s = str(s).strip()
    return s.replace("/", "_").replace("\\", "_").replace(":", "_")


def _build_matrix(df: pd.DataFrame, object_names: list[str]) -> pd.DataFrame:
    pivot = df.pivot_table(
        index="Target_Year",
        columns="ObjectName",
        values="Value",
        aggfunc="sum",
        fill_value=0.0
    )
    for n in object_names:
        if n not in pivot.columns:
            pivot[n] = 0.0
    pivot = pivot[object_names].reset_index().rename(columns={"Target_Year": "Year"})
    try:
        pivot["Year"] = pivot["Year"].astype(int)
        pivot = pivot.sort_values("Year")
    except Exception:
        pivot = pivot.sort_values("Year")
    return pivot


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _ensure_membership_bi(db, collection_enum, a: str, b: str) -> bool:
    """
    Try to create membership in both directions. Returns True if one direction worked.
    """
    try:
        _ensure_membership(db, collection_enum, a, b)
        return True
    except Exception:
        pass
    try:
        _ensure_membership(db, collection_enum, b, a)
        return True
    except Exception:
        return False


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _resolve_power2x_maxload_enum_optional(db, SystemNS) -> Optional[int]:
    prop_candidates = [
        "Max Load",
        "Maximum Load",
        "MaxLoad",
        "Max Capacity",
        "Capacity",
        "Max Power",
        "Maximum Power",
        "Max Input",
        "Maximum Input",
        "Upper Limit",
    ]
    trials = []
    for p in prop_candidates:
        trials.extend([
            ("System", "Power2X", "SystemPower2X", p),
            ("System", "Power2X", "Power2X", p),
            ("System", "Power2X", "Power2Xs", p),
            ("System", "Power 2X", "SystemPower2X", p),
            ("System", "Power 2X", "System Power 2X", p),
        ])
    return _resolve_enum_id_optional(db, SystemNS, trials, label="Power2X.Max Load")


def _ensure_generator_category(db, ClassEnum, category_name: str):
    """
    Create generator category (best effort). Safe to call repeatedly.
    """
    if not category_name:
        return
    try:
        db.AddCategory(ClassEnum.Generator, str(category_name))
    except Exception:
        pass


def _find_collection_enum(CollectionEnum, required_tokens_lc: list[str], preferred_names: list[str], fallback_id: Optional[int] = None):
    """
    Resolve a CollectionEnum value robustly across different PLEXOS versions/builds.
    """
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]} -> {int(chosen[2]) if str(chosen[2]).isdigit() else chosen[2]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


# -----------------------------
# MAIN STEP
# -----------------------------
def step_generators_power2x_and_maxcap_datafile():
    allowed = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No regions found in Bidding_Zone_List.xlsx")

    tech_to_fuel = _read_tech_fuel_mapping(TECH_FUEL_MAP_XLSX)

    df = pd.read_csv(GEN_CAP_CSV, dtype=str)
    df = _normalize_columns(df)

    required = ["data_version", "Target_Year", "Bidding_Zone", "Technology", "Operation Status", "Value"]
    for r in required:
        if r not in df.columns:
            raise RuntimeError(f"Missing required column {r!r}. Found: {list(df.columns)}")

    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0.0)
    for c in ["Bidding_Zone", "Technology", "Operation Status", "data_version", "Target_Year"]:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df["data_version"] == FILTER_DATA_VERSION) &
        (df["Operation Status"] == FILTER_OPERATION_STATUS) &
        (df["Bidding_Zone"].isin(allowed))
    ].copy()

    # -----------------------------
    # ONLY CHANGE: Exclude specific technologies
    # -----------------------------
    before_count = len(df)
    df = df[~df["Technology"].isin(EXCLUDED_TECHNOLOGIES)].copy()
    removed = before_count - len(df)
    if removed > 0:
        print(f"Excluded {removed} rows due to excluded technologies: {sorted(EXCLUDED_TECHNOLOGIES)}")

    if df.empty:
        raise RuntimeError("No rows remain after filters.")

    target_years = sorted({int(y) for y in df["Target_Year"].dropna().unique().tolist() if str(y).strip().isdigit()})

    df["MappedFuel"] = df["Technology"].map(lambda t: tech_to_fuel.get(t, "")).fillna("").astype(str).str.strip()
    df["ObjectClass"] = df["MappedFuel"].map(lambda f: "Power2X" if f.lower() == PTX_TOKEN.lower() else "Generator")
    df["ObjectName"] = df.apply(lambda r: _sanitize_name(f"{r['Bidding_Zone']}_{r['Technology']}"), axis=1)

    object_names = df["ObjectName"].dropna().unique().tolist()

    matrix = _build_matrix(df[["Target_Year", "ObjectName", "Value"]], object_names)
    matrix.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Wrote matrix CSV: {OUTPUT_CSV_PATH}")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        # Collections (robust + fallbacks)
        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID
        )
        fuel_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "fuel"],
            ["SystemFuels", "Fuels"],
            fallback_id=FALLBACK_SYSTEM_FUELS_COLLECTION_ID
        )
        datafile_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "data", "file"],
            ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
            fallback_id=FALLBACK_SYSTEM_DATAFILES_COLLECTION_ID
        )

        # From your diagnostics:
        p2x_mem_collection = getattr(CollectionEnum, "SystemPower2X") if hasattr(CollectionEnum, "SystemPower2X") else None
        p2x_node_collection = getattr(CollectionEnum, "Power2XNodes") if hasattr(CollectionEnum, "Power2XNodes") else None
        p2x_fuel_collection = getattr(CollectionEnum, "Power2XFuels") if hasattr(CollectionEnum, "Power2XFuels") else None

        # Generator node membership collection:
        gen_node_collection = None
        if hasattr(CollectionEnum, "NodeGenerators"):
            gen_node_collection = getattr(CollectionEnum, "NodeGenerators")
        elif hasattr(CollectionEnum, "GeneratorNodes"):
            gen_node_collection = getattr(CollectionEnum, "GeneratorNodes")
        elif hasattr(CollectionEnum, "PowerStationNodes"):
            gen_node_collection = getattr(CollectionEnum, "PowerStationNodes")

        # Generator fuel membership (best effort)
        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")

        # DataFile filename enum
        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        # Generator enums
        gen_units_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Generator", "Generators", PROP_UNITS),
            ("System", "Generator", "SystemGenerators", PROP_UNITS),
        ])
        gen_maxcap_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Generator", "Generators", PROP_GENERATOR_MAXCAP),
            ("System", "Generator", "SystemGenerators", PROP_GENERATOR_MAXCAP),
        ])

        # Power2X enums
        p2x_units_enum = None
        p2x_maxload_enum = None
        if p2x_mem_collection is not None:
            p2x_units_enum = _resolve_enum_id_optional(db, SystemNS, [
                ("System", "Power2X", "SystemPower2X", PROP_UNITS),
                ("System", "Power2X", "Power2X", PROP_UNITS),
                ("System", "Power2X", "Power2Xs", PROP_UNITS),
                ("System", "Power 2X", "SystemPower2X", PROP_UNITS),
                ("System", "Power 2X", "System Power 2X", PROP_UNITS),
            ], label="Power2X.Units")

            p2x_maxload_enum = _resolve_power2x_maxload_enum_optional(db, SystemNS)
            print(f"Power2X enum resolution: Units={p2x_units_enum}, Max Load={p2x_maxload_enum}")
        else:
            print("[WARN] CollectionEnum.SystemPower2X not found; Power2X objects will be skipped.")

        # Fuel property enums (optional)
        fuel_price_enum, _, _, _ = _resolve_fuel_property_enum_optional(db, SystemNS, "Fuel Price")
        fuel_emiss_enum, _, _, _ = _resolve_fuel_property_enum_optional(db, SystemNS, "Emission Fuels Production Rate")

        # DataFile object + filename property (store relative path in DataFile slot)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME)
        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
            Scenario=scenario_str
        )

        # Ensure special fuels and set properties to 0 for each target year (only if enums resolved)
        if fuel_price_enum is not None or fuel_emiss_enum is not None:
            for fuel_name in sorted(NEW_FUELS_ZEROED):
                _ensure_object(db, ClassEnum.Fuel, fuel_name)
                f_mem_id = _ensure_membership(db, fuel_mem_collection, "System", fuel_name)

                if target_years:
                    for y in target_years:
                        dt_from = NetDateTime(int(y), 1, 1, 0, 0, 0)
                        if fuel_price_enum is not None:
                            _add_property_row(db, fuel_price_enum, f_mem_id, 1, 0.0, DateFrom=dt_from, Scenario=scenario_str)
                        if fuel_emiss_enum is not None:
                            _add_property_row(db, fuel_emiss_enum, f_mem_id, 1, 0.0, DateFrom=dt_from, Scenario=scenario_str)
                else:
                    if fuel_price_enum is not None:
                        _add_property_row(db, fuel_price_enum, f_mem_id, 1, 0.0, Scenario=scenario_str)
                    if fuel_emiss_enum is not None:
                        _add_property_row(db, fuel_emiss_enum, f_mem_id, 1, 0.0, Scenario=scenario_str)
        else:
            print("[WARN] Fuel property enums for price/emissions could not be resolved; Biomass/Waste/Geothermal will be created but not parameterized.")

        # Caches
        nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))

        fuels_in_db_lc = {}
        for f in fuels_in_db:
            k = str(f).strip().lower()
            if k and k not in fuels_in_db_lc:
                fuels_in_db_lc[k] = f

        gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))

        has_p2x_enum = hasattr(ClassEnum, "Power2X")
        p2x_in_db = set()
        if has_p2x_enum:
            p2x_in_db = set(_get_objects_safe(db, getattr(ClassEnum, "Power2X")))

        # Create objects and link properties/memberships
        created_g = created_p = linked_props = linked_nodes = linked_fuels = 0
        gen_node_links = 0

        unique_objs = df.drop_duplicates(subset=["ObjectName"])[
            ["ObjectName", "ObjectClass", "Bidding_Zone", "MappedFuel"]
        ].copy()

        for _, r in unique_objs.iterrows():
            obj_name = str(r["ObjectName"]).strip()
            obj_class = str(r["ObjectClass"]).strip()
            bz = str(r["Bidding_Zone"]).strip()
            mapped_fuel = str(r["MappedFuel"]).strip()

            if obj_class == "Generator":
                _ensure_generator_category(db, ClassEnum, bz)

                if obj_name not in gens_in_db:
                    _add_object(db, ClassEnum.Generator, obj_name, add_to_system=True, category=bz, description="")
                    gens_in_db.add(obj_name)
                    created_g += 1

                mem_id = _ensure_membership(db, gen_mem_collection, "System", obj_name)

                _add_property_row(db, gen_units_enum, mem_id, 1, 1.0, Scenario=scenario_str)
                _add_property_row(
                    db, gen_maxcap_enum, mem_id, 1, 0.0,
                    DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                    Scenario=scenario_str
                )
                linked_props += 1

                if bz in nodes_in_db and gen_node_collection is not None:
                    if _ensure_membership_bi(db, gen_node_collection, bz, obj_name):
                        linked_nodes += 1
                        gen_node_links += 1

                if mapped_fuel and mapped_fuel.lower() not in ("none", PTX_TOKEN.lower()):
                    if gen_fuel_collection is not None:
                        mf_lc = mapped_fuel.strip().lower()
                        mf_canon = FUEL_NAME_ALIASES_LC.get(mf_lc, mapped_fuel.strip())
                        actual_fuel = fuels_in_db_lc.get(mf_canon.lower())

                        if actual_fuel:
                            try:
                                _ensure_membership(db, gen_fuel_collection, obj_name, actual_fuel)
                                linked_fuels += 1
                            except Exception:
                                try:
                                    _ensure_membership(db, gen_fuel_collection, actual_fuel, obj_name)
                                    linked_fuels += 1
                                except Exception:
                                    pass
                        else:
                            print(f"[WARN] Fuel '{mapped_fuel}' (canon='{mf_canon}') not found in DB fuels; cannot assign to generator '{obj_name}'.")

            else:
                if not has_p2x_enum or p2x_mem_collection is None:
                    continue

                p2x_enum = getattr(ClassEnum, "Power2X")
                if obj_name not in p2x_in_db:
                    _add_object(db, p2x_enum, obj_name, add_to_system=True, category="", description="")
                    p2x_in_db.add(obj_name)
                    created_p += 1

                mem_id = _ensure_membership(db, p2x_mem_collection, "System", obj_name)

                if p2x_units_enum is not None:
                    _add_property_row(db, p2x_units_enum, mem_id, 1, 1.0, Scenario=scenario_str)

                if p2x_maxload_enum is not None:
                    _add_property_row(
                        db, p2x_maxload_enum, mem_id, 1, 0.0,
                        DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                        Scenario=scenario_str
                    )
                    linked_props += 1

                if bz in nodes_in_db and p2x_node_collection is not None:
                    if _ensure_membership_bi(db, p2x_node_collection, bz, obj_name):
                        linked_nodes += 1

                if mapped_fuel and mapped_fuel.lower() not in ("none", PTX_TOKEN.lower()) and p2x_fuel_collection is not None:
                    if mapped_fuel in fuels_in_db:
                        try:
                            _ensure_membership(db, p2x_fuel_collection, obj_name, mapped_fuel)
                            linked_fuels += 1
                        except Exception:
                            try:
                                _ensure_membership(db, p2x_fuel_collection, mapped_fuel, obj_name)
                                linked_fuels += 1
                            except Exception:
                                pass

        print("Done.")
        print(f"Created Generators: {created_g}")
        print(f"Created Power2X: {created_p}")
        print(f"Linked MaxCap/MaxLoad properties: {linked_props}")
        print(f"Linked Node memberships (total): {linked_nodes}")
        print(f"  - Generator node memberships: {gen_node_links}")
        print(f"Linked Fuel memberships: {linked_fuels}")

        if gen_node_collection is None:
            print("[WARN] Could not find a Generator<->Node membership collection enum (NodeGenerators/GeneratorNodes/PowerStationNodes). Generators may still be missing node memberships.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_generators_power2x_and_maxcap_datafile()




In [ ]:
# ============================================================
# Thermal Property Additions
# - Adds Generator "Heat Rate" based on Common Data.xlsx efficiencies
# - Reads ONLY: sheet "Common Data", rows 15-41 (inclusive)
# - Uses: Fuel=Column C, Desc=Column D (CCS detection), Efficiency=Column F (percent)
# - Heat Rate = 3.6 / EfficiencyFraction   where EfficiencyFraction is in [0,1]
# - Override rule: Biomass & Waste use the Hard Coal heat rate (and later same rule for other properties)
# ============================================================
import os
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd
import math
import copy

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# PROPERTY (UI label) â€” per your screenshot this is "Heat Rate"
# -----------------------------
PROP_HEAT_RATE_UI_CANDIDATES = [
    "Heat Rate",            # confirmed in UI screenshot
    "Heat rate",
    "Generator Heat Rate",  # fallback
    "Generator heat rate",
]

# -----------------------------
# Excel target range
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 15
ROW_END_1BASED = 41  # inclusive

# Table uses columns B-L, but we reference by Excel letters:
# Fuel = C, Desc = D, Efficiency = F
COL_FUEL_IDX = 2  # C (0-based)
COL_DESC_IDX = 3  # D
COL_EFF_IDX  = 5  # F

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "  # CCS fuel object naming rule: "<Fuel> - <Desc>"

# Override mapping: these fuels should use Hard Coal's computed values
OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (defensive patterns consistent with earlier sections)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    """
    Safe membership lookup in either direction; returns None if not found.
    """
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))


def _resolve_enum_id_optional(db, SystemNS, trials, label: str) -> Optional[int]:
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    print(f"[WARN] Could not resolve EnumId for {label}. Last attempt: {last}")
    return None


def _resolve_generator_property_enum_optional(db, SystemNS, ui_property_candidates: List[str]) -> Optional[Tuple[int, str]]:
    child_variants = ["Generator", "Generators"]
    collection_variants = ["Generators", "SystemGenerators"]

    for prop in ui_property_candidates:
        prop = (prop or "").strip()
        if not prop:
            continue

        trials = []
        for child in child_variants:
            for col in collection_variants:
                trials.append(("System", child, col, prop))

        enum_id = _resolve_enum_id_optional(db, SystemNS, trials, label=f"Generator.{prop}")
        if enum_id is not None:
            return enum_id, prop

    return None


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _parse_efficiency_to_fraction(x) -> Optional[float]:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None

    s = str(x).strip()
    if not s:
        return None

    if "%" in s:
        s2 = s.replace("%", "").strip()
        try:
            v = float(s2)
        except Exception:
            return None
        frac = v / 100.0
        return float(frac) if 0.0 < frac <= 1.0 else None

    try:
        v = float(s)
    except Exception:
        return None

    if 0.0 < v <= 1.0:
        return float(v)
    if 1.0 < v <= 100.0:
        frac = v / 100.0
        return float(frac) if 0.0 < frac <= 1.0 else None

    return None


def _read_common_data_efficiencies_table(xlsx: Path) -> pd.DataFrame:
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, COL_EFF_IDX)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "EfficiencyCell": sub.iloc[:, COL_EFF_IDX],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()
    out["Efficiency"] = out["EfficiencyCell"].map(_parse_efficiency_to_fraction)

    out = out[(out["Fuel"] != "") & out["Fuel"].notna() & out["Efficiency"].notna()].copy()
    out = out[(out["Efficiency"] > 0.0) & (out["Efficiency"] <= 1.0)].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)

    return out


def _build_fuel_to_heatrate_map(df_eff: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    mapping: Dict[str, Dict[str, Any]] = {}

    non_ccs = df_eff[~df_eff["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)["Efficiency"].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg_eff = float(r["mean"])
            cnt = int(r["count"])
            if fuel and avg_eff > 0:
                mapping[fuel] = {
                    "heatrate": float(3.6 / avg_eff),
                    "avg_eff": avg_eff,
                    "count": cnt,
                    "is_ccs": False,
                    "fuel": fuel,
                    "desc": "",
                }

    ccs = df_eff[df_eff["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)["Efficiency"].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg_eff = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc and avg_eff > 0:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {
                    "heatrate": float(3.6 / avg_eff),
                    "avg_eff": avg_eff,
                    "count": cnt,
                    "is_ccs": True,
                    "fuel": fuel,
                    "desc": desc,
                }

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set) -> None:
    """
    Mutates mapping: sets override_fuels entries equal to reference_fuel entry (deep copy),
    but preserves the 'fuel' field as the override fuel name.
    Case-insensitive match for the reference and overrides.
    """
    if not mapping:
        return

    # Find reference key in mapping (case-insensitive)
    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys: {list(mapping.keys())}")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue

        # Overwrite / create entry
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values (heatrate={ref_stats['heatrate']:.4f} GJ/MWh).")


# ============================================================
# MAIN
# ============================================================
def thermal_property_additions_step():
    # 1) Read table slice and compute mappings
    df_eff = _read_common_data_efficiencies_table(COMMON_DATA_XLSX)
    if df_eff.empty:
        raise RuntimeError(
            f"No usable efficiency rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}."
        )

    fuel_to_stats = _build_fuel_to_heatrate_map(df_eff)
    if not fuel_to_stats:
        raise RuntimeError("No heat rate mappings computed from the specified table slice.")

    # Apply Biomass/Waste override => Hard Coal
    _apply_override_fuels_use_reference(
        fuel_to_stats,
        reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE,
        override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )

    print(f"Using sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    print(f"Efficiency rows used (after parsing): {len(df_eff)}")
    print(f"Unique fuels in table slice: {sorted(df_eff['Fuel'].unique().tolist())}")
    print(f"Computed heat rates for {len(fuel_to_stats)} fuel keys (incl. CCS variants + overrides).")

    # 2) Open PLEXOS DB
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        # Collections
        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )
        fuel_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "fuel"],
            candidates=["SystemFuels", "Fuels"],
        )

        # Generator<->Fuel membership collection (best effort)
        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(
                CollectionEnum,
                must_contain_tokens=["fuel", "generator"],
                candidates=[]
            )

        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")
        if fuel_mem_collection is None:
            raise RuntimeError("Could not resolve Fuel membership collection (SystemFuels/Fuels).")
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generator<->Fuel membership collection (GeneratorFuels/FuelGenerators).")

        # Resolve Heat Rate enum
        resolved = _resolve_generator_property_enum_optional(db, SystemNS, PROP_HEAT_RATE_UI_CANDIDATES)
        if resolved is None:
            raise RuntimeError(
                "Could not resolve enum for Generator Heat Rate property. "
                f"Tried labels: {PROP_HEAT_RATE_UI_CANDIDATES}"
            )
        gen_heatrate_enum, resolved_label = resolved
        print(f"Resolved Generator property enum: {resolved_label!r} -> EnumId={gen_heatrate_enum}")

        # Cache existing objects
        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))

        # Diagnostics: computed keys missing in DB
        missing_fuels = [f for f in fuel_to_stats.keys() if f not in fuels_in_db]
        if missing_fuels:
            print("[WARN] Some computed fuel keys do not exist as Fuel objects in the DB (they will be skipped):")
            for f in missing_fuels[:40]:
                print(f"  - {f}")
            if len(missing_fuels) > 40:
                print(f"  ... (+{len(missing_fuels)-40} more)")

        # 3) Apply to generators with membership to each fuel
        writes = 0
        fuels_applied = 0
        skipped_no_memberships = 0

        for fuel_name in sorted(fuel_to_stats.keys()):
            st = fuel_to_stats[fuel_name]
            hr = round(float(st["heatrate"]), 1)

            if fuel_name not in fuels_in_db:
                continue

            matched_gens = []
            for g in gens_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, fuel_name)
                if mem is not None:
                    matched_gens.append(g)

            if not matched_gens:
                skipped_no_memberships += 1
                continue

            fuels_applied += 1

            for g in matched_gens:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                _add_property_row(
                    db, gen_heatrate_enum, g_mem_id, 1, hr,
                    Scenario=scenario_str
                )
                writes += 1

            ccs_tag = " (CCS)" if st["is_ccs"] else ""
            print(
                f"Applied Heat Rate {hr:.4f} GJ/MWh to {len(matched_gens)} generator(s) for Fuel='{fuel_name}'{ccs_tag} "
                f"[avg_eff={st['avg_eff']*100:.2f}%, n={st['count']}]"
            )

        print("Done. (Thermal Property Additions)")
        print(f"Fuel keys computed: {len(fuel_to_stats)}")
        print(f"Fuel keys applied (had Fuel object + >=1 generator membership): {fuels_applied}")
        print(f"Fuel keys skipped (no generator memberships found): {skipped_no_memberships}")
        print(f"Generator Heat Rate writes: {writes}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
thermal_property_additions_step()





In [ ]:
# ============================================================
# VO&M Property Additions
# - Adds Generator "VO&M Charge" based on Common Data.xlsx
# - Reads ONLY: sheet "Common Data", rows 15-41 (inclusive)
# - Uses: Fuel=Column C, Desc=Column D (CCS detection), VO&M Charge=Column H
# - Non-CCS: average VO&M by Fuel
# - CCS: average VO&M by (Fuel, Desc) and write to Fuel object name "<Fuel> - <Desc>"
# - Override rule: Biomass & Waste use Hard Coal's computed VO&M value
# - Writes property to System->Generator membership (same as earlier logic)
# - Uses known enum_id from DB Browser: VO&M Charge = 63
# ============================================================

import os
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd
import copy
import math

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# PROPERTY ENUM (from DB Browser for SQLite)
# -----------------------------
VOM_ENUM_ID = 64  # VO&M Charge

# -----------------------------
# Excel target range
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 15
ROW_END_1BASED = 41  # inclusive

# Fuel = C, Desc = D, VO&M = H
COL_FUEL_IDX = 2  # C (0-based)
COL_DESC_IDX = 3  # D
COL_VOM_IDX  = 7  # H (0-based)

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "  # CCS fuel object naming rule: "<Fuel> - <Desc>"

# Override mapping: these fuels should use Hard Coal's computed values
OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (copied from earlier logic for identical behavior)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _read_common_data_vom_table(xlsx: Path) -> pd.DataFrame:
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, COL_VOM_IDX)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "VOMCell": sub.iloc[:, COL_VOM_IDX],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()

    # Numeric parse for VO&M
    def _to_float(x):
        if x is None:
            return None
        if isinstance(x, float) and math.isnan(x):
            return None
        s = str(x).strip()
        if not s:
            return None
        try:
            return float(s)
        except Exception:
            return None

    out["VOM"] = out["VOMCell"].map(_to_float)
    out = out[(out["Fuel"] != "") & out["Fuel"].notna() & out["VOM"].notna()].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)
    return out


def _build_fuel_to_vom_map(df_vom: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    mapping: Dict[str, Dict[str, Any]] = {}

    non_ccs = df_vom[~df_vom["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)["VOM"].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel:
                mapping[fuel] = {
                    "vom": float(avg),
                    "count": cnt,
                    "is_ccs": False,
                    "fuel": fuel,
                    "desc": "",
                }

    ccs = df_vom[df_vom["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)["VOM"].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {
                    "vom": float(avg),
                    "count": cnt,
                    "is_ccs": True,
                    "fuel": fuel,
                    "desc": desc,
                }

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set) -> None:
    if not mapping:
        return

    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys: {list(mapping.keys())}")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values (VO&M={ref_stats['vom']:.6g}).")


# ============================================================
# MAIN
# ============================================================
def vom_property_additions_step():
    # 1) Read table slice and compute mappings
    df_vom = _read_common_data_vom_table(COMMON_DATA_XLSX)
    if df_vom.empty:
        raise RuntimeError(
            f"No usable VO&M rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}."
        )

    fuel_to_stats = _build_fuel_to_vom_map(df_vom)
    if not fuel_to_stats:
        raise RuntimeError("No VO&M mappings computed from the specified table slice.")

    # Apply Biomass/Waste override => Hard Coal
    _apply_override_fuels_use_reference(
        fuel_to_stats,
        reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE,
        override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )

    print(f"Using sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    print(f"VO&M rows used (after parsing): {len(df_vom)}")
    print(f"Computed VO&M for {len(fuel_to_stats)} fuel keys (incl. CCS variants + overrides).")

    # Preview first ~10 mapping entries
    print("Computed VO&M mapping preview:")
    for i, k in enumerate(sorted(fuel_to_stats.keys(), key=lambda s: s.casefold())):
        if i >= 10:
            break
        st = fuel_to_stats[k]
        tag = " (CCS)" if st["is_ccs"] else ""
        print(f"  {k:<35} VO&M={st['vom']:.6g} n={st['count']}{tag}")

    # 2) Open PLEXOS DB
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    VOM_ENUM_ID = _resolve_semantic_enum("System", "Generator", "Generators", "VO&M Charge", 64)
    

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        # Collections (identical to earlier logic)
        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )

        # Generator<->Fuel membership collection (identical to earlier logic)
        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(
                CollectionEnum,
                must_contain_tokens=["fuel", "generator"],
                candidates=[]
            )

        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generator<->Fuel membership collection (GeneratorFuels/FuelGenerators).")

        print(f"Using VO&M enum id: {VOM_ENUM_ID}")

        # Cache existing objects
        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))

        # Diagnostics: computed keys missing in DB
        missing_fuels = [f for f in fuel_to_stats.keys() if f not in fuels_in_db]
        if missing_fuels:
            print("[WARN] Some computed fuel keys do not exist as Fuel objects in the DB (they will be skipped):")
            for f in missing_fuels[:40]:
                print(f"  - {f}")
            if len(missing_fuels) > 40:
                print(f"  ... (+{len(missing_fuels)-40} more)")

        # 3) Apply to generators with membership to each fuel (identical to earlier logic)
        writes = 0
        fuels_applied = 0
        skipped_no_memberships = 0
        fuels_with_no_gens: List[str] = []

        for fuel_name in sorted(fuel_to_stats.keys()):
            st = fuel_to_stats[fuel_name]
            vom = round(float(st["vom"]), 1)

            if fuel_name not in fuels_in_db:
                continue

            matched_gens = []
            for g in gens_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, fuel_name)
                if mem is not None:
                    matched_gens.append(g)

            if not matched_gens:
                skipped_no_memberships += 1
                fuels_with_no_gens.append(fuel_name)
                continue

            fuels_applied += 1

            for g in matched_gens:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                _add_property_row(
                    db, VOM_ENUM_ID, g_mem_id, 1, vom,
                    Scenario=scenario_str
                )
                writes += 1

            ccs_tag = " (CCS)" if st["is_ccs"] else ""
            print(
                f"Applied VO&M {vom:.6g} to {len(matched_gens)} generator(s) for Fuel='{fuel_name}'{ccs_tag} "
                f"[n={st['count']}]"
            )

        if fuels_with_no_gens:
            print("\n[WARN] Fuels found in DB but with no generators having membership (nothing written):")
            for f in fuels_with_no_gens[:60]:
                print(f"  - {f}")
            if len(fuels_with_no_gens) > 60:
                print(f"  ... (+{len(fuels_with_no_gens)-60} more)")

        print("Done. (VO&M Property Additions)")
        print(f"Fuel keys computed: {len(fuel_to_stats)}")
        print(f"Fuel keys applied (had Fuel object + >=1 generator membership): {fuels_applied}")
        print(f"Fuel keys skipped (no generator memberships found): {skipped_no_memberships}")
        print(f"Generator VO&M Charge writes: {writes}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
vom_property_additions_step()





In [ ]:
# ============================================================
# Min Up/Down Time Property Additions
# - Adds Generator "Min Up Time" and "Min Down Time" based on Common Data.xlsx
# - Reads ONLY: sheet "Common Data", rows 15-41 (inclusive)
# - Uses: Fuel=Column C, Desc=Column D (CCS detection)
# - Non-CCS: average value by Fuel (Column C)
# - CCS: average by (Fuel, Desc) and map to Fuel object name "<Fuel> - <Desc>"
# - Override rule: Biomass & Waste use Hard Coal's computed values
# - Writes property to System->Generator membership (same mechanics as the earlier property logic)
# - Uses known enum_ids (from DB Browser): Min Up Time = 85, Min Down Time = 87
# - Rounds to 1 decimal before writing
# ============================================================

import os
from pathlib import Path
from typing import Optional, Dict, Any, List
import pandas as pd
import copy
import math

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

# Optional scenario
SCENARIO_NAME: Optional[str] = "Min_Up_Down_Time"  # e.g. "36_WS"

# -----------------------------
# PROPERTY ENUMS (from DB Browser)
# -----------------------------
ENUM_MIN_UP = 86  # "Min Up Time"
ENUM_MIN_DOWN = 88  # "Min Down Time"

# -----------------------------
# Excel target range
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 15
ROW_END_1BASED = 41  # inclusive

# Fuel = C, Desc = D
COL_FUEL_IDX = 2  # C (0-based)
COL_DESC_IDX = 3  # D

# ---- Plug-in indices (0-based) ----
# Min Up Time = Column I => 8
# Min Down Time = Column J => 9
COL_MIN_UP_IDX   = 8  # I (0-based)
COL_MIN_DOWN_IDX = 9  # J (0-based)

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "  # CCS fuel object naming rule: "<Fuel> - <Desc>"

# Override mapping: these fuels should use Hard Coal's computed values
OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (copied from earlier logic for identical behavior)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _read_common_data_table_for_two_props(xlsx: Path, col_a_idx: int, col_b_idx: int) -> pd.DataFrame:
    """
    Reads the earlier fixed slice and returns:
      Fuel (C), Desc (D), PropACell, PropBCell, PropA, PropB, IsCCS
    """
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, col_a_idx, col_b_idx)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "PropACell": sub.iloc[:, col_a_idx],
        "PropBCell": sub.iloc[:, col_b_idx],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()

    def _to_float(x):
        if x is None:
            return None
        if isinstance(x, float) and math.isnan(x):
            return None
        s = str(x).strip()
        if not s:
            return None
        try:
            return float(s)
        except Exception:
            return None

    out["PropA"] = out["PropACell"].map(_to_float)
    out["PropB"] = out["PropBCell"].map(_to_float)

    # Keep rows where Fuel exists and at least one property parsed; mapping functions will filter per-property.
    out = out[(out["Fuel"] != "") & out["Fuel"].notna()].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)
    return out


def _build_fuel_to_value_map(df: pd.DataFrame, value_col: str, *, value_label: str) -> Dict[str, Dict[str, Any]]:
    """
    Builds mapping key -> stats:
      - Non-CCS: key = Fuel
      - CCS: key = "<Fuel> - <Desc>"
    Stats include: value, count, is_ccs, fuel, desc
    """
    mapping: Dict[str, Dict[str, Any]] = {}

    usable = df[(df["Fuel"] != "") & df["Fuel"].notna() & df[value_col].notna()].copy()
    if usable.empty:
        return mapping

    non_ccs = usable[~usable["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel:
                mapping[fuel] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": False,
                    "fuel": fuel,
                    "desc": "",
                    "label": value_label,
                }

    ccs = usable[usable["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": True,
                    "fuel": fuel,
                    "desc": desc,
                    "label": value_label,
                }

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set, value_key: str = "value") -> None:
    if not mapping:
        return

    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys: {list(mapping.keys())}")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(
        f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values "
        f"({ref_stats.get('label','Value')}={float(ref_stats[value_key]):.6g})."
    )


def _print_mapping_preview(mapping: Dict[str, Dict[str, Any]], *, title: str, value_key: str = "value", limit: int = 10) -> None:
    print(title)
    for i, k in enumerate(sorted(mapping.keys(), key=lambda s: s.casefold())):
        if i >= limit:
            break
        st = mapping[k]
        tag = " (CCS)" if st.get("is_ccs") else ""
        print(f"  {k:<35} {st.get('label','Value')}={float(st[value_key]):.6g} n={int(st.get('count',0))}{tag}")


# ============================================================
# MAIN
# ============================================================
def min_up_down_time_property_additions_step():
    # 1) Read table slice once and compute mappings for both properties
    df = _read_common_data_table_for_two_props(COMMON_DATA_XLSX, COL_MIN_UP_IDX, COL_MIN_DOWN_IDX)

    if df.empty:
        raise RuntimeError(
            f"No usable rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}."
        )

    fuel_to_min_up = _build_fuel_to_value_map(df, "PropA", value_label="Min Up Time")
    fuel_to_min_down = _build_fuel_to_value_map(df, "PropB", value_label="Min Down Time")

    if not fuel_to_min_up:
        raise RuntimeError("No Min Up Time mappings computed from the specified table slice.")
    if not fuel_to_min_down:
        raise RuntimeError("No Min Down Time mappings computed from the specified table slice.")

    # Apply Biomass/Waste override => Hard Coal (independently, same rule as previous steps)
    _apply_override_fuels_use_reference(
        fuel_to_min_up,
        reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE,
        override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )
    _apply_override_fuels_use_reference(
        fuel_to_min_down,
        reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE,
        override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )

    used_up = int(df["PropA"].notna().sum())
    used_dn = int(df["PropB"].notna().sum())

    print(f"Using sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    print(f"Rows with Min Up Time parsed: {used_up}")
    print(f"Rows with Min Down Time parsed: {used_dn}")
    print(f"Computed Min Up Time for {len(fuel_to_min_up)} fuel keys (incl. CCS variants + overrides).")
    print(f"Computed Min Down Time for {len(fuel_to_min_down)} fuel keys (incl. CCS variants + overrides).")

    _print_mapping_preview(fuel_to_min_up, title="Computed Min Up Time mapping preview:")
    _print_mapping_preview(fuel_to_min_down, title="Computed Min Down Time mapping preview:")

    # 2) Open PLEXOS DB
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_MIN_UP = _resolve_semantic_enum("System", "Generator", "Generators", "Min Up Time", 86)
    ENUM_MIN_DOWN = _resolve_semantic_enum("System", "Generator", "Generators", "Min Down Time", 88)
    

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        # Collections (identical to earlier logic)
        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )

        # Generator<->Fuel membership collection (identical to earlier logic)
        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(
                CollectionEnum,
                must_contain_tokens=["fuel", "generator"],
                candidates=[]
            )

        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generator<->Fuel membership collection (GeneratorFuels/FuelGenerators).")

        print(f"Using enum id: Min Up Time = {ENUM_MIN_UP}, Min Down Time = {ENUM_MIN_DOWN}")

        # Cache existing objects
        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))

        def _diagnose_missing_fuels(mapping: Dict[str, Dict[str, Any]], label: str) -> None:
            missing = [f for f in mapping.keys() if f not in fuels_in_db]
            if missing:
                print(f"[WARN] Some computed fuel keys for {label} do not exist as Fuel objects in the DB (they will be skipped):")
                for f in missing[:40]:
                    print(f"  - {f}")
                if len(missing) > 40:
                    print(f"  ... (+{len(missing)-40} more)")

        _diagnose_missing_fuels(fuel_to_min_up, "Min Up Time")
        _diagnose_missing_fuels(fuel_to_min_down, "Min Down Time")

        def _apply_mapping_to_generators(mapping: Dict[str, Dict[str, Any]], *, enum_id: int, prop_label: str) -> int:
            writes = 0
            fuels_applied = 0
            skipped_no_memberships = 0
            fuels_with_no_gens: List[str] = []

            for fuel_name in sorted(mapping.keys()):
                st = mapping[fuel_name]
                val = round(float(st["value"]), 1)

                if fuel_name not in fuels_in_db:
                    continue

                matched_gens: List[str] = []
                for g in gens_in_db:
                    mem = _get_membership_id_optional(db, gen_fuel_collection, g, fuel_name)
                    if mem is not None:
                        matched_gens.append(g)

                if not matched_gens:
                    skipped_no_memberships += 1
                    fuels_with_no_gens.append(fuel_name)
                    continue

                fuels_applied += 1

                for g in matched_gens:
                    g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                    _add_property_row(
                        db, enum_id, g_mem_id, 1, val,
                        Scenario=scenario_str
                    )
                    writes += 1

                ccs_tag = " (CCS)" if st.get("is_ccs") else ""
                print(
                    f"Applied {prop_label} {val:.6g} to {len(matched_gens)} generator(s) for Fuel='{fuel_name}'{ccs_tag} "
                    f"[n={int(st.get('count', 0))}]"
                )

            if fuels_with_no_gens:
                print(f"\n[WARN] Fuels found in DB but with no generators having membership for {prop_label} (nothing written):")
                for f in fuels_with_no_gens[:60]:
                    print(f"  - {f}")
                if len(fuels_with_no_gens) > 60:
                    print(f"  ... (+{len(fuels_with_no_gens)-60} more)")

            print(f"Done. ({prop_label} Property Additions)")
            print(f"Fuel keys computed: {len(mapping)}")
            print(f"Fuel keys applied (had Fuel object + >=1 generator membership): {fuels_applied}")
            print(f"Fuel keys skipped (no generator memberships found): {skipped_no_memberships}")
            print(f"Generator {prop_label} writes: {writes}")
            return writes

        writes_up = _apply_mapping_to_generators(fuel_to_min_up, enum_id=ENUM_MIN_UP, prop_label="Min Up Time")
        print("\n" + "=" * 60 + "\n")
        writes_dn = _apply_mapping_to_generators(fuel_to_min_down, enum_id=ENUM_MIN_DOWN, prop_label="Min Down Time")

        print("Summary:")
        print(f"  Min Up Time writes:   {writes_up}")
        print(f"  Min Down Time writes: {writes_dn}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
min_up_down_time_property_additions_step()





In [ ]:
# ============================================================
# Start Fuel Membership + DataFile-scaled Offtake at Start + DataFile-scaled Start Cost
#
# OUTPUTS CREATED:
#   - Generator_Start_Cost.csv        (same structure as Generator_Max_Capacities.csv)
#   - Generator_Offtake_at_Start.csv  (same structure as Generator_Max_Capacities.csv, BUT headers updated to "Gen~StartFuel")
#
# PLEXOS:
#   - Start Fuel memberships: copy Generatorâ†”Fuel -> Generatorâ†”StartFuel
#   - Offtake at Start (enum=1): on Generator.Start Fuels collection (Generatorâ†”Fuel membership)
#   - Start Cost (enum=71): on Systemâ†’Generator membership (Generator property)
# ============================================================

import os
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd
import copy
import math

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

# Generator MaxCap datafile (created earlier)
GEN_MAXCAP_DIR = DATA_FILES_ROOT / "Generator Data"
GEN_MAXCAP_CSV_PATH = GEN_MAXCAP_DIR / "Generator_Max_Capacities.csv"

# Where to write new datafiles (same folder as MaxCap)
OUTPUT_DIR = GEN_MAXCAP_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_START_COST_CSV_NAME = "Generator_Start_Cost.csv"
OUT_OFFTAKE_CSV_NAME    = "Generator_Offtake_at_Start.csv"

OUT_START_COST_CSV_PATH = OUTPUT_DIR / OUT_START_COST_CSV_NAME
OUT_OFFTAKE_CSV_PATH    = OUTPUT_DIR / OUT_OFFTAKE_CSV_NAME

PLEXOS_REL_START_COST_PATH = r"Data Files\Generator Data\Generator_Start_Cost.csv"
PLEXOS_REL_OFFTAKE_PATH    = r"Data Files\Generator Data\Generator_Offtake_at_Start.csv"

DATAFILE_OBJ_START_COST = "Generator_Start_Cost"
DATAFILE_OBJ_OFFTAKE    = "Generator_Offtake_at_Start"

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# PROVIDED ENUMS / IDS
# -----------------------------
ENUM_OFFTAKE_AT_START = 1    # Offtake at Start (Generator.Start Fuels membership property)
ENUM_START_COST = 72   # Start Cost (Generator property on Systemâ†’Generator)

# Start Fuel collection ids to try (user-provided)
START_FUEL_COLLECTION_IDS_TO_TRY = [8, 393]

# -----------------------------
# Excel target range / columns
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 15
ROW_END_1BASED = 41  # inclusive

COL_FUEL_IDX = 2  # C (0-based)
COL_DESC_IDX = 3  # D
COL_OFFTAKE_BASE_IDX = 10  # K (0-based)
COL_START_COST_BASE_IDX = 11  # L (0-based)

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "  # "<Fuel> - <Desc>" for CCS

OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (kept consistent with earlier steps)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _resolve_start_fuel_collection(CollectionEnum):
    name_candidates = [
        "GeneratorStartFuels",
        "GeneratorStartFuel",
        "StartFuels",
        "StartFuelGenerators",
        "GeneratorStartFuelFuels",
        "GeneratorStartFuelFuel",
    ]
    for cand in name_candidates:
        if hasattr(CollectionEnum, cand):
            return getattr(CollectionEnum, cand), f"name:{cand}"

    for cid in START_FUEL_COLLECTION_IDS_TO_TRY:
        try:
            return CollectionEnum(int(cid)), f"id:{cid}"
        except Exception:
            continue

    return None, None


def _read_common_data_table_for_two_props(xlsx: Path, col_a_idx: int, col_b_idx: int) -> pd.DataFrame:
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, col_a_idx, col_b_idx)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "PropACell": sub.iloc[:, col_a_idx],
        "PropBCell": sub.iloc[:, col_b_idx],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()

    def _to_float(x):
        if x is None:
            return None
        if isinstance(x, float) and math.isnan(x):
            return None
        s = str(x).strip()
        if not s:
            return None
        try:
            return float(s)
        except Exception:
            return None

    out["PropA"] = out["PropACell"].map(_to_float)
    out["PropB"] = out["PropBCell"].map(_to_float)

    out = out[(out["Fuel"] != "") & out["Fuel"].notna()].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)
    return out


def _build_fuel_to_value_map(df: pd.DataFrame, value_col: str, *, value_label: str) -> Dict[str, Dict[str, Any]]:
    mapping: Dict[str, Dict[str, Any]] = {}

    usable = df[(df["Fuel"] != "") & df["Fuel"].notna() & df[value_col].notna()].copy()
    if usable.empty:
        return mapping

    non_ccs = usable[~usable["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel:
                mapping[fuel] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": False,
                    "fuel": fuel,
                    "desc": "",
                    "label": value_label,
                }

    ccs = usable[usable["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": True,
                    "fuel": fuel,
                    "desc": desc,
                    "label": value_label,
                }

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set, value_key: str = "value") -> None:
    if not mapping:
        return

    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys.")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(
        f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values "
        f"({ref_stats.get('label','Value')}={float(ref_stats[value_key]):.6g})."
    )


def _print_mapping_preview(mapping: Dict[str, Dict[str, Any]], *, title: str, limit: int = 10) -> None:
    print(title)
    for i, k in enumerate(sorted(mapping.keys(), key=lambda s: s.casefold())):
        if i >= limit:
            break
        st = mapping[k]
        tag = " (CCS)" if st.get("is_ccs") else ""
        print(f"  {k:<35} {st.get('label','Value')}={float(st['value']):.6g} n={int(st.get('count',0))}{tag}")


def _read_maxcap_matrix(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Generator max capacity CSV not found: {path}")
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise RuntimeError(f"Maxcap CSV has unexpected structure: {path}")
    year_col = df.columns[0]
    df = df.copy()
    df.rename(columns={year_col: "Year"}, inplace=True)
    try:
        df["Year"] = df["Year"].astype(int)
    except Exception:
        pass
    return df


def _write_matrix_like_maxcap(maxcap_df: pd.DataFrame,
                             values_by_gen: Dict[str, List[float]],
                             out_path: Path,
                             header_map: Optional[Dict[str, str]] = None) -> None:
    out = pd.DataFrame({"Year": maxcap_df["Year"].tolist()})
    gen_cols = [c for c in maxcap_df.columns if c != "Year"]

    # Build with original generator column names as keys
    for g in gen_cols:
        col_vals = values_by_gen.get(g)
        if col_vals is None:
            out[g] = [0.0] * len(maxcap_df)
        else:
            out[g] = col_vals

    # âœ… Apply header renaming right before write (robust)
    if isinstance(header_map, dict) and header_map:
        out.rename(columns=header_map, inplace=True)

    out.to_csv(out_path, index=False)
    print(f"Wrote matrix CSV: {out_path}")


# ============================================================
# MAIN
# ============================================================
def step_start_fuel_offtake_startcost_datafiles():
    # 1) Read common data slice and compute per-fuel base mappings
    df_cd = _read_common_data_table_for_two_props(COMMON_DATA_XLSX, COL_OFFTAKE_BASE_IDX, COL_START_COST_BASE_IDX)
    if df_cd.empty:
        raise RuntimeError(
            f"No usable rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}."
        )

    fuel_to_offtake_base = _build_fuel_to_value_map(df_cd, "PropA", value_label="Offtake at Start base (per MW)")
    fuel_to_start_cost_base = _build_fuel_to_value_map(df_cd, "PropB", value_label="Start Cost base (per MW)")

    if not fuel_to_offtake_base:
        raise RuntimeError("No Offtake-at-Start base mappings computed from Common Data slice.")
    if not fuel_to_start_cost_base:
        raise RuntimeError("No Start Cost base mappings computed from Common Data slice.")

    _apply_override_fuels_use_reference(
        fuel_to_offtake_base, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )
    _apply_override_fuels_use_reference(
        fuel_to_start_cost_base, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL
    )

    print(f"Using sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    print(f"Computed Offtake base keys: {len(fuel_to_offtake_base)}")
    print(f"Computed Start Cost base keys: {len(fuel_to_start_cost_base)}")
    _print_mapping_preview(fuel_to_offtake_base, title="Computed Offtake-at-Start base mapping preview:")
    _print_mapping_preview(fuel_to_start_cost_base, title="Computed Start Cost base mapping preview:")

    # 2) Read MaxCap matrix (years x generators)
    maxcap_df = _read_maxcap_matrix(GEN_MAXCAP_CSV_PATH)
    years = maxcap_df["Year"].tolist()
    gen_cols = [c for c in maxcap_df.columns if c != "Year"]
    print(f"Loaded MaxCap matrix: years={len(years)}, generators={len(gen_cols)}")

    # 3) Open DB and build generator->fuel assignment via Generatorâ†”Fuel membership
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_START_COST = _resolve_semantic_enum("System", "Generator", "Generators", "Start Cost", 72)
    

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )
        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")

        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(CollectionEnum, ["fuel", "generator"], [])
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generatorâ†”Fuel membership collection (GeneratorFuels/FuelGenerators).")

        start_fuel_collection, start_fuel_resolved_by = _resolve_start_fuel_collection(CollectionEnum)
        if start_fuel_collection is None:
            raise RuntimeError("Could not resolve Start Fuel collection (tried ids 8 and 392, plus common enum names).")
        print(f"Resolved Start Fuel collection via {start_fuel_resolved_by}")

        datafile_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "data"],
            candidates=["SystemDataFiles", "DataFiles", "SystemDataFile"],
        )
        if datafile_mem_collection is None:
            raise RuntimeError("Could not resolve Systemâ†”DataFile membership collection (SystemDataFiles/DataFiles).")

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(
            f"Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, "
            f"collection={df_col_used!r}, property={df_prop_used!r}"
        )

        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))

        # Build generator -> list of fuel memberships
        gen_to_fuels: Dict[str, List[str]] = {}
        for g in gens_in_db:
            matched = []
            for f in fuels_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, f)
                if mem is not None:
                    matched.append(f)
            if matched:
                gen_to_fuels[g] = matched

        db_gen_set = set(gens_in_db)

        # Choose ONE start fuel per generator (same as before)
        gen_assigned_fuel: Dict[str, str] = {}
        mapping_keys_union = set(fuel_to_offtake_base.keys()) | set(fuel_to_start_cost_base.keys())

        for g in gen_cols:
            fuels = gen_to_fuels.get(g, [])
            if not fuels:
                continue
            pref = [f for f in fuels if f in mapping_keys_union]
            chosen_list = pref if pref else fuels
            gen_assigned_fuel[g] = chosen_list[0]  # this is the StartFuel name weâ€™ll put after "~"

        # 4) Ensure Start Fuel memberships (copy Generatorâ†”Fuel -> Generatorâ†”StartFuel)
        created_sf = 0
        already_sf = 0
        for g, fuels in gen_to_fuels.items():
            for f in fuels:
                before = _get_membership_id_optional(db, start_fuel_collection, g, f)
                _ensure_membership(db, start_fuel_collection, g, f)
                after = _get_membership_id_optional(db, start_fuel_collection, g, f)
                if before is None and after is not None:
                    created_sf += 1
                else:
                    already_sf += 1
        print(f"Start Fuel membership copy: created={created_sf}, already_existed={already_sf}")

        # 5) Build scaled matrices
        start_cost_values_by_gen: Dict[str, List[float]] = {}
        offtake_values_by_gen: Dict[str, List[float]] = {}

        maxcap_numeric = {}
        for g in gen_cols:
            s = pd.to_numeric(maxcap_df[g], errors="coerce").fillna(0.0)
            maxcap_numeric[g] = s.tolist()

        for g in gen_cols:
            fuel = gen_assigned_fuel.get(g)
            if not fuel:
                start_cost_values_by_gen[g] = [0.0] * len(years)
                offtake_values_by_gen[g] = [0.0] * len(years)
                continue

            base_offtake = float(fuel_to_offtake_base.get(fuel, {}).get("value", 0.0))
            base_sc      = float(fuel_to_start_cost_base.get(fuel, {}).get("value", 0.0))

            caps = maxcap_numeric[g]
            start_cost_values_by_gen[g] = [round(float(base_sc) * float(c), 1) for c in caps]
            offtake_values_by_gen[g] = [round(float(base_offtake) * float(c), 1) for c in caps]

        # âœ… KEY CHANGE: Offtake header map is built for ALL generator columns where we know a start fuel
        # Resulting header: "<GeneratorName>~<StartFuelName>"
        offtake_header_map: Dict[str, str] = {}
        for g in gen_cols:
            sf = gen_assigned_fuel.get(g)
            if sf:
                offtake_header_map[g] = f"{g}~{sf}"

        # Start Cost CSV unchanged
        _write_matrix_like_maxcap(maxcap_df, start_cost_values_by_gen, OUT_START_COST_CSV_PATH)

        # Offtake CSV with renamed headers
        _write_matrix_like_maxcap(maxcap_df, offtake_values_by_gen, OUT_OFFTAKE_CSV_PATH, header_map=offtake_header_map)

        # 6) Create DataFile objects and set filename property (relative path)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJ_START_COST)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJ_OFFTAKE)

        df_mem_id_sc = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJ_START_COST)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_sc, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_START_COST_PATH),
            Scenario=scenario_str
        )

        df_mem_id_of = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJ_OFFTAKE)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_of, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_OFFTAKE_PATH),
            Scenario=scenario_str
        )

        # 7) Link Start Cost property (enum 71) via DataFile
        start_cost_links = 0
        valid_fuels_for_start_cost = set(fuel_to_start_cost_base.keys())
        target_gens_for_start_cost = sorted(
            g for g, f in gen_assigned_fuel.items()
            if (g in db_gen_set) and (f in valid_fuels_for_start_cost)
        )
        for g in target_gens_for_start_cost:
            g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
            _add_property_row(
                db, ENUM_START_COST, g_mem_id, 1, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_START_COST_PATH),
                Scenario=scenario_str
            )
            start_cost_links += 1
        print(f"Linked Start Cost (enum={ENUM_START_COST}) via DataFile to {start_cost_links} generator(s).")

        # 8) Link Offtake at Start property (enum 1) to Generatorâ†”StartFuel membership using DataFile
        offtake_links = 0
        missing_sf_mem = 0

        for g, fuels in gen_to_fuels.items():
            if g not in db_gen_set:
                continue
            for f in fuels:
                try:
                    sf_mem_id = _ensure_membership(db, start_fuel_collection, g, f)
                except Exception:
                    missing_sf_mem += 1
                    continue

                _add_property_row(
                    db, ENUM_OFFTAKE_AT_START, sf_mem_id, 1, 0.0,
                    DataFile=SystemNS.String(PLEXOS_REL_OFFTAKE_PATH),
                    Scenario=scenario_str
                )
                offtake_links += 1

        print(
            f"Linked Offtake at Start (enum={ENUM_OFFTAKE_AT_START}) via DataFile to "
            f"{offtake_links} Generatorâ†”StartFuel membership(s)."
        )
        if missing_sf_mem:
            print(f"[WARN] Failed to create/read {missing_sf_mem} StartFuel memberships while linking Offtake.")

        print("Done.")
        print("Summary:")
        print(f"  StartFuel memberships created: {created_sf}")
        print(f"  StartFuel memberships already existed: {already_sf}")
        print(f"  Start Cost DataFile object: {DATAFILE_OBJ_START_COST} -> {PLEXOS_REL_START_COST_PATH}")
        print(f"  Offtake DataFile object: {DATAFILE_OBJ_OFFTAKE} -> {PLEXOS_REL_OFFTAKE_PATH}")
        print(f"  Start Cost links written: {start_cost_links}")
        print(f"  Offtake-at-Start links written: {offtake_links}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_start_fuel_offtake_startcost_datafiles()





In [ ]:
# ============================================================
# Outage Property Additions (Forced/Maintenance + MTTR)
#
# EXISTING (keep untouched):
#   - Band 1 (Forced Outage Rate + MTTR) written to Scenario "Forced_Outage"
#   - Band 2 (Maintenance Rate + MTTR)   written to Scenario "Optimized_Maintenance"
#
# NEW (added, without changing the above behavior):
#   - Create TWO generator daily profile CSVs from Maintenance_profile.csv:
#       1) Generator_Planned_Maintenance.csv  -> Generator property "Outage Rating" (enum 239)
#       2) Generator_Units_Out.csv            -> Generator property "Units Out"    (enum 232)
#   - Link these CSVs via the DataFile column (same pattern as Max Capacities)
#   - Write them under a NEW scenario (default): "Maintenance"
#
# FIX (date parsing):
#   - Maintenance_profile.csv "Date" is in "DD/MM/YYYY" format.
#     We parse with dayfirst=True to ensure ALL days are represented correctly.
#
# UPDATE (this change only):
#   - Outage Rating (239) and Units Out (232) are linked on BAND 3 (not band 1).
#
# NEW UPDATE (this change only):
#   - Generator_Units_Out.csv is derived from Generator_Planned_Maintenance.csv:
#       For each Year and each Generator column:
#         UnitsOut = 1 if PlannedMaintenanceValue != (max of that generator column within that Year)
#                    else 0
#     (Year/Month/Day columns stay unchanged; headers align 1:1)
# ============================================================

import os
from pathlib import Path
from typing import Optional, Dict, Any, List
import pandas as pd
import copy
import math

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

# -----------------------------
# SCENARIOS (UNCHANGED FOR EXISTING BEHAVIOR)
# -----------------------------
# Band 1 scenario for Forced Outage Rate + MTTR
SCENARIO_FOR_FORCED: Optional[str] = "Forced_Outage"

# Band 2 scenario for Maintenance Rate + MTTR
SCENARIO_FOR_MAINT: Optional[str] = "Optimized_Maintenance"

# NEW scenario for the daily maintenance profile (Outage Rating + Units Out data files)
SCENARIO_FOR_DAILY_MAINT_PROFILE: Optional[str] = "Planned_Maintenance"

# -----------------------------
# ENUMS (provided)
# -----------------------------
ENUM_FORCED_OUTAGE_RATE = 230
ENUM_MAINT_RATE = 234
ENUM_MTTR = 244  # Mean Time to Repair

# NEW generator outage profile enums
ENUM_UNITS_OUT = 233
ENUM_OUTAGE_RATING = 240

# -----------------------------
# Excel target range (outage property table)
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 49
ROW_END_1BASED = 75  # inclusive

# Fuel/Desc
COL_FUEL_IDX = 2  # C (0-based)
COL_DESC_IDX = 3  # D

# Table columns:
# Forced Outage Rate = E => 4
# MTTR (days)        = F => 5
# Maintenance Rate   = H => 7
COL_FORCED_IDX = 4  # E
COL_MTTR_IDX   = 5  # F
COL_MAINT_IDX  = 7  # H

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "

OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# Unit conversions
RATE_FRACTION_TO_PERCENT_UNITS = 100.0  # 0.05 -> 5.0
MTTR_DAYS_TO_HOURS = 24.0

# -----------------------------
# NEW: Maintenance profile inputs/outputs
# -----------------------------
MAINT_PROFILE_CSV = DASHBOARD_RAWDATA_DIR / "Maintenance_profile.csv"

GEN_MAX_CAP_CSV = DATA_FILES_ROOT / r"Generator Data\Generator_Max_Capacities.csv"

OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTAGE_RATING_CSV_NAME = "Generator_Planned_Maintenance.csv"
UNITS_OUT_CSV_NAME     = "Generator_Units_Out.csv"
OUTAGE_RATING_CSV_PATH = OUTPUT_DIR / OUTAGE_RATING_CSV_NAME
UNITS_OUT_CSV_PATH     = OUTPUT_DIR / UNITS_OUT_CSV_NAME

# PLEXOS relative paths (must match how you referenced Generator_Max_Capacities.csv)
PLEXOS_REL_OUTAGE_RATING_CSV = r"Data Files\Generator Data\Generator_Planned_Maintenance.csv"
PLEXOS_REL_UNITS_OUT_CSV     = r"Data Files\Generator Data\Generator_Units_Out.csv"

DATAFILE_OBJECT_OUTAGE_RATING = "Generator_Planned_Maintenance"
DATAFILE_OBJECT_UNITS_OUT     = "Generator_Units_Out"

# Technology -> Generator suffix rule (based on your examples)
TECH_TO_GEN_SUFFIX = {
    "Gas": "Natural gas",
    "Hard coal": "Hard coal",
    "Heavy oil": "Heavy oil",
    "Hydrogen": "Hydrogen",
    "Lignite": "Lignite",
    "Light oil": "Light oil",
    "Shale oil": "Shale oil",
    "Nuclear": "Nuclear",
}
ALLOWED_TECHS_LC = {k.strip().lower() for k in TECH_TO_GEN_SUFFIX.keys()}

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (existing ones kept, plus a few new ones)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _read_common_data_outage_table(xlsx: Path) -> pd.DataFrame:
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, COL_FORCED_IDX, COL_MTTR_IDX, COL_MAINT_IDX)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "ForcedCell": sub.iloc[:, COL_FORCED_IDX],
        "MTTRCell": sub.iloc[:, COL_MTTR_IDX],
        "MaintCell": sub.iloc[:, COL_MAINT_IDX],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()

    def _to_float(x):
        if x is None:
            return None
        if isinstance(x, float) and math.isnan(x):
            return None
        s = str(x).strip()
        if not s:
            return None
        try:
            return float(s)
        except Exception:
            return None

    # Parse raw numbers
    out["Forced_raw"] = out["ForcedCell"].map(_to_float)
    out["Maint_raw"]  = out["MaintCell"].map(_to_float)
    out["MTTR_Days"]  = out["MTTRCell"].map(_to_float)

    # Convert rates to percent-units BEFORE averaging
    out["Forced"] = out["Forced_raw"].map(lambda v: None if v is None else float(v) * RATE_FRACTION_TO_PERCENT_UNITS)
    out["Maint"]  = out["Maint_raw"].map(lambda v: None if v is None else float(v) * RATE_FRACTION_TO_PERCENT_UNITS)

    out = out[(out["Fuel"] != "") & out["Fuel"].notna()].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)
    return out


def _build_fuel_to_value_map(df: pd.DataFrame, value_col: str, *, value_label: str) -> Dict[str, Dict[str, Any]]:
    mapping: Dict[str, Dict[str, Any]] = {}

    usable = df[(df["Fuel"] != "") & df["Fuel"].notna() & df[value_col].notna()].copy()
    if usable.empty:
        return mapping

    non_ccs = usable[~usable["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel:
                mapping[fuel] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": False,
                    "fuel": fuel,
                    "desc": "",
                    "label": value_label,
                }

    ccs = usable[usable["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {
                    "value": float(avg),
                    "count": cnt,
                    "is_ccs": True,
                    "fuel": fuel,
                    "desc": desc,
                    "label": value_label,
                }

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set, value_key: str = "value") -> None:
    if not mapping:
        return

    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys.")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(
        f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values "
        f"({ref_stats.get('label','Value')}={float(ref_stats[value_key]):.6g})."
    )


def _print_mapping_preview(mapping: Dict[str, Dict[str, Any]], *, title: str, limit: int = 10) -> None:
    print(title)
    for i, k in enumerate(sorted(mapping.keys(), key=lambda s: s.casefold())):
        if i >= limit:
            break
        st = mapping[k]
        tag = " (CCS)" if st.get("is_ccs") else ""
        print(f"  {k:<35} {st.get('label','Value')}={float(st['value']):.6g} n={int(st.get('count',0))}{tag}")


def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Bidding zone list not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _tech_to_generator_name(bz: str, tech: str) -> Optional[str]:
    bz = str(bz).strip()
    tech = str(tech).strip()
    if not bz or not tech:
        return None

    tech_lc = tech.lower().strip()
    match_key = None
    for k in TECH_TO_GEN_SUFFIX.keys():
        if k.lower().strip() == tech_lc:
            match_key = k
            break
    if match_key is None:
        return None

    suffix = TECH_TO_GEN_SUFFIX[match_key]
    return f"{bz}_{suffix}"


def _build_daily_maintenance_profile_csvs(
    maint_profile_csv: Path,
    bidding_zone_xlsx: Path,
    gen_max_cap_csv: Path,
    out_outage_rating_csv: Path,
    out_units_out_csv: Path,
) -> List[str]:
    if not maint_profile_csv.exists():
        raise FileNotFoundError(f"Maintenance profile CSV not found: {maint_profile_csv}")
    if not gen_max_cap_csv.exists():
        raise FileNotFoundError(f"Generator max capacities CSV not found: {gen_max_cap_csv}")

    allowed_bzs = _read_bidding_zones(bidding_zone_xlsx)
    if not allowed_bzs:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    dfm = pd.read_csv(maint_profile_csv, dtype=str)
    dfm.columns = [str(c).strip() for c in dfm.columns]

    col_date = None
    col_bz = None
    col_tech = None
    col_maint = None
    for c in dfm.columns:
        lc = c.lower().strip()
        if lc == "date":
            col_date = c
        elif lc in ("bidding zone", "bidding_zone", "biddingzone", "market node", "market_node", "node", "region"):
            col_bz = c
        elif lc == "technology":
            col_tech = c
        elif lc in ("maintenance (gw)", "maintenance_gw", "maintenance", "maintenance gw"):
            col_maint = c

    missing = [n for n, v in [("Date", col_date), ("Bidding Zone", col_bz), ("Technology", col_tech), ("Maintenance (GW)", col_maint)] if v is None]
    if missing:
        raise RuntimeError(f"Maintenance_profile.csv missing required columns: {missing}. Found: {list(dfm.columns)}")

    dfm = dfm[[col_date, col_bz, col_tech, col_maint]].copy()
    dfm.rename(columns={col_date: "Date", col_bz: "Bidding_Zone", col_tech: "Technology", col_maint: "MaintGW"}, inplace=True)

    dfm["Bidding_Zone"] = dfm["Bidding_Zone"].astype(str).str.strip()
    dfm["Technology"] = dfm["Technology"].astype(str).str.strip()

    dfm = dfm[dfm["Bidding_Zone"].isin(allowed_bzs)].copy()
    dfm = dfm[dfm["Technology"].str.lower().isin(ALLOWED_TECHS_LC)].copy()

    if dfm.empty:
        raise RuntimeError("No rows remain in Maintenance_profile.csv after filtering bidding zones + allowed technologies.")

    dfm["Date"] = pd.to_datetime(dfm["Date"], errors="coerce", dayfirst=True)
    dfm["MaintGW"] = pd.to_numeric(dfm["MaintGW"], errors="coerce").fillna(0.0)

    dfm = dfm[dfm["Date"].notna()].copy()
    if dfm.empty:
        raise RuntimeError("No valid dates found in Maintenance_profile.csv after parsing Date column.")

    dfm["Generator"] = dfm.apply(lambda r: _tech_to_generator_name(r["Bidding_Zone"], r["Technology"]), axis=1)
    dfm = dfm[dfm["Generator"].notna()].copy()
    if dfm.empty:
        raise RuntimeError("No valid (Bidding Zone, Technology) -> Generator mappings could be created from Maintenance_profile.csv.")

    dfm["MaintMW"] = dfm["MaintGW"].astype(float) * 1000.0

    maint_pivot = dfm.pivot_table(index="Date", columns="Generator", values="MaintMW", aggfunc="sum", fill_value=0.0)

    cap = pd.read_csv(gen_max_cap_csv)
    cap.columns = [str(c).strip() for c in cap.columns]
    if "Year" not in cap.columns:
        raise RuntimeError(f"Generator_Max_Capacities.csv must have a 'Year' column. Found: {list(cap.columns)}")

    cap["Year"] = pd.to_numeric(cap["Year"], errors="coerce")
    cap = cap[cap["Year"].notna()].copy()
    cap["Year"] = cap["Year"].astype(int)

    generators = sorted(list(maint_pivot.columns))

    dates = pd.Index(sorted(pd.to_datetime(maint_pivot.index).unique()))
    out = pd.DataFrame({"Date": dates})
    out["Year"] = out["Date"].dt.year.astype(int)
    out["Month"] = out["Date"].dt.month.astype(int)
    out["Day"] = out["Date"].dt.day.astype(int)

    maint_pivot = maint_pivot.reindex(dates, fill_value=0.0)
    cap_by_year = cap.set_index("Year")

    planned_df = out[["Year", "Month", "Day"]].copy()

    for g in generators:
        cap_vals = []
        for y in out["Year"].tolist():
            if y in cap_by_year.index and g in cap_by_year.columns:
                try:
                    v = float(cap_by_year.loc[y, g])
                except Exception:
                    v = 0.0
            else:
                v = 0.0
            cap_vals.append(v)

        cap_series = pd.Series(cap_vals, index=dates, dtype=float)
        maint_series = maint_pivot[g].astype(float) if g in maint_pivot.columns else pd.Series(0.0, index=dates)

        planned = (cap_series - maint_series).clip(lower=0.0)
        planned_df[g] = planned.values

    units_df = planned_df[["Year", "Month", "Day"]].copy()
    gen_cols = [c for c in planned_df.columns if c not in ("Year", "Month", "Day")]

    planned_numeric = planned_df.copy()
    for c in gen_cols:
        planned_numeric[c] = pd.to_numeric(planned_numeric[c], errors="coerce").fillna(0.0)

    year_max = planned_numeric.groupby("Year")[gen_cols].transform("max")
    units_vals = (planned_numeric[gen_cols] != year_max).astype(int)

    for c in gen_cols:
        units_df[c] = units_vals[c].values

    planned_df.to_csv(out_outage_rating_csv, index=False)
    print(f"Wrote: {out_outage_rating_csv}")

    units_df.to_csv(out_units_out_csv, index=False)
    print(f"Wrote: {out_units_out_csv}")

    years = sorted(out["Year"].unique().tolist())
    for y in years[:10]:
        n = int((out["Year"] == y).sum())
        print(f"[INFO] Output rows for Year={y}: {n} (expected 365 or 366 if full-year input)")

    return generators


# ============================================================
# MAIN
# ============================================================
def outage_property_additions_step():
    generators_for_profile = _build_daily_maintenance_profile_csvs(
        MAINT_PROFILE_CSV,
        BIDDING_ZONE_XLSX,
        GEN_MAX_CAP_CSV,
        OUTAGE_RATING_CSV_PATH,
        UNITS_OUT_CSV_PATH,
    )

    df_out = _read_common_data_outage_table(COMMON_DATA_XLSX)
    if df_out.empty:
        raise RuntimeError(
            f"No usable rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}."
        )

    raw_forced = pd.to_numeric(df_out["Forced_raw"], errors="coerce").dropna()
    raw_maint = pd.to_numeric(df_out["Maint_raw"], errors="coerce").dropna()
    if not raw_forced.empty:
        print(f"Forced raw sample: min={raw_forced.min():.6g}, median={raw_forced.median():.6g}, max={raw_forced.max():.6g}")
    if not raw_maint.empty:
        print(f"Maint  raw sample: min={raw_maint.min():.6g}, median={raw_maint.median():.6g}, max={raw_maint.max():.6g}")
    conv_forced = pd.to_numeric(df_out["Forced"], errors="coerce").dropna()
    conv_maint = pd.to_numeric(df_out["Maint"], errors="coerce").dropna()
    if not conv_forced.empty:
        print(f"Forced %unit sample: min={conv_forced.min():.6g}, median={conv_forced.median():.6g}, max={conv_forced.max():.6g}")
    if not conv_maint.empty:
        print(f"Maint  %unit sample: min={conv_maint.min():.6g}, median={conv_maint.median():.6g}, max={conv_maint.max():.6g}")

    fuel_to_forced = _build_fuel_to_value_map(df_out, "Forced", value_label="Forced Outage Rate (%)")
    fuel_to_maint  = _build_fuel_to_value_map(df_out, "Maint",  value_label="Maintenance Rate (%)")
    fuel_to_mttr_d = _build_fuel_to_value_map(df_out, "MTTR_Days", value_label="Mean Time to Repair (days)")

    if not fuel_to_forced:
        raise RuntimeError("No Forced Outage Rate mappings computed from the specified table slice.")
    if not fuel_to_maint:
        raise RuntimeError("No Maintenance Rate mappings computed from the specified table slice.")
    if not fuel_to_mttr_d:
        raise RuntimeError("No Mean Time to Repair mappings computed from the specified table slice.")

    _apply_override_fuels_use_reference(fuel_to_forced, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)
    _apply_override_fuels_use_reference(fuel_to_maint,  reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)
    _apply_override_fuels_use_reference(fuel_to_mttr_d, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)

    print(f"\nUsing sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    print(f"Rows with Forced parsed: {int(df_out['Forced'].notna().sum())}")
    print(f"Rows with Maint parsed:  {int(df_out['Maint'].notna().sum())}")
    print(f"Rows with MTTR parsed:   {int(df_out['MTTR_Days'].notna().sum())}")

    _print_mapping_preview(fuel_to_forced, title="Computed Forced Outage Rate mapping preview (% units):")
    _print_mapping_preview(fuel_to_maint,  title="Computed Maintenance Rate mapping preview (% units):")

    mttr_hours_preview = {}
    for k, st in fuel_to_mttr_d.items():
        mttr_hours_preview[k] = dict(st)
        mttr_hours_preview[k]["value"] = float(st["value"]) * MTTR_DAYS_TO_HOURS
        mttr_hours_preview[k]["label"] = "Mean Time to Repair (hours)"
    _print_mapping_preview(mttr_hours_preview, title="Computed MTTR mapping preview (hours):")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)

    ENUM_FORCED_OUTAGE_RATE = _resolve_semantic_enum("System", "Generator", "Generators", "Forced Outage Rate", 230)
    ENUM_MAINT_RATE = _resolve_semantic_enum("System", "Generator", "Generators", "Maintenance Rate", 234)
    ENUM_MTTR = _resolve_semantic_enum("System", "Generator", "Generators", "Mean Time to Repair", 244)
    ENUM_UNITS_OUT = _resolve_semantic_enum("System", "Generator", "Generators", "Units Out", 233)
    ENUM_OUTAGE_RATING = _resolve_semantic_enum("System", "Generator", "Generators", "Outage Rating", 240)

    try:
        scenario_forced_str = None
        scenario_maint_str = None
        scenario_profile_str = None

        if SCENARIO_FOR_FORCED:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_FOR_FORCED)
            scenario_forced_str = SystemNS.String(SCENARIO_FOR_FORCED)

        if SCENARIO_FOR_MAINT:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_FOR_MAINT)
            scenario_maint_str = SystemNS.String(SCENARIO_FOR_MAINT)

        if SCENARIO_FOR_DAILY_MAINT_PROFILE:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_FOR_DAILY_MAINT_PROFILE)
            scenario_profile_str = SystemNS.String(SCENARIO_FOR_DAILY_MAINT_PROFILE)

        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )

        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(CollectionEnum, ["fuel", "generator"], [])

        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generator<->Fuel membership collection (GeneratorFuels/FuelGenerators).")

        datafile_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "data", "file"],
            candidates=["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
        )
        if datafile_mem_collection is None:
            raise RuntimeError("Could not resolve System<->DataFiles membership collection (SystemDataFiles/DataFiles).")

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"[INFO] Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))

        writes_forced = 0
        writes_maint = 0
        writes_mttr_band1 = 0
        writes_mttr_band2 = 0
        fuels_with_no_gens: List[str] = []

        fuel_keys = sorted(set(fuel_to_forced.keys()) | set(fuel_to_maint.keys()) | set(fuel_to_mttr_d.keys()))

        for fuel_name in fuel_keys:
            if fuel_name not in fuels_in_db:
                continue

            matched_gens: List[str] = []
            for g in gens_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, fuel_name)
                if mem is not None:
                    matched_gens.append(g)

            if not matched_gens:
                fuels_with_no_gens.append(fuel_name)
                continue

            forced_val = round(float(fuel_to_forced[fuel_name]["value"]), 1) if fuel_name in fuel_to_forced else None
            maint_val  = round(float(fuel_to_maint[fuel_name]["value"]), 1) if fuel_name in fuel_to_maint else None
            mttr_hours = round(float(fuel_to_mttr_d[fuel_name]["value"]) * MTTR_DAYS_TO_HOURS, 1) if fuel_name in fuel_to_mttr_d else None

            for g in matched_gens:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)

                if forced_val is not None:
                    _add_property_row(db, ENUM_FORCED_OUTAGE_RATE, g_mem_id, 1, forced_val, Scenario=scenario_forced_str)
                    writes_forced += 1
                if mttr_hours is not None:
                    _add_property_row(db, ENUM_MTTR, g_mem_id, 1, mttr_hours, Scenario=scenario_forced_str)
                    writes_mttr_band1 += 1

                if maint_val is not None:
                    _add_property_row(db, ENUM_MAINT_RATE, g_mem_id, 2, maint_val, Scenario=scenario_maint_str)
                    writes_maint += 1
                if mttr_hours is not None:
                    _add_property_row(db, ENUM_MTTR, g_mem_id, 2, mttr_hours, Scenario=scenario_maint_str)
                    writes_mttr_band2 += 1

            print(
                f"Fuel='{fuel_name}': gens={len(matched_gens)} "
                f"Forced%={forced_val if forced_val is not None else 'NA'} (band1), "
                f"Maint%={maint_val if maint_val is not None else 'NA'} (band2), "
                f"MTTRh={mttr_hours if mttr_hours is not None else 'NA'} (bands1&2)"
            )

        if fuels_with_no_gens:
            print("\n[WARN] Fuels found in DB but with no generators having membership (nothing written):")
            for f in fuels_with_no_gens[:60]:
                print(f"  - {f}")
            if len(fuels_with_no_gens) > 60:
                print(f"  ... (+{len(fuels_with_no_gens)-60} more)")

        print("Done. (Outage Property Additions)")
        print("Writes summary:")
        print(f"  Forced Outage Rate writes (band 1): {writes_forced}")
        print(f"  Maintenance Rate writes (band 2):   {writes_maint}")
        print(f"  MTTR writes (band 1):               {writes_mttr_band1}")
        print(f"  MTTR writes (band 2):               {writes_mttr_band2}")

        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_OUTAGE_RATING)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_UNITS_OUT)

        df_mem_id_outage = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_OUTAGE_RATING)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_outage, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_OUTAGE_RATING_CSV),
            Scenario=scenario_profile_str
        )

        df_mem_id_units = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_UNITS_OUT)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_units, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_UNITS_OUT_CSV),
            Scenario=scenario_profile_str
        )

        gens_in_db_set = set(gens_in_db)
        linked_outage_rating = 0
        linked_units_out = 0
        skipped_missing_generators: List[str] = []

        for g in generators_for_profile:
            if g not in gens_in_db_set:
                skipped_missing_generators.append(g)
                continue

            g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)

            _add_property_row(
                db, ENUM_OUTAGE_RATING, g_mem_id, 3, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_OUTAGE_RATING_CSV),
                Scenario=scenario_profile_str
            )
            linked_outage_rating += 1

            _add_property_row(
                db, ENUM_UNITS_OUT, g_mem_id, 3, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_UNITS_OUT_CSV),
                Scenario=scenario_profile_str
            )
            linked_units_out += 1

        print("\nDone. (NEW Daily Maintenance Profile DataFiles)")
        print(f"  Linked Outage Rating (enum {ENUM_OUTAGE_RATING}) to datafile for generators: {linked_outage_rating}")
        print(f"  Linked Units Out   (enum {ENUM_UNITS_OUT})   to datafile for generators: {linked_units_out}")

        if skipped_missing_generators:
            print("\n[WARN] These generators were referenced by the maintenance profile but do not exist in the DB (skipped):")
            for s in skipped_missing_generators[:80]:
                print(f"  - {s}")
            if len(skipped_missing_generators) > 80:
                print(f"  ... (+{len(skipped_missing_generators)-80} more)")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
outage_property_additions_step()

In [ ]:
# ============================================================
# Min Stable Factor + DataFile-scaled Max Ramp Up/Down
#
# Reads the same outage-style table used earlier:
#   Common Data.xlsx, sheet "Common Data", rows 49â€“75 (inclusive)
#   Fuel = C, Desc = D (CCS detection)
#   Non-CCS: avg by Fuel
#   CCS: avg by (Fuel, Desc) -> Fuel object "<Fuel> - <Desc>"
#   Override: Biomass & Waste use Hard Coal's computed values
#
# Properties:
#   1) Min Stable Factor (enum 55) from Column I
#      - Table is fraction -> convert to percent-units by *100 BEFORE averaging
#      - Write to Systemâ†’Generator membership (band 1)
#
#   2) Max Ramp Up (enum 104) from Column J
#      - Table is fraction of MaxCap -> convert to percent-units by *100 BEFORE averaging
#      - Then for each generator and each year: (avg_percent_units/100) * MaxCap(year)
#      - Write via DataFile matrix (Year x Generator) like Generator_Max_Capacities.csv
#      - DataFile object: "Generator_Max_Ramp_Up"
#      - CSV: ...\Generator_Max_Ramp_Up.csv (same directory)
#
#   3) Max Ramp Down (enum 108) from Column K
#      - Same logic as Max Ramp Up
#      - DataFile object: "Generator_Max_Ramp_Down"
#      - CSV: ...\Generator_Max_Ramp_Down.csv
#
# NOTE:
# - We scale ramp % of max capacity into an ABSOLUTE ramp value (MW/h etc.) by multiplying by MaxCap.
# - Rounds to 1 decimal before writing to PLEXOS / before writing CSV values.
#
# ============================================================

import os
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd
import copy
import math

# -----------------------------
# USER PATHS
# -----------------------------
COMMON_DATA_XLSX = COMMON_DATA_DIR / "Common Data.xlsx"

GEN_DATA_DIR = DATA_FILES_ROOT / "Generator Data"
GEN_DATA_DIR.mkdir(parents=True, exist_ok=True)

GEN_MAXCAP_CSV_PATH = GEN_DATA_DIR / "Generator_Max_Capacities.csv"

OUT_RAMP_UP_CSV_NAME = "Generator_Max_Ramp_Up.csv"
OUT_RAMP_DN_CSV_NAME = "Generator_Max_Ramp_Down.csv"

OUT_RAMP_UP_CSV_PATH = GEN_DATA_DIR / OUT_RAMP_UP_CSV_NAME
OUT_RAMP_DN_CSV_PATH = GEN_DATA_DIR / OUT_RAMP_DN_CSV_NAME

PLEXOS_REL_RAMP_UP_PATH = r"Data Files\Generator Data\Generator_Max_Ramp_Up.csv"
PLEXOS_REL_RAMP_DN_PATH = r"Data Files\Generator Data\Generator_Max_Ramp_Down.csv"

DATAFILE_OBJ_RAMP_UP = "Generator_Max_Ramp_Up"
DATAFILE_OBJ_RAMP_DN = "Generator_Max_Ramp_Down"

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# ENUMS (provided)
# -----------------------------
ENUM_MIN_STABLE_FACTOR = 56
ENUM_MAX_RAMP_UP = 105
ENUM_MAX_RAMP_DOWN = 109

# -----------------------------
# Excel table slice (same as earlier outage logic)
# -----------------------------
SHEET_NAME = "Common Data"
ROW_START_1BASED = 49
ROW_END_1BASED = 75  # inclusive

COL_FUEL_IDX = 2  # C
COL_DESC_IDX = 3  # D

# New columns:
# Min Stable Factor = I => 8
# Max Ramp Up       = J => 9
# Max Ramp Down     = K => 10
COL_MIN_STABLE_IDX = 8   # I
COL_RAMP_UP_IDX    = 9   # J
COL_RAMP_DN_IDX    = 10  # K

CCS_TOKEN = "CCS"
FUEL_SEPARATOR = " - "

OVERRIDE_FUELS_USE_HARD_COAL = {"Biomass", "Waste"}
REFERENCE_FUEL_FOR_OVERRIDE = "Hard Coal"

# Unit conversions
FRACTION_TO_PERCENT_UNITS = 100.0  # 0.15 -> 15.0

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS (kept consistent with Steps 4.x)
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], candidates: List[str]):
    for c in candidates:
        if hasattr(CollectionEnum, c):
            return getattr(CollectionEnum, c)

    toks = [t.lower().strip() for t in (must_contain_tokens or []) if t]
    try:
        names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    except Exception:
        names = []

    for n in names:
        ln = n.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, n)
            except Exception:
                continue
    return None


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _read_common_data_step_table(xlsx: Path) -> pd.DataFrame:
    if not xlsx.exists():
        raise FileNotFoundError(f"Common Data file not found: {xlsx}")

    df = pd.read_excel(xlsx, sheet_name=SHEET_NAME, header=None)

    max_idx = max(COL_FUEL_IDX, COL_DESC_IDX, COL_MIN_STABLE_IDX, COL_RAMP_UP_IDX, COL_RAMP_DN_IDX)
    if df.shape[1] <= max_idx:
        raise RuntimeError(f"Sheet {SHEET_NAME!r} has only {df.shape[1]} columns; expected at least {max_idx+1}.")

    start0 = ROW_START_1BASED - 1
    end0_excl = ROW_END_1BASED
    sub = df.iloc[start0:end0_excl, :].copy()

    out = pd.DataFrame({
        "Fuel": sub.iloc[:, COL_FUEL_IDX],
        "Desc": sub.iloc[:, COL_DESC_IDX],
        "MinStableCell": sub.iloc[:, COL_MIN_STABLE_IDX],
        "RampUpCell": sub.iloc[:, COL_RAMP_UP_IDX],
        "RampDnCell": sub.iloc[:, COL_RAMP_DN_IDX],
    })

    out["Fuel"] = out["Fuel"].astype(str).str.strip()
    out["Desc"] = out["Desc"].astype(str).fillna("").str.strip()

    def _to_float(x):
        if x is None:
            return None
        if isinstance(x, float) and math.isnan(x):
            return None
        s = str(x).strip()
        if not s:
            return None
        try:
            return float(s)
        except Exception:
            return None

    # Parse raw (likely fractions) then convert to percent-units BEFORE averaging where needed
    out["MinStable_raw"] = out["MinStableCell"].map(_to_float)
    out["RampUp_raw"]    = out["RampUpCell"].map(_to_float)   # fraction of maxcap
    out["RampDn_raw"]    = out["RampDnCell"].map(_to_float)   # fraction of maxcap

    out["MinStable_pct"] = out["MinStable_raw"].map(lambda v: None if v is None else float(v) * FRACTION_TO_PERCENT_UNITS)
    out["RampUp_pct"]    = out["RampUp_raw"].map(lambda v: None if v is None else float(v) * FRACTION_TO_PERCENT_UNITS)
    out["RampDn_pct"]    = out["RampDn_raw"].map(lambda v: None if v is None else float(v) * FRACTION_TO_PERCENT_UNITS)

    out = out[(out["Fuel"] != "") & out["Fuel"].notna()].copy()
    out["IsCCS"] = out["Desc"].str.contains(CCS_TOKEN, case=False, na=False)
    return out


def _build_fuel_to_value_map(df: pd.DataFrame, value_col: str, *, value_label: str) -> Dict[str, Dict[str, Any]]:
    mapping: Dict[str, Dict[str, Any]] = {}

    usable = df[(df["Fuel"] != "") & df["Fuel"].notna() & df[value_col].notna()].copy()
    if usable.empty:
        return mapping

    non_ccs = usable[~usable["IsCCS"]].copy()
    if not non_ccs.empty:
        g = non_ccs.groupby("Fuel", dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g.iterrows():
            fuel = str(r["Fuel"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel:
                mapping[fuel] = {"value": float(avg), "count": cnt, "is_ccs": False, "fuel": fuel, "desc": "", "label": value_label}

    ccs = usable[usable["IsCCS"]].copy()
    if not ccs.empty:
        g2 = ccs.groupby(["Fuel", "Desc"], dropna=True)[value_col].agg(["mean", "count"]).reset_index()
        for _, r in g2.iterrows():
            fuel = str(r["Fuel"]).strip()
            desc = str(r["Desc"]).strip()
            avg = float(r["mean"])
            cnt = int(r["count"])
            if fuel and desc:
                key = f"{fuel}{FUEL_SEPARATOR}{desc}"
                mapping[key] = {"value": float(avg), "count": cnt, "is_ccs": True, "fuel": fuel, "desc": desc, "label": value_label}

    return mapping


def _apply_override_fuels_use_reference(mapping: Dict[str, Dict[str, Any]], *, reference_fuel: str, override_fuels: set) -> None:
    if not mapping:
        return

    ref_key = None
    ref_lc = reference_fuel.strip().lower()
    for k in mapping.keys():
        if str(k).strip().lower() == ref_lc:
            ref_key = k
            break

    if ref_key is None:
        print(f"[WARN] Override requested but reference fuel {reference_fuel!r} not found in computed mapping keys.")
        return

    ref_stats = mapping[ref_key]
    for f in override_fuels:
        f_clean = str(f).strip()
        if not f_clean:
            continue
        new_stats = copy.deepcopy(ref_stats)
        new_stats["fuel"] = f_clean
        new_stats["desc"] = ""
        new_stats["is_ccs"] = False
        mapping[f_clean] = new_stats

    print(
        f"Override applied: {sorted(list(override_fuels))} now use {reference_fuel!r} computed values "
        f"({ref_stats.get('label','Value')}={float(ref_stats['value']):.6g})."
    )


def _print_mapping_preview(mapping: Dict[str, Dict[str, Any]], *, title: str, limit: int = 10) -> None:
    print(title)
    for i, k in enumerate(sorted(mapping.keys(), key=lambda s: s.casefold())):
        if i >= limit:
            break
        st = mapping[k]
        tag = " (CCS)" if st.get("is_ccs") else ""
        print(f"  {k:<35} {st.get('label','Value')}={float(st['value']):.6g} n={int(st.get('count',0))}{tag}")


def _read_maxcap_matrix(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Generator max capacity CSV not found: {path}")
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise RuntimeError(f"Maxcap CSV has unexpected structure: {path}")
    year_col = df.columns[0]
    df = df.copy()
    df.rename(columns={year_col: "Year"}, inplace=True)
    try:
        df["Year"] = df["Year"].astype(int)
    except Exception:
        pass
    return df


def _write_matrix_like_maxcap(maxcap_df: pd.DataFrame, values_by_gen: Dict[str, List[float]], out_path: Path) -> None:
    out = pd.DataFrame({"Year": maxcap_df["Year"].tolist()})
    gen_cols = [c for c in maxcap_df.columns if c != "Year"]
    for g in gen_cols:
        col_vals = values_by_gen.get(g)
        if col_vals is None:
            out[g] = [0.0] * len(maxcap_df)
        else:
            out[g] = col_vals
    out.to_csv(out_path, index=False)
    print(f"Wrote matrix CSV: {out_path}")


# ============================================================
# MAIN
# ============================================================
def step_min_stable_and_ramps():
    # 1) Read Common Data slice and compute per-fuel averages (percent-units)
    df = _read_common_data_step_table(COMMON_DATA_XLSX)
    if df.empty:
        raise RuntimeError(f"No usable rows found in sheet {SHEET_NAME!r}, rows {ROW_START_1BASED}-{ROW_END_1BASED}.")

    fuel_to_min_stable = _build_fuel_to_value_map(df, "MinStable_pct", value_label="Min Stable Factor (%)")
    fuel_to_ramp_up_pct = _build_fuel_to_value_map(df, "RampUp_pct", value_label="Max Ramp Up (% of MaxCap)")
    fuel_to_ramp_dn_pct = _build_fuel_to_value_map(df, "RampDn_pct", value_label="Max Ramp Down (% of MaxCap)")

    if not fuel_to_min_stable:
        raise RuntimeError("No Min Stable Factor mappings computed.")
    if not fuel_to_ramp_up_pct:
        raise RuntimeError("No Max Ramp Up mappings computed.")
    if not fuel_to_ramp_dn_pct:
        raise RuntimeError("No Max Ramp Down mappings computed.")

    _apply_override_fuels_use_reference(fuel_to_min_stable, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)
    _apply_override_fuels_use_reference(fuel_to_ramp_up_pct, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)
    _apply_override_fuels_use_reference(fuel_to_ramp_dn_pct, reference_fuel=REFERENCE_FUEL_FOR_OVERRIDE, override_fuels=OVERRIDE_FUELS_USE_HARD_COAL)

    print(f"Using sheet={SHEET_NAME!r}, rows={ROW_START_1BASED}-{ROW_END_1BASED} (inclusive).")
    _print_mapping_preview(fuel_to_min_stable, title="Computed Min Stable Factor mapping preview (% units):")
    _print_mapping_preview(fuel_to_ramp_up_pct, title="Computed Ramp Up rate mapping preview (% of MaxCap):")
    _print_mapping_preview(fuel_to_ramp_dn_pct, title="Computed Ramp Down rate mapping preview (% of MaxCap):")

    # 2) Load MaxCap matrix
    maxcap_df = _read_maxcap_matrix(GEN_MAXCAP_CSV_PATH)
    years = maxcap_df["Year"].tolist()
    gen_cols = [c for c in maxcap_df.columns if c != "Year"]
    print(f"Loaded MaxCap matrix: years={len(years)}, generators={len(gen_cols)}")

    # 3) Open DB and determine generator->fuel assignment via Generatorâ†”Fuel membership
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_MIN_STABLE_FACTOR = _resolve_semantic_enum("System", "Generator", "Generators", "Min Stable Factor", 56)
    ENUM_MAX_RAMP_UP = _resolve_semantic_enum("System", "Generator", "Generators", "Max Ramp Up", 105)
    ENUM_MAX_RAMP_DOWN = _resolve_semantic_enum("System", "Generator", "Generators", "Max Ramp Down", 109)
    

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        gen_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "generator"],
            candidates=["SystemGenerators", "Generators"],
        )
        if gen_mem_collection is None:
            raise RuntimeError("Could not resolve Generator membership collection (SystemGenerators/Generators).")

        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")
        else:
            gen_fuel_collection = _find_collection_enum_optional(CollectionEnum, ["fuel", "generator"], [])
        if gen_fuel_collection is None:
            raise RuntimeError("Could not resolve Generatorâ†”Fuel membership collection (GeneratorFuels/FuelGenerators).")

        datafile_mem_collection = _find_collection_enum_optional(
            CollectionEnum,
            must_contain_tokens=["system", "data"],
            candidates=["SystemDataFiles", "DataFiles", "SystemDataFile"],
        )
        if datafile_mem_collection is None:
            raise RuntimeError("Could not resolve Systemâ†”DataFile membership collection (SystemDataFiles/DataFiles).")

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(
            f"Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, "
            f"collection={df_col_used!r}, property={df_prop_used!r}"
        )

        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))
        db_gen_set = set(gens_in_db)

        # Diagnostics: computed fuels missing in DB
        def _warn_missing(mapping: Dict[str, Dict[str, Any]], label: str):
            missing = [k for k in mapping.keys() if k not in fuels_in_db]
            if missing:
                print(f"[WARN] Some computed fuel keys for {label} are not Fuel objects in DB (skipped in membership matching):")
                for f in missing[:40]:
                    print(f"  - {f}")

        _warn_missing(fuel_to_min_stable, "Min Stable Factor")
        _warn_missing(fuel_to_ramp_up_pct, "Max Ramp Up")
        _warn_missing(fuel_to_ramp_dn_pct, "Max Ramp Down")

        # Build generator -> fuels (probing style consistent with earlier steps)
        gen_to_fuels: Dict[str, List[str]] = {}
        for g in gens_in_db:
            matched = []
            for f in fuels_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, f)
                if mem is not None:
                    matched.append(f)
            if matched:
                gen_to_fuels[g] = matched

        # Choose a single assigned fuel key per generator (warn if ambiguous)
        mapping_keys_union = set(fuel_to_min_stable.keys()) | set(fuel_to_ramp_up_pct.keys()) | set(fuel_to_ramp_dn_pct.keys())
        gen_assigned_fuel: Dict[str, str] = {}
        ambiguous: List[Tuple[str, List[str]]] = []
        no_fuel: List[str] = []

        for g in gen_cols:
            fuels = gen_to_fuels.get(g, [])
            if not fuels:
                no_fuel.append(g)
                continue

            pref = [f for f in fuels if f in mapping_keys_union]
            chosen_list = pref if pref else fuels
            gen_assigned_fuel[g] = chosen_list[0]

            if len(chosen_list) > 1:
                ambiguous.append((g, chosen_list[:10]))

        if no_fuel:
            print("[WARN] Some generators have no Fuel membership found; their ramp values will be zeros:")
            for g in no_fuel[:40]:
                print(f"  - {g}")

        if ambiguous:
            print("[WARN] Some generators matched multiple candidate fuels; first was chosen (verify if you expect multi-fuel generators):")
            for g, lst in ambiguous[:30]:
                print(f"  - {g}: chosen={lst[0]!r}, candidates={lst}")

        # 4) Write Min Stable Factor to Systemâ†’Generator (band 1)
        writes_msf = 0
        fuels_with_no_gens: List[str] = []

        for fuel_name in sorted(fuel_to_min_stable.keys()):
            if fuel_name not in fuels_in_db:
                continue

            matched_gens = []
            for g in gens_in_db:
                mem = _get_membership_id_optional(db, gen_fuel_collection, g, fuel_name)
                if mem is not None:
                    matched_gens.append(g)

            if not matched_gens:
                fuels_with_no_gens.append(fuel_name)
                continue

            msf = round(float(fuel_to_min_stable[fuel_name]["value"]), 1)

            for g in matched_gens:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                _add_property_row(db, ENUM_MIN_STABLE_FACTOR, g_mem_id, 1, msf, Scenario=scenario_str)
                writes_msf += 1

            print(f"Applied Min Stable Factor {msf:.6g}% to {len(matched_gens)} generator(s) for Fuel='{fuel_name}'")

        if fuels_with_no_gens:
            print("\n[WARN] Fuels found in DB but with no generators having membership (Min Stable Factor nothing written):")
            for f in fuels_with_no_gens[:60]:
                print(f"  - {f}")

        # 5) Build YearÃ—Generator ramp matrices (absolute values) and write CSVs
        maxcap_numeric = {}
        for g in gen_cols:
            s = pd.to_numeric(maxcap_df[g], errors="coerce").fillna(0.0)
            maxcap_numeric[g] = s.tolist()

        ramp_up_values_by_gen: Dict[str, List[float]] = {}
        ramp_dn_values_by_gen: Dict[str, List[float]] = {}

        missing_ru: Dict[str, int] = {}
        missing_rd: Dict[str, int] = {}

        for g in gen_cols:
            fuel = gen_assigned_fuel.get(g)
            caps = maxcap_numeric[g]

            if not fuel:
                ramp_up_values_by_gen[g] = [0.0] * len(years)
                ramp_dn_values_by_gen[g] = [0.0] * len(years)
                continue

            ru_pct_units = None
            if fuel in fuel_to_ramp_up_pct:
                ru_pct_units = float(fuel_to_ramp_up_pct[fuel]["value"])
            else:
                missing_ru[fuel] = missing_ru.get(fuel, 0) + 1
                ru_pct_units = 0.0

            rd_pct_units = None
            if fuel in fuel_to_ramp_dn_pct:
                rd_pct_units = float(fuel_to_ramp_dn_pct[fuel]["value"])
            else:
                missing_rd[fuel] = missing_rd.get(fuel, 0) + 1
                rd_pct_units = 0.0

            # Convert percent-units to fraction for multiplication with MaxCap
            ru_frac = float(ru_pct_units) / 100.0
            rd_frac = float(rd_pct_units) / 100.0

            ramp_up_values_by_gen[g] = [round(ru_frac * float(c), 1) for c in caps]
            ramp_dn_values_by_gen[g] = [round(rd_frac * float(c), 1) for c in caps]

        if missing_ru:
            print("[WARN] Some generator-assigned fuels not found in Ramp Up mapping; their Ramp Up values are zeroed:")
            for f, n in list(missing_ru.items())[:40]:
                print(f"  - {f}: affected_generators={n}")
        if missing_rd:
            print("[WARN] Some generator-assigned fuels not found in Ramp Down mapping; their Ramp Down values are zeroed:")
            for f, n in list(missing_rd.items())[:40]:
                print(f"  - {f}: affected_generators={n}")

        _write_matrix_like_maxcap(maxcap_df, ramp_up_values_by_gen, OUT_RAMP_UP_CSV_PATH)
        _write_matrix_like_maxcap(maxcap_df, ramp_dn_values_by_gen, OUT_RAMP_DN_CSV_PATH)

        # 6) Create DataFile objects and set their filename properties (relative paths)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJ_RAMP_UP)
        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJ_RAMP_DN)

        df_mem_id_ru = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJ_RAMP_UP)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_ru, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_RAMP_UP_PATH),
            Scenario=scenario_str
        )

        df_mem_id_rd = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJ_RAMP_DN)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id_rd, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_RAMP_DN_PATH),
            Scenario=scenario_str
        )

        # 7) Link ramp properties only to generators that matched a Common Data fuel rule
        ramp_up_links = 0
        ramp_dn_links = 0

        valid_fuels_for_ramp_up = set(fuel_to_ramp_up_pct.keys())
        valid_fuels_for_ramp_dn = set(fuel_to_ramp_dn_pct.keys())

        target_gens_for_ramp_up = sorted(
            g for g, f in gen_assigned_fuel.items()
            if (g in db_gen_set) and (f in valid_fuels_for_ramp_up)
        )
        target_gens_for_ramp_dn = sorted(
            g for g, f in gen_assigned_fuel.items()
            if (g in db_gen_set) and (f in valid_fuels_for_ramp_dn)
        )

        for g in target_gens_for_ramp_up:
            g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
            _add_property_row(
                db, ENUM_MAX_RAMP_UP, g_mem_id, 1, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_RAMP_UP_PATH),
                Scenario=scenario_str
            )
            ramp_up_links += 1

        for g in target_gens_for_ramp_dn:
            g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
            _add_property_row(
                db, ENUM_MAX_RAMP_DOWN, g_mem_id, 1, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_RAMP_DN_PATH),
                Scenario=scenario_str
            )
            ramp_dn_links += 1


        print("Done.")
        print("Summary:")
        print(f"  Min Stable Factor writes: {writes_msf}")
        print(f"  Ramp Up DataFile:   {DATAFILE_OBJ_RAMP_UP} -> {PLEXOS_REL_RAMP_UP_PATH} (links={ramp_up_links})")
        print(f"  Ramp Down DataFile: {DATAFILE_OBJ_RAMP_DN} -> {PLEXOS_REL_RAMP_DN_PATH} (links={ramp_dn_links})")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_min_stable_and_ramps()





#### 5. Battery Import

In [ ]:
# ============================================================
# STEP â€” Batteries import (FULL copy-pasteable script)
# ============================================================
# Paste this entire cell into your notebook / VS and run.
# It expects your existing helpers to be available in the same kernel:
#   _bootstrap_plexos_net, _add_object, _ensure_membership_bi, _add_property_row, _get_objects_safe
# These are defined elsewhere in your notebook (as per earlier cells you shared).
#
# This script:
#  - reads Batteries additional information.csv
#  - filters by bidding zones listed in Bidding_Zone_List.xlsx
#  - filters OP_STAT == "Available on market" and data_version == "ERAA 2025 final"
#  - creates Battery objects named "{BiddingZone}_Market_Battery"
#  - couples Battery <-> Node (bi-directional) using same node name as bidding zone (like generators)
#  - ensures System->Battery membership using Collection ID 81 (coerced to CollectionEnum)
#  - writes Units (enum 25), Initial SoC (32), Charge Eff (33), Discharge Eff (34) once per battery
#  - writes Max Power (28) and Capacity (26) per TARGET_YEAR with DateFrom = YYYY-01-01
#  - sums duplicate rows per (MARKET_NODE, TARGET_YEAR)
#
# Adjust constants at the top if needed.
# ============================================================

from pathlib import Path
from typing import Optional, Set, List, Any
import pandas as pd
import math

# -----------------------------
# User-editable configuration
# -----------------------------
BATTERIES_CSV = DASHBOARD_RAWDATA_DIR / "Batteries additional information.csv"

FILTER_DATA_VERSION = "ERAA 2025 final"
FILTER_OPERATION_STATUS = "Available on market"

# CSV column names (must match file)
COL_NODE = "MARKET_NODE"
COL_OP_STAT = "OP_STAT"
COL_MAX_POWER = "MAX LOAD CAP (MW)"
COL_CAPACITY = "STORAGE CAPACITY (MWh)"
COL_YEAR = "TARGET_YEAR"
COL_DATA_VERSION = "data_version"

# Property enum ids
ENUM_UNITS = 25
ENUM_CAPACITY = 26
ENUM_MAX_POWER = 28

# Additional static battery properties (percent values, adjustable)
INITIAL_SOC_PERCENT = 50
CHARGE_EFFICIENCY_PERCENT = 98
DISCHARGE_EFFICIENCY_PERCENT = 98

ENUM_INITIAL_SOC = 32
ENUM_CHARGE_EFF = 33
ENUM_DISCHARGE_EFF = 34

# System -> Battery collection id (given)
SYSTEM_BATTERY_COLLECTION_ID = 81

# Optional scenario name (None or string)
SCENARIO_NAME: Optional[str] = None

# -----------------------------
# Local helper functions
# -----------------------------
def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _round_1dp(x):
    fx = _as_float(x)
    if fx is None:
        return None
    return round(fx, 1)

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals

def _find_battery_node_collection(CollectionEnum) -> Optional[Any]:
    """
    Best-effort: find a Node<->Battery collection enum by trying common names.
    If your CollectionEnum contains a specific name for battery-node, extend the list below.
    """
    candidates = [
        "NodeBatteries", "BatteryNodes", "StorageNodes", "NodeStorages",
        "NodeStorage", "StoragesNodes", "NodeStorageUnits", "BatteryNode"
    ]
    for name in candidates:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return None

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> int:
    """
    Ensure a membership and return membership id (int).
    Tries different overloads and coerces int->CollectionEnum where possible.
    Raises RuntimeError when failing to ensure membership.
    """
    last_exc = None
    candidates = []

    # Try to coerce raw to CollectionEnum if it's an int
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        # Try plain GetMembershipID
        try:
            mid = int(db.GetMembershipID(col_enum, parent_name, child_name))
            return mid
        except Exception as e:
            last_exc = e

        # Try SystemNS.String wrappers
        try:
            mid = int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
            return mid
        except Exception as e:
            last_exc = e

        # Mixed wrappers
        try:
            mid = int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), child_name))
            return mid
        except Exception as e:
            last_exc = e
        try:
            mid = int(db.GetMembershipID(col_enum, parent_name, SystemNS.String(child_name)))
            return mid
        except Exception as e:
            last_exc = e

        # Try to add membership then fetch
        try:
            try:
                db.AddMembership(col_enum, parent_name, child_name)
            except Exception:
                db.AddMembership(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name))
            # After AddMembership, try to fetch ID
            try:
                return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
            except Exception:
                try:
                    return int(db.GetMembershipID(col_enum, parent_name, child_name))
                except Exception as e:
                    last_exc = e
                    continue
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(
        f"_ensure_membership_robust: failed to ensure membership for parent={parent_name}, child={child_name}, "
        f"collection_raw={collection_enum_raw}. Last error: {last_exc}"
    )

# -----------------------------
# Main implementation
# -----------------------------
def step_batteries_import_full():
    # Read bidding zones
    allowed = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    # Read CSV
    df = pd.read_csv(BATTERIES_CSV)

    required = [COL_NODE, COL_OP_STAT, COL_MAX_POWER, COL_CAPACITY, COL_YEAR, COL_DATA_VERSION]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in Batteries CSV: {missing}\nFound: {list(df.columns)}")

    # Normalize and parse
    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_OP_STAT] = df[COL_OP_STAT].astype(str).map(_clean_cell)
    df[COL_DATA_VERSION] = df[COL_DATA_VERSION].astype(str).map(_clean_cell)
    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_MAX_POWER] = df[COL_MAX_POWER].apply(_as_float)
    df[COL_CAPACITY] = df[COL_CAPACITY].apply(_as_float)

    # Apply filters
    df = df[
        (df[COL_DATA_VERSION] == FILTER_DATA_VERSION) &
        (df[COL_OP_STAT] == FILTER_OPERATION_STATUS) &
        (df[COL_NODE].isin(allowed)) &
        (df[COL_YEAR].notna())
    ].copy()

    if df.empty:
        print("[STEP] No rows remain after filters. Nothing to import.")
        return

    # Aggregate duplicates per (node, year)
    dup = df.groupby([COL_NODE, COL_YEAR]).size().reset_index(name="count")
    dup = dup[dup["count"] > 1]
    if not dup.empty:
        print("[STEP] WARNING: duplicate rows per (MARKET_NODE, TARGET_YEAR) found; summing Max Power and Capacity.")
        try:
            display(dup.head(20))
        except Exception:
            print(dup.head(20).to_string(index=False))

    df_g = (
        df.groupby([COL_NODE, COL_YEAR], as_index=False)
          .agg({COL_MAX_POWER: "sum", COL_CAPACITY: "sum"})
          .sort_values([COL_NODE, COL_YEAR])
    )

    print("\n[STEP] Preview (first ~10 zone-year rows after filters + aggregation):")
    try:
        display(df_g.head(10))
    except Exception:
        print(df_g.head(10).to_string(index=False))

    # Bootstrap PLEXOS and connect
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    # Scenario string if needed
    scenario_str = None
    if SCENARIO_NAME:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

    # Best-effort find node<->battery collection enum
    bat_node_collection = _find_battery_node_collection(CollectionEnum)

    # Get nodes and existing batteries in DB
    nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
    try:
        batteries_in_db = set(_get_objects_safe(db, ClassEnum.Battery))
    except Exception:
        batteries_in_db = set()

    created_batteries = 0
    created_memberships = 0
    props_written = 0
    node_links = 0
    warnings: List[str] = []

    zone_names = sorted(df_g[COL_NODE].unique().tolist())
    print(f"\n[STEP] Batteries to import (one per MARKET_NODE): {len(zone_names)}")
    print("Sample:", zone_names[:20])

    try:
        sys_bat_collection_raw = SYSTEM_BATTERY_COLLECTION_ID

        for zone in zone_names:
            batt_name = f"{zone}_Market_Battery"

            # Create Battery object if missing (DO NOT create categories)
            if batt_name not in batteries_in_db:
                try:
                    _add_object(db, ClassEnum.Battery, batt_name, add_to_system=True, category="", description="")
                    batteries_in_db.add(batt_name)
                    created_batteries += 1
                except Exception as e:
                    warnings.append(f"Failed to create Battery '{batt_name}': {e}")
                    continue

            # Ensure System -> Battery membership (robust coercion)
            try:
                mem_id = _ensure_membership_robust(
                    db, CollectionEnum, SystemNS,
                    sys_bat_collection_raw,
                    "System",
                    batt_name
                )
                created_memberships += 1
            except Exception as e:
                warnings.append(f"Failed System->Battery membership for '{batt_name}': {e}")
                mem_id = None

            # Write non-time-dependent properties once (on System->Battery membership)
            if mem_id is not None:
                try:
                    # Units = 1
                    _add_property_row(db, ENUM_UNITS, mem_id, 1, 1.0, Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Units write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_INITIAL_SOC, mem_id, 1, float(INITIAL_SOC_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Initial SoC write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_CHARGE_EFF, mem_id, 1, float(CHARGE_EFFICIENCY_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Charge Efficiency write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_DISCHARGE_EFF, mem_id, 1, float(DISCHARGE_EFFICIENCY_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Discharge Efficiency write for '{batt_name}': {e}")
            else:
                warnings.append(f"No valid System->Battery membership for '{batt_name}'; skipping static property writes.")

            # Node coupling (bi-directional) like generators
            node_name = zone
            if bat_node_collection is not None:
                if node_name in nodes_in_db:
                    try:
                        if _ensure_membership_bi(db, bat_node_collection, node_name, batt_name):
                            node_links += 1
                    except Exception as e:
                        warnings.append(f"Failed Battery<->Node link for '{batt_name}' <-> '{node_name}': {e}")
                else:
                    warnings.append(f"Node '{node_name}' not found in DB; cannot couple Battery '{batt_name}' to Node.")
            else:
                warnings.append("Battery<->Node collection enum not found; node coupling skipped for batteries.")

            # Write year-based Max Power + Capacity with DateFrom per TARGET_YEAR
            sub = df_g[df_g[COL_NODE] == zone]
            for _, r in sub.iterrows():
                y = int(r[COL_YEAR])
                dt_from = NetDateTime(int(y), 1, 1, 0, 0, 0)

                max_p = _round_1dp(r[COL_MAX_POWER])
                cap = _round_1dp(r[COL_CAPACITY])

                if mem_id is None:
                    warnings.append(f"No System->Battery membership for '{batt_name}'; skipping property writes for year {y}.")
                    continue

                if max_p is not None:
                    try:
                        _add_property_row(db, ENUM_MAX_POWER, mem_id, 1, float(max_p), DateFrom=dt_from, Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Max Power write for '{batt_name}' year={y}: {e}")
                else:
                    warnings.append(f"Blank Max Power for '{batt_name}' year={y}; skipped.")

                if cap is not None:
                    try:
                        _add_property_row(db, ENUM_CAPACITY, mem_id, 1, float(cap), DateFrom=dt_from, Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Capacity write for '{batt_name}' year={y}: {e}")
                else:
                    warnings.append(f"Blank Capacity for '{batt_name}' year={y}; skipped.")

        # Summary
        print("\n[STEP] Done.")
        print(f"  Created Battery objects:          {created_batteries}")
        print(f"  Memberships created/ensured:      {created_memberships}")
        print(f"  Node links created (bi-dir):      {node_links}")
        print(f"  Properties written:               {props_written}")
        print(f"  Zone-year rows processed:         {len(df_g)}")

        if warnings:
            print("\n[STEP] Warnings (first 80):")
            for w in warnings[:80]:
                print("  - " + w)
            if len(warnings) > 80:
                print(f"  ... and {len(warnings) - 80} more")
        else:
            print("\n[STEP] No warnings.")
    finally:
        try:
            db.Close()
        except Exception:
            pass

# Run the step
step_batteries_import_full()





#### 6. Storage Import

In [ ]:
# ============================================================
# STEP â€” Storages import (UPDATED FULL copy-pasteable script)
#
# CHANGES vs the original version:
#  - REMOVED: Min Volume Penalty + Max Volume Penalty (handled later in the workflow now)
#  - FIX: If a "Min Volume" property is being set to 1 in this step (or you want a default),
#         this step now writes Min Volume (Enum 21) = 0 ONCE (no DateFrom) for created storages.
#  - KEEP: "Reservoir" and "Pondage" create ONLY Head storage (no Tail)
#  - KEEP: Head Storage "Max Spill" (Enum 46) set to 1e30
#  - KEEP: Units (Enum 17) = 1, End Effects Method (Enum 4) = 1, Decomposition Method (Enum 6) = 1
#
# NEW CHANGE (ONLY, per your latest instruction in THIS message):
#  - Do NOT add Pump Units (Enum 128), Pump Efficiency (Enum 124), or Pump Load (Enum 125)
#    for technologies in ONLY_HEAD_PLANT_TYPES (Reservoir, Pondage).
#
# Notes:
# - Storages are created with add_to_system=True so a System->Storage membership exists
#   (we only READ membership ID from collection 99 to write properties; we do NOT add/ensure it).
# ============================================================
# Expects existing helpers:
#   _bootstrap_plexos_net, _add_object, _ensure_membership_bi, _add_property_row, _get_objects_safe
# ============================================================

from pathlib import Path
from typing import Optional, Set, List, Any
import pandas as pd

# -----------------------------
# User-editable configuration
# -----------------------------
HYDRO_CSV = DASHBOARD_RAWDATA_DIR / "Hydro additional information.csv"

FILTER_DATA_VERSION = "ERAA 2025 final"
OMIT_PLANT_TYPES = {"Run of river"}

# Only-head technologies
ONLY_HEAD_PLANT_TYPES = {"Reservoir", "Pondage"}

# CSV column names (must match file)
COL_NODE = "MARKET_NODE"
COL_PLANT = "PEMMDB_PLANT_TYPE"
COL_PUMP_CAP = "MAX PUMPING CAP (MW)"
COL_STORAGE_TWH = "Storage Capacity [TWh]"
COL_YEAR = "TARGET_YEAR"
COL_DATA_VERSION = "data_version"

# ---- Storage class id (you said Class ID of storages is 9) ----
STORAGE_CLASS_ID = 9

# Storage property enum ids (given)
ENUM_END_EFFECTS_METHOD = 4          # End Effects Method
ENUM_DECOMPOSITION_METHOD = 6        # Decomposition Method
ENUM_UNITS = 17                      # Units
ENUM_MAX_VOLUME = 18                 # Max Volume (GWh)
ENUM_MIN_VOLUME = 21                 # Min Volume  (GWh)  <-- default to 0 here
ENUM_INITIAL_VOLUME = 20             # Initial Volume (GWh)

# If this fails in your DB, just change this number.
ENUM_MAX_SPILL = 46
MAX_SPILL_DEFAULT = 1.0e30

# Values for storage static properties
END_EFFECTS_METHOD_VALUE = 1         # Free
DECOMPOSITION_METHOD_VALUE = 1       # Targets
UNITS_VALUE = 1.0                    # Units = 1
MIN_VOLUME_DEFAULT_VALUE = 0.0       # Default Min Volume = 0 (no DateFrom)

# Generator pump property enum ids (given)
ENUM_PUMP_EFFICIENCY = 125
ENUM_PUMP_LOAD = 126
ENUM_PUMP_UNITS = 129

PUMP_EFFICIENCY_VALUE = 75.0
PUMP_UNITS_VALUE = 1.0

# Storage collection ID (given) â€” used ONLY to FETCH membership id (NOT to add/ensure)
SYSTEM_STORAGE_COLLECTION_ID = 99

# Optional scenario name (None or string)
SCENARIO_NAME: Optional[str] = None

# -----------------------------
# Local helper functions
# -----------------------------
def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _round_1dp(x):
    fx = _as_float(x)
    if fx is None:
        return None
    return round(fx, 1)

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals

def _find_collection_enum(CollectionEnum, candidates: List[str]) -> Optional[Any]:
    for name in candidates:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return None

def _find_node_storage_collection(CollectionEnum) -> Optional[Any]:
    return _find_collection_enum(CollectionEnum, [
        "NodeStorages",
        "NodeStorage",
        "StorageNodes",
        "StorageNode",
        "NodeReservoirs",
        "ReservoirNodes",
        "NodeStorageUnits",
        "StoragesNodes",
    ])

def _find_generator_head_storage_collection(CollectionEnum) -> Optional[Any]:
    return _find_collection_enum(CollectionEnum, [
        "GeneratorHeadStorages",
        "GeneratorHeadStorage",
        "HeadStorageGenerators",
        "GeneratorsHeadStorages",
        "GeneratorHeadReservoirs",
        "GeneratorHeadReservoir",
    ])

def _find_generator_tail_storage_collection(CollectionEnum) -> Optional[Any]:
    return _find_collection_enum(CollectionEnum, [
        "GeneratorTailStorages",
        "GeneratorTailStorage",
        "TailStorageGenerators",
        "GeneratorsTailStorages",
        "GeneratorTailReservoirs",
        "GeneratorTailReservoir",
    ])

def _find_system_generator_collection(CollectionEnum) -> Optional[Any]:
    return _find_collection_enum(CollectionEnum, [
        "SystemGenerators",
        "SystemGenerator",
        "Generators",
    ])

def _ensure_storage_category(db, ClassEnum, category_name: str) -> bool:
    if not category_name:
        return True
    if not hasattr(db, "AddCategory"):
        return False

    attempts = [
        (ClassEnum.Storage, str(category_name)),
        (STORAGE_CLASS_ID, str(category_name)),
        (ClassEnum.Storage, str(category_name), ""),
        (STORAGE_CLASS_ID, str(category_name), ""),
    ]

    for args in attempts:
        try:
            db.AddCategory(*args)
            return True
        except Exception:
            pass
    return False

def _get_membership_id_only(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> Optional[int]:
    candidates = []
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), child_name))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, parent_name, SystemNS.String(child_name)))
        except Exception:
            pass
    return None

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> int:
    last_exc = None
    candidates = []

    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e

        try:
            try:
                db.AddMembership(col_enum, parent_name, child_name)
            except Exception:
                db.AddMembership(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name))

            try:
                return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
            except Exception:
                return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(
        f"_ensure_membership_robust: failed membership parent={parent_name}, child={child_name}, "
        f"collection_raw={collection_enum_raw}. Last error: {last_exc}"
    )

# -----------------------------
# Main implementation
# -----------------------------
def step_storages_import_full():
    allowed = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    df = pd.read_csv(HYDRO_CSV)

    required = [COL_NODE, COL_PLANT, COL_PUMP_CAP, COL_STORAGE_TWH, COL_YEAR, COL_DATA_VERSION]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in Hydro CSV: {missing}\nFound: {list(df.columns)}")

    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_PLANT] = df[COL_PLANT].astype(str).map(_clean_cell)
    df[COL_DATA_VERSION] = df[COL_DATA_VERSION].astype(str).map(_clean_cell)

    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_PUMP_CAP] = df[COL_PUMP_CAP].apply(_as_float)
    df[COL_STORAGE_TWH] = df[COL_STORAGE_TWH].apply(_as_float)

    df = df[
        (df[COL_DATA_VERSION] == FILTER_DATA_VERSION) &
        (df[COL_NODE].isin(allowed)) &
        (df[COL_YEAR].notna()) &
        (~df[COL_PLANT].isin(OMIT_PLANT_TYPES))
    ].copy()

    if df.empty:
        print("[STEP] No rows remain after filters. Nothing to import.")
        return

    df_g = (
        df.groupby([COL_NODE, COL_PLANT, COL_YEAR], as_index=False)
          .agg({COL_PUMP_CAP: "sum", COL_STORAGE_TWH: "sum"})
          .sort_values([COL_NODE, COL_PLANT, COL_YEAR])
    )

    print("\n[STEP] Preview (first ~10 rows after filters + aggregation):")
    try:
        display(df_g.head(10))
    except Exception:
        print(df_g.head(10).to_string(index=False))

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_PUMP_EFFICIENCY = _resolve_semantic_enum("System", "Generator", "Generators", "Pump Efficiency", 125)
    ENUM_PUMP_LOAD = _resolve_semantic_enum("System", "Generator", "Generators", "Pump Load", 126)
    ENUM_PUMP_UNITS = _resolve_semantic_enum("System", "Generator", "Generators", "Pump Units", 129)
    

    scenario_str = SystemNS.String(SCENARIO_NAME) if SCENARIO_NAME else None

    node_storage_collection = _find_node_storage_collection(CollectionEnum)
    gen_head_storage_collection = _find_generator_head_storage_collection(CollectionEnum)
    gen_tail_storage_collection = _find_generator_tail_storage_collection(CollectionEnum)
    system_generator_collection = _find_system_generator_collection(CollectionEnum)

    nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
    try:
        storages_in_db = set(_get_objects_safe(db, ClassEnum.Storage))
    except Exception:
        storages_in_db = set()
    try:
        generators_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
    except Exception:
        generators_in_db = set()

    created_storages = 0
    created_memberships = 0
    node_links = 0
    generator_links = 0
    props_written = 0
    warnings: List[str] = []
    pump_static_written_for: Set[str] = set()
    categories_done: Set[str] = set()

    pairs = (
        df_g[[COL_NODE, COL_PLANT]]
        .drop_duplicates()
        .sort_values([COL_NODE, COL_PLANT])
        .values
        .tolist()
    )

    print(f"\n[STEP] Storage pairs to import (MARKET_NODE x PEMMDB_PLANT_TYPE): {len(pairs)}")
    print("Sample:", pairs[:20])

    try:
        for zone, plant in pairs:
            base_name = f"{zone}_{plant}"
            head_storage_name = f"{base_name} Head"
            tail_storage_name = f"{base_name} Tail"

            create_tail = (plant not in ONLY_HEAD_PLANT_TYPES)
            storage_names = [head_storage_name] + ([tail_storage_name] if create_tail else [])

            # Ensure category once per zone
            if zone not in categories_done:
                ok = _ensure_storage_category(db, ClassEnum, zone)
                if not ok:
                    warnings.append("db.AddCategory not available for Storage categories on this API surface; storages will be created without categories.")
                categories_done.add(zone)

            # Create storages
            for sname in storage_names:
                if sname not in storages_in_db:
                    try:
                        try:
                            _add_object(db, ClassEnum.Storage, sname, add_to_system=True, category=zone, description="")
                        except Exception:
                            _add_object(db, ClassEnum.Storage, sname, add_to_system=True, category="", description="")
                            warnings.append(f"Created '{sname}' without category (category '{zone}' still not recognized).")
                        storages_in_db.add(sname)
                        created_storages += 1
                    except Exception as e:
                        warnings.append(f"Failed to create Storage '{sname}': {e}")

            # Read membership ids for property writing
            mem_head = _get_membership_id_only(db, CollectionEnum, SystemNS, SYSTEM_STORAGE_COLLECTION_ID, "System", head_storage_name)
            mem_tail = None
            if create_tail:
                mem_tail = _get_membership_id_only(db, CollectionEnum, SystemNS, SYSTEM_STORAGE_COLLECTION_ID, "System", tail_storage_name)

            # Static storage props (NO penalties here):
            for sname, mid, is_head in (
                (head_storage_name, mem_head, True),
                *([(tail_storage_name, mem_tail, False)] if create_tail else []),
            ):
                if mid is None:
                    warnings.append(f"Could not read System->Storage membership id for '{sname}' (collection {SYSTEM_STORAGE_COLLECTION_ID}); skipping storage property writes.")
                    continue

                try:
                    _add_property_row(db, ENUM_UNITS, mid, 1, float(UNITS_VALUE), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Units write for '{sname}': {e}")

                try:
                    _add_property_row(db, ENUM_END_EFFECTS_METHOD, mid, 1, float(END_EFFECTS_METHOD_VALUE), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed End Effects Method write for '{sname}': {e}")

                try:
                    _add_property_row(db, ENUM_DECOMPOSITION_METHOD, mid, 1, float(DECOMPOSITION_METHOD_VALUE), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Decomposition Method write for '{sname}': {e}")

                # Default Min Volume to 0 (no DateFrom)
                try:
                    _add_property_row(db, ENUM_MIN_VOLUME, mid, 1, float(MIN_VOLUME_DEFAULT_VALUE), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Min Volume default=0 write for '{sname}' (enum {ENUM_MIN_VOLUME}): {e}")

                if is_head:
                    try:
                        _add_property_row(db, ENUM_MAX_SPILL, mid, 1, float(MAX_SPILL_DEFAULT), Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(
                            f"Failed Max Spill write for '{sname}' (enum {ENUM_MAX_SPILL}, value {MAX_SPILL_DEFAULT:.3g}). "
                            f"If enum is wrong in your build, update ENUM_MAX_SPILL. Error: {e}"
                        )

            # Node coupling (bi-directional)
            if node_storage_collection is not None:
                node_name = zone
                if node_name in nodes_in_db:
                    for sname in storage_names:
                        if sname in storages_in_db:
                            try:
                                if _ensure_membership_bi(db, node_storage_collection, node_name, sname):
                                    node_links += 1
                            except Exception as e:
                                warnings.append(f"Failed Storage<->Node link for '{sname}' <-> '{node_name}': {e}")
                else:
                    warnings.append(f"Node '{node_name}' not found in DB; cannot couple storages for '{base_name}'.")
            else:
                warnings.append("Node<->Storage collection enum not found; node coupling skipped for storages.")

            # Generator coupling + pump property writes
            gen_name = base_name
            gen_mem_id = None

            if gen_name not in generators_in_db:
                warnings.append(f"Generator '{gen_name}' not found in DB; skipping generator head/tail coupling and pump properties.")
            else:
                if gen_head_storage_collection is not None:
                    try:
                        _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_head_storage_collection, gen_name, head_storage_name)
                        created_memberships += 1
                        generator_links += 1
                    except Exception as e:
                        warnings.append(f"Failed Generator->HeadStorage membership for '{gen_name}' -> '{head_storage_name}': {e}")
                else:
                    warnings.append("Generator->HeadStorage collection enum not found; head storage coupling skipped.")

                if create_tail:
                    if gen_tail_storage_collection is not None:
                        try:
                            _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_tail_storage_collection, gen_name, tail_storage_name)
                            created_memberships += 1
                            generator_links += 1
                        except Exception as e:
                            warnings.append(f"Failed Generator->TailStorage membership for '{gen_name}' -> '{tail_storage_name}': {e}")
                    else:
                        warnings.append("Generator->TailStorage collection enum not found; tail storage coupling skipped.")

                if system_generator_collection is not None:
                    try:
                        gen_mem_id = _ensure_membership_robust(db, CollectionEnum, SystemNS, system_generator_collection, "System", gen_name)
                        created_memberships += 1
                    except Exception as e:
                        warnings.append(f"Failed System->Generator membership for '{gen_name}' (needed for pump props): {e}")
                        gen_mem_id = None
                else:
                    warnings.append("System->Generator collection enum not found; pump property writes will be skipped.")

                # NEW CHANGE (ONLY): Skip Pump Units + Pump Efficiency for Reservoir/Pondage technologies
                if gen_mem_id is not None and gen_name not in pump_static_written_for and plant not in ONLY_HEAD_PLANT_TYPES:
                    try:
                        _add_property_row(db, ENUM_PUMP_UNITS, gen_mem_id, 1, float(PUMP_UNITS_VALUE), Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Pump Units write for generator '{gen_name}': {e}")

                    try:
                        _add_property_row(db, ENUM_PUMP_EFFICIENCY, gen_mem_id, 1, float(PUMP_EFFICIENCY_VALUE), Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Pump Efficiency write for generator '{gen_name}': {e}")

                    pump_static_written_for.add(gen_name)

            # Year-based writes: Max/Initial Volume (Head + optional Tail), Pump Load for generator
            sub = df_g[(df_g[COL_NODE] == zone) & (df_g[COL_PLANT] == plant)]
            for _, r in sub.iterrows():
                y = int(r[COL_YEAR])
                dt_from = NetDateTime(int(y), 1, 1, 0, 0, 0)

                cap_twh = _as_float(r[COL_STORAGE_TWH])
                max_vol_gwh = None if cap_twh is None else _round_1dp(cap_twh * 1000.0)  # TWh -> GWh
                init_vol_gwh = None if max_vol_gwh is None else _round_1dp(max_vol_gwh * 0.5)

                pump_load = _round_1dp(r[COL_PUMP_CAP])

                storage_mems = [(head_storage_name, mem_head)]
                if create_tail:
                    storage_mems.append((tail_storage_name, mem_tail))

                for sname, mid in storage_mems:
                    if mid is None:
                        warnings.append(f"Could not read System->Storage membership id for '{sname}'; skipping volume writes for year {y}.")
                        continue

                    if max_vol_gwh is not None:
                        try:
                            _add_property_row(db, ENUM_MAX_VOLUME, mid, 1, float(max_vol_gwh), DateFrom=dt_from, Scenario=scenario_str)
                            props_written += 1
                        except Exception as e:
                            warnings.append(f"Failed Max Volume write for '{sname}' year={y}: {e}")

                    if init_vol_gwh is not None:
                        try:
                            _add_property_row(db, ENUM_INITIAL_VOLUME, mid, 1, float(init_vol_gwh), DateFrom=dt_from, Scenario=scenario_str)
                            props_written += 1
                        except Exception as e:
                            warnings.append(f"Failed Initial Volume write for '{sname}' year={y}: {e}")

                # Keep the previous change: Skip Pump Load writes for Reservoir/Pondage technologies
                if plant not in ONLY_HEAD_PLANT_TYPES:
                    if gen_mem_id is not None and pump_load is not None:
                        try:
                            _add_property_row(db, ENUM_PUMP_LOAD, gen_mem_id, 1, float(pump_load), DateFrom=dt_from, Scenario=scenario_str)
                            props_written += 1
                        except Exception as e:
                            warnings.append(f"Failed Pump Load write for generator '{gen_name}' year={y}: {e}")

        print("\n[STEP] Done.")
        print(f"  Created Storage objects:               {created_storages}")
        print(f"  Memberships created/ensured:           {created_memberships}")
        print(f"  Storage<->Node links created (bi-dir): {node_links}")
        print(f"  Generator<->Storage links created:     {generator_links}")
        print(f"  Properties written:                    {props_written}")
        print(f"  Rows processed (node-plant-year):      {len(df_g)}")

        if warnings:
            print("\n[STEP] Warnings (first 180):")
            for w in warnings[:180]:
                print("  - " + w)
            if len(warnings) > 180:
                print(f"  ... and {len(warnings) - 180} more")
        else:
            print("\n[STEP] No warnings.")
    finally:
        try:
            db.Close()
        except Exception:
            pass

# Run the step
step_storages_import_full()




#### 7. Reserves Import

In [ ]:
# ============================================================
# STEP â€” Reserve Objects Import (UPDATED FULL copy-pasteable script)
#   FIX (Thermal generator selection):
#   - Thermal generators are now selected by checking REAL Generator<->Fuel memberships
#     against Fuel objects in the database (NOT by generator name tokens).
#   - We restrict the Fuel scan to the specific fuels you listed (tokens), to keep it fast:
#       Nuclear, Hard coal, Lignite, Gas, Light oil, Heavy oil, Oil shale/Shale oil, Hydrogen
#
# Still includes:
#   - Target years derived from filtered CSV (no hardcoding), optional allowlist
#   - Reserve categories per zone; reserves assigned to zone category
#   - Reserve <-> Generator memberships for selected generators:
#       * Thermal (Generator has membership to one of the target fuels)
#       * *_DSR generators (excluding iDSR)
#       * Pump/reservoir hydro technologies (name keyword-based)
#
# Requires existing helpers:
#   _bootstrap_plexos_net, _add_object, _add_property_row, _get_objects_safe
# ============================================================

from pathlib import Path
from typing import Optional, Set, List, Any, Dict, Tuple
import pandas as pd

# -----------------------------
# User-editable configuration
# -----------------------------
RESERVE_REQ_CSV = DASHBOARD_RAWDATA_DIR / "Reserve requirements.csv"

FILTER_DATA_VERSION = "ERAA 2025 final"

# OPTIONAL: restrict imported years to a known set (set to None to accept all years in filtered CSV)
ALLOWED_YEARS: Optional[Set[int]] = None
# Example:
# ALLOWED_YEARS = {2028, 2030, 2033, 2035}

# CSV column names (must match file)
COL_DATA_VERSION = "data_version"
COL_YEAR = "YEAR"
COL_NODE = "MARKET_NODE"
COL_CATEGORY = "Category"
COL_VALUE = "Value"

# System -> Reserve collection id (given) used to READ membership id for writing properties
SYSTEM_RESERVE_COLLECTION_ID = 163
SYSTEM_NAME = "System"

# Property enum ids (given)
ENUM_TYPE = 1
ENUM_MUTUALLY_EXCLUSIVE = 2
ENUM_SHARING_ENABLED = 10
ENUM_MIN_PROVISION = 14
ENUM_TIMEFRAME = 16
ENUM_DURATION = 17
ENUM_VORS = 22

# Static values
STATIC_FCR = {
    ENUM_TYPE: 3.0,               # Regulation Raise (3)
    ENUM_MUTUALLY_EXCLUSIVE: 1.0, # Yes (1)
    ENUM_SHARING_ENABLED: -1.0,   # Yes (-1)
    ENUM_TIMEFRAME: 30.0,
    ENUM_DURATION: 300.0,
    ENUM_VORS: 10000.0,
}
STATIC_FRR = {
    ENUM_TYPE: 1.0,               # Raise (1)
    ENUM_MUTUALLY_EXCLUSIVE: 1.0, # Yes (1)
    ENUM_SHARING_ENABLED: -1.0,   # Yes (-1)
    ENUM_TIMEFRAME: 900.0,
    ENUM_DURATION: 3600.0,
    ENUM_VORS: 10000.0,
}

# Optional scenario name (None or string)
SCENARIO_NAME: Optional[str] = None

# Generator selection rules
# - DSR: include if contains "_DSR" but does NOT contain "iDSR"
DSR_INCLUDE_TOKEN = "_DSR"
DSR_EXCLUDE_TOKEN = "iDSR"

# - Hydro pump/reservoir technologies: name contains any of these tokens
HYDRO_NAME_TOKENS = [
    "RESERVOIR",
    "OPEN LOOP",
    "CLOSED LOOP",
    "PUMPING",
    "PUMP",
]

# - Thermal fuels to include (Fuel object name contains any token below)
THERMAL_FUEL_TOKENS = [
    "NUCLEAR",
    "HARD COAL",
    "LIGNITE",
    "GAS",
    "LIGHT OIL",
    "HEAVY OIL",
    "OIL SHALE",
    "SHALE OIL",
    "HYDROGEN",
    "BIOFUEL",
    "WASTE",
    "BIOMASS",
    "SMALL BIOMASS",
]

# -----------------------------
# Local helper functions (same style as earlier)
# -----------------------------
def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _round_1dp(x):
    fx = _as_float(x)
    if fx is None:
        return None
    return round(fx, 1)

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    # Defensive: remove common header strings if present
    for bad in ("Bidding_Zone", "Bidding Zone", "BIDDING_ZONE", "ZONE", "Zone"):
        if bad in vals:
            vals.remove(bad)
    return vals

def _category_to_reserve_type(cat: str) -> Optional[str]:
    if cat is None:
        return None
    s = str(cat).strip().upper()
    if not s:
        return None
    if "FCR" in s:
        return "FCR"
    if "FRR" in s:
        return "FRR"
    return None

def _find_collection_enum(CollectionEnum, candidates: List[str]) -> Optional[Any]:
    for name in candidates:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return None

def _get_membership_id_only(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> Optional[int]:
    """
    ONLY tries to read membership id. Does NOT add membership.
    """
    candidates = []
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), child_name))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, parent_name, SystemNS.String(child_name)))
        except Exception:
            pass

    return None

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> int:
    """
    Ensure a membership and return membership id (int).
    """
    last_exc = None
    candidates = []

    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        # Try read first
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e

        # Try add then read
        try:
            try:
                db.AddMembership(col_enum, parent_name, child_name)
            except Exception:
                db.AddMembership(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name))

            try:
                return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
            except Exception:
                return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(
        f"_ensure_membership_robust: failed membership parent={parent_name}, child={child_name}, "
        f"collection_raw={collection_enum_raw}. Last error: {last_exc}"
    )

def _ensure_reserve_category(db, ClassEnum, category_name: str) -> bool:
    """
    Create Reserve category (best effort). Safe to call repeatedly.
    Mirrors the earlier pattern: db.AddCategory(ClassEnum.Reserve, "DE00") + safe fallbacks.
    """
    if not category_name:
        return True
    if not hasattr(db, "AddCategory"):
        return False

    try:
        reserve_class_raw = ClassEnum.Reserve
    except Exception:
        reserve_class_raw = None

    attempts = []
    if reserve_class_raw is not None:
        attempts.extend([
            (reserve_class_raw, str(category_name)),
            (reserve_class_raw, str(category_name), ""),
        ])
        try:
            attempts.extend([
                (int(reserve_class_raw), str(category_name)),
                (int(reserve_class_raw), str(category_name), ""),
            ])
        except Exception:
            pass

    for args in attempts:
        try:
            db.AddCategory(*args)
            return True
        except Exception:
            pass

    return False

def _create_reserve_object_robust(db, ClassEnum, name: str, add_to_system: bool, category: str, description: str) -> bool:
    """
    Create Reserve object using your existing _add_object helper first, then a few safe fallbacks.
    """
    try:
        _add_object(db, ClassEnum.Reserve, name, add_to_system=add_to_system, category=category, description=description)
        return True
    except Exception:
        pass

    attempts = [
        (ClassEnum.Reserve, str(name), bool(add_to_system), str(category), str(description)),
        (ClassEnum.Reserve, str(name)),
        (ClassEnum.Reserve, str(name), bool(add_to_system)),
        (str(name), ClassEnum.Reserve, bool(add_to_system), str(category), str(description)),
        (str(name), ClassEnum.Reserve),
    ]
    last_exc = None
    for args in attempts:
        try:
            db.AddObject(*args)
            return True
        except Exception as e:
            last_exc = e

    raise RuntimeError(f"Reserve AddObject failed for '{name}'. Last error: {last_exc}")

def _matches_any_token(name: str, tokens: List[str]) -> bool:
    u = str(name).upper()
    return any(t in u for t in tokens)

def _is_selected_generator_for_reserves_basic(gen_name: str, zone: str) -> Tuple[bool, str]:
    """
    Name-based selections (DSR + hydro techs). Thermal handled separately via Fuel memberships.
    """
    if not gen_name or not zone:
        return (False, "invalid names")
    if not str(gen_name).startswith(f"{zone}_"):
        return (False, "not in zone (name prefix)")

    g_upper = str(gen_name).upper()

    if DSR_INCLUDE_TOKEN.upper() in g_upper and DSR_EXCLUDE_TOKEN.upper() not in g_upper:
        return (True, "DSR")

    for tok in HYDRO_NAME_TOKENS:
        if tok in g_upper:
            return (True, f"hydro token '{tok}'")

    return (False, "not basic-selected")

def _is_thermal_by_fuel_membership(
    db,
    CollectionEnum,
    SystemNS,
    generator_fuel_collection: Optional[Any],
    gen_name: str,
    candidate_fuels: List[str]
) -> Tuple[bool, str]:
    """
    Thermal selection by REAL membership:
      Returns True if generator has membership to ANY fuel in candidate_fuels (subset of Fuel objects).
    """
    if generator_fuel_collection is None:
        return (False, "Generator<->Fuel collection enum not found")

    # Try only the relevant fuels list for speed (stop on first hit)
    for fuel_name in candidate_fuels:
        mid = _get_membership_id_only(db, CollectionEnum, SystemNS, generator_fuel_collection, gen_name, fuel_name)
        if mid is not None and mid > 0:
            return (True, f"thermal (Fuel='{fuel_name}')")

    return (False, "no matching Fuel membership")

# -----------------------------
# Main implementation
# -----------------------------
def step_reserves_import_full():
    allowed = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx (first sheet, flat list).")

    df = pd.read_csv(RESERVE_REQ_CSV)

    required = [COL_DATA_VERSION, COL_YEAR, COL_NODE, COL_CATEGORY, COL_VALUE]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in Reserve requirements CSV: {missing}\nFound: {list(df.columns)}")

    # Clean + base filter
    df[COL_DATA_VERSION] = df[COL_DATA_VERSION].astype(str).map(_clean_cell)
    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_CATEGORY] = df[COL_CATEGORY].astype(str).map(_clean_cell)

    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_VALUE] = df[COL_VALUE].apply(_as_float)

    df = df[
        (df[COL_DATA_VERSION] == FILTER_DATA_VERSION) &
        (df[COL_NODE].isin(allowed)) &
        (df[COL_YEAR].notna())
    ].copy()

    if df.empty:
        print("[STEP] No rows remain after filters (data_version, zones). Nothing to import.")
        return

    # Derive years from CSV (like earlier logic), optionally restrict
    years_in_data = set(int(x) for x in df[COL_YEAR].astype(int).unique().tolist())
    if ALLOWED_YEARS is not None:
        years_final = sorted(set(years_in_data).intersection(set(ALLOWED_YEARS)))
    else:
        years_final = sorted(years_in_data)

    df = df[df[COL_YEAR].astype(int).isin(years_final)].copy()
    if df.empty:
        print("[STEP] No rows remain after year filter. Nothing to import.")
        print(f"         Years in data: {sorted(years_in_data)}")
        print(f"         ALLOWED_YEARS: {sorted(ALLOWED_YEARS) if ALLOWED_YEARS else None}")
        return

    # Map Category -> ReserveType and drop unrecognized
    df["RESERVE_TYPE"] = df[COL_CATEGORY].apply(_category_to_reserve_type)
    unk = df[df["RESERVE_TYPE"].isna()]
    if not unk.empty:
        print(f"[STEP] WARNING: {len(unk)} rows have unrecognized Category (no FCR/FRR match) and will be skipped.")
    df = df[df["RESERVE_TYPE"].notna()].copy()

    if df.empty:
        print("[STEP] All rows were filtered out due to Category mapping. Nothing to import.")
        return

    # Defensive duplicate aggregation: sum by (zone, year, reserve_type)
    dup = df.groupby([COL_NODE, COL_YEAR, "RESERVE_TYPE"]).size().reset_index(name="count")
    dup = dup[dup["count"] > 1]
    if not dup.empty:
        print("[STEP] WARNING: duplicate rows per (MARKET_NODE, YEAR, RESERVE_TYPE) found; summing Value.")
        try:
            display(dup.head(20))
        except Exception:
            print(dup.head(20).to_string(index=False))

    df_g = (
        df.groupby([COL_NODE, COL_YEAR, "RESERVE_TYPE"], as_index=False)
          .agg({COL_VALUE: "sum"})
          .sort_values([COL_NODE, COL_YEAR, "RESERVE_TYPE"])
    )
    df_g[COL_VALUE] = df_g[COL_VALUE].apply(_round_1dp)

    print("\n[STEP] Preview (first ~10 rows after filters + aggregation):")
    try:
        display(df_g.head(10))
    except Exception:
        print(df_g.head(10).to_string(index=False))

    # Bootstrap (earlier pattern)
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    scenario_str = SystemNS.String(SCENARIO_NAME) if SCENARIO_NAME else None

    # Find relevant collections (best effort, minimal candidate lists)
    reserve_generator_collection = _find_collection_enum(CollectionEnum, [
        "ReserveGenerators",
        "ReserveGenerator",
        "GeneratorReserves",
        "GeneratorReserve",
        "ReservesGenerators",
        "GeneratorsReserves",
    ])

    generator_fuel_collection = _find_collection_enum(CollectionEnum, [
        "GeneratorFuels",
        "GeneratorFuel",
        "FuelsGenerators",
        "FuelGenerators",
        "GeneratorPrimaryFuels",
        "GeneratorPrimaryFuel",
    ])

    # Existing objects snapshot
    try:
        reserves_in_db = set(_get_objects_safe(db, ClassEnum.Reserve))
    except Exception:
        reserves_in_db = set()

    try:
        generators_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
    except Exception:
        generators_in_db = set()

    try:
        fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
    except Exception:
        fuels_in_db = set()

    # Build candidate fuel names (subset) based on your token list
    candidate_fuels = sorted([f for f in (str(x) for x in fuels_in_db) if _matches_any_token(f, THERMAL_FUEL_TOKENS)])
    if not candidate_fuels:
        # This is not fatal, but it will prevent thermal selection
        print("[STEP][WARN] No Fuel objects matched THERMAL_FUEL_TOKENS; thermal generator selection may select none.")
        print("               Consider adjusting THERMAL_FUEL_TOKENS to match your DB fuel naming.")

    created_reserves = 0
    categories_created = 0
    memberships_found = 0
    static_props_written = 0
    minprov_props_written = 0
    rows_processed = 0
    reserve_generator_links = 0
    warnings: List[str] = []
    static_written_for: Set[str] = set()
    categories_done: Set[str] = set()

    # Diagnostics counters for selection
    selected_basic = 0
    selected_thermal = 0
    selected_total = 0

    reserve_names: List[str] = []
    for z in sorted(allowed):
        reserve_names.append(f"{z}_FCR")
        reserve_names.append(f"{z}_FRR")

    print(f"\n[STEP] Years imported (derived): {years_final}")
    print(f"[STEP] Zones in filter list:     {len(allowed)}")
    print(f"[STEP] Reserves to ensure/create (2 per zone): {len(reserve_names)}")
    print(f"[STEP] Candidate fuels for thermal selection: {len(candidate_fuels)}")
    print("  Sample fuels:", candidate_fuels[:15])

    try:
        # 0) Ensure Reserve categories per zone
        for zone in sorted(allowed):
            if zone in categories_done:
                continue
            ok = _ensure_reserve_category(db, ClassEnum, zone)
            if ok:
                categories_created += 1
            else:
                warnings.append(
                    "db.AddCategory not available / failed for Reserve categories on this API surface; "
                    "reserves may be created without categories."
                )
            categories_done.add(zone)

        # 1) Ensure objects exist (category = zone)
        for rname in reserve_names:
            zone = rname.split("_", 1)[0]
            if rname not in reserves_in_db:
                try:
                    _create_reserve_object_robust(db, ClassEnum, rname, add_to_system=True, category=zone, description="")
                    created_reserves += 1
                    reserves_in_db.add(rname)
                except Exception as e:
                    warnings.append(f"Failed to create Reserve '{rname}': {e}")

        # 2) Read membership ids (System->Reserve) for property rows
        mem_id_by_reserve: Dict[str, Optional[int]] = {}
        for rname in reserve_names:
            try:
                mid = _get_membership_id_only(db, CollectionEnum, SystemNS, SYSTEM_RESERVE_COLLECTION_ID, SYSTEM_NAME, rname)
                mem_id_by_reserve[rname] = mid
                if mid is not None:
                    memberships_found += 1
                else:
                    warnings.append(
                        f"Could not read System->Reserve membership id for '{rname}' (collection {SYSTEM_RESERVE_COLLECTION_ID}); "
                        f"property writes may be skipped for this reserve."
                    )
            except Exception as e:
                mem_id_by_reserve[rname] = None
                warnings.append(f"Error reading membership id for '{rname}': {e}")

        # 3) Write static props ONCE per reserve (via membership id)
        for rname in reserve_names:
            mid = mem_id_by_reserve.get(rname)
            if mid is None:
                continue
            if rname in static_written_for:
                continue

            static_map = STATIC_FCR if rname.endswith("_FCR") else STATIC_FRR
            for enum_id, v in static_map.items():
                try:
                    _add_property_row(db, int(enum_id), int(mid), 1, float(_round_1dp(v)), Scenario=scenario_str)
                    static_props_written += 1
                except Exception as e:
                    warnings.append(f"Failed static prop write (enum {enum_id}) for '{rname}': {e}")

            static_written_for.add(rname)

        # 4) Write Min Provision time-dependent rows
        for _, r in df_g.iterrows():
            rows_processed += 1
            zone = str(r[COL_NODE]).strip()
            year = int(r[COL_YEAR])
            rtype = str(r["RESERVE_TYPE"]).strip().upper()
            val = r[COL_VALUE]

            rname = f"{zone}_{rtype}"
            mid = mem_id_by_reserve.get(rname)

            if mid is None:
                warnings.append(f"Skipping Min Provision for '{rname}' year={year}: no System->Reserve membership id.")
                continue

            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)
            if val is None:
                continue

            try:
                _add_property_row(db, int(ENUM_MIN_PROVISION), int(mid), 1, float(val), DateFrom=dt_from, Scenario=scenario_str)
                minprov_props_written += 1
            except Exception as e:
                warnings.append(f"Failed Min Provision write for '{rname}' year={year}: {e}")

        # 5) Reserve <-> Generator memberships (selected generators only)
        if reserve_generator_collection is None:
            warnings.append(
                "Reserve<->Generator collection enum not found (tried common names). "
                "Skipping reserve-generator link creation."
            )
        else:
            # Build per-zone generator lists (by name prefix)
            gens_by_zone: Dict[str, List[str]] = {z: [] for z in allowed}
            for g in generators_in_db:
                gs = str(g)
                for z in allowed:
                    if gs.startswith(f"{z}_"):
                        gens_by_zone[z].append(gs)

            if generator_fuel_collection is None:
                warnings.append(
                    "Generator<->Fuel collection enum not found; thermal generator selection will be skipped "
                    "(DSR/hydro name-based selection still applies)."
                )

            for zone in sorted(allowed):
                r_fcr = f"{zone}_FCR"
                r_frr = f"{zone}_FRR"
                zone_reserves = [rn for rn in (r_fcr, r_frr) if rn in reserves_in_db]
                if not zone_reserves:
                    continue

                selected: List[Tuple[str, str]] = []
                for gname in gens_by_zone.get(zone, []):
                    # 5a) name-based selections (DSR + hydro)
                    ok_basic, reason_basic = _is_selected_generator_for_reserves_basic(gname, zone)
                    if ok_basic:
                        selected.append((gname, reason_basic))
                        selected_basic += 1
                        continue

                    # 5b) thermal selection by REAL fuel memberships (specific fuels)
                    ok_th, reason_th = _is_thermal_by_fuel_membership(
                        db=db,
                        CollectionEnum=CollectionEnum,
                        SystemNS=SystemNS,
                        generator_fuel_collection=generator_fuel_collection,
                        gen_name=gname,
                        candidate_fuels=candidate_fuels
                    )
                    if ok_th:
                        selected.append((gname, reason_th))
                        selected_thermal += 1

                # de-duplicate by generator name (keep first reason)
                seen = set()
                selected_unique: List[Tuple[str, str]] = []
                for gname, reason in selected:
                    if gname in seen:
                        continue
                    seen.add(gname)
                    selected_unique.append((gname, reason))

                selected_total += len(selected_unique)

                if not selected_unique:
                    continue

                for rname in zone_reserves:
                    for gname, reason in selected_unique:
                        try:
                            _ensure_membership_robust(db, CollectionEnum, SystemNS, reserve_generator_collection, rname, gname)
                            reserve_generator_links += 1
                        except Exception:
                            try:
                                _ensure_membership_robust(db, CollectionEnum, SystemNS, reserve_generator_collection, gname, rname)
                                reserve_generator_links += 1
                            except Exception as e2:
                                warnings.append(
                                    f"Failed Reserve<->Generator link for zone={zone}: reserve='{rname}' gen='{gname}' ({reason}). Error: {e2}"
                                )

        print("\n[STEP] Done.")
        print(f"  Created Reserve categories (attempted): {categories_created}")
        print(f"  Created Reserve objects:               {created_reserves}")
        print(f"  System->Reserve memberships found:     {memberships_found} / {len(reserve_names)}")
        print(f"  Static property rows written:          {static_props_written}")
        print(f"  Min Provision rows processed:          {rows_processed}")
        print(f"  Min Provision property rows written:   {minprov_props_written}")
        print(f"  Reserve<->Generator links created:     {reserve_generator_links}")
        print(f"  Generator selection counts:")
        print(f"    - Selected (basic DSR/hydro name):   {selected_basic}")
        print(f"    - Selected (thermal via Fuel mem):  {selected_thermal}")
        print(f"    - Selected unique total:            {selected_total}")

        if warnings:
            print("\n[STEP] Warnings (first 300):")
            for w in warnings[:300]:
                print("  - " + w)
            if len(warnings) > 300:
                print(f"  ... and {len(warnings) - 300} more")
        else:
            print("\n[STEP] No warnings.")
    finally:
        try:
            db.Close()
        except Exception:
            pass

# Run the step
step_reserves_import_full()





#### 8. Power2X Modelling Setup (Market Objects, Heat Nodes, Activation Price & Efficiency)

In [ ]:
# ============================================================
# STEP â€” Markets + Heat Nodes + memberships + property writes
#
# Includes:
#  - Market.Price computed:
#       Price_Input = Price_File / (Efficiency/100) / 3.6
#  - Written Market.Price and Power2X.Efficiency rounded to 2 decimals
#  - No dummy probe rows; probe uses first real year/value and skip rewriting it
#
# FIX (new):
#  - Make column name constants LOCAL inside the main function so they cannot be
#    overwritten by prior notebook state (prevents KeyError: 0).
#
# NOTHING ELSE CHANGED.
# ============================================================

import os
from pathlib import Path
from typing import Optional, List, Tuple, Any
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
DISPATCHABLE_CONS_CSV = DEMAND_DATA_DIR / "Additional dispatchable consumption.csv"

SCENARIO_NAME: Optional[str] = None

# -----------------------------
# Your confirmed Enums
# -----------------------------
MARKET_PRICE_ENUM = 12
P2X_EFF_ENUM = 15

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# Helpers
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _norm_header(s: str) -> str:
    s = str(s).strip().lower()
    for ch in [" ", "\t", "-", ".", "(", ")", "[", "]"]:
        s = s.replace(ch, "")
    return s


def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())


def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None


def _norm_plant_type(s: str) -> str:
    t = _clean_cell(s)
    tl = t.lower()
    if "electro" in tl:
        return "Electrolyser"
    if "power" in tl and "heat" in tl:
        return "Power to heat"
    return t


def _market_name(zone: str, plant_type_norm: str) -> Optional[str]:
    tl = plant_type_norm.lower()
    if "electro" in tl:
        return f"{zone}_H2"
    if "power" in tl and "heat" in tl:
        return f"{zone}_Heat"
    return None


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, existing_cache: Optional[set] = None) -> bool:
    if existing_cache is not None and name in existing_cache:
        return True
    try:
        existing = set(_get_objects_safe(db, class_enum_value))
        if name in existing:
            if existing_cache is not None:
                existing_cache.add(name)
            return True
        _add_object(db, class_enum_value, name, add_to_system=True, category="", description="")
        if existing_cache is not None:
            existing_cache.add(name)
        return True
    except Exception:
        return False


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


# -----------------------------
# Membership creation â€” DO NOT CHANGE (working)
# -----------------------------
def _ensure_membership_bruteforce(db, CollectionEnum, parent: str, child: str) -> Optional[Tuple[Any, int]]:
    for enum_name in ["HeatNodeMarkets", "Power2XHeatNodes"]:
        if hasattr(CollectionEnum, enum_name):
            col_enum = getattr(CollectionEnum, enum_name)
            for a, b in [(parent, child), (child, parent)]:
                try:
                    db.AddMembership(col_enum, a, b)
                except Exception:
                    pass
                try:
                    mid = int(db.GetMembershipID(col_enum, a, b))
                    print(f"[INFO] Created membership using resolved collection '{enum_name}' for parent={a} child={b} -> mem_id={mid}")
                    return (col_enum, mid)
                except Exception:
                    try:
                        mid = int(db.GetMembershipID(col_enum, b, a))
                        print(f"[INFO] Created membership using resolved collection '{enum_name}' for parent={b} child={a} -> mem_id={mid}")
                        return (col_enum, mid)
                    except Exception:
                        pass

    all_attrs = [a for a in dir(CollectionEnum) if not a.startswith("_")]
    for attr in all_attrs:
        try:
            col_val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        for a, b in [(parent, child), (child, parent)]:
            try:
                db.AddMembership(col_val, a, b)
            except Exception:
                pass
            try:
                mid = int(db.GetMembershipID(col_val, a, b))
                print(f"[INFO] Brute-force succeeded: collectionEnumAttr={attr} -> mem_id={mid} (parent={a}, child={b})")
                return (col_val, mid)
            except Exception:
                try:
                    mid = int(db.GetMembershipID(col_val, b, a))
                    print(f"[INFO] Brute-force succeeded: collectionEnumAttr={attr} -> mem_id={mid} (parent={b}, child={a})")
                    return (col_val, mid)
                except Exception:
                    pass

    return (None, None)


# -----------------------------
# Property-target membership discovery USING FIRST REAL WRITE (no dummy)
# -----------------------------
def _get_membership_id_readonly(db, col_enum, parent: str, child: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(col_enum, parent, child))
    except Exception:
        return None


def _find_property_target_membership_using_real_first_write(
    db,
    CollectionEnum,
    object_name: str,
    enum_id: int,
    trial_pairs: List[Tuple[Any, str, str]],
    *,
    probe_value: float,
    probe_dt_from,
    probe_scenario,
    debug_label: str
) -> Tuple[int, bool]:
    for col_enum, parent, child in trial_pairs:
        mid = _get_membership_id_readonly(db, col_enum, parent, child)
        if mid is None:
            continue
        try:
            _add_property_row(db, enum_id, mid, 1, float(probe_value), DateFrom=probe_dt_from, Scenario=probe_scenario)
            print(f"[INFO] {debug_label}: Found target mem_id={mid} (collection={int(col_enum)} parent={parent} child={child}) using FIRST REAL write")
            return int(mid), True
        except Exception:
            pass

    candidates = [object_name, "System"]
    all_attrs = [a for a in dir(CollectionEnum) if not a.startswith("_")]
    for attr in all_attrs:
        try:
            col_val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        for p in candidates:
            mid = _get_membership_id_readonly(db, col_val, p, object_name)
            if mid is not None:
                try:
                    _add_property_row(db, enum_id, mid, 1, float(probe_value), DateFrom=probe_dt_from, Scenario=probe_scenario)
                    print(f"[INFO] {debug_label}: Scan found mem_id={mid} using CollectionEnum.{attr} (id={int(col_val)}) via FIRST REAL write")
                    return int(mid), True
                except Exception:
                    pass

            mid2 = _get_membership_id_readonly(db, col_val, object_name, p)
            if mid2 is not None:
                try:
                    _add_property_row(db, enum_id, mid2, 1, float(probe_value), DateFrom=probe_dt_from, Scenario=probe_scenario)
                    print(f"[INFO] {debug_label}: Scan found mem_id={mid2} using CollectionEnum.{attr} (id={int(col_val)}) via FIRST REAL write")
                    return int(mid2), True
                except Exception:
                    pass

    raise RuntimeError(
        f"{debug_label}: Could not find ANY existing membership for object '{object_name}' "
        f"that accepts Enum={enum_id}. (No skipping.)"
    )


# ============================================================
# MAIN STEP
# ============================================================
def step_markets_heatnodes_and_ptx_properties():
    # ---- LOCAL column constants to prevent notebook state contamination ----
    COL_NODE  = "MARKET_NODE"
    COL_PLANT = "PEMMDB_PLANT_TYPE"
    COL_EFF   = "Weighted Average Electrolyser Efficiency"
    COL_PRICE = "Weighted Average Activation Price"
    COL_YEAR  = "TARGET_YEAR"

    bz_df = pd.read_excel(BIDDING_ZONE_XLSX, sheet_name=0, header=None)
    allowed = {str(x).strip() for x in bz_df.values.ravel() if not pd.isna(x)}
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    df = pd.read_csv(DISPATCHABLE_CONS_CSV, dtype=str)

    colmap = {}
    for c in df.columns:
        cn = _norm_header(c)
        if cn in ("market_node", "marketnode"):
            colmap[c] = COL_NODE
        elif cn in ("pemmdb_plant_type", "pemmdb_plant_typ", "pemmdbplanttype"):
            colmap[c] = COL_PLANT
        elif cn in ("target_year", "targetyear", "year"):
            colmap[c] = COL_YEAR
        elif "electro" in cn and "eff" in cn:
            colmap[c] = COL_EFF
        elif "activation" in cn and "price" in cn:
            colmap[c] = COL_PRICE
    df.rename(columns=colmap, inplace=True)
    print(df.columns)

    required = [COL_NODE, COL_PLANT, COL_EFF, COL_PRICE, COL_YEAR]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in CSV: {missing}. Found: {list(df.columns)}")

    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_PLANT] = df[COL_PLANT].astype(str).map(_clean_cell)
    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_EFF] = df[COL_EFF].apply(_as_float)
    df[COL_PRICE] = df[COL_PRICE].apply(_as_float)

    df = df[(df[COL_NODE].isin(allowed)) & (df[COL_YEAR].notna())].copy()
    if df.empty:
        print("[STEP] No rows after filters.")
        return

    df["PlantTypeNorm"] = df[COL_PLANT].map(_norm_plant_type)
    df["Power2XNameWanted"] = df.apply(lambda r: f"{r[COL_NODE]}_{r['PlantTypeNorm']}", axis=1)
    df["MarketName"] = df.apply(lambda r: _market_name(r[COL_NODE], r["PlantTypeNorm"]), axis=1)
    df = df[df["MarketName"].notna()].copy()
    if df.empty:
        print("[STEP] No Electrolyser/Power-to-heat rows after plant filter.")
        return

    df_g = (
        df.groupby([COL_NODE, "PlantTypeNorm", COL_YEAR], as_index=False)
          .agg({"Power2XNameWanted":"first","MarketName":"first", COL_EFF:"mean", COL_PRICE:"mean"})
          .sort_values([COL_NODE, "PlantTypeNorm", COL_YEAR])
    )

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        scenario_str = SystemNS.String(SCENARIO_NAME) if SCENARIO_NAME else None

        if not hasattr(ClassEnum, "Power2X"):
            raise RuntimeError("ClassEnum.Power2X not found.")
        p2x_class = getattr(ClassEnum, "Power2X")

        market_class = getattr(ClassEnum, "Market") if hasattr(ClassEnum, "Market") else None
        if market_class is None:
            for attr in dir(ClassEnum):
                if attr.lower().startswith("market"):
                    market_class = getattr(ClassEnum, attr)
                    break
        if market_class is None:
            raise RuntimeError("Could not resolve Market class.")

        heatnode_class = getattr(ClassEnum, "HeatNode") if hasattr(ClassEnum, "HeatNode") else None
        if heatnode_class is None:
            for attr in dir(ClassEnum):
                if "heat" in attr.lower() and "node" in attr.lower():
                    heatnode_class = getattr(ClassEnum, attr)
                    break
        if heatnode_class is None:
            raise RuntimeError("Could not resolve HeatNode class.")

        markets_in_db = set(_get_objects_safe(db, market_class))
        heatnodes_in_db = set(_get_objects_safe(db, heatnode_class))
        p2x_in_db = set(_get_objects_safe(db, p2x_class))
        p2x_lc = {str(n).strip().lower(): str(n).strip() for n in p2x_in_db if str(n).strip()}

        unique_pairs = df_g[[COL_NODE, "PlantTypeNorm", "MarketName", "Power2XNameWanted"]].drop_duplicates().values.tolist()
        print(f"[STEP] Unique pairs to process: {len(unique_pairs)}")

        created_markets = created_heat = 0
        link_count = 0
        writes_price = 0
        writes_eff = 0

        likely_market_col_enums = []
        for n in ["SystemMarkets", "Markets", "Market"]:
            if hasattr(CollectionEnum, n):
                likely_market_col_enums.append(getattr(CollectionEnum, n))

        likely_p2x_col_enums = []
        for n in ["SystemPower2X", "Power2X", "Power2Xs"]:
            if hasattr(CollectionEnum, n):
                likely_p2x_col_enums.append(getattr(CollectionEnum, n))

        for zone, plant_norm, market_name, p2x_wanted in unique_pairs:
            zone = str(zone).strip()
            plant_norm = str(plant_norm).strip()
            market_name = str(market_name).strip()
            p2x_wanted = str(p2x_wanted).strip()
            p2x_actual = p2x_lc.get(p2x_wanted.lower())

            if not p2x_actual:
                raise RuntimeError(f"Power2X '{p2x_wanted}' not found in DB but is required. (No skipping.)")

            if market_name not in markets_in_db:
                if _ensure_object(db, market_class, market_name, existing_cache=markets_in_db):
                    created_markets += 1
            if market_name not in heatnodes_in_db:
                if _ensure_object(db, heatnode_class, market_name, existing_cache=heatnodes_in_db):
                    created_heat += 1

            col_mid = _ensure_membership_bruteforce(db, CollectionEnum, market_name, market_name)
            if col_mid[0] is None:
                raise RuntimeError(f"Failed to create Market<->HeatNode membership for '{market_name}'")
            link_count += 1

            col_mid2 = _ensure_membership_bruteforce(db, CollectionEnum, p2x_actual, market_name)
            if col_mid2[0] is None:
                raise RuntimeError(f"Failed to create HeatNode<->Power2X membership for '{market_name}' <-> '{p2x_actual}'")
            link_count += 1

            sub = df_g[(df_g[COL_NODE] == zone) & (df_g["PlantTypeNorm"] == plant_norm)].sort_values(COL_YEAR).copy()
            if sub.empty:
                continue

            first = sub.iloc[0]
            first_year = int(first[COL_YEAR])
            first_dt = NetDateTime(int(first_year), 1, 1, 0, 0, 0)

            first_price_file = _as_float(first[COL_PRICE])
            first_eff_raw = _as_float(first[COL_EFF])
            if first_price_file is None or first_eff_raw is None:
                raise RuntimeError(f"Missing first-year values for '{market_name}' / '{p2x_actual}' (No skipping.)")
            if float(first_eff_raw) == 0.0:
                raise RuntimeError(f"Efficiency is zero for '{p2x_actual}' in year {first_year}; cannot divide by zero.")

            first_price_input = first_price_file / (float(first_eff_raw) / 100.0) / 3.6
            first_price_input = round(float(first_price_input), 2)
            first_eff_to_write = round(float(first_eff_raw), 2)

            market_trials = []
            for ce in likely_market_col_enums:
                market_trials.append((ce, "System", market_name))
                market_trials.append((ce, market_name, "System"))
                market_trials.append((ce, market_name, market_name))

            mem_market_for_price, _ = _find_property_target_membership_using_real_first_write(
                db, CollectionEnum, market_name, MARKET_PRICE_ENUM,
                trial_pairs=market_trials,
                probe_value=float(first_price_input),
                probe_dt_from=first_dt,
                probe_scenario=scenario_str,
                debug_label="Market.Price"
            )
            writes_price += 1
            print(f"[OK] Wrote Market.Price (first/probe): {market_name} year={first_year} price={first_price_input} mem_id={mem_market_for_price}")

            p2x_trials = []
            for ce in likely_p2x_col_enums:
                p2x_trials.append((ce, "System", p2x_actual))
                p2x_trials.append((ce, p2x_actual, "System"))
                p2x_trials.append((ce, p2x_actual, p2x_actual))

            mem_p2x_for_eff, _ = _find_property_target_membership_using_real_first_write(
                db, CollectionEnum, p2x_actual, P2X_EFF_ENUM,
                trial_pairs=p2x_trials,
                probe_value=float(first_eff_to_write),
                probe_dt_from=first_dt,
                probe_scenario=scenario_str,
                debug_label="Power2X.Efficiency"
            )
            writes_eff += 1
            print(f"[OK] Wrote Power2X.Efficiency (first/probe): {p2x_actual} year={first_year} eff={first_eff_to_write} mem_id={mem_p2x_for_eff}")

            for i in range(1, len(sub)):
                rr = sub.iloc[i]
                y = int(rr[COL_YEAR])
                dt_from = NetDateTime(int(y), 1, 1, 0, 0, 0)

                price_file = _as_float(rr[COL_PRICE])
                eff_raw = _as_float(rr[COL_EFF])
                if price_file is None or eff_raw is None:
                    raise RuntimeError(f"Missing values for '{market_name}' / '{p2x_actual}' year={y} (No skipping.)")
                if float(eff_raw) == 0.0:
                    raise RuntimeError(f"Efficiency is zero for '{p2x_actual}' in year {y}; cannot divide by zero.")

                price_input = price_file / (float(eff_raw) / 100.0) / 3.6
                price_input = round(float(price_input), 2)
                eff_to_write = round(float(eff_raw), 2)

                _add_property_row(db, MARKET_PRICE_ENUM, mem_market_for_price, 1, float(price_input), DateFrom=dt_from, Scenario=scenario_str)
                writes_price += 1
                print(f"[OK] Wrote Market.Price: {market_name} year={y} price={price_input} mem_id={mem_market_for_price}")

                _add_property_row(db, P2X_EFF_ENUM, mem_p2x_for_eff, 1, float(eff_to_write), DateFrom=dt_from, Scenario=scenario_str)
                writes_eff += 1
                print(f"[OK] Wrote Power2X.Efficiency: {p2x_actual} year={y} eff={eff_to_write} mem_id={mem_p2x_for_eff}")

        print("\n[STEP] Completed.")
        print(f"  Created Markets:              {created_markets}")
        print(f"  Created HeatNodes:            {created_heat}")
        print(f"  Relationship memberships made:{link_count}")
        print(f"  Market.Price writes:          {writes_price}")
        print(f"  Power2X.Efficiency writes:    {writes_eff}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_markets_heatnodes_and_ptx_properties()




#### 9. Inelastic Import

In [ ]:
# ============================================================
# Inelastic Supply (Generators with Fixed Load + Max Capacity)
#
# CHANGE vs the prior working version (THIS UPDATE ONLY):
#   - If no rows remain after filters (e.g. for a specific bidding zone),
#     PRINT A WARNING and EXIT CLEANLY (no exception).
#
# Everything else unchanged.
# ============================================================

import os
from pathlib import Path
from typing import Optional
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
GEN_CAP_CSV = DASHBOARD_RAWDATA_DIR / "GenerationCapacities.csv"

OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_NAME = "Generator_Inelastic.csv"
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PLEXOS_REL_CSV_PATH = r"Data Files\Generator Data\Generator_Inelastic.csv"

DATAFILE_OBJECT_NAME = "Generator_Inelastic"

# -----------------------------
# CHANGEABLE FILTERS
# -----------------------------
FILTER_DATA_VERSION = "ERAA 2025 final"
FILTER_OPERATION_STATUS_INELASTIC = "Inelastic supply / fixed profile"  # Column F in your description

# Optional scenario
SCENARIO_NAME: Optional[str] = None  # e.g. "36_WS"

# -----------------------------
# PROPERTY / ENUMS
# -----------------------------
FIXED_LOAD_ENUM_ID = 96     # Fixed Load
MAX_CAPACITY_ENUM_ID = 52   # Max Capacity (as requested)

PROP_UNITS = "Units"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# COLLECTION ENUM FALLBACK IDS
# -----------------------------
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1
FALLBACK_SYSTEM_DATAFILES_COLLECTION_ID = None

# -----------------------------
# HELPERS
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _sanitize_name(s: str) -> str:
    s = str(s).strip()
    return s.replace("/", "_").replace("\\", "_").replace(":", "_")


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc in ("data_version", "data version"):
            rename[c] = "data_version"
        elif lc in ("target_year", "target year", "year"):
            rename[c] = "Target_Year"
        elif lc in ("market_node", "market node"):
            rename[c] = "Market_Node"
        elif lc == "technology":
            rename[c] = "Technology"
        elif lc in ("operational_status", "operational status", "operation status"):
            rename[c] = "Operation Status"
        elif lc == "value":
            rename[c] = "Value"

    df.rename(columns=rename, inplace=True)
    return df


def _build_matrix(df: pd.DataFrame, object_names: list[str]) -> pd.DataFrame:
    pivot = df.pivot_table(
        index="Target_Year",
        columns="ObjectName",
        values="Value",
        aggfunc="sum",
        fill_value=0.0
    )
    for n in object_names:
        if n not in pivot.columns:
            pivot[n] = 0.0
    pivot = pivot[object_names].reset_index().rename(columns={"Target_Year": "Year"})
    try:
        pivot["Year"] = pivot["Year"].astype(int)
        pivot = pivot.sort_values("Year")
    except Exception:
        pivot = pivot.sort_values("Year")
    return pivot


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _ensure_membership_bi(db, collection_enum, a: str, b: str) -> bool:
    try:
        _ensure_membership(db, collection_enum, a, b)
        return True
    except Exception:
        pass
    try:
        _ensure_membership(db, collection_enum, b, a)
        return True
    except Exception:
        return False


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _find_collection_enum(CollectionEnum, required_tokens_lc: list[str], preferred_names: list[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


def _ensure_generator_category(db, ClassEnum, category_name: str):
    if not category_name:
        return
    try:
        db.AddCategory(ClassEnum.Generator, str(category_name))
    except Exception:
        pass


# -----------------------------
# STEP MAIN
# -----------------------------
def step_inelastic_supply_generators():
    allowed_nodes = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed_nodes:
        raise RuntimeError("No regions found in Bidding_Zone_List.xlsx")

    df = pd.read_csv(GEN_CAP_CSV, dtype=str)
    df = _normalize_columns(df)

    required = ["data_version", "Target_Year", "Market_Node", "Technology", "Operation Status", "Value"]
    for r in required:
        if r not in df.columns:
            raise RuntimeError(f"Missing required column {r!r}. Found: {list(df.columns)}")

    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0.0)
    for c in ["data_version", "Target_Year", "Market_Node", "Technology", "Operation Status"]:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df["data_version"] == FILTER_DATA_VERSION) &
        (df["Operation Status"] == FILTER_OPERATION_STATUS_INELASTIC) &
        (df["Market_Node"].isin(allowed_nodes))
    ].copy()

    # --- CHANGE: warn + clean exit if nothing matched ---
    if df.empty:
        print("[STEP] WARNING: No rows remain after filters. Nothing to do.")
        print(f"  data_version filter:        {FILTER_DATA_VERSION!r}")
        print(f"  Operation Status filter:    {FILTER_OPERATION_STATUS_INELASTIC!r}")
        print(f"  Allowed nodes count:        {len(allowed_nodes)}")
        print("  Tip: verify Operation Status spelling/casing in GenerationCapacities.csv for this zone.")
        return
    # ----------------------------------------------------

    df["ObjectName"] = df.apply(lambda r: _sanitize_name(f"{r['Market_Node']}_{r['Technology']}_Inelastic"), axis=1)
    object_names = df["ObjectName"].dropna().unique().tolist()

    matrix = _build_matrix(df[["Target_Year", "ObjectName", "Value"]], object_names)
    matrix.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Wrote inelastic matrix CSV: {OUTPUT_CSV_PATH}")

    db, ClassEnum, CollectionEnum, SystemNS = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    FIXED_LOAD_ENUM_ID = _resolve_semantic_enum("System", "Generator", "Generators", "Fixed Load", 96)
    MAX_CAPACITY_ENUM_ID = _resolve_semantic_enum("System", "Generator", "Generators", "Max Capacity", 52)
    

    try:
        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID
        )
        datafile_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "data", "file"],
            ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
            fallback_id=FALLBACK_SYSTEM_DATAFILES_COLLECTION_ID
        )

        gen_node_collection = None
        if hasattr(CollectionEnum, "NodeGenerators"):
            gen_node_collection = getattr(CollectionEnum, "NodeGenerators")
        elif hasattr(CollectionEnum, "GeneratorNodes"):
            gen_node_collection = getattr(CollectionEnum, "GeneratorNodes")
        elif hasattr(CollectionEnum, "PowerStationNodes"):
            gen_node_collection = getattr(CollectionEnum, "PowerStationNodes")

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        gen_units_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Generator", "Generators", PROP_UNITS),
            ("System", "Generator", "SystemGenerators", PROP_UNITS),
        ])

        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME)
        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
            Scenario=scenario_str
        )

        gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
        nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))

        created = linked_nodes = linked_props = 0

        unique = df.drop_duplicates(subset=["ObjectName"])[["ObjectName", "Market_Node"]].copy()

        for _, r in unique.iterrows():
            gen_name = str(r["ObjectName"]).strip()
            node_name = str(r["Market_Node"]).strip()

            _ensure_generator_category(db, ClassEnum, node_name)

            if gen_name not in gens_in_db:
                _add_object(db, ClassEnum.Generator, gen_name, add_to_system=True, category=node_name, description="")
                gens_in_db.add(gen_name)
                created += 1

            mem_id = _ensure_membership(db, gen_mem_collection, "System", gen_name)

            _add_property_row(db, gen_units_enum, mem_id, 1, 1.0, Scenario=scenario_str)

            # Fixed Load (Enum 95) via DataFile
            _add_property_row(
                db,
                FIXED_LOAD_ENUM_ID,
                mem_id,
                1,
                0.0,
                DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                Scenario=scenario_str
            )

            # Max Capacity (Enum 52) via SAME DataFile
            _add_property_row(
                db,
                MAX_CAPACITY_ENUM_ID,
                mem_id,
                1,
                0.0,
                DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                Scenario=scenario_str
            )

            linked_props += 2

            if gen_node_collection is not None and node_name in nodes_in_db:
                if _ensure_membership_bi(db, gen_node_collection, node_name, gen_name):
                    linked_nodes += 1

        print("Done.")
        print(f"Created Inelastic Generators: {created}")
        print(f"Linked Fixed Load (Enum {FIXED_LOAD_ENUM_ID}) + Max Capacity (Enum {MAX_CAPACITY_ENUM_ID}) via DataFile: {linked_props}")
        print(f"Linked Node memberships: {linked_nodes}")

        if gen_node_collection is None:
            print("[WARN] Could not find a Generator<->Node membership collection enum (NodeGenerators/GeneratorNodes/PowerStationNodes). Node coupling may be missing.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_inelastic_supply_generators()




#### 10. Battery Adjusted Demand and Object

In [ ]:
# Battery/EV/HP demand adjustments (ROW-ALIGNED, no DATE parsing) + PLEXOS variables
import os, csv, shutil, re
from pathlib import Path
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
DEMAND_FINAL_DIR = DATA_FILES_ROOT / "Final Demand Time Series"
BATTERY_ADJUSTED_SUBFOLDER = "Battery Adjusted Demand"
BATTERY_ADJUSTED_DIR = DEMAND_FINAL_DIR / BATTERY_ADJUSTED_SUBFOLDER

IDSR_RATIO_CSV = DEMAND_DATA_DIR / "iDSR_ratio.csv"
BATTERIES_SRC_DIR = DEMAND_DATA_DIR / r"Behind the meter PV-Battery system additions\Batteries"
DETAILED_DEMAND_DIR = DEMAND_DATA_DIR / "Detailed demand"


SCENARIO_NAME = "Battery_Adjusted_Demand"

OVERWRITE_BATTERY_ADJUSTED = True

# PLEXOS API
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# -----------------------------
# CSV HELPERS
# -----------------------------
def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:8192]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _parse_region_year_from_filename(fname: str):
    m_region = re.match(r"([A-Z]{2}\d{2})_", fname)
    m_year = re.search(r"(\d{4})", fname)
    region = m_region.group(1) if m_region else None
    year = int(m_year.group(1)) if m_year else None
    return region, year


def _to_float_safe(x):
    try:
        return float(str(x).strip())
    except Exception:
        return 0.0


# -----------------------------
# Addition-file loader (ROW-ALIGNED)
# -----------------------------
def _read_addition_matrix(add_path: Path) -> pd.DataFrame | None:
    """
    Reads addition file and returns a dataframe with numeric columns '1'..'36'
    and the SAME NUMBER OF ROWS as in the file. NO date parsing.
    We only use the WS columns.
    """
    if not add_path.exists():
        return None

    delim = _detect_csv_delimiter(add_path)
    df = pd.read_csv(add_path, sep=delim, engine="python")

    ws_cols = [f"WS{i:02d}" for i in range(1, 37)]
    num_cols = [str(i) for i in range(1, 37)]

    if all(c in df.columns for c in ws_cols):
        mat = df[ws_cols].copy()
        mat = mat.rename(columns={f"WS{i:02d}": str(i) for i in range(1, 37)})
    elif all(c in df.columns for c in num_cols):
        mat = df[num_cols].copy()
    else:
        # case-insensitive WS mapping
        upper_map = {c.strip().upper(): c for c in df.columns}
        if all(f"WS{i:02d}" in upper_map for i in range(1, 37)):
            mat = df[[upper_map[f"WS{i:02d}"] for i in range(1, 37)]].copy()
            mat.columns = [str(i) for i in range(1, 37)]
        else:
            return None

    # coerce to float
    for c in num_cols:
        mat[c] = pd.to_numeric(mat[c], errors="coerce").fillna(0.0).astype("float32")

    return mat


def _apply_addition_row_aligned(demand_df: pd.DataFrame, add_mat: pd.DataFrame, factor: float, label: str) -> pd.DataFrame:
    """
    Adds factor * add_mat row-by-row to demand_df columns '1'..'36'.
    Assumes the addition file rows correspond 1:1 with demand file rows.
    If row counts differ, we truncate to the smaller size and warn.
    """
    if factor <= 0 or add_mat is None or add_mat.empty:
        return demand_df

    num_cols = [str(i) for i in range(1, 37)]

    n_d = len(demand_df)
    n_a = len(add_mat)
    n = min(n_d, n_a)
    if n_d != n_a:
        print(f"  Warning: row mismatch for {label}: demand rows={n_d}, add rows={n_a}. Truncating to {n} rows.")

    out = demand_df.copy()
    # make sure demand numeric cols are float
    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype("float32")

    # add scaled values (vectorized, no merge)
    out.loc[out.index[:n], num_cols] = (
        out.loc[out.index[:n], num_cols].values
        + add_mat.loc[add_mat.index[:n], num_cols].values * float(factor)
    )

    print(f"  Applied {label}: factor={factor} (row-aligned)")
    return out


# -----------------------------
# LOAD iDSR RATIOS
# -----------------------------
idf_delim = _detect_csv_delimiter(IDSR_RATIO_CSV)
idr = pd.read_csv(IDSR_RATIO_CSV, sep=idf_delim, engine="python", dtype=str).fillna("")

idsr_map = {}
for _, row in idr.iterrows():
    node = str(row.get("MARKET_NODE", "")).strip()
    try:
        year = int(str(row.get("TARGET_YEAR", "")).strip())
    except Exception:
        continue
    idsr_map[(node, year)] = {
        "ev": _to_float_safe(row.get("Ratio of price-sensitive consumer - Evs", 0.0)),
        "hp": _to_float_safe(row.get("Ratio of price-sensitive consumer - Heat Pumps", 0.0)),
        "bat": _to_float_safe(row.get("Ratio of price-sensitive consumer - Non-Market Participating Batteries", 0.0)),
    }


# -----------------------------
# COPY DEMAND FILES -> Battery Adjusted Demand + APPLY ADDITIONS
# -----------------------------
BATTERY_ADJUSTED_DIR.mkdir(parents=True, exist_ok=True)

allowed = _read_bidding_zones(BIDDING_ZONE_XLSX)
demand_files = []
for p in sorted(DEMAND_FINAL_DIR.glob("*.csv")):
    region, year = _parse_region_year_from_filename(p.name)
    if region and year and region in allowed and p.parent != BATTERY_ADJUSTED_DIR:
        demand_files.append((p, region, year))

# UPDATED: warn instead of failing
if not demand_files:
    print(
        f"\nWARNING: No demand CSVs found in {DEMAND_FINAL_DIR} matching bidding zones.\n"
        f"This can happen if a bidding zone has no demand in this dataset.\n"
        f"No battery-adjusted demand files will be created."
    )

copied_files = []
for src_path, region, year in demand_files:
    dst_path = BATTERY_ADJUSTED_DIR / src_path.name
    if dst_path.exists() and not OVERWRITE_BATTERY_ADJUSTED:
        print(f"Skipping existing adjusted file: {dst_path.name}")
    else:
        shutil.copy2(src_path, dst_path)

    delim = _detect_csv_delimiter(dst_path)
    demand_df = pd.read_csv(dst_path, sep=delim, engine="python")

    # Validate demand schema (Year, Month, Day, Period, 1..36)
    num_cols = [str(i) for i in range(1, 37)]
    required = {"Year", "Month", "Day", "Period"} | set(num_cols)
    if not required.issubset(set(demand_df.columns)):
        raise RuntimeError(
            f"{dst_path.name}: expected columns Year, Month, Day, Period, 1..36 not found.\n"
            f"Columns: {demand_df.columns.tolist()}"
        )

    ratios = idsr_map.get((region, year), {"ev": 0.0, "hp": 0.0, "bat": 0.0})
    print(f"\nProcessing {region} {year}: ratios={ratios}")

    # Batteries
    if ratios["bat"] > 0:
        add_path = BATTERIES_SRC_DIR / f"{region}_Batteries_{year}_National Trends.csv"
        add_mat = _read_addition_matrix(add_path)
        if add_mat is None:
            print(f"  Warning: could not read Batteries file: {add_path}")
        else:
            demand_df = _apply_addition_row_aligned(demand_df, add_mat, ratios["bat"], "Batteries")

    # EVs
    if ratios["ev"] > 0:
        add_path = DETAILED_DEMAND_DIR / f"{region}_Demand_EVpart_{year}_National Trends.csv"
        add_mat = _read_addition_matrix(add_path)
        if add_mat is None:
            print(f"  Warning: could not read EV file: {add_path}")
        else:
            demand_df = _apply_addition_row_aligned(demand_df, add_mat, ratios["ev"], "EV")

    # Heat Pumps
    if ratios["hp"] > 0:
        add_path = DETAILED_DEMAND_DIR / f"{region}_Demand_HPpart_{year}_National Trends.csv"
        add_mat = _read_addition_matrix(add_path)
        if add_mat is None:
            print(f"  Warning: could not read HP file: {add_path}")
        else:
            demand_df = _apply_addition_row_aligned(demand_df, add_mat, ratios["hp"], "Heat Pumps")

    # Save adjusted demand
    demand_df.to_csv(dst_path, index=False, sep=delim)
    copied_files.append((dst_path, region, year))

print(f"\nCreated/updated {len(copied_files)} adjusted demand files in: {BATTERY_ADJUSTED_DIR}")


# -----------------------------
# ONE-LINE VALIDATION (AT00 2028 Period=1, column '1')
# -----------------------------
try:
    if copied_files:
        p = BATTERY_ADJUSTED_DIR / "AT00_Demand_total_2028_National Trends.csv"
        if p.exists():
            vdf = pd.read_csv(p, sep=_detect_csv_delimiter(p), engine="python")
            val = vdf.loc[
                (vdf["Year"] == 2028) & (vdf["Month"] == 1) & (vdf["Day"] == 1) & (vdf["Period"] == 1),
                ["1"],
            ].head(1)
            print("\nVALIDATION (AT00 2028, 2028-01-01 Period=1, column '1'):")
            print(val)
        else:
            print("\nVALIDATION skipped: AT00 2028 adjusted file not found.")
    else:
        print("\nVALIDATION skipped: no adjusted demand files created.")
except Exception as e:
    print("\nVALIDATION failed:", e)


# -----------------------------
# PLEXOS: create variables + link profiles + Region.Load, all under scenario Battery_Adjusted_Demand
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True)


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))
        except Exception as e:
            last = (a, b, c, d, str(e))
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _find_system_regions_collection_enum(CollectionEnum):
    if hasattr(CollectionEnum, "SystemRegions"):
        return getattr(CollectionEnum, "SystemRegions")
    names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    candidates = [n for n in names if ("system" in n.lower() and "region" in n.lower())]
    if not candidates:
        raise RuntimeError("Could not find System->Regions collection enum")
    return getattr(CollectionEnum, candidates[0])


def _resolve_variable_profile_enum(db, SystemNS):
    prop_candidates = ["Profile", "Profiles", "Profile Data", "Profile File", "Profile Filename", "Profile Data File", "Data File", "DataFile", "File", "Path", "Time Series"]
    collection_candidates = ["Variables", "SystemVariables", "Variable"]
    last = None
    for col in collection_candidates:
        for prop in prop_candidates:
            try:
                enum_id = int(db.PropertyName2EnumId(
                    SystemNS.String("System"),
                    SystemNS.String("Variable"),
                    SystemNS.String(col),
                    SystemNS.String(prop),
                ))
                return enum_id, col, prop
            except Exception as e:
                last = (col, prop, str(e))
    raise RuntimeError(f"Could not resolve Variable profile enum. Last attempt: {last}")


# UPDATED: If no adjusted files, skip PLEXOS section cleanly
if not copied_files:
    print("\nNo adjusted demand files were created. Skipping PLEXOS variable creation.")
else:
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    try:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)

        region_load_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Region", "Regions", "Load"),
            ("System", "Region", "SystemRegions", "Load"),
            ("Region", "System", "Regions", "Load"),
            ("System", "Regions", "Region", "Load"),
            ("System", "SystemRegions", "Region", "Load"),
        ])
        sampling_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Variable", "Variables", "Sampling Method"),
            ("System", "Variable", "Variables", "SamplingMethod"),
            ("System", "Variable", "SystemVariables", "Sampling Method"),
            ("System", "Variable", "SystemVariables", "SamplingMethod"),
            ("Variable", "System", "Variables", "Sampling Method"),
            ("Variable", "System", "Variables", "SamplingMethod"),
        ])
        profile_enum, profile_col, profile_prop = _resolve_variable_profile_enum(db, SystemNS)

        sys_regions_enum = _find_system_regions_collection_enum(CollectionEnum)
        if hasattr(CollectionEnum, "SystemVariables"):
            var_mem_collection = getattr(CollectionEnum, "SystemVariables")
        elif hasattr(CollectionEnum, "Variables"):
            var_mem_collection = getattr(CollectionEnum, "Variables")
        else:
            raise RuntimeError("Could not find CollectionEnum.SystemVariables or CollectionEnum.Variables")

        regions_in_db = set(_get_objects_safe(db, ClassEnum.Region))
        variables_in_db = set(_get_objects_safe(db, ClassEnum.Variable))

        created_vars = 0
        written_sampling = 0
        written_profiles = 0
        written_load = 0

        for dst_path, region, year in copied_files:
            var_name = f"{region}_Battery_Adjusted_Demand_{year}"
            dt_from = NetDateTime(year, 1, 1, 0, 0, 0)

            if var_name not in variables_in_db:
                _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category="", description="")
                variables_in_db.add(var_name)
                created_vars += 1

            try:
                db.GetMembershipID(var_mem_collection, "System", var_name)
            except Exception:
                try:
                    db.AddMembership(var_mem_collection, "System", var_name)
                except Exception:
                    pass

            var_mem_id = int(db.GetMembershipID(var_mem_collection, "System", var_name))

            db.AddProperty(
                var_mem_id,
                sampling_enum,
                1,
                2.0,
                dt_from,
                None,
                None,
                None,
                None,
                SystemNS.String(SCENARIO_NAME),
                None
            )
            written_sampling += 1

            plexos_rel = str(Path("Data Files") / "Final Demand Time Series" / BATTERY_ADJUSTED_SUBFOLDER / dst_path.name)
            for band in range(1, 37):
                db.AddProperty(
                    var_mem_id,
                    profile_enum,
                    int(band),
                    0.0,
                    dt_from,
                    None,
                    None,
                    SystemNS.String(plexos_rel),
                    None,
                    SystemNS.String(SCENARIO_NAME),
                    None
                )
                written_profiles += 1

            if region in regions_in_db:
                region_mem = int(db.GetMembershipID(sys_regions_enum, "System", region))
                db.AddProperty(
                    region_mem,
                    region_load_enum,
                    1,
                    0.0,
                    dt_from,
                    None,
                    SystemNS.String(var_name),
                    None,
                    None,
                    SystemNS.String(SCENARIO_NAME),
                    None
                )
                written_load += 1

        print("\nPLEXOS variable creation done.")
        print(f"Variables created: {created_vars}")
        print(f"Sampling rows: {written_sampling}")
        print(f"Profile rows: {written_profiles}")
        print(f"Region.Load rows: {written_load}")

    finally:
        try:
            db.Close()
        except Exception:
            pass




In [ ]:
# ============================================================
# STEP â€” OOM Batteries import (FULL copy-pasteable script)
# ============================================================
# This script:
#  - reads Batteries additional information.csv
#  - restricts to bidding zones listed in Bidding_Zone_List.xlsx
#  - filters OP_STAT == "Out of market - for PV/battery dispatch optimization"
#    and data_version == "ERAA 2025 final"
#  - creates Battery objects named "{BiddingZone}_OOM_Battery" (e.g., DE00_OOM_Battery)
#  - couples Battery <-> Node (bi-directional) using node name = bidding zone (best effort)
#  - ensures System->Battery membership using Collection ID 81
#  - multiplies MAX LOAD CAP (MW) and STORAGE CAPACITY (MWh) by iDSR ratio column:
#      "Ratio of price-sensitive consumer - Non-Market Participating Batteries"
#    for matching (MARKET_NODE, TARGET_YEAR)
#  - writes Max Power (enum 28) and Capacity (enum 26) per TARGET_YEAR with DateFrom = YYYY-01-01
#  - writes Units (enum 25), Initial SoC (32), Charge Eff (33), Discharge Eff (34) once per battery
#  - marks ALL rows with Scenario "Battery_Adjusted_Demand"
#
# Paste this entire cell and run.
# ============================================================

from pathlib import Path
from typing import Optional, Set, List, Any, Dict, Tuple
import os, csv, re
import pandas as pd

# -----------------------------
# User-editable configuration
# -----------------------------
BATTERIES_CSV = DASHBOARD_RAWDATA_DIR / "Batteries additional information.csv"
IDSR_RATIO_CSV = DEMAND_DATA_DIR / "iDSR_ratio.csv"

FILTER_DATA_VERSION = "ERAA 2025 final"
FILTER_OPERATION_STATUS = "Out of market - for PV/battery dispatch optimization"

# CSV column names (must match file)
COL_NODE = "MARKET_NODE"
COL_OP_STAT = "OP_STAT"
COL_MAX_POWER = "MAX LOAD CAP (MW)"
COL_CAPACITY = "STORAGE CAPACITY (MWh)"
COL_YEAR = "TARGET_YEAR"
COL_DATA_VERSION = "data_version"

# iDSR ratio column (column E in your description)
COL_RATIO_BAT = "Ratio of price-sensitive consumer - Non-Market Participating Batteries"

# Property enum ids
ENUM_UNITS = 25
ENUM_CAPACITY = 26
ENUM_MAX_POWER = 28

# Static battery properties (percent values)
INITIAL_SOC_PERCENT = 50.0
CHARGE_EFFICIENCY_PERCENT = 95.91
DISCHARGE_EFFICIENCY_PERCENT = 95.91

ENUM_INITIAL_SOC = 32
ENUM_CHARGE_EFF = 33
ENUM_DISCHARGE_EFF = 34

# System -> Battery collection id (given)
SYSTEM_BATTERY_COLLECTION_ID = 81

# Scenario (required)
SCENARIO_NAME = "Battery_Adjusted_Demand"

# If True, write zero-sized batteries too (when ratio==0). If False, skip those batteries/years.
WRITE_ZERO_WHEN_RATIO_MISSING_OR_ZERO = False

# -----------------------------
# CSV helpers
# -----------------------------
def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:8192]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","

def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _round_1dp(x):
    fx = _as_float(x)
    if fx is None:
        return None
    return round(fx, 1)

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals

# -----------------------------
# PLEXOS bootstrap + minimal helpers (standalone)
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime

def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []

def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    # AddObject(string, ClassEnum, bool, string, string)
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)

def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category="", description="")

def _add_property_row(db, EnumId: int, MembershipId: int, BandId: int, Value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None, Pattern=None,
                      Scenario=None, Action=None):
    # Database.AddProperty(MembershipId, EnumId, BandId, Value, DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action)
    db.AddProperty(
        int(MembershipId),
        int(EnumId),
        int(BandId),
        float(Value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )

def _find_battery_node_collection(CollectionEnum) -> Optional[Any]:
    candidates = [
        "NodeBatteries", "BatteryNodes", "NodeBattery", "BatteryNode",
        "NodeStorages", "StorageNodes", "NodeStorage", "StoragesNodes",
        "NodeStorageUnits"
    ]
    for name in candidates:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return None

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> int:
    last_exc = None
    candidates = []
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e

        # Try to add then fetch
        try:
            try:
                db.AddMembership(col_enum, parent_name, child_name)
            except Exception:
                db.AddMembership(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name))
            try:
                return int(db.GetMembershipID(col_enum, parent_name, child_name))
            except Exception:
                return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(
        f"_ensure_membership_robust: failed membership parent={parent_name}, child={child_name}, "
        f"collection_raw={collection_enum_raw}. Last error: {last_exc}"
    )

def _ensure_membership_bi(db, CollectionEnum, SystemNS, collection_enum, a: str, b: str) -> bool:
    """
    Best-effort bi-directional membership. Some collections are directional; this is safe.
    Returns True if at least one direction ensured.
    """
    ok = False
    try:
        _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum, a, b)
        ok = True
    except Exception:
        pass
    try:
        _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum, b, a)
        ok = True
    except Exception:
        pass
    return ok

# -----------------------------
# Implementation
# -----------------------------
def step_oom_batteries_import():
    allowed = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    # Read iDSR ratios
    rdelim = _detect_csv_delimiter(IDSR_RATIO_CSV)
    rdf = pd.read_csv(IDSR_RATIO_CSV, sep=rdelim, engine="python", dtype=str).fillna("")
    if COL_RATIO_BAT not in rdf.columns:
        raise RuntimeError(f"iDSR_ratio.csv missing required column: {COL_RATIO_BAT}\nFound: {list(rdf.columns)}")

    ratio_map: Dict[Tuple[str, int], float] = {}
    for _, row in rdf.iterrows():
        node = _clean_cell(row.get("MARKET_NODE", ""))
        y_raw = _clean_cell(row.get("TARGET_YEAR", ""))
        try:
            y = int(float(y_raw))
        except Exception:
            continue
        ratio = _to_float_safe(row.get(COL_RATIO_BAT, "0"))
        ratio_map[(node, y)] = ratio

    # Read Batteries additional information
    df = pd.read_csv(BATTERIES_CSV)

    required = [COL_NODE, COL_OP_STAT, COL_MAX_POWER, COL_CAPACITY, COL_YEAR, COL_DATA_VERSION]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in Batteries CSV: {missing}\nFound: {list(df.columns)}")

    # Normalize and parse
    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_OP_STAT] = df[COL_OP_STAT].astype(str).map(_clean_cell)
    df[COL_DATA_VERSION] = df[COL_DATA_VERSION].astype(str).map(_clean_cell)
    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_MAX_POWER] = df[COL_MAX_POWER].apply(_as_float)
    df[COL_CAPACITY] = df[COL_CAPACITY].apply(_as_float)

    # Filters
    df = df[
        (df[COL_DATA_VERSION] == FILTER_DATA_VERSION) &
        (df[COL_OP_STAT] == FILTER_OPERATION_STATUS) &
        (df[COL_NODE].isin(allowed)) &
        (df[COL_YEAR].notna())
    ].copy()

    if df.empty:
        print("[STEP] No rows remain after filters. Nothing to import.")
        return

    # Add ratio and scale
    def _lookup_ratio(r):
        node = r[COL_NODE]
        y = int(r[COL_YEAR])
        return float(ratio_map.get((node, y), 0.0))

    df["__ratio_bat__"] = df.apply(_lookup_ratio, axis=1)

    # Optionally skip rows with ratio <= 0
    if not WRITE_ZERO_WHEN_RATIO_MISSING_OR_ZERO:
        df = df[df["__ratio_bat__"] > 0].copy()
        if df.empty:
            print("[STEP] All rows had ratio<=0 (or missing). Nothing to import (per WRITE_ZERO...=False).")
            return

    df["__max_power_scaled__"] = df[COL_MAX_POWER].fillna(0.0) * df["__ratio_bat__"]
    df["__capacity_scaled__"] = df[COL_CAPACITY].fillna(0.0) * df["__ratio_bat__"]

    # Aggregate duplicates per (node, year) by SUM (scaled values)
    df_g = (
        df.groupby([COL_NODE, COL_YEAR], as_index=False)
          .agg({"__max_power_scaled__": "sum", "__capacity_scaled__": "sum", "__ratio_bat__": "max"})
          .sort_values([COL_NODE, COL_YEAR])
    )

    print("\n[STEP] Preview (first ~10 zone-year rows after filters + scaling + aggregation):")
    try:
        display(df_g.head(10))
    except Exception:
        print(df_g.head(10).to_string(index=False))

    # Connect PLEXOS
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    scenario_str = SystemNS.String(SCENARIO_NAME)
    _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)

    # Best-effort find node<->battery collection
    bat_node_collection = _find_battery_node_collection(CollectionEnum)

    nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
    try:
        batteries_in_db = set(_get_objects_safe(db, ClassEnum.Battery))
    except Exception:
        batteries_in_db = set()

    created_batteries = 0
    memberships_ensured = 0
    node_links = 0
    props_written = 0
    warnings: List[str] = []

    try:
        sys_bat_collection_raw = SYSTEM_BATTERY_COLLECTION_ID

        zone_names = sorted(df_g[COL_NODE].unique().tolist())
        print(f"\n[STEP] OOM Batteries to import (one per MARKET_NODE): {len(zone_names)}")
        print("Sample:", zone_names[:20])

        for zone in zone_names:
            batt_name = f"{zone}_OOM_Battery"

            # Create battery object if missing
            if batt_name not in batteries_in_db:
                try:
                    _add_object(db, ClassEnum.Battery, batt_name, add_to_system=True, category="", description="")
                    batteries_in_db.add(batt_name)
                    created_batteries += 1
                except Exception as e:
                    warnings.append(f"Failed to create Battery '{batt_name}': {e}")
                    continue

            # Ensure System->Battery membership
            try:
                mem_id = _ensure_membership_robust(
                    db, CollectionEnum, SystemNS,
                    sys_bat_collection_raw,
                    "System",
                    batt_name
                )
                memberships_ensured += 1
            except Exception as e:
                warnings.append(f"Failed System->Battery membership for '{batt_name}': {e}")
                mem_id = None

            # Static props once
            if mem_id is not None:
                try:
                    _add_property_row(db, ENUM_UNITS, mem_id, 1, 1.0, Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Units write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_INITIAL_SOC, mem_id, 1, float(INITIAL_SOC_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Initial SoC write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_CHARGE_EFF, mem_id, 1, float(CHARGE_EFFICIENCY_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Charge Efficiency write for '{batt_name}': {e}")

                try:
                    _add_property_row(db, ENUM_DISCHARGE_EFF, mem_id, 1, float(DISCHARGE_EFFICIENCY_PERCENT), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Discharge Efficiency write for '{batt_name}': {e}")
            else:
                warnings.append(f"No valid System->Battery membership for '{batt_name}'; skipping static property writes.")

            # Node coupling (best effort)
            if bat_node_collection is not None:
                node_name = zone
                if node_name in nodes_in_db:
                    try:
                        if _ensure_membership_bi(db, CollectionEnum, SystemNS, bat_node_collection, node_name, batt_name):
                            node_links += 1
                    except Exception as e:
                        warnings.append(f"Failed Battery<->Node link for '{batt_name}' <-> '{node_name}': {e}")
                else:
                    warnings.append(f"Node '{node_name}' not found in DB; cannot couple Battery '{batt_name}' to Node.")
            else:
                warnings.append("Battery<->Node collection enum not found; node coupling skipped for OOM batteries.")

            # Year-specific properties (scaled)
            if mem_id is None:
                continue

            sub = df_g[df_g[COL_NODE] == zone]
            for _, r in sub.iterrows():
                y = int(r[COL_YEAR])
                dt_from = NetDateTime(y, 1, 1, 0, 0, 0)

                max_p = _round_1dp(r["__max_power_scaled__"])
                cap = _round_1dp(r["__capacity_scaled__"])

                if (max_p is None or max_p == 0.0) and (cap is None or cap == 0.0) and not WRITE_ZERO_WHEN_RATIO_MISSING_OR_ZERO:
                    continue

                if max_p is not None:
                    try:
                        _add_property_row(db, ENUM_MAX_POWER, mem_id, 1, float(max_p), DateFrom=dt_from, Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Max Power write for '{batt_name}' year={y}: {e}")

                if cap is not None:
                    try:
                        _add_property_row(db, ENUM_CAPACITY, mem_id, 1, float(cap), DateFrom=dt_from, Scenario=scenario_str)
                        props_written += 1
                    except Exception as e:
                        warnings.append(f"Failed Capacity write for '{batt_name}' year={y}: {e}")

        # Summary
        print("\n[STEP] Done.")
        print(f"  Created OOM Battery objects:      {created_batteries}")
        print(f"  System->Battery ensured:          {memberships_ensured}")
        print(f"  Node links created (bi-dir):      {node_links}")
        print(f"  Properties written:               {props_written}")
        print(f"  Zone-year rows processed:         {len(df_g)}")

        if warnings:
            print("\n[STEP] Warnings (first 80):")
            for w in warnings[:80]:
                print("  - " + w)
            if len(warnings) > 80:
                print(f"  ... and {len(warnings) - 80} more")
        else:
            print("\n[STEP] No warnings.")
    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_oom_batteries_import()




In [ ]:
# ============================================================
# STEP â€” iDSR Batteries import from GenerationCapacities.csv (FULL copy-pasteable script)
# Updated tech mapping:
#   EV technologies  == "EV - iDSR"
#   HP technologies  == "HP - iDSR"
# ============================================================

from pathlib import Path
from typing import Optional, Set, List, Any, Dict, Tuple
import os, csv
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
GEN_CAP_CSV = DASHBOARD_RAWDATA_DIR / "GenerationCapacities.csv"
IDSR_RATIO_CSV = DEMAND_DATA_DIR / "iDSR_ratio.csv"

FILTER_DATA_VERSION = "ERAA 2025 final"
SCENARIO_NAME = "Battery_Adjusted_Demand"

# -----------------------------
# iDSR ratio columns
# -----------------------------
COL_RATIO_EV = "Ratio of price-sensitive consumer - Evs"
COL_RATIO_HP = "Ratio of price-sensitive consumer - Heat Pumps"

# -----------------------------
# PLEXOS Property Enums
# -----------------------------
ENUM_RECHARGE_TIMEFRAME = 5     # hours
ENUM_UNITS = 25
ENUM_CAPACITY = 26
ENUM_MAX_POWER = 28
ENUM_INITIAL_SOC = 32
ENUM_CHARGE_EFF = 33
ENUM_DISCHARGE_EFF = 34

# Static values
UNITS_VALUE = 1.0
CAPACITY_ARBITRARY = 100000.0
INITIAL_SOC_PERCENT = 50.0
CHARGE_EFF_PERCENT = 100.0
DISCHARGE_EFF_PERCENT = 100.0
RECHARGE_TIMEFRAME_HOURS = 6.0

# System -> Battery collection id
SYSTEM_BATTERY_COLLECTION_ID = 81

# If True, writes Max Power even if scaled value == 0 (not recommended)
WRITE_ZERO_MAX_POWER = False

# -----------------------------
# CSV helpers
# -----------------------------
def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:8192]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","

def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _to_float_safe(x):
    try:
        return float(str(x).strip())
    except Exception:
        return 0.0

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals

def _pick_col(df: pd.DataFrame, candidates: List[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    raise RuntimeError(f"Could not find any of columns {candidates} in file. Found columns: {list(df.columns)}")

def _classify_tech_to_type(tech: str) -> Optional[str]:
    """
    EXACT mapping per your spec:
      - EV technologies are "EV - iDSR"
      - HP technologies are "HP - iDSR"
    (case/whitespace tolerant)
    """
    t = _clean_cell(tech).lower()
    if t == "ev - idsr":
        return "EV"
    if t == "hp - idsr":
        return "HP"
    return None

# -----------------------------
# PLEXOS bootstrap + minimal helpers (standalone)
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime

def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []

def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)

def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category="", description="")

def _add_property_row(db, EnumId: int, MembershipId: int, BandId: int, Value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None, Pattern=None,
                      Scenario=None, Action=None):
    db.AddProperty(
        int(MembershipId),
        int(EnumId),
        int(BandId),
        float(Value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )

def _find_battery_node_collection(CollectionEnum) -> Optional[Any]:
    candidates = [
        "NodeBatteries", "BatteryNodes", "NodeBattery", "BatteryNode",
        "NodeStorages", "StorageNodes", "NodeStorage", "StoragesNodes",
        "NodeStorageUnits"
    ]
    for name in candidates:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return None

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> int:
    last_exc = None
    candidates = []
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception as e:
            last_exc = e
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e

        try:
            try:
                db.AddMembership(col_enum, parent_name, child_name)
            except Exception:
                db.AddMembership(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name))
            try:
                return int(db.GetMembershipID(col_enum, parent_name, child_name))
            except Exception:
                return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception as e:
            last_exc = e
            continue

    raise RuntimeError(
        f"_ensure_membership_robust: failed membership parent={parent_name}, child={child_name}, "
        f"collection_raw={collection_enum_raw}. Last error: {last_exc}"
    )

def _ensure_membership_bi(db, CollectionEnum, SystemNS, collection_enum, a: str, b: str) -> bool:
    ok = False
    try:
        _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum, a, b)
        ok = True
    except Exception:
        pass
    try:
        _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_enum, b, a)
        ok = True
    except Exception:
        pass
    return ok

# -----------------------------
# STEP implementation
# -----------------------------
def step_idsr_batteries_from_generationcapacities():
    allowed = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    # Read iDSR ratios
    rdelim = _detect_csv_delimiter(IDSR_RATIO_CSV)
    rdf = pd.read_csv(IDSR_RATIO_CSV, sep=rdelim, engine="python", dtype=str).fillna("")
    for col in ["MARKET_NODE", "TARGET_YEAR", COL_RATIO_EV, COL_RATIO_HP]:
        if col not in rdf.columns:
            raise RuntimeError(f"iDSR_ratio.csv missing required column: {col}\nFound: {list(rdf.columns)}")

    ratio_ev: Dict[Tuple[str, int], float] = {}
    ratio_hp: Dict[Tuple[str, int], float] = {}
    for _, row in rdf.iterrows():
        node = _clean_cell(row.get("MARKET_NODE", ""))
        y_raw = _clean_cell(row.get("TARGET_YEAR", ""))
        try:
            y = int(float(y_raw))
        except Exception:
            continue
        ratio_ev[(node, y)] = _to_float_safe(row.get(COL_RATIO_EV, "0"))
        ratio_hp[(node, y)] = _to_float_safe(row.get(COL_RATIO_HP, "0"))

    # Read GenerationCapacities.csv
    gdelim = _detect_csv_delimiter(GEN_CAP_CSV)
    gdf = pd.read_csv(GEN_CAP_CSV, sep=gdelim, engine="python")

    # Resolve columns robustly
    col_data_version = _pick_col(gdf, ["data_version", "Data_Version", "DATA_VERSION"])
    col_target_year  = _pick_col(gdf, ["Target year", "TARGET_YEAR", "Target Year", "Year"])
    col_market_node  = _pick_col(gdf, ["Market_Node", "MARKET_NODE", "Market Node", "Node", "Bidding Zone"])
    col_technology   = _pick_col(gdf, ["Technology", "TECHNOLOGY", "Tech"])
    col_value        = _pick_col(gdf, ["Value", "VALUE"])

    # Clean and filter
    gdf[col_data_version] = gdf[col_data_version].map(_clean_cell)
    gdf[col_market_node]  = gdf[col_market_node].map(_clean_cell)
    gdf[col_technology]   = gdf[col_technology].map(_clean_cell)
    gdf[col_target_year]  = pd.to_numeric(gdf[col_target_year], errors="coerce")
    gdf[col_value]        = gdf[col_value].apply(_as_float)

    gdf = gdf[
        (gdf[col_data_version] == FILTER_DATA_VERSION) &
        (gdf[col_market_node].isin(allowed)) &
        (gdf[col_target_year].notna()) &
        (gdf[col_value].notna())
    ].copy()

    if gdf.empty:
        print("[STEP] No rows remain after filters (data_version/allowed nodes/year/value). Nothing to import.")
        return

    # Classify tech to EV/HP (exact names)
    gdf["__type__"] = gdf[col_technology].apply(_classify_tech_to_type)
    gdf = gdf[gdf["__type__"].notna()].copy()
    if gdf.empty:
        print("[STEP] No rows matched Technology == 'EV - iDSR' or 'HP - iDSR'. Nothing to import.")
        return

    # Attach ratios and scale Value -> Max Power
    def _ratio_for_row(r) -> float:
        node = r[col_market_node]
        y = int(r[col_target_year])
        t = r["__type__"]
        return float(ratio_ev.get((node, y), 0.0)) if t == "EV" else float(ratio_hp.get((node, y), 0.0))

    gdf["__ratio__"] = gdf.apply(_ratio_for_row, axis=1)
    gdf = gdf[gdf["__ratio__"] > 0].copy()  # ONLY import when relevant ratio > 0
    if gdf.empty:
        print("[STEP] All EV/HP rows had ratio<=0 (or missing). Nothing to import.")
        return

    gdf["__max_power_scaled__"] = gdf[col_value].astype(float) * gdf["__ratio__"].astype(float)
    if not WRITE_ZERO_MAX_POWER:
        gdf = gdf[gdf["__max_power_scaled__"] > 0].copy()
        if gdf.empty:
            print("[STEP] All scaled Max Power values were 0. Nothing to import (WRITE_ZERO_MAX_POWER=False).")
            return

    # Aggregate duplicates per (node, year, type) by sum
    gdf_g = (
        gdf.groupby([col_market_node, col_target_year, "__type__"], as_index=False)
           .agg({"__max_power_scaled__": "sum", "__ratio__": "max"})
           .sort_values([col_market_node, col_target_year, "__type__"])
    )

    print("\n[STEP] Preview (first ~15 rows after filters + scaling + aggregation):")
    try:
        display(gdf_g.head(15))
    except Exception:
        print(gdf_g.head(15).to_string(index=False))

    # Connect PLEXOS
    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    scenario_str = SystemNS.String(SCENARIO_NAME)
    _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)

    bat_node_collection = _find_battery_node_collection(CollectionEnum)

    nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
    try:
        batteries_in_db = set(_get_objects_safe(db, ClassEnum.Battery))
    except Exception:
        batteries_in_db = set()

    created_batteries = 0
    memberships_ensured = 0
    node_links = 0
    props_written = 0
    warnings: List[str] = []

    try:
        sys_bat_collection_raw = SYSTEM_BATTERY_COLLECTION_ID

        pairs = sorted({(r[col_market_node], r["__type__"]) for _, r in gdf_g.iterrows()})
        print(f"\n[STEP] iDSR batteries to import (node,type pairs): {len(pairs)}")
        print("Sample:", pairs[:20])

        for node, typ in pairs:
            suffix = "EV" if typ == "EV" else "HP"
            batt_name = f"{node}_iDSR_{suffix}"

            # Create battery object if missing
            if batt_name not in batteries_in_db:
                try:
                    _add_object(db, ClassEnum.Battery, batt_name, add_to_system=True, category="", description="")
                    batteries_in_db.add(batt_name)
                    created_batteries += 1
                except Exception as e:
                    warnings.append(f"Failed to create Battery '{batt_name}': {e}")
                    continue

            # Ensure System->Battery membership
            try:
                mem_id = _ensure_membership_robust(
                    db, CollectionEnum, SystemNS,
                    sys_bat_collection_raw,
                    "System",
                    batt_name
                )
                memberships_ensured += 1
            except Exception as e:
                warnings.append(f"Failed System->Battery membership for '{batt_name}': {e}")
                mem_id = None

            if mem_id is None:
                continue

            # Static props once (Scenario applied)
            def _w(enum_id: int, value: float, label: str):
                nonlocal props_written
                try:
                    _add_property_row(db, enum_id, mem_id, 1, float(value), Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed {label} write for '{batt_name}': {e}")

            _w(ENUM_UNITS, UNITS_VALUE, "Units")
            _w(ENUM_CAPACITY, CAPACITY_ARBITRARY, "Capacity")
            _w(ENUM_INITIAL_SOC, INITIAL_SOC_PERCENT, "Initial SoC")
            _w(ENUM_CHARGE_EFF, CHARGE_EFF_PERCENT, "Charge Eff")
            _w(ENUM_DISCHARGE_EFF, DISCHARGE_EFF_PERCENT, "Discharge Eff")
            _w(ENUM_RECHARGE_TIMEFRAME, RECHARGE_TIMEFRAME_HOURS, "Recharge Timeframe")

            # Node coupling (best effort)
            if bat_node_collection is not None and node in nodes_in_db:
                try:
                    if _ensure_membership_bi(db, CollectionEnum, SystemNS, bat_node_collection, node, batt_name):
                        node_links += 1
                except Exception as e:
                    warnings.append(f"Failed Battery<->Node link for '{batt_name}' <-> '{node}': {e}")

            # Year-specific Max Power rows
            sub = gdf_g[(gdf_g[col_market_node] == node) & (gdf_g["__type__"] == typ)]
            for _, rr in sub.iterrows():
                y = int(rr[col_target_year])
                dt_from = NetDateTime(y, 1, 1, 0, 0, 0)

                mp = float(rr["__max_power_scaled__"])
                if (mp == 0.0) and (not WRITE_ZERO_MAX_POWER):
                    continue
                try:
                    _add_property_row(db, ENUM_MAX_POWER, mem_id, 1, mp, DateFrom=dt_from, Scenario=scenario_str)
                    props_written += 1
                except Exception as e:
                    warnings.append(f"Failed Max Power write for '{batt_name}' year={y}: {e}")

        # Summary
        print("\n[STEP] Done.")
        print(f"  Created iDSR Battery objects:     {created_batteries}")
        print(f"  System->Battery ensured:          {memberships_ensured}")
        print(f"  Node links created (bi-dir):      {node_links}")
        print(f"  Properties written:               {props_written}")
        print(f"  Zone-year-type rows processed:    {len(gdf_g)}")

        if warnings:
            print("\n[STEP] Warnings (first 80):")
            for w in warnings[:80]:
                print("  - " + w)
            if len(warnings) > 80:
                print(f"  ... and {len(warnings) - 80} more")
        else:
            print("\n[STEP] No warnings.")
    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_idsr_batteries_from_generationcapacities()




#### 11. PECD Renewables + Demand Category Creation

In [ ]:
import os, csv, re, shutil
from pathlib import Path
import pandas as pd

# ============================================================
# STEP â€” PECD Capacity Factors -> Final PECD Time Series CSVs
#         + PLEXOS Variables + Generator Rating Factor (Enum 83)
#         + Variable Categories (Solar / Wind Onshore / Wind Offshore)
#
# CHANGE (ONLY, per your latest instruction in THIS message):
#   - The "8760 vs 8784 row mismatch auto-fix" is ONLY allowed for TARGET YEAR 2028.
#     * If (year==2028) AND (template rows == 8784) AND (PECD WS rows == 8760):
#         - append (duplicate) the FIRST 24 PECD rows to the END (rows 8761..8784)
#         - print a WARNING that the PECD file only had 8760 rows and initial-day values were used
#   - For ALL OTHER years:
#       * It is correct that WS has 8760 rows (non-leap years)
#       * If the chosen template happens to be 8784 rows, we must trim Feb 29 to get 8760
#       * No "append 24 rows" behavior is applied.
#
# Existing behavior kept:
#   - Removed ALL "Demand" category creation + Demand categorization logic.
#   - For target year 2028 (leap year), insert Feb 29 (24 rows) after Feb 28 Period 24,
#     duplicating Feb 28 values for all bands (1..36).
#   - "Wind offshore floating" generators reuse the "Wind offshore fixed" PECD variable/file.
#   - Support bidding zones that contain underscores (e.g. *_OFF) in parsing/matching.
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
PECD_SRC_DIR = PECD_RES_DIR / "Capacity Factors_250716"
PECD_TECH_MAP_XLSX = BASE_DIR / "PECD_Technology_Mapping.xlsx"

DEMAND_TEMPLATE_DIR = DATA_FILES_ROOT / "Final Demand Time Series"

FINAL_PECD_DIR = DATA_FILES_ROOT / "Final PECD Time Series"
PLEXOS_REL_PREFIX = r"Data Files\Final PECD Time Series"


CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

SCENARIO_NAME = "36_WS"
OVERWRITE_FINAL_FILES = True

RATING_FACTOR_ENUM = 84

CAT_SOLAR = "Solar"
CAT_WIND_ON = "Wind Onshore"
CAT_WIND_OFF = "Wind Offshore"


# -----------------------------
# HELPERS: Input parsing
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _read_pecd_tech_mapping(xlsx_path: Path) -> dict[str, str]:
    df = pd.read_excel(xlsx_path, sheet_name=0)
    if df.shape[1] < 2:
        raise RuntimeError("PECD_Technology_Mapping.xlsx must have at least two columns (A=PECD tech, B=PLEXOS tech).")
    a = df.columns[0]
    b = df.columns[1]
    out = {}
    for _, row in df.iterrows():
        k = str(row[a]).strip() if not pd.isna(row[a]) else ""
        v = str(row[b]).strip() if not pd.isna(row[b]) else ""
        if k and v:
            out[k] = v
    return out


def _parse_pecd_filename(name: str):
    """
    allow bidding-zone tokens containing underscores, e.g. DKKF_OFF.
    Expected: <BIDDING_ZONE>_CapacityFactors_<TECH>_<YEAR>.csv
    """
    m = re.match(r"^(.+?)_CapacityFactors_(.+)_(\d{4})\.csv$", name)
    if not m:
        return None, None, None
    bz = m.group(1).strip()
    tech = m.group(2).strip()
    year = int(m.group(3))
    return bz, tech, year


def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:4096]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _load_demand_template():
    candidates = sorted(DEMAND_TEMPLATE_DIR.glob("*.csv"))
    if not candidates:
        raise RuntimeError(f"No template CSVs found in DEMAND_TEMPLATE_DIR: {DEMAND_TEMPLATE_DIR}")
    tmpl = candidates[0]
    delim = _detect_csv_delimiter(tmpl)
    df = pd.read_csv(tmpl, sep=delim, engine="python")

    cols_lower = {c.lower(): c for c in df.columns}
    required = ["year", "month", "day", "period"]
    missing = [c for c in required if c not in cols_lower]
    if missing:
        raise RuntimeError(
            f"Template CSV {tmpl.name} must contain columns {required}.\n"
            f"Found columns: {list(df.columns)}"
        )

    for i in range(1, 37):
        if str(i) not in df.columns:
            raise RuntimeError(f"Template CSV {tmpl.name} missing band column '{i}'")
    return df, delim


def _read_pecd_ws_matrix(path: Path) -> pd.DataFrame:
    delim = _detect_csv_delimiter(path)

    try:
        dfh = pd.read_csv(path, sep=delim, engine="python", header=0)
        cols = list(dfh.columns)
        if all(f"WS{i:02d}" in cols for i in range(1, 37)):
            df_slice = dfh.iloc[10:10 + 8760].copy()
            if df_slice.shape[0] != 8760:
                raise RuntimeError(f"{path.name}: expected 8760 data rows after slicing, got {df_slice.shape[0]}")
            out = pd.DataFrame({str(i): pd.to_numeric(df_slice[f"WS{i:02d}"], errors="coerce") for i in range(1, 37)})
            if out.isna().any().any():
                bad = int(out.isna().sum().sum())
                raise RuntimeError(f"{path.name}: {bad} non-numeric/blank values found in WS columns slice.")
            return out * 100.0
    except Exception:
        pass

    dfr = pd.read_csv(path, sep=delim, engine="python", header=None)
    dfr_slice = dfr.iloc[11:11 + 8760, :].copy()
    if dfr_slice.shape[0] != 8760:
        raise RuntimeError(f"{path.name}: expected 8760 rows from 12..8771, got {dfr_slice.shape[0]}")

    start_col = 2
    end_col_exclusive = start_col + 36
    if dfr_slice.shape[1] < end_col_exclusive:
        raise RuntimeError(
            f"{path.name}: expected at least {end_col_exclusive} columns to read WS01..WS36, "
            f"but found {dfr_slice.shape[1]} columns."
        )

    out = pd.DataFrame()
    for i in range(1, 37):
        src_col = start_col + (i - 1)
        out[str(i)] = pd.to_numeric(dfr_slice.iloc[:, src_col], errors="coerce")

    if out.isna().any().any():
        bad = int(out.isna().sum().sum())
        raise RuntimeError(f"{path.name}: {bad} non-numeric/blank values found in assumed WS columns slice.")

    return out * 100.0


def _category_for_variable_name(var_name: str) -> str:
    s = var_name.lower()
    if ("solar" in s) or ("pv" in s) or ("csp" in s) or ("thermal" in s):
        return CAT_SOLAR
    if "onshore" in s:
        return CAT_WIND_ON
    if "offshore" in s:
        return CAT_WIND_OFF
    return ""


def _insert_leap_day_2028(df: pd.DataFrame) -> pd.DataFrame:
    """
    Insert Feb 29 (24 rows) right after Feb 28 Period 24, duplicating Feb 28 values.
    Expects columns Year/Month/Day/Period (case-insensitive) and band columns "1".."36".
    """
    cols_lower = {c.lower(): c for c in df.columns}
    ycol = cols_lower.get("year")
    mcol = cols_lower.get("month")
    dcol = cols_lower.get("day")
    pcol = cols_lower.get("period")
    if not all([ycol, mcol, dcol, pcol]):
        raise RuntimeError("Template/out DF missing one of required columns: Year, Month, Day, Period")

    mask_28 = (df[mcol] == 2) & (df[dcol] == 28)
    idx_28 = df.index[mask_28].tolist()
    if len(idx_28) < 24:
        raise RuntimeError("Could not find 24 rows for Feb 28 in the template-based time series.")

    feb28_rows = df.loc[idx_28].copy()
    feb28_rows = feb28_rows.sort_values(by=[pcol]).head(24)

    feb29_rows = feb28_rows.copy()
    feb29_rows[ycol] = 2028
    feb29_rows[mcol] = 2
    feb29_rows[dcol] = 29
    feb29_rows[pcol] = list(range(1, 25))

    mask_28_p24 = mask_28 & (df[pcol] == 24)
    idx_insert_after = df.index[mask_28_p24].tolist()
    if not idx_insert_after:
        insert_pos = max(idx_28)
        left = df.loc[:insert_pos].copy()
        right = df.loc[insert_pos + 1:].copy()
        out = pd.concat([left, feb29_rows, right], ignore_index=True)
        return out

    insert_pos = idx_insert_after[-1]
    left = df.loc[:insert_pos].copy()
    right = df.loc[insert_pos + 1:].copy()
    out = pd.concat([left, feb29_rows, right], ignore_index=True)
    return out


# -----------------------------
# NEW: Make a per-year template view (trim Feb 29 for non-2028 if template is 8784)
# -----------------------------
def _template_for_year(template_df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    If template has 8784 rows (includes Feb 29) but year is NOT 2028,
    trim Feb 29 rows to produce a 8760-row template for non-leap years.

    If year is 2028, return template unchanged (may be 8760 or 8784).
    """
    base = template_df.copy()
    if int(year) == 2028:
        return base

    if int(base.shape[0]) == 8784:
        cols_lower = {c.lower(): c for c in base.columns}
        mcol = cols_lower.get("month")
        dcol = cols_lower.get("day")
        if not (mcol and dcol):
            return base  # cannot safely trim
        mask_feb29 = (base[mcol] == 2) & (base[dcol] == 29)
        trimmed = base.loc[~mask_feb29].copy()
        if int(trimmed.shape[0]) != 8760:
            # If something odd, keep original to avoid silent corruption
            return base
        return trimmed.reset_index(drop=True)

    return base


# -----------------------------
# UPDATED: Handle template vs WS row mismatches
#   - The special "append 24 rows" fix is ONLY for 2028
# -----------------------------
def _align_ws_to_template_rows(ws_mat: pd.DataFrame, template_rows: int, *, src_name: str, year: int) -> pd.DataFrame:
    """
    If ws rows != template rows:
      - ONLY if (year==2028) and (template_rows==8784) and (ws_rows==8760):
          append first 24 ws rows to end (warn)
      - If template_rows == 8760 and ws_rows == 8784: truncate to first 8760 (warn)
      - Else best-effort:
          * if ws_rows > template_rows: truncate (warn)
          * if ws_rows < template_rows: repeat first rows until match (warn)
    """
    ws_rows = int(ws_mat.shape[0])
    if ws_rows == template_rows:
        return ws_mat

    if int(year) == 2028 and template_rows == 8784 and ws_rows == 8760:
        print(
            f"WARNING: Row mismatch for {src_name} (year={year}): template has 8784 rows but PECD WS matrix has 8760 rows. "
            f"Appending first 24 rows to bottom (8761..8784) using initial-year values."
        )
        extra = ws_mat.iloc[:24, :].copy()
        out = pd.concat([ws_mat, extra], ignore_index=True)
        return out

    if template_rows == 8760 and ws_rows == 8784:
        print(
            f"WARNING: Row mismatch for {src_name} (year={year}): template has 8760 rows but PECD WS matrix has 8784 rows. "
            f"Truncating to first 8760 rows."
        )
        return ws_mat.iloc[:8760, :].copy()

    if ws_rows > template_rows:
        print(
            f"WARNING: Row mismatch for {src_name} (year={year}): template has {template_rows} rows but PECD WS matrix has {ws_rows} rows. "
            f"Truncating to {template_rows} rows."
        )
        return ws_mat.iloc[:template_rows, :].copy()

    # ws_rows < template_rows (generic fill)
    missing = template_rows - ws_rows
    print(
        f"WARNING: Row mismatch for {src_name} (year={year}): template has {template_rows} rows but PECD WS matrix has {ws_rows} rows. "
        f"Filling missing {missing} rows by repeating from the start of the year."
    )
    reps = (missing // ws_rows) + 1
    filler = pd.concat([ws_mat] * reps, ignore_index=True).iloc[:missing, :].copy()
    out = pd.concat([ws_mat, filler], ignore_index=True)
    return out


# -----------------------------
# NEW: Offshore floating reuse helper
# -----------------------------
def _looks_like_offshore_floating_generator(gen_name: str) -> bool:
    s = gen_name.lower()
    return ("offshore" in s) and ("float" in s)


def _pick_offshore_fixed_token(tech_map: dict[str, str]) -> str | None:
    """
    Find which PECD tech token in tech_map corresponds to *offshore fixed* in PLEXOS naming.
    We will reuse that token's variable/file for offshore floating generators.
    """
    for k, v in tech_map.items():
        vs = str(v).lower()
        if ("offshore" in vs) and ("wind" in vs) and ("fixed" in vs) and ("float" not in vs):
            return k

    for k, v in tech_map.items():
        vs = str(v).lower()
        if ("offshore" in vs) and ("wind" in vs) and ("float" not in vs):
            return k

    for k in tech_map.keys():
        ks = str(k).lower()
        if ("offshore" in ks) and ("float" not in ks):
            return k

    return None


# -----------------------------
# UPDATED: Generator -> bidding-zone extraction (supports underscores in BZ)
# -----------------------------
def _build_gens_by_bz(generators: list[str], allowed_bzs: set[str]) -> dict[str, list[str]]:
    """
    match the LONGEST bidding zone prefix (from Bidding_Zone_List) at start of generator name,
    using either exact match or '<BZ>_' prefix.
    """
    bzs_sorted = sorted([bz for bz in allowed_bzs if bz], key=len, reverse=True)
    out: dict[str, list[str]] = {}
    for g in generators:
        g_str = str(g)
        matched_bz = None
        for bz in bzs_sorted:
            if g_str == bz or g_str.startswith(bz + "_"):
                matched_bz = bz
                break
        if matched_bz is None:
            continue
        out.setdefault(matched_bz, []).append(g_str)
    return out


# -----------------------------
# PLEXOS_NET LOADER + HELPERS
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_category_for_class(db, ClassEnum, SystemNS, class_enum_value, category_name: str):
    if not category_name:
        return

    if hasattr(db, "AddCategory"):
        fn = getattr(db, "AddCategory")
        attempts = [
            lambda: fn(class_enum_value, category_name),
            lambda: fn(int(class_enum_value), category_name),
            lambda: fn(category_name, class_enum_value),
            lambda: fn(category_name, int(class_enum_value)),
            lambda: fn(SystemNS.String("Variable"), SystemNS.String(category_name)),
            lambda: fn(SystemNS.String(category_name), SystemNS.String("Variable")),
        ]
        for a in attempts:
            try:
                a()
                return
            except Exception:
                pass

    if hasattr(ClassEnum, "Category"):
        try:
            _add_object(db, getattr(ClassEnum, "Category"), category_name, add_to_system=True, category="", description="")
            return
        except Exception:
            pass

    print(f"WARNING: Could not pre-create category '{category_name}' for this PLEXOS API build.")


def _get_object_id_best_effort(db, class_enum_value, name: str):
    for fn_name, attempts in [
        ("GetObjectID", [
            lambda fn: int(fn(class_enum_value, name)),
            lambda fn: int(fn(int(class_enum_value), name)),
            lambda fn: int(fn(name, class_enum_value)),
            lambda fn: int(fn(name, int(class_enum_value))),
            lambda fn: int(fn(name)),
        ]),
        ("ObjectName2ID", [
            lambda fn: int(fn(class_enum_value, name)),
            lambda fn: int(fn(int(class_enum_value), name)),
            lambda fn: int(fn(name, class_enum_value)),
            lambda fn: int(fn(name, int(class_enum_value))),
        ]),
        ("GetObjectId", [
            lambda fn: int(fn(class_enum_value, name)),
            lambda fn: int(fn(int(class_enum_value), name)),
            lambda fn: int(fn(name, class_enum_value)),
            lambda fn: int(fn(name, int(class_enum_value))),
            lambda fn: int(fn(name)),
        ]),
    ]:
        if hasattr(db, fn_name):
            fn = getattr(db, fn_name)
            for a in attempts:
                try:
                    return a(fn)
                except Exception:
                    pass
    return None


def _set_object_category_best_effort(db, SystemNS, class_enum_value, name: str, category: str) -> bool:
    if not category:
        return True

    obj_id = _get_object_id_best_effort(db, class_enum_value, name)
    class_int = None
    try:
        class_int = int(class_enum_value)
    except Exception:
        class_int = None

    str_vals = [name, category, ""]
    bool_vals = [True, False]
    int_vals = [v for v in [obj_id, class_int] if v is not None]

    if hasattr(db, "UpdateObject"):
        fn = getattr(db, "UpdateObject")
        explicit = [
            lambda: fn(name, class_enum_value, True, category, ""),
            lambda: fn(name, class_enum_value, True, category),
            lambda: fn(name, class_enum_value, category, ""),
            lambda: fn(name, class_enum_value, category),
        ]
        if class_int is not None:
            explicit += [
                lambda: fn(name, class_int, True, category, ""),
                lambda: fn(name, class_int, True, category),
                lambda: fn(name, class_int, category, ""),
                lambda: fn(name, class_int, category),
            ]
        if obj_id is not None:
            explicit += [
                lambda: fn(obj_id, class_enum_value, True, category, ""),
                lambda: fn(obj_id, class_enum_value, True, category),
                lambda: fn(obj_id, class_enum_value, category, ""),
                lambda: fn(obj_id, class_enum_value, category),
            ]
            if class_int is not None:
                explicit += [
                    lambda: fn(obj_id, class_int, True, category, ""),
                    lambda: fn(obj_id, class_int, True, category),
                    lambda: fn(obj_id, class_int, category, ""),
                    lambda: fn(obj_id, class_int, category),
                ]
        for a in explicit:
            try:
                a()
                return True
            except Exception:
                pass

    try:
        import System  # noqa
        from System.Reflection import BindingFlags
        t = db.GetType()
        methods = [m for m in t.GetMethods(BindingFlags.Public | BindingFlags.Instance)]
    except Exception:
        methods = []

    candidate_names = set(["UpdateObject", "UpdateObjectCategory", "SetObjectCategory", "SetCategory", "UpdateCategory"])

    for m in methods:
        if m.Name not in candidate_names:
            continue
        try:
            ps = list(m.GetParameters())
        except Exception:
            continue

        per_param_options = []
        ok = True
        for p in ps:
            pt = p.ParameterType
            tname = pt.FullName or str(pt)

            if "System.String" in tname:
                per_param_options.append(str_vals)
            elif "System.Boolean" in tname:
                per_param_options.append(bool_vals)
            elif ("System.Int32" in tname) or ("System.Int16" in tname) or ("System.Int64" in tname):
                if not int_vals:
                    ok = False
                    break
                per_param_options.append(int_vals)
            else:
                if "PLEXOS_NET.Enums.ClassEnum" in tname:
                    per_param_options.append([class_enum_value])
                elif "System.Object" in tname:
                    per_param_options.append([None])
                else:
                    ok = False
                    break
        if not ok:
            continue

        def _product(lists):
            if not lists:
                yield []
                return
            first, rest = lists[0], lists[1:]
            for v in first:
                for tail in _product(rest):
                    yield [v] + tail

        if len(per_param_options) > 6:
            continue

        tried = 0
        for args in _product(per_param_options):
            tried += 1
            if tried > 500:
                break
            if category not in [a for a in args if isinstance(a, str)]:
                continue
            if any(isinstance(a, str) for a in args) and (name not in [a for a in args if isinstance(a, str)]):
                continue
            try:
                m.Invoke(db, args)
                return True
            except Exception:
                continue

    return False


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _resolve_variable_profile_enum(db, SystemNS):
    prop_candidates = [
        "Profile", "Profiles", "Profile Data", "Profile File", "Profile Filename",
        "Profile File Name", "Profile Data File", "Profile DataFile",
        "Data File", "DataFile", "Filename", "File Name", "File", "Path", "Data",
        "Time Series",
    ]
    collection_candidates = ["Variables", "SystemVariables", "Variable"]

    last = None
    for col in collection_candidates:
        for prop in prop_candidates:
            try:
                enum_id = int(db.PropertyName2EnumId(
                    SystemNS.String("System"),
                    SystemNS.String("Variable"),
                    SystemNS.String(col),
                    SystemNS.String(prop),
                ))
                return enum_id, col, prop
            except Exception as e:
                last = (col, prop, str(e))
                continue

    raise RuntimeError(
        "Could not resolve a Variable profile/file property.\n"
        f"Last attempt: {last}\n"
        "Please confirm the exact property name shown in PLEXOS UI for Variable profile."
    )


def _find_system_generators_collection_enum(CollectionEnum):
    if hasattr(CollectionEnum, "SystemGenerators"):
        return getattr(CollectionEnum, "SystemGenerators")
    names = [n for n in dir(CollectionEnum) if not n.startswith("_")]
    candidates = [n for n in names if ("system" in n.lower() and "gen" in n.lower())]
    if not candidates:
        raise RuntimeError("Could not find System->Generators collection enum (SystemGenerators)")
    return getattr(CollectionEnum, candidates[0])


# -----------------------------
# MAIN STEP
# -----------------------------
def step_pecd_to_rating_factors():
    allowed_bzs = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed_bzs:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    tech_map = _read_pecd_tech_mapping(PECD_TECH_MAP_XLSX)
    if not tech_map:
        raise RuntimeError("No PECD technology mappings found in PECD_Technology_Mapping.xlsx")

    offshore_fixed_token = _pick_offshore_fixed_token(tech_map)
    if not offshore_fixed_token:
        print("WARNING: Could not identify an 'offshore fixed' token in PECD_Technology_Mapping.xlsx. "
              "Offshore floating generators will NOT be auto-covered.")
    else:
        print(f"Offshore floating generators will reuse offshore token: {offshore_fixed_token!r} -> {tech_map[offshore_fixed_token]!r}")

    template_df, template_delim = _load_demand_template()

    picked = []
    for p in sorted(PECD_SRC_DIR.glob("*.csv")):
        bz, tech_token, year = _parse_pecd_filename(p.name)
        if not bz or not tech_token or not year:
            continue
        if bz not in allowed_bzs:
            continue
        if tech_token not in tech_map:
            continue
        picked.append((p, bz, tech_token, year))

    if not picked:
        raise RuntimeError(f"No PECD files matched allowed bidding zones + mapping in: {PECD_SRC_DIR}")

    FINAL_PECD_DIR.mkdir(parents=True, exist_ok=True)

    copied = []
    for src, bz, tech_token, year in picked:
        dst = FINAL_PECD_DIR / src.name

        # Per-year template sizing:
        # - Non-2028: if template includes Feb29 (8784), trim it to 8760
        # - 2028: keep template as-is (may be 8760 or 8784)
        template_year_df = _template_for_year(template_df, int(year))
        template_rows = int(template_year_df.shape[0])

        ws_mat = _read_pecd_ws_matrix(src)

        # Row mismatch handling:
        # - The "append 24 rows" fix is ONLY applied for year==2028
        ws_mat = _align_ws_to_template_rows(ws_mat, template_rows, src_name=src.name, year=int(year))

        if int(ws_mat.shape[0]) != template_rows:
            raise RuntimeError(
                f"Row alignment failed for {src.name} (year={year}): "
                f"template has {template_rows} rows but aligned WS has {ws_mat.shape[0]} rows."
            )

        out = template_year_df.copy()
        cols_lower = {c.lower(): c for c in out.columns}
        out[cols_lower["year"]] = int(year)

        for i in range(1, 37):
            out[str(i)] = ws_mat[str(i)].values

        # Keep existing behavior: for 2028, insert Feb 29 by duplicating Feb 28 values.
        # (This covers cases where template_year_df was 8760 rows.)
        if int(year) == 2028 and int(out.shape[0]) == 8760:
            out = _insert_leap_day_2028(out)

        if dst.exists() and (not OVERWRITE_FINAL_FILES):
            pass
        out.to_csv(dst, index=False, sep=template_delim)
        copied.append((dst, bz, tech_token, year))

    print(f"Built {len(copied)} PECD demand-format CSVs into: {FINAL_PECD_DIR}")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    RATING_FACTOR_ENUM = _resolve_semantic_enum("System", "Generator", "Generators", "Rating Factor", 84)
    

    try:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)

        _ensure_category_for_class(db, ClassEnum, SystemNS, ClassEnum.Variable, CAT_SOLAR)
        _ensure_category_for_class(db, ClassEnum, SystemNS, ClassEnum.Variable, CAT_WIND_ON)
        _ensure_category_for_class(db, ClassEnum, SystemNS, ClassEnum.Variable, CAT_WIND_OFF)

        sampling_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Variable", "Variables", "Sampling Method"),
            ("System", "Variable", "Variables", "SamplingMethod"),
            ("System", "Variable", "SystemVariables", "Sampling Method"),
            ("System", "Variable", "SystemVariables", "SamplingMethod"),
            ("Variable", "System", "Variables", "Sampling Method"),
            ("Variable", "System", "Variables", "SamplingMethod"),
        ])
        print("Resolved EnumId for Variable.Sampling Method:", sampling_enum)

        profile_enum, profile_col, profile_prop = _resolve_variable_profile_enum(db, SystemNS)
        print(f"Resolved Variable profile property: collection={profile_col!r} property={profile_prop!r} enum_id={profile_enum}")

        if hasattr(CollectionEnum, "SystemVariables"):
            var_mem_collection = getattr(CollectionEnum, "SystemVariables")
        elif hasattr(CollectionEnum, "Variables"):
            var_mem_collection = getattr(CollectionEnum, "Variables")
        else:
            raise RuntimeError("Could not find CollectionEnum.SystemVariables or CollectionEnum.Variables")

        sys_gens_enum = _find_system_generators_collection_enum(CollectionEnum)

        variables_list_in_db = list(_get_objects_safe(db, ClassEnum.Variable))
        variables_in_db = set(variables_list_in_db)

        generators_in_db = list(_get_objects_safe(db, ClassEnum.Generator))
        gens_by_bz = _build_gens_by_bz(generators_in_db, allowed_bzs)

        created_vars = 0
        written_sampling = 0
        written_profiles = 0
        written_rating_factor = 0
        matched_generators = 0

        for dst, bz, tech_token, year in copied:
            plexos_tech = tech_map[tech_token]
            var_name = f"{bz}_PECD_{tech_token}_{year}"
            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

            var_category = _category_for_variable_name(var_name)

            if var_name not in variables_in_db:
                try:
                    _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category=var_category, description="")
                except Exception as e:
                    print(f"WARNING: AddObject failed with category='{var_category}' for '{var_name}'. Creating without category. Error: {e}")
                    _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category="", description="")
                    if var_category:
                        _set_object_category_best_effort(db, SystemNS, ClassEnum.Variable, var_name, var_category)

                variables_in_db.add(var_name)
                created_vars += 1
            else:
                if var_category:
                    _set_object_category_best_effort(db, SystemNS, ClassEnum.Variable, var_name, var_category)

            try:
                db.GetMembershipID(var_mem_collection, "System", var_name)
            except Exception:
                try:
                    db.AddMembership(var_mem_collection, "System", var_name)
                except Exception:
                    pass

            var_mem_id = int(db.GetMembershipID(var_mem_collection, "System", var_name))

            db.AddProperty(
                var_mem_id,
                sampling_enum,
                1,
                2.0,
                dt_from,
                None,
                None,
                None,
                None,
                SystemNS.String(SCENARIO_NAME),
                None
            )
            written_sampling += 1

            plexos_rel = str(Path(PLEXOS_REL_PREFIX) / dst.name)
            for band in range(1, 37):
                db.AddProperty(
                    var_mem_id,
                    profile_enum,
                    int(band),
                    0.0,
                    dt_from,
                    None,
                    None,
                    SystemNS.String(plexos_rel),
                    None,
                    SystemNS.String(SCENARIO_NAME),
                    None
                )
                written_profiles += 1

            bz_gens = gens_by_bz.get(bz, [])
            target_gens = [g for g in bz_gens if plexos_tech.lower() in g.lower()]

            if offshore_fixed_token and (tech_token == offshore_fixed_token):
                floating_gens = [g for g in bz_gens if _looks_like_offshore_floating_generator(g)]
                for g in floating_gens:
                    if g not in target_gens:
                        target_gens.append(g)

            if not target_gens:
                print(f"WARNING: No generators matched for bz='{bz}' tech_token='{tech_token}' (PLEXOS tech='{plexos_tech}')")
                continue

            for gen_name in target_gens:
                try:
                    db.GetMembershipID(sys_gens_enum, "System", gen_name)
                except Exception:
                    try:
                        db.AddMembership(sys_gens_enum, "System", gen_name)
                    except Exception:
                        pass

                gen_mem_id = int(db.GetMembershipID(sys_gens_enum, "System", gen_name))

                db.AddProperty(
                    gen_mem_id,
                    int(RATING_FACTOR_ENUM),
                    1,
                    0.0,
                    dt_from,
                    None,
                    SystemNS.String(var_name),
                    None,
                    None,
                    SystemNS.String(SCENARIO_NAME),
                    None
                )
                written_rating_factor += 1
                matched_generators += 1

        print(f"Variables created: {created_vars}")
        print(f"Sampling Method rows written: {written_sampling}")
        print(f"Variable profile rows written (bands): {written_profiles}")
        print(f"Generator Rating Factor rows written: {written_rating_factor}")
        print(f"Total generator matches updated: {matched_generators}")
        print("Done.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_pecd_to_rating_factors()





#### 12. Hydro Constraints & Inflow

In [ ]:
import os, csv, re
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import pandas as pd

# ============================================================
# STEP â€” Hydro Constraints (Week/Day) -> Data Files + PLEXOS property linking
#
# CHANGE (ONLY, per your latest instruction):
#   - When Min Volume Penalty (Enum 22) and Max Volume Penalty (Enum 19) is added,
#     also add Target Penalty (Enum 51), value 1,000,000, in the exact same manner.
#
# ALSO FIX (from your error):
#   - _bootstrap_plexos_net() must return 6 values:
#       (db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime)
#     Your failing version returned only 5.
#
# Everything else unchanged.
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
HYDRO_SRC_DIR = PECD_RES_DIR / "Hydro Constraints_250716"
PECD_TECH_MAP_XLSX = BASE_DIR / "PECD_Technology_Mapping.xlsx"

HYDRO_INFO_CSV = DASHBOARD_RAWDATA_DIR / "Hydro additional information.csv"
HYDRO_INFO_DATA_VERSION = "ERAA 2025 final"

HYDRO_PATTERNS_XLSX = BASE_DIR / "Hydro_Patterns.xlsx"


OUT_DIR = DATA_FILES_ROOT / "Hydro Constraints"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLEXOS_REL_PREFIX = r"Data Files\Hydro Constraints"

SCENARIO_NAME: str = "Hydro_Constraints"
DATAFILE_CATEGORY = "Hydro_Constraints"

# -----------------------------
# CONSTRAINT -> (Class, Enum)
# -----------------------------
CONSTRAINT_MAP = {
    "MAX_PUMP":    ("Generator", 126),  # Pump Load
    "MIN_PUMP":    ("Generator", 130),  # Min Pump Load
    "MAX_TURB":    ("Generator", 182),  # Max Energy Week/Day (PeriodTypeId selects Week/Day)
    "MIN_TURB":    ("Generator", 183),  # Min Energy Week/Day (PeriodTypeId selects Week/Day)
    "MAX_TURB_EN": ("Generator", 182),  # Max Energy Week/Day (PeriodTypeId selects Week/Day)
    "MIN_TURB_EN": ("Generator", 183),  # Min Energy Week/Day (PeriodTypeId selects Week/Day)
    "MAX_RES":     ("Storage",   18),   # Max Volume
    "MIN_RES":     ("Storage",   21),   # Min Volume
}

GW_TO_MW_HEADERS = {"MAX_PUMP", "MIN_PUMP"}
RES_RATIO_HEADERS = {"MAX_RES", "MIN_RES"}  # ratio * MaxVolumeGWh(from Hydro additional information)
ENERGY_EN_HEADERS = {"MAX_TURB", "MIN_TURB", "MAX_TURB_EN", "MIN_TURB_EN"}  # needs PeriodTypeId Week/Day

# -----------------------------
# Penalties (later section only)
# -----------------------------
MIN_VOLUME_PENALTY_ENUM = 22
MAX_VOLUME_PENALTY_ENUM = 19
TARGET_PENALTY_ENUM     = 51
PENALTY_VALUE = 1000000.0

# -----------------------------
# PLEXOS API DLLS
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS
# ============================================================
def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:4096]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _read_pecd_tech_mapping(xlsx_path: Path) -> dict[str, str]:
    df = pd.read_excel(xlsx_path, sheet_name=0)
    if df.shape[1] < 2:
        raise RuntimeError("PECD_Technology_Mapping.xlsx must have at least two columns (A=PECD tech, B=PLEXOS tech).")
    a = df.columns[0]
    b = df.columns[1]
    out = {}
    for _, row in df.iterrows():
        k = str(row[a]).strip() if not pd.isna(row[a]) else ""
        v = str(row[b]).strip() if not pd.isna(row[b]) else ""
        if k and v:
            out[k] = v
    return out


def _parse_hydro_filename(name: str):
    m = re.match(r"^([A-Za-z0-9]+)_Hydro_Constraints_([^_]+)_(\d{4})\.csv$", name)
    if not m:
        return None, None, None
    return m.group(1).strip(), m.group(2).strip(), int(m.group(3))


def _normalize_header_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())


def _as_float(x) -> Optional[float]:
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None


def _load_hydro_patterns() -> Tuple[str, List[str], str, List[str]]:
    if not HYDRO_PATTERNS_XLSX.exists():
        raise FileNotFoundError(f"Hydro patterns file not found: {HYDRO_PATTERNS_XLSX}")

    df = pd.read_excel(HYDRO_PATTERNS_XLSX, sheet_name=0)

    if df.shape[1] < 2:
        raise RuntimeError(
            "Hydro_Patterns.xlsx must have at least two columns:\n"
            "  Column A for Week patterns, Column B for Day patterns."
        )

    week_header = str(df.columns[0]).strip()
    day_header  = str(df.columns[1]).strip()

    week_vals = [str(v) for v in df.iloc[:, 0].tolist() if not pd.isna(v)]
    day_vals  = [str(v) for v in df.iloc[:, 1].tolist() if not pd.isna(v)]

    return week_header, week_vals, day_header, day_vals


def _apply_patterns_to_first_column(out: pd.DataFrame, gran: str,
                                   week_header: str, week_vals: List[str],
                                   day_header: str, day_vals: List[str]) -> pd.DataFrame:
    out = out.copy()
    target_header = "Pattern"  # ALWAYS use "Pattern"

    if gran == "Week":
        n = len(out)
        if len(week_vals) < n:
            raise RuntimeError(f"Hydro_Patterns.xlsx Column A has {len(week_vals)} rows but need {n} for Week outputs.")
        out.insert(0, target_header, week_vals[:n])
        if "Week" in out.columns:
            out.drop(columns=["Week"], inplace=True)
        return out

    else:  # Day
        n = len(out)
        if len(day_vals) < n:
            raise RuntimeError(f"Hydro_Patterns.xlsx Column B has {len(day_vals)} rows but need {n} for Day outputs.")
        out.insert(0, target_header, day_vals[:n])
        if "Day" in out.columns:
            out.drop(columns=["Day"], inplace=True)
        return out


def _load_max_volume_lookup_from_hydro_info() -> Dict[Tuple[str, str, int], float]:
    if not HYDRO_INFO_CSV.exists():
        raise FileNotFoundError(f"Hydro additional information file not found: {HYDRO_INFO_CSV}")

    df = pd.read_csv(HYDRO_INFO_CSV)

    required = ["MARKET_NODE", "PEMMDB_PLANT_TYPE", "Storage Capacity [TWh]", "TARGET_YEAR", "data_version"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in Hydro additional information.csv: {missing}\nFound: {list(df.columns)}")

    df = df.copy()
    df["MARKET_NODE"] = df["MARKET_NODE"].map(_clean_cell)
    df["PEMMDB_PLANT_TYPE"] = df["PEMMDB_PLANT_TYPE"].map(_clean_cell)
    df["data_version"] = df["data_version"].map(_clean_cell)

    df = df[df["data_version"] == HYDRO_INFO_DATA_VERSION].copy()
    if df.empty:
        raise RuntimeError(f"No rows found in Hydro additional information.csv for data_version='{HYDRO_INFO_DATA_VERSION}'")

    df["TARGET_YEAR"] = pd.to_numeric(df["TARGET_YEAR"], errors="coerce")
    df["Storage Capacity [TWh]"] = df["Storage Capacity [TWh]"].apply(_as_float)

    df = df[df["TARGET_YEAR"].notna()].copy()
    df = df[df["Storage Capacity [TWh]"].notna()].copy()
    if df.empty:
        raise RuntimeError("No valid rows with numeric TARGET_YEAR and Storage Capacity [TWh] after filtering.")

    g = (
        df.groupby(["MARKET_NODE", "PEMMDB_PLANT_TYPE", "TARGET_YEAR"], as_index=False)
          .agg({"Storage Capacity [TWh]": "sum"})
    )

    out: Dict[Tuple[str, str, int], float] = {}
    for _, r in g.iterrows():
        node = str(r["MARKET_NODE"]).strip()
        plant = str(r["PEMMDB_PLANT_TYPE"]).strip()
        year = int(r["TARGET_YEAR"])
        cap_twh = float(r["Storage Capacity [TWh]"])
        out[(node, plant, year)] = cap_twh * 1000.0  # -> GWh

    return out


# ============================================================
# PLEXOS_NET BOOTSTRAP + DB HELPERS
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum, PeriodEnum
    import System as SystemNS
    from System import DateTime as NetDateTime

    # IMPORTANT: return 6 values (matches caller unpacking)
    return Database(), ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _find_collection_enum(CollectionEnum, required_tokens_lc: list[str], preferred_names: list[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][2]

    if fallback_id is not None:
        return int(fallback_id)

    raise RuntimeError(f"Could not resolve CollectionEnum for tokens={required_tokens_lc}, preferred={preferred_names}")


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(f"Could not resolve DataFile filename enum id. Last attempt: {last}")


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    """
    AddProperty with optional PeriodTypeId (PeriodEnum).
    If the API surface doesn't accept the extra argument, fall back to the 11-arg call.
    """
    try:
        if PeriodTypeId is not None:
            db.AddProperty(
                int(mem_id),
                int(enum_id),
                int(band),
                float(value),
                DateFrom,
                DateTo,
                Variable,
                DataFile,
                Pattern,
                Scenario,
                Action,
                PeriodTypeId
            )
            return
    except Exception:
        pass

    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _ensure_datafile_category(db, ClassEnum, category_name: str) -> None:
    if not category_name:
        return
    if hasattr(db, "AddCategory"):
        try:
            db.AddCategory(ClassEnum.DataFile, str(category_name))
            return
        except Exception:
            pass
        try:
            db.AddCategory(int(ClassEnum.DataFile), str(category_name))
            return
        except Exception:
            pass


# ============================================================
# BUILD placeholder outputs from source hydro files
# ============================================================
def _build_constraint_outputs() -> Tuple[Dict[Tuple[int, str, str], pd.DataFrame], Dict[Tuple[int, str, str], Dict[str, int]]]:
    tech_map = _read_pecd_tech_mapping(PECD_TECH_MAP_XLSX)

    outputs: Dict[Tuple[int, str, str], pd.DataFrame] = {}
    meta: Dict[Tuple[int, str, str], Dict[str, int]] = {}

    src_files = sorted(HYDRO_SRC_DIR.glob("*.csv"))
    if not src_files:
        raise RuntimeError(f"No .csv files found in: {HYDRO_SRC_DIR}")

    for p in src_files:
        bz, pecd_tech, year = _parse_hydro_filename(p.name)
        if not bz:
            continue
        if pecd_tech not in tech_map:
            continue

        delim = _detect_csv_delimiter(p)
        df = pd.read_csv(p, sep=delim, engine="python")
        df = _normalize_header_cols(df)

        cols = set(df.columns)
        has_week = "WEEK" in cols
        has_day = "DAY" in cols
        if (not has_week) and (not has_day):
            continue
        if has_week and has_day:
            continue

        gran = "Week" if has_week else "Day"
        time_col = "WEEK" if has_week else "DAY"
        out_time_col = "Week" if has_week else "Day"

        df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
        df = df.dropna(subset=[time_col]).copy()
        df[time_col] = df[time_col].astype(int)

        constraint_cols = [c for c in df.columns if c in CONSTRAINT_MAP]
        if not constraint_cols:
            continue

        plexos_tech_token = tech_map[pecd_tech]

        for c in constraint_cols:
            obj_class, enum_id = CONSTRAINT_MAP[c]
            key = (year, gran, c)

            if key not in outputs:
                max_n = 53 if gran == "Week" else 366
                outputs[key] = pd.DataFrame({out_time_col: list(range(1, max_n + 1))})
                meta[key] = {"enum": int(enum_id), "class": obj_class}

            out_df = outputs[key]
            placeholder = f"__SRC__{bz}__{pecd_tech}__{plexos_tech_token}__{obj_class}"

            series = df[[time_col, c]].copy()
            series[c] = pd.to_numeric(series[c], errors="coerce")

            tmp = pd.DataFrame({out_time_col: series[time_col].astype(int).values, placeholder: series[c].values})
            out_df = out_df.merge(tmp, on=out_time_col, how="left")

            dup_x = f"{placeholder}_x"
            dup_y = f"{placeholder}_y"
            if dup_x in out_df.columns and dup_y in out_df.columns:
                out_df[placeholder] = out_df[dup_x].combine_first(out_df[dup_y])
                out_df.drop(columns=[dup_x, dup_y], inplace=True)

            outputs[key] = out_df

    if not outputs:
        raise RuntimeError("No hydro constraint data matched the expected patterns (mapping + WEEK/DAY + constraint headers).")

    return outputs, meta


# ============================================================
# Expand placeholders -> real PLEXOS object columns + apply scaling + write CSVs
# ============================================================
def _expand_scale_and_write_csvs(outputs, meta, db, ClassEnum, maxvol_lookup: Dict[Tuple[str, str, int], float]) -> Dict[Tuple[int, str, str], Path]:
    week_header, week_vals, day_header, day_vals = _load_hydro_patterns()

    gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))

    gens_by_bz: Dict[str, List[str]] = {}
    for g in gens_in_db:
        if "_" in g:
            prefix = g.split("_", 1)[0]
            gens_by_bz.setdefault(prefix, []).append(g)

    written_paths: Dict[Tuple[int, str, str], Path] = {}
    missing_maxvol_warnings = []

    for key, df in outputs.items():
        year, gran, constraint = key
        out_time_col = "Week" if gran == "Week" else "Day"
        obj_class = meta[key]["class"]

        placeholder_cols = [c for c in df.columns if str(c).startswith("__SRC__")]
        if not placeholder_cols:
            continue

        out = df[[out_time_col]].copy()

        for ph in placeholder_cols:
            parts = ph.split("__")
            if len(parts) < 6:
                continue
            bz = parts[2]
            plexos_plant = parts[4]
            ph_class = parts[5]

            if ph_class != obj_class:
                continue

            vals = pd.to_numeric(df[ph], errors="coerce")

            bz_gens = gens_by_bz.get(bz, [])
            matched_gens = [g for g in bz_gens if plexos_plant.lower() in g.lower()]
            if not matched_gens:
                continue

            if obj_class == "Generator":
                if constraint in GW_TO_MW_HEADERS:
                    vals = vals * 1000.0
                for g in matched_gens:
                    out[g] = vals.values

            else:
                mv = maxvol_lookup.get((bz, plexos_plant, int(year)))
                for g in matched_gens:
                    head_name = f"{g} Head"
                    if mv is None:
                        missing_maxvol_warnings.append((bz, plexos_plant, int(year), head_name))
                        out[head_name] = vals.values
                    else:
                        out[head_name] = (vals * float(mv)).values

        obj_cols = [c for c in out.columns if c != out_time_col]
        keep_cols = [out_time_col]
        for c in obj_cols:
            if out[c].notna().any():
                keep_cols.append(c)
        out = out[keep_cols]

        out = _apply_patterns_to_first_column(out, gran, week_header, week_vals, day_header, day_vals)

        out_name = f"Hydro_{gran}_{constraint}_{year}.csv"
        out_path = OUT_DIR / out_name
        out.to_csv(out_path, index=False)

        written_paths[key] = out_path

    if missing_maxvol_warnings:
        uniq = sorted(set(missing_maxvol_warnings))
        print(f"[WARN] Missing Max Volume from Hydro additional information for {len(uniq)} (bz,plant,year) combos.")
        print("       For these, MAX_RES/MIN_RES were written UN-SCALED (ratios) to avoid empty files.")
        for bz, plant, y, head in uniq[:120]:
            print(f"   - Missing: MARKET_NODE={bz}, PEMMDB_PLANT_TYPE={plant}, TARGET_YEAR={y} (head={head})")
        if len(uniq) > 120:
            print(f"   ... and {len(uniq)-120} more")

    if not written_paths:
        raise RuntimeError("No constraint outputs produced any matched/scaled PLEXOS object columns (check mapping + DB object names).")

    return written_paths


# ============================================================
# Push into PLEXOS: DataFile objects + property rows (ALL under scenario)
# Penalties:
#   - Max Volume Penalty (Enum 19): ONCE per storage we touch, no DateFrom
#   - Target Penalty   (Enum 51): ONCE per storage we touch, no DateFrom (same as Max)
#   - Min Volume Penalty (Enum 22): ONCE per storage that receives MIN_RES, no DateFrom
# ============================================================
def _push_to_plexos(written_paths, meta, db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime):
    _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
    scenario_str = SystemNS.String(SCENARIO_NAME)

    _ensure_datafile_category(db, ClassEnum, DATAFILE_CATEGORY)

    gen_mem_collection = _find_collection_enum(
        CollectionEnum, ["system", "generator"], ["SystemGenerators", "Generators"], fallback_id=None
    )
    stor_mem_collection = _find_collection_enum(
        CollectionEnum, ["system", "stor"], ["SystemStorages", "Storages", "SystemStorage"], fallback_id=None
    )
    datafile_mem_collection = _find_collection_enum(
        CollectionEnum, ["system", "data", "file"], ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"], fallback_id=None
    )

    datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
    print(f"Resolved DataFile filename EnumId={datafile_filename_enum} (child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r})")
    print(f"Scenario used: {SCENARIO_NAME}")
    print(f"DataFile category used: {DATAFILE_CATEGORY}")

    gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
    stor_in_db = set(_get_objects_safe(db, ClassEnum.Storage)) if hasattr(ClassEnum, "Storage") else set()

    property_rows = 0
    skipped_missing_objs = 0
    datafiles_updated = 0
    datafiles_categorized = 0

    min_vol_penalty_rows = 0
    max_vol_penalty_rows = 0
    target_penalty_rows = 0

    min_penalty_written: set[str] = set()
    max_penalty_written: set[str] = set()
    target_penalty_written: set[str] = set()

    existing_datafiles_cache = set(_get_objects_safe(db, ClassEnum.DataFile))

    for key, out_path in written_paths.items():
        year, gran, constraint = key
        enum_id = int(meta[key]["enum"])
        obj_class = meta[key]["class"]

        df_name = out_path.stem
        plexos_rel = str(Path(PLEXOS_REL_PREFIX) / out_path.name)

        # Ensure DataFile object in category
        if df_name not in existing_datafiles_cache:
            try:
                _add_object(db, ClassEnum.DataFile, df_name, add_to_system=True, category=DATAFILE_CATEGORY, description="")
                datafiles_categorized += 1
            except Exception:
                _add_object(db, ClassEnum.DataFile, df_name, add_to_system=True, category="", description="")
            existing_datafiles_cache.add(df_name)
        else:
            if hasattr(db, "UpdateObject"):
                try:
                    db.UpdateObject(df_name, ClassEnum.DataFile, True, DATAFILE_CATEGORY, "")
                    datafiles_categorized += 1
                except Exception:
                    try:
                        db.UpdateObject(df_name, int(ClassEnum.DataFile), True, DATAFILE_CATEGORY, "")
                        datafiles_categorized += 1
                    except Exception:
                        pass

        # DataFile filename row (scenario)
        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", df_name)
        _add_property_row(
            db, datafile_filename_enum, df_mem_id, 1, 0.0,
            DataFile=SystemNS.String(plexos_rel),
            Scenario=scenario_str
        )
        datafiles_updated += 1

        out_df = pd.read_csv(out_path, engine="python")
        first_col = out_df.columns[0]  # Pattern
        obj_cols = [c for c in out_df.columns if c != first_col]

        dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

        period_type = None
        if constraint in ENERGY_EN_HEADERS:
            period_type = PeriodEnum.Week if gran == "Week" else PeriodEnum.Day

        for obj_name in obj_cols:
            if obj_class == "Generator":
                if obj_name not in gens_in_db:
                    skipped_missing_objs += 1
                    continue
                mem_id = _ensure_membership(db, gen_mem_collection, "System", obj_name)
            else:
                if obj_name not in stor_in_db:
                    skipped_missing_objs += 1
                    continue
                mem_id = _ensure_membership(db, stor_mem_collection, "System", obj_name)

            # Link object -> constraint datafile (year-specific)
            _add_property_row(
                db,
                enum_id,
                mem_id,
                1,
                0.0,
                DateFrom=dt_from,
                DataFile=SystemNS.String(plexos_rel),
                Scenario=scenario_str,
                PeriodTypeId=period_type
            )
            property_rows += 1

            # Penalties (Storage only) â€” NO DateFrom, written ONCE
            if obj_class == "Storage":
                # Max Volume Penalty (always for touched head storages)
                if obj_name not in max_penalty_written:
                    _add_property_row(db, MAX_VOLUME_PENALTY_ENUM, mem_id, 1, float(PENALTY_VALUE), Scenario=scenario_str)
                    max_penalty_written.add(obj_name)
                    max_vol_penalty_rows += 1

                # Target Penalty (same as max penalty: always for touched head storages)
                if obj_name not in target_penalty_written:
                    _add_property_row(db, TARGET_PENALTY_ENUM, mem_id, 1, float(PENALTY_VALUE), Scenario=scenario_str)
                    target_penalty_written.add(obj_name)
                    target_penalty_rows += 1

                # Min Volume Penalty (only for MIN_RES rule)
                if constraint == "MIN_RES" and obj_name not in min_penalty_written:
                    _add_property_row(db, MIN_VOLUME_PENALTY_ENUM, mem_id, 1, float(PENALTY_VALUE), Scenario=scenario_str)
                    min_penalty_written.add(obj_name)
                    min_vol_penalty_rows += 1

    print("Done.")
    print(f"DataFile filename rows written: {datafiles_updated}")
    print(f"Property rows written (object->constraint): {property_rows}")
    print(f"Max Volume Penalty rows written (Enum {MAX_VOLUME_PENALTY_ENUM}): {max_vol_penalty_rows}")
    print(f"Target Penalty rows written (Enum {TARGET_PENALTY_ENUM}): {target_penalty_rows}")
    print(f"Min Volume Penalty rows written (Enum {MIN_VOLUME_PENALTY_ENUM}): {min_vol_penalty_rows}")
    print(f"DataFiles created/updated in category '{DATAFILE_CATEGORY}': {datafiles_categorized}")
    if skipped_missing_objs:
        print(f"[WARN] Skipped {skipped_missing_objs} object columns because object not found in DB (Generators/Storages).")


# ============================================================
# MAIN
# ============================================================
def step_hydro_constraints_week_day_to_plexos():
    if not HYDRO_SRC_DIR.exists():
        raise RuntimeError(f"HYDRO_SRC_DIR not found: {HYDRO_SRC_DIR}")

    maxvol_lookup = _load_max_volume_lookup_from_hydro_info()
    print(f"[INFO] Loaded Max Volume lookup entries: {len(maxvol_lookup)} (MARKET_NODE, PEMMDB_PLANT_TYPE, TARGET_YEAR)")

    outputs, meta = _build_constraint_outputs()

    db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        written_paths = _expand_scale_and_write_csvs(outputs, meta, db, ClassEnum, maxvol_lookup)
        print(f"Wrote {len(written_paths)} hydro constraint CSV(s) into: {OUT_DIR}")

        _push_to_plexos(written_paths, meta, db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime)

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_hydro_constraints_week_day_to_plexos()





In [ ]:
# ============================================================
# STEP â€” Hydro Inflows -> CSV transform + PLEXOS Variables (36 bands, User sampling)
#          + link Variables to Generator.Fixed Load (HRR) or Storage.Natural Inflow (HOL/HCL/HRI/HPI)
#
# CHANGES (ONLY, per your instruction in THIS message):
#   - If NO hydro inflow files match (because some zones have no hydro inflows),
#     print a WARNING and exit the step cleanly (no exception).
#
# Everything else unchanged.
# ============================================================

import os, csv, re
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
HYDRO_INFLOW_SRC_DIR = PECD_RES_DIR / "Hydro Inflows_250704"
PECD_TECH_MAP_XLSX    = BASE_DIR / "PECD_Technology_Mapping.xlsx"

# Patterns file
HYDRO_PATTERNS_XLSX = BASE_DIR / "Hydro_Patterns.xlsx"


OUT_DIR = DATA_FILES_ROOT / "Hydro Inflows"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLEXOS_REL_PREFIX = r"Data Files\Hydro Inflows"

# Scenario + Variable Category
SCENARIO_NAME = "Hydro_Inflows"
VARIABLE_CATEGORY = "Hydro_Inflows"

# -----------------------------
# TECHS / ENUMS
# -----------------------------
TECH_HRR = "HRR"  # Run of river (Generator -> Fixed Load)
TECH_HEAD_STOR = {"HOL", "HCL", "HRI", "HPI"}  # Storage "<Gen> Head" -> Natural Inflow

ENUM_FIXED_LOAD = 96
ENUM_FIXED_LOAD_PENALTY = 97
FIXED_LOAD_PENALTY_VALUE = 1_000_000.0

ENUM_NATURAL_INFLOW = 30

# -----------------------------
# PLEXOS API DLLS
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


# ============================================================
# HELPERS: file + mapping + parsing
# ============================================================
def _detect_csv_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="ignore")[:4096]
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _read_pecd_tech_mapping(xlsx_path: Path) -> dict[str, str]:
    df = pd.read_excel(xlsx_path, sheet_name=0)
    if df.shape[1] < 2:
        raise RuntimeError("PECD_Technology_Mapping.xlsx must have at least two columns (A=PECD tech, B=PLEXOS tech).")
    a = df.columns[0]
    b = df.columns[1]
    out = {}
    for _, row in df.iterrows():
        k = str(row[a]).strip() if not pd.isna(row[a]) else ""
        v = str(row[b]).strip() if not pd.isna(row[b]) else ""
        if k and v:
            out[k] = v
    return out


def _extract_year_from_name(name: str) -> Optional[int]:
    m = re.search(r"(19\d{2}|20\d{2})", name)
    if not m:
        return None
    return int(m.group(1))


def _extract_bz_prefix(name: str) -> Optional[str]:
    if "_" not in name:
        return None
    return name.split("_", 1)[0].strip() or None


def _extract_tech_token(name: str, allowed_tokens: set[str]) -> Optional[str]:
    upper = name.upper()
    for tok in allowed_tokens:
        if f"_{tok}_" in upper or upper.endswith(f"_{tok}.CSV") or upper.endswith(f"_{tok}") or f"_{tok}-" in upper:
            return tok
    for tok in allowed_tokens:
        if tok in upper:
            return tok
    return None


def _rename_ws_headers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ren = {}
    for i in range(1, 37):
        ws = f"WS{i:02d}"
        if ws in df.columns:
            ren[ws] = str(i)
    if ren:
        df.rename(columns=ren, inplace=True)
    return df


def _divide_numeric_band_columns(df: pd.DataFrame, divisor: float) -> pd.DataFrame:
    """
    Divide only numeric band columns "1".."36" if present.
    Leaves all other columns unchanged.
    """
    df = df.copy()
    for i in range(1, 37):
        c = str(i)
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().any():
                df[c] = s / float(divisor)
    return df


def _infer_storage_divisor_from_first_header(df: pd.DataFrame) -> float:
    """
    For HOL/HCL/HRI/HPI:
      - If the *first header* is WEEK/week => divide by 168
      - If the *first header* is DAY/day  => divide by 24
      - Otherwise default to 24
    """
    if df.columns.size == 0:
        return 24.0
    first = str(df.columns[0]).strip().lower()
    if first == "week":
        return 168.0
    if first == "day":
        return 24.0
    return 24.0


# Patterns loader + applier
def _load_hydro_patterns() -> Tuple[str, List[str], str, List[str]]:
    if not HYDRO_PATTERNS_XLSX.exists():
        raise FileNotFoundError(f"Hydro patterns file not found: {HYDRO_PATTERNS_XLSX}")

    df = pd.read_excel(HYDRO_PATTERNS_XLSX, sheet_name=0)
    if df.shape[1] < 2:
        raise RuntimeError(
            "Hydro_Patterns.xlsx must have at least two columns:\n"
            "  Column A for Week patterns, Column B for Day patterns."
        )

    week_header = str(df.columns[0]).strip()
    day_header  = str(df.columns[1]).strip()

    week_vals = [str(v) for v in df.iloc[:, 0].tolist() if not pd.isna(v)]
    day_vals  = [str(v) for v in df.iloc[:, 1].tolist() if not pd.isna(v)]

    return week_header, week_vals, day_header, day_vals


def _apply_patterns_to_first_column_inplace(
    df: pd.DataFrame,
    week_header: str,
    week_vals: List[str],
    day_header: str,
    day_vals: List[str],
) -> pd.DataFrame:
    df = df.copy()

    if df.columns.size == 0:
        return df

    first_col_name = str(df.columns[0]).strip()
    first_lc = first_col_name.lower()

    target_header = "Pattern"

    if first_lc == "week":
        n = len(df)
        if len(week_vals) < n:
            raise RuntimeError(
                f"Hydro_Patterns.xlsx Column A has {len(week_vals)} rows but need {n} for WEEK inflow files."
            )
        df.rename(columns={df.columns[0]: target_header}, inplace=True)
        df[target_header] = week_vals[:n]
        return df

    if first_lc == "day":
        n = len(df)
        if len(day_vals) < n:
            raise RuntimeError(
                f"Hydro_Patterns.xlsx Column B has {len(day_vals)} rows but need {n} for DAY inflow files."
            )
        df.rename(columns={df.columns[0]: target_header}, inplace=True)
        df[target_header] = day_vals[:n]
        return df

    return df


def _transform_and_write_inflow_csv(
    src_path: Path,
    dst_path: Path,
    tech_token: str,
    week_header: str,
    week_vals: List[str],
    day_header: str,
    day_vals: List[str],
) -> None:
    delim = _detect_csv_delimiter(src_path)
    df = pd.read_csv(src_path, sep=delim, engine="python", header=0)

    # Decide divisor BEFORE pattern replace, because storage divisor depends on WEEK/DAY header
    if tech_token == TECH_HRR:
        divisor = 24.0
    elif tech_token in TECH_HEAD_STOR:
        divisor = _infer_storage_divisor_from_first_header(df)
    else:
        divisor = 1.0

    df = _rename_ws_headers(df)

    if divisor != 1.0:
        df = _divide_numeric_band_columns(df, divisor)

    # Replace first column WEEK/DAY header+values with Hydro_Patterns.xlsx and set header = "Pattern"
    df = _apply_patterns_to_first_column_inplace(df, week_header, week_vals, day_header, day_vals)

    df.to_csv(dst_path, index=False, sep=delim)


# ============================================================
# PLEXOS_NET BOOTSTRAP + DB HELPERS
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _find_collection_enum(CollectionEnum, required_tokens_lc: list[str], preferred_names: list[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][2]

    if fallback_id is not None:
        return int(fallback_id)

    raise RuntimeError(f"Could not resolve CollectionEnum for tokens={required_tokens_lc}, preferred={preferred_names}")


def _try_enum(db, SystemNS, a, b, c, d) -> int:
    return int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))


def _resolve_enum_id(db, SystemNS, trials):
    last = None
    for a, b, c, d in trials:
        try:
            return _try_enum(db, SystemNS, a, b, c, d)
        except Exception as e:
            last = (a, b, c, d, str(e))
            continue
    raise RuntimeError(f"Could not resolve EnumId. Last attempt: {last}")


def _resolve_variable_profile_enum(db, SystemNS):
    prop_candidates = [
        "Profile", "Profiles", "Profile Data", "Profile File", "Profile Filename",
        "Profile File Name", "Profile Data File", "Profile DataFile",
        "Data File", "DataFile", "Filename", "File Name", "File", "Path", "Data",
        "Time Series",
    ]
    collection_candidates = ["Variables", "SystemVariables", "Variable"]

    last = None
    for col in collection_candidates:
        for prop in prop_candidates:
            try:
                enum_id = int(db.PropertyName2EnumId(
                    SystemNS.String("System"),
                    SystemNS.String("Variable"),
                    SystemNS.String(col),
                    SystemNS.String(prop),
                ))
                return enum_id, col, prop
            except Exception as e:
                last = (col, prop, str(e))
                continue

    raise RuntimeError(
        "Could not resolve a Variable profile/file property.\n"
        f"Last attempt: {last}\n"
        "Please confirm the exact property name shown in PLEXOS UI for Variable profile."
    )


def _ensure_category_for_class(db, ClassEnum, SystemNS, class_enum_value, category_name: str):
    if not category_name:
        return

    if hasattr(db, "AddCategory"):
        fn = getattr(db, "AddCategory")
        attempts = [
            lambda: fn(class_enum_value, category_name),
            lambda: fn(int(class_enum_value), category_name),
            lambda: fn(category_name, class_enum_value),
            lambda: fn(category_name, int(class_enum_value)),
            lambda: fn(SystemNS.String("Variable"), SystemNS.String(category_name)),
            lambda: fn(SystemNS.String(category_name), SystemNS.String("Variable")),
        ]
        for a in attempts:
            try:
                a()
                return
            except Exception:
                pass

    if hasattr(ClassEnum, "Category"):
        try:
            _add_object(db, getattr(ClassEnum, "Category"), category_name, add_to_system=True, category="", description="")
            return
        except Exception:
            pass

    print(f"WARNING: Could not pre-create category '{category_name}' for this PLEXOS API build.")


# ============================================================
# MAIN
# ============================================================
def step_hydro_inflows_to_plexos():
    if not HYDRO_INFLOW_SRC_DIR.exists():
        raise RuntimeError(f"HYDRO_INFLOW_SRC_DIR not found: {HYDRO_INFLOW_SRC_DIR}")

    allowed_bzs = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed_bzs:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    tech_map = _read_pecd_tech_mapping(PECD_TECH_MAP_XLSX)
    if not tech_map:
        raise RuntimeError("No PECD technology mappings found in PECD_Technology_Mapping.xlsx")

    week_header, week_vals, day_header, day_vals = _load_hydro_patterns()

    allowed_tokens = {TECH_HRR, *TECH_HEAD_STOR}

    picked: List[Tuple[Path, str, str, int]] = []  # (src, bz, tech_token, year)

    for p in sorted(HYDRO_INFLOW_SRC_DIR.glob("*.csv")):
        bz = _extract_bz_prefix(p.stem)
        if not bz or bz not in allowed_bzs:
            continue

        year = _extract_year_from_name(p.stem)
        if year is None:
            continue

        tech_token = _extract_tech_token(p.stem, allowed_tokens)
        if not tech_token:
            continue

        if tech_token not in tech_map:
            print(f"WARNING: File '{p.name}' tech token '{tech_token}' not found in PECD_Technology_Mapping.xlsx. Skipping.")
            continue

        picked.append((p, bz, tech_token, year))

    # UPDATED: warn instead of failing, then exit cleanly
    if not picked:
        print(
            f"\nWARNING: No hydro inflow files matched (allowed bidding zones + tokens {sorted(allowed_tokens)} + year).\n"
            f"Source: {HYDRO_INFLOW_SRC_DIR}\n"
            f"This can happen if the imported bidding zones have no hydro inflows in the dataset.\n"
            f"This section will exit without creating variables or linking properties."
        )
        return

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    written: List[Tuple[Path, str, str, int]] = []  # (dst, bz, tech_token, year)
    for src, bz, tech_token, year in picked:
        dst = OUT_DIR / src.name
        _transform_and_write_inflow_csv(src, dst, tech_token, week_header, week_vals, day_header, day_vals)
        written.append((dst, bz, tech_token, year))

    print(f"Transformed + wrote {len(written)} inflow CSV(s) into: {OUT_DIR}")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_FIXED_LOAD = _resolve_semantic_enum("System", "Generator", "Generators", "Fixed Load", 96)
    ENUM_FIXED_LOAD_PENALTY = _resolve_semantic_enum("System", "Generator", "Generators", "Fixed Load Penalty", 97)
    

    try:
        # Ensure scenario
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        # Ensure a Variable category (best effort)
        _ensure_category_for_class(db, ClassEnum, SystemNS, ClassEnum.Variable, VARIABLE_CATEGORY)

        # Resolve Variable enums (Sampling Method + Profile/Data file property)
        sampling_enum = _resolve_enum_id(db, SystemNS, [
            ("System", "Variable", "Variables", "Sampling Method"),
            ("System", "Variable", "Variables", "SamplingMethod"),
            ("System", "Variable", "SystemVariables", "Sampling Method"),
            ("System", "Variable", "SystemVariables", "SamplingMethod"),
            ("Variable", "System", "Variables", "Sampling Method"),
            ("Variable", "System", "Variables", "SamplingMethod"),
        ])
        print("Resolved EnumId for Variable.Sampling Method:", sampling_enum)

        profile_enum, profile_col, profile_prop = _resolve_variable_profile_enum(db, SystemNS)
        print(f"Resolved Variable profile property: collection={profile_col!r} property={profile_prop!r} enum_id={profile_enum}")

        # Collections
        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=None
        )
        stor_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "stor"],
            ["SystemStorages", "Storages", "SystemStorage"],
            fallback_id=None
        )

        if hasattr(CollectionEnum, "SystemVariables"):
            var_mem_collection = getattr(CollectionEnum, "SystemVariables")
        elif hasattr(CollectionEnum, "Variables"):
            var_mem_collection = getattr(CollectionEnum, "Variables")
        else:
            raise RuntimeError("Could not find CollectionEnum.SystemVariables or CollectionEnum.Variables")

        # Existing objects
        gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
        stors_in_db = set(_get_objects_safe(db, ClassEnum.Storage)) if hasattr(ClassEnum, "Storage") else set()
        vars_in_db = set(_get_objects_safe(db, ClassEnum.Variable))

        gens_by_bz: Dict[str, List[str]] = {}
        for g in gens_in_db:
            if "_" in g:
                prefix = g.split("_", 1)[0]
                gens_by_bz.setdefault(prefix, []).append(g)

        created_vars = 0
        sampling_rows = 0
        profile_rows = 0
        prop_rows = 0
        penalty_rows = 0
        skipped_missing_objs = 0
        no_match_warnings = 0

        for dst, bz, tech_token, year in written:
            plexos_tech = tech_map[tech_token]
            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

            var_name = f"{bz}_HYDRO_{tech_token}_{year}"

            if var_name not in vars_in_db:
                try:
                    _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category=VARIABLE_CATEGORY, description="")
                except Exception as e:
                    print(f"WARNING: AddObject failed with category='{VARIABLE_CATEGORY}' for '{var_name}'. Creating without category. Error: {e}")
                    _add_object(db, ClassEnum.Variable, var_name, add_to_system=True, category="", description="")
                vars_in_db.add(var_name)
                created_vars += 1

            var_mem_id = _ensure_membership(db, var_mem_collection, "System", var_name)

            db.AddProperty(
                int(var_mem_id),
                int(sampling_enum),
                1,
                2.0,
                dt_from,
                None,
                None,
                None,
                None,
                scenario_str,
                None
            )
            sampling_rows += 1

            plexos_rel = str(Path(PLEXOS_REL_PREFIX) / dst.name)
            for band in range(1, 37):
                db.AddProperty(
                    int(var_mem_id),
                    int(profile_enum),
                    int(band),
                    0.0,
                    dt_from,
                    None,
                    None,
                    SystemNS.String(plexos_rel),
                    None,
                    scenario_str,
                    None
                )
                profile_rows += 1

            bz_gens = gens_by_bz.get(bz, [])
            matched_gens = [g for g in bz_gens if plexos_tech.lower() in g.lower()]

            if not matched_gens:
                print(f"WARNING: No generators matched for inflow file '{dst.name}' (bz={bz}, tech_token={tech_token}, plexos_tech='{plexos_tech}')")
                no_match_warnings += 1
                continue

            if tech_token == TECH_HRR:
                for gen_name in matched_gens:
                    if gen_name not in gens_in_db:
                        skipped_missing_objs += 1
                        continue
                    gen_mem_id = _ensure_membership(db, gen_mem_collection, "System", gen_name)

                    db.AddProperty(
                        int(gen_mem_id),
                        int(ENUM_FIXED_LOAD),
                        1,
                        0.0,
                        dt_from,
                        None,
                        SystemNS.String(var_name),
                        None,
                        None,
                        scenario_str,
                        None
                    )
                    prop_rows += 1

                    db.AddProperty(
                        int(gen_mem_id),
                        int(ENUM_FIXED_LOAD_PENALTY),
                        1,
                        float(FIXED_LOAD_PENALTY_VALUE),
                        dt_from,
                        None,
                        None,
                        None,
                        None,
                        scenario_str,
                        None
                    )
                    penalty_rows += 1

            elif tech_token in TECH_HEAD_STOR:
                for gen_name in matched_gens:
                    head_name = f"{gen_name} Head"
                    if head_name not in stors_in_db:
                        skipped_missing_objs += 1
                        continue

                    stor_mem_id = _ensure_membership(db, stor_mem_collection, "System", head_name)

                    db.AddProperty(
                        int(stor_mem_id),
                        int(ENUM_NATURAL_INFLOW),
                        1,
                        0.0,
                        dt_from,
                        None,
                        SystemNS.String(var_name),
                        None,
                        None,
                        scenario_str,
                        None
                    )
                    prop_rows += 1

            else:
                print(f"WARNING: Unexpected tech_token '{tech_token}' for file '{dst.name}'. Skipping linking.")
                continue

        print("Done.")
        print(f"Transformed inflow CSVs written: {len(written)}")
        print(f"Variables created: {created_vars}")
        print(f"Sampling Method rows written: {sampling_rows}")
        print(f"Variable profile rows written (bands): {profile_rows}")
        print(f"Property rows written (Variable links): {prop_rows}")
        print(f"Fixed Load Penalty rows written (HRR only): {penalty_rows}")
        if skipped_missing_objs:
            print(f"[WARN] Skipped {skipped_missing_objs} links because object not found in DB (Generators/Storages).")
        if no_match_warnings:
            print(f"[WARN] {no_match_warnings} inflow files had no generator matches (check mapping + generator naming).")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_hydro_inflows_to_plexos()





#### 13. Other Data (Derating, Detailed DSR, FORs, Must-run, Reservoir levels)

In [ ]:
import os
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import pandas as pd

# ============================================================
# STEP â€” Generator Rating Factor (Derating Ratios)
#   UPDATE (ONLY, per your latest instruction):
#     - Associate Rating Factor property rows (and DataFile filename row) with
#       Scenario = "Derating_Thermal"
#
#   NEW UPDATE (ONLY, per your latest instruction in THIS message):
#     - If no matches are found between Derating_data.csv and existing generator names,
#       do NOT fail. Print a warning and still write an output CSV (Year-only).
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
DERATING_SRC_CSV = OTHER_DATA_DIR / "Derating_data.csv"


OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_NAME = "Generator_Derating_Ratios.csv"
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PLEXOS_REL_CSV_PATH = r"Data Files\Generator Data\Generator_Derating_Ratios.csv"

DATAFILE_OBJECT_NAME = "Generator_Derating_Ratios"
DATAFILE_CATEGORY_NAME = "Other_Detailed_Data"

# NEW: scenario for Rating Factor linking
SCENARIO_NAME = "Derating_Thermal"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# Enums / constants
# -----------------------------
RATING_FACTOR_ENUM_ID = 84  # per your instruction

# -----------------------------
# HELPERS
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, category: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category=category, description="")


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    if PeriodTypeId is None:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action
        )
    else:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action,
            PeriodTypeId
        )


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _find_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


def _normalize_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _canon_fuel_for_generator_name(pemmdb_fuel_type: str) -> str:
    f = _normalize_str(pemmdb_fuel_type).lower()
    mapping = {
        "gas": "Natural gas",
        "natural gas": "Natural gas",
        "hard coal": "Hard coal",
        "coal": "Hard coal",
        "lignite": "Lignite",
        "oil": "Oil",
        "fuel oil": "Oil",
        "diesel": "Oil",
        "nuclear": "Nuclear",
        "biomass": "Biomass",
        "waste": "Waste",
        "geothermal": "Geothermal",
        "shale oil": "Oil Shale",
        "oil shale": "Oil Shale",
        "hydro": "Hydro",
        "water": "Hydro",
        "wind": "Wind",
        "solar": "Solar",
    }
    if f in mapping:
        return mapping[f]
    return _normalize_str(pemmdb_fuel_type).title()


def _try_match_generators(existing_generators: set[str], market_node: str, fuel_text: str) -> List[str]:
    mn = _normalize_str(market_node)
    ft = _normalize_str(fuel_text)
    if not mn or not ft:
        return []

    exact = f"{mn}_{ft}"
    if exact in existing_generators:
        return [exact]

    mn_lc = mn.lower()
    ft_lc = ft.lower()

    hits = []
    for g in existing_generators:
        glc = g.lower()
        if not glc.startswith(mn_lc + "_"):
            continue
        if ft_lc in glc:
            hits.append(g)

    hits.sort()
    return hits


# -----------------------------
# MAIN STEP
# -----------------------------
def step_generator_derating_rating_factor():
    df = pd.read_csv(DERATING_SRC_CSV, dtype=str)

    required_cols = ["MARKET_NODE", "PEMMDB_FUEL_TYPE", "TARGET_YEAR", "Weighted Average Derating Ratio"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Derating source is missing columns: {missing}. Found: {list(df.columns)}")

    df["MARKET_NODE"] = df["MARKET_NODE"].map(_normalize_str)
    df["PEMMDB_FUEL_TYPE"] = df["PEMMDB_FUEL_TYPE"].map(_normalize_str)
    df["TARGET_YEAR"] = df["TARGET_YEAR"].map(_normalize_str)
    df["Weighted Average Derating Ratio"] = pd.to_numeric(df["Weighted Average Derating Ratio"], errors="coerce").fillna(0.0)

    target_years = sorted({int(y) for y in df["TARGET_YEAR"].unique().tolist() if str(y).strip().isdigit()})
    if not target_years:
        raise RuntimeError("No valid TARGET_YEAR values found in Derating_data.csv")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    RATING_FACTOR_ENUM_ID = _resolve_semantic_enum("System", "Generator", "Generators", "Rating Factor", 84)
    

    try:
        # Ensure scenario exists + scenario handle
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        existing_generators = set(_get_objects_safe(db, ClassEnum.Generator))
        if not existing_generators:
            raise RuntimeError("No Generator objects found in the PLEXOS DB (ClassEnum.Generator).")

        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=1
        )
        datafile_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "data", "file"],
            ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
            fallback_id=None
        )

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        value_map: Dict[Tuple[str, int], float] = {}
        matched_generators: set[str] = set()
        unmatched_pairs: Dict[Tuple[str, str], int] = {}

        for _, r in df.iterrows():
            mn = _normalize_str(r["MARKET_NODE"])
            ft_raw = _normalize_str(r["PEMMDB_FUEL_TYPE"])
            y_raw = _normalize_str(r["TARGET_YEAR"])
            ratio = float(r["Weighted Average Derating Ratio"])

            if not (mn and ft_raw and y_raw and y_raw.isdigit()):
                continue
            y = int(y_raw)

            fuel_for_name = _canon_fuel_for_generator_name(ft_raw)

            gens = _try_match_generators(existing_generators, mn, fuel_for_name)
            if not gens:
                gens = _try_match_generators(existing_generators, mn, ft_raw)

            if not gens:
                key = (mn, ft_raw)
                unmatched_pairs[key] = unmatched_pairs.get(key, 0) + 1
                continue

            rf_val = ratio * 100.0
            for g in gens:
                matched_generators.add(g)
                key = (g, y)
                value_map[key] = value_map.get(key, 0.0) + rf_val

        # Build output CSV:
        # - If there are matches: Year + generator columns
        # - If no matches: Year only (still write file; do not fail)
        if not matched_generators:
            print(
                "[WARN] No matches found between Derating_data.csv and existing generator names.\n"
                "       The script will NOT fail. It will write a Year-only output CSV and still create/link the DataFile object.\n"
                "       (Generator Rating Factor linking will be skipped.)"
            )
            if unmatched_pairs:
                top_unmatched = sorted(unmatched_pairs.items(), key=lambda kv: kv[1], reverse=True)[:15]
                print("[WARN] Example unmatched MARKET_NODE / PEMMDB_FUEL_TYPE pairs (top 15 by frequency):")
                for (mn, ft), cnt in top_unmatched:
                    print(f"       - {mn} / {ft}  (rows: {cnt})")

            out = pd.DataFrame({"Year": target_years})
            out.to_csv(OUTPUT_CSV_PATH, index=False)
            print(f"Wrote derating matrix CSV (Year-only): {OUTPUT_CSV_PATH}")

            # Ensure DataFile category + DataFile object + filename property (WITH scenario)
            try:
                db.AddCategory(ClassEnum.DataFile, str(DATAFILE_CATEGORY_NAME))
            except Exception:
                pass

            _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME, category=DATAFILE_CATEGORY_NAME)
            df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)

            _add_property_row(
                db, datafile_filename_enum, df_mem_id, 1, 0.0,
                DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                Scenario=scenario_str
            )

            print("Done.")
            print(f"Scenario used: {SCENARIO_NAME}")
            print(f"Linked Rating Factor (Enum {RATING_FACTOR_ENUM_ID}) to generators: 0")
            return

        generator_columns = sorted(matched_generators)

        out = pd.DataFrame({"Year": target_years})
        for g in generator_columns:
            out[g] = 0.0

        year_to_idx = {y: i for i, y in enumerate(target_years)}
        for (g, y), v in value_map.items():
            if y in year_to_idx and g in out.columns:
                out.at[year_to_idx[y], g] = float(v)

        out.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"Wrote derating matrix CSV: {OUTPUT_CSV_PATH}")
        print(f"Matched generators (columns created): {len(generator_columns)}")

        # DataFile category + DataFile object + filename property (now WITH scenario)
        try:
            db.AddCategory(ClassEnum.DataFile, str(DATAFILE_CATEGORY_NAME))
        except Exception:
            pass

        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME, category=DATAFILE_CATEGORY_NAME)
        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)

        _add_property_row(
            db, datafile_filename_enum, df_mem_id, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
            Scenario=scenario_str
        )

        # Link Rating Factor (Enum 83) to matched generators WITH scenario
        linked = 0
        for g in generator_columns:
            try:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                _add_property_row(
                    db, RATING_FACTOR_ENUM_ID, g_mem_id, 1, 0.0,
                    DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                    Scenario=scenario_str
                )
                linked += 1
            except Exception:
                pass

        print("Done.")
        print(f"Scenario used: {SCENARIO_NAME}")
        print(f"Linked Rating Factor (Enum {RATING_FACTOR_ENUM_ID}) to generators: {linked}")

        if unmatched_pairs:
            # Show a few unmatched pairs even when there ARE matches (useful debugging)
            top_unmatched = sorted(unmatched_pairs.items(), key=lambda kv: kv[1], reverse=True)[:10]
            print("[INFO] Some unmatched MARKET_NODE / PEMMDB_FUEL_TYPE pairs (top 10 by frequency):")
            for (mn, ft), cnt in top_unmatched:
                print(f"       - {mn} / {ft}  (rows: {cnt})")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_generator_derating_rating_factor()




In [ ]:
import os
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import pandas as pd

# ============================================================
# STEP â€” Explicit DSR Detailed -> NEW DSR Generators + Direct Properties
#
# CHANGE (ONLY):
#   - If no rows remain after filtering to allowed MARKET_NODEs,
#     DO NOT raise RuntimeError. Print a warning and exit gracefully.
#
# Everything else unchanged.
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
SRC_CSV = OTHER_DATA_DIR / "Explicit DSR detailed.csv"

# -----------------------------
# Scenario
# -----------------------------
SCENARIO_NAME = "DSR_Detailed"

# -----------------------------
# ENUMS (given by you)
# -----------------------------
ENUM_UNITS = 51
ENUM_MAX_CAPACITY = 52
ENUM_OFFER_QTY = 199
ENUM_OFFER_PRICE = 200

# -----------------------------
# Constraint-related enums (given by you)
# -----------------------------
ENUM_CONSTRAINT_OPERATING_HOURS_COEFF = 6       # on Constraint<->Generator (collection id 32)
ENUM_CONSTRAINT_SENSE = 1                       # on Constraint (System<->Constraint membership)
ENUM_CONSTRAINT_RHS_DAY = 16                    # on Constraint (System<->Constraint membership)
SENSE_LESS_EQUAL = -1.0                         # <=

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# HELPERS
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum, PeriodEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, category: str = "", description: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category=category, description=description)


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _ensure_membership_bi(db, collection_enum, a: str, b: str) -> bool:
    try:
        _ensure_membership(db, collection_enum, a, b)
        return True
    except Exception:
        pass
    try:
        _ensure_membership(db, collection_enum, b, a)
        return True
    except Exception:
        return False


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    if PeriodTypeId is None:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action
        )
    else:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action,
            PeriodTypeId
        )


def _find_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


def _resolve_class_enum_by_tokens(ClassEnum, required_tokens_lc: List[str], preferred_names: List[str]) -> Optional[int]:
    for n in preferred_names:
        if hasattr(ClassEnum, n):
            return getattr(ClassEnum, n)

    candidates = []
    for attr in dir(ClassEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(ClassEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            candidates.append((len(name_lc), attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved ClassEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    return None


def _norm_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _sanitize_name(s: str) -> str:
    s = str(s).strip()
    s = s.replace("/", "_").replace("\\", "_").replace(":", "_")
    s = s.replace("(", "").replace(")", "")
    s = s.replace("[", "").replace("]", "")
    s = s.replace("{", "").replace("}", "")
    s = s.replace(" ", "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s


def _format_price_token(v) -> str:
    try:
        x = float(v)
    except Exception:
        return _sanitize_name(str(v))

    if abs(x) < 1e-12:
        x = 0.0

    sign = "m" if x < 0 else ""
    x = abs(x)

    if float(int(round(x))) == x:
        return f"{sign}{int(x)}"

    s = f"{x:.6f}".rstrip("0").rstrip(".")
    s = s.replace(".", "p")
    return f"{sign}{s}"


def _unique_name(base: str, existing: set[str]) -> str:
    if base not in existing:
        return base
    i = 2
    while True:
        cand = f"{base}_{i}"
        if cand not in existing:
            return cand
        i += 1


def _hours_group(hours_val: float) -> Optional[float]:
    try:
        h = float(hours_val)
    except Exception:
        return None
    if abs(h - 24.0) <= 1e-9:
        return None
    return float(round(h, 6))


# -----------------------------
# MAIN STEP
# -----------------------------
def step_explicit_dsr_detailed_generators():
    allowed_nodes = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed_nodes:
        raise RuntimeError("No nodes found in Bidding_Zone_List.xlsx")

    df = pd.read_csv(SRC_CSV, dtype=str)

    required_cols = [
        "MARKET_NODE",
        "DA activation price (EUR/MWh)",
        "Max hours dispatched per day [h]",
        "MAX CAP (MW)",
        "TARGET_YEAR",
        "Country",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in source CSV: {missing}. Found: {list(df.columns)}")

    df["MARKET_NODE"] = df["MARKET_NODE"].map(_norm_str)
    df["TARGET_YEAR"] = df["TARGET_YEAR"].map(_norm_str)

    df["DA activation price (EUR/MWh)"] = pd.to_numeric(df["DA activation price (EUR/MWh)"], errors="coerce").fillna(0.0)
    df["Max hours dispatched per day [h]"] = pd.to_numeric(df["Max hours dispatched per day [h]"], errors="coerce").fillna(24.0)
    df["MAX CAP (MW)"] = pd.to_numeric(df["MAX CAP (MW)"], errors="coerce").fillna(0.0)

    before = len(df)
    df = df[df["MARKET_NODE"].isin(allowed_nodes)].copy()
    removed = before - len(df)
    if removed > 0:
        print(f"Filtered out {removed} rows because MARKET_NODE not in Bidding_Zone_List.xlsx")

    # =========================
    # CHANGE (ONLY): do NOT crash if empty after filtering
    # =========================
    if df.empty:
        print("[WARN] No rows remain after filtering to allowed MARKET_NODEs.")
        print("       This can be expected if no Explicit DSR entries exist for the current bidding zone list.")
        print("       No DSR generators/constraints/reserve memberships were created or updated.")
        return
    # =========================

    df = df[df["TARGET_YEAR"].str.match(r"^\d+$", na=False)].copy()
    if df.empty:
        raise RuntimeError("No rows remain after filtering to numeric TARGET_YEARs.")

    db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_OFFER_QTY = _resolve_semantic_enum("System", "Generator", "Generators", "Offer Quantity", 199)
    ENUM_OFFER_PRICE = _resolve_semantic_enum("System", "Generator", "Generators", "Offer Price", 200)
    ENUM_UNITS = _resolve_semantic_enum("System", "Generator", "Generators", "Units", 51)
    ENUM_MAX_CAPACITY = _resolve_semantic_enum("System", "Generator", "Generators", "Max Capacity", 52)
    

    try:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=1
        )

        gen_node_collection = None
        if hasattr(CollectionEnum, "NodeGenerators"):
            gen_node_collection = getattr(CollectionEnum, "NodeGenerators")
        elif hasattr(CollectionEnum, "GeneratorNodes"):
            gen_node_collection = getattr(CollectionEnum, "GeneratorNodes")
        elif hasattr(CollectionEnum, "PowerStationNodes"):
            gen_node_collection = getattr(CollectionEnum, "PowerStationNodes")

        nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
        gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))

        # --- Reserve linking ---
        reserve_class = _resolve_class_enum_by_tokens(
            ClassEnum,
            ["reserve"],
            ["Reserve", "Reserves"]
        )
        reserves_in_db = set(_get_objects_safe(db, reserve_class)) if reserve_class is not None else set()

        reserve_gen_collection = None
        if hasattr(CollectionEnum, "ReserveGenerators"):
            reserve_gen_collection = getattr(CollectionEnum, "ReserveGenerators")
        elif hasattr(CollectionEnum, "GeneratorReserves"):
            reserve_gen_collection = getattr(CollectionEnum, "GeneratorReserves")
        else:
            try:
                reserve_gen_collection = _find_collection_enum(
                    CollectionEnum,
                    ["reserve", "generator"],
                    ["ReserveGenerators", "GeneratorReserves"],
                    fallback_id=None
                )
            except Exception:
                reserve_gen_collection = None

        # --- Constraint class + collections ---
        constraint_class = _resolve_class_enum_by_tokens(
            ClassEnum,
            ["constraint"],
            ["Constraint", "Constraints"]
        )

        system_constraint_collection = None
        try:
            system_constraint_collection = _find_collection_enum(
                CollectionEnum,
                ["system", "constraint"],
                ["SystemConstraints", "Constraints"],
                fallback_id=759
            )
        except Exception:
            system_constraint_collection = int(759)

        constraint_generator_collection = None
        try:
            constraint_generator_collection = _find_collection_enum(
                CollectionEnum,
                ["constraint", "generator"],
                ["ConstraintGenerators", "GeneratorConstraints"],
                fallback_id=32
            )
        except Exception:
            constraint_generator_collection = int(32)

        constraints_in_db = set(_get_objects_safe(db, constraint_class)) if constraint_class is not None else set()

        # 1) Existing <MARKET_NODE>_DSR generators: add Units=0 with Scenario only
        updated_existing = 0
        for mn in sorted(set(df["MARKET_NODE"].tolist())):
            gname = f"{mn}_DSR"
            if gname not in gens_in_db:
                continue
            mem_id = _ensure_membership(db, gen_mem_collection, "System", gname)
            _add_property_row(db, ENUM_UNITS, mem_id, 1, 0.0, Scenario=scenario_str)
            updated_existing += 1

        # 2) New generators: group by (MARKET_NODE, price_token, hours_group)
        created_new = 0
        linked_nodes = 0
        prop_rows_added = 0

        created_gen_to_mn: Dict[str, str] = {}
        gen_daily_activation_hours: Dict[str, float] = {}

        df["_price_tok"] = df["DA activation price (EUR/MWh)"].map(_format_price_token)
        df["_hours_group"] = df["Max hours dispatched per day [h]"].map(_hours_group)

        df = df.sort_values(["MARKET_NODE", "_price_tok", "_hours_group", "TARGET_YEAR"]).reset_index(drop=True)

        for (mn, price_tok, hgrp), gdf in df.groupby(["MARKET_NODE", "_price_tok", "_hours_group"], dropna=False):
            mn = str(mn).strip()
            price_tok = str(price_tok).strip()

            if pd.isna(hgrp):
                hgrp = None
            else:
                try:
                    hgrp = float(hgrp)
                except Exception:
                    hgrp = None

            base = _sanitize_name(f"{mn}_DSR_{price_tok}")
            if hgrp is not None:
                htok = _sanitize_name(f"h{_format_price_token(hgrp)}")
                base = _sanitize_name(f"{base}_{htok}")

            new_name = _unique_name(base, gens_in_db)

            # Create in Category = MARKET_NODE
            if new_name not in gens_in_db:
                _add_object(db, ClassEnum.Generator, new_name, add_to_system=True, category=mn, description="")
                gens_in_db.add(new_name)
                created_new += 1
                created_gen_to_mn[new_name] = mn

            mem_id = _ensure_membership(db, gen_mem_collection, "System", new_name)

            # Units for NEW generators:
            _add_property_row(db, ENUM_UNITS, mem_id, 1, 0.0)
            _add_property_row(db, ENUM_UNITS, mem_id, 1, 1.0, Scenario=scenario_str)
            prop_rows_added += 2

            # Node membership to MARKET_NODE
            if gen_node_collection is not None and mn in nodes_in_db:
                if _ensure_membership_bi(db, gen_node_collection, mn, new_name):
                    linked_nodes += 1

            # If hours != 24 for this generator group, remember to create constraint
            if hgrp is not None:
                gen_daily_activation_hours[new_name] = float(hgrp)

            # Year-specific property rows (Offer qty/price, max capacity)
            for _, r in gdf.iterrows():
                y = int(str(r["TARGET_YEAR"]).strip())
                dt_from = NetDateTime(y, 1, 1, 0, 0, 0)

                offer_qty = float(r["MAX CAP (MW)"])
                offer_price = float(r["DA activation price (EUR/MWh)"])

                _add_property_row(db, ENUM_OFFER_QTY, mem_id, 1, offer_qty, DateFrom=dt_from)
                _add_property_row(db, ENUM_OFFER_PRICE, mem_id, 1, offer_price, DateFrom=dt_from)
                _add_property_row(db, ENUM_MAX_CAPACITY, mem_id, 1, offer_qty, DateFrom=dt_from)
                prop_rows_added += 3

        # Create Constraints for generators with daily activation (hours != 24)
        constraints_created = 0
        constraint_gen_links = 0
        constraint_prop_rows = 0

        if constraint_class is None:
            print("[WARN] Could not resolve ClassEnum for Constraint objects. Daily activation constraints were not created.")
        else:
            for gname, hours in gen_daily_activation_hours.items():
                cname = f"{gname}_Daily_Activation"

                if cname not in constraints_in_db:
                    _add_object(db, constraint_class, cname, add_to_system=True, category="", description="")
                    constraints_in_db.add(cname)
                    constraints_created += 1

                c_sys_mem = _ensure_membership(db, system_constraint_collection, "System", cname)

                # --- KEY CHANGE: force "RHS Day" by writing RHS enum with PeriodTypeId=Day ---
                _add_property_row(
                    db,
                    ENUM_CONSTRAINT_RHS_DAY,
                    c_sys_mem,
                    1,
                    float(hours),
                    PeriodTypeId=PeriodEnum.Day
                )
                _add_property_row(db, ENUM_CONSTRAINT_SENSE, c_sys_mem, 1, float(SENSE_LESS_EQUAL))
                constraint_prop_rows += 2

                if _ensure_membership_bi(db, constraint_generator_collection, cname, gname):
                    try:
                        cg_mem = int(db.GetMembershipID(constraint_generator_collection, cname, gname))
                    except Exception:
                        cg_mem = int(db.GetMembershipID(constraint_generator_collection, gname, cname))

                    _add_property_row(db, ENUM_CONSTRAINT_OPERATING_HOURS_COEFF, cg_mem, 1, 1.0)
                    constraint_prop_rows += 1
                    constraint_gen_links += 1

        # Reserve memberships between created DSR generators and matching Reserve objects
        reserve_links_added = 0
        if reserve_gen_collection is None:
            print("[WARN] Could not resolve a Reserve<->Generator membership collection enum (ReserveGenerators/GeneratorReserves). Reserve memberships were not created.")
        elif reserve_class is None:
            print("[WARN] Could not resolve ClassEnum for Reserve objects. Reserve memberships were not created.")
        elif not reserves_in_db:
            print("[WARN] No Reserve objects found in the database. Reserve memberships were not created.")
        else:
            reserves_by_mn: Dict[str, List[str]] = {}
            for rname in reserves_in_db:
                try:
                    r = str(rname)
                except Exception:
                    continue
                if "_" not in r:
                    continue
                mn_prefix = r.split("_", 1)[0].strip()
                if not mn_prefix:
                    continue
                reserves_by_mn.setdefault(mn_prefix, []).append(r)

            for gname, mn in created_gen_to_mn.items():
                cand_reserves = reserves_by_mn.get(mn, [])
                for rname in cand_reserves:
                    if _ensure_membership_bi(db, reserve_gen_collection, rname, gname):
                        reserve_links_added += 1

        print("Done.")
        print(f"Scenario ensured: {SCENARIO_NAME}")
        print(f"Allowed MARKET_NODE count (from Bidding_Zone_List.xlsx): {len(allowed_nodes)}")
        print(f"Updated existing <MARKET_NODE>_DSR generators (Units=0 with scenario only): {updated_existing}")
        print(f"Created NEW DSR detailed generators: {created_new}")
        print(f"Linked Node memberships (new generators): {linked_nodes}")
        print(f"Daily Activation constraints created: {constraints_created}")
        print(f"Constraint<->Generator links created: {constraint_gen_links}")
        print(f"Constraint property rows added (RHS Day, Sense, Coeff): {constraint_prop_rows}")
        print(f"Reserve memberships created (total links): {reserve_links_added}")
        print(f"Total generator property rows added: {prop_rows_added}")

        if gen_node_collection is None:
            print("[WARN] Could not find a Generator<->Node membership collection enum (NodeGenerators/GeneratorNodes/PowerStationNodes). Node memberships may be missing.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_explicit_dsr_detailed_generators()




In [ ]:
import os
from pathlib import Path
from typing import Optional, Dict, Tuple, List, Set
import pandas as pd

# ============================================================
# STEP â€” Forced Outage Rate (Detailed) -> Data File + PLEXOS property linking
#
# ONLY CHANGE vs prior version:
#   - MTTR written to Band 4 is now fixed by technology:
#       * Nuclear -> 168
#       * All other technologies -> 24
#     MTTR is written with Scenario "Forced_Outage_Detailed" (same as this step).
#
# Everything else unchanged.
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
FORCED_OUTAGE_SRC_CSV = Path(
    str(OTHER_DATA_DIR / "Forced_outage_rate_extraction.csv")
)
BIDDING_ZONE_XLSX = Path(
    str(BIDDING_ZONE_XLSX)
)


OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_NAME = "Generator_Forced_Outage_Rate_Detailed.csv"
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PLEXOS_REL_CSV_PATH = r"Data Files\Generator Data\Generator_Forced_Outage_Rate_Detailed.csv"

DATAFILE_OBJECT_NAME = "Generator_Forced_Outage_Rate_Detailed"
DATAFILE_CATEGORY_NAME = "Other_Detailed_Data"

# Scenario for all rows created in this step
SCENARIO_NAME = "Forced_Outage_Detailed"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# HELPERS
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum, PeriodEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, category: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category=category, description="")


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    if PeriodTypeId is None:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action
        )
    else:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action,
            PeriodTypeId
        )


def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = [
        "Data Files", "DataFiles", "SystemDataFiles", "System Data Files",
        "SystemDataFile", "DataFile"
    ]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue
    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _find_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


def _normalize_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _canon_tech_for_generator_name(tech: str) -> str:
    t = _normalize_str(tech)
    tl = t.lower()
    mapping = {
        "gas": "Natural gas",
        "natural gas": "Natural gas",
        "hard coal": "Hard coal",
        "coal": "Hard coal",
        "lignite": "Lignite",
        "oil": "Oil",
        "fuel oil": "Oil",
        "diesel": "Oil",
        "nuclear": "Nuclear",
        "biomass": "Biomass",
        "waste": "Waste",
        "geothermal": "Geothermal",
        "oil shale": "Oil Shale",
        "shale oil": "Oil Shale",
        "wind": "Wind",
        "solar": "Solar",
        "hydro": "Hydro",
        "water": "Hydro",
        "ccgt": "CCGT",
        "ocgt": "OCGT",
        "chp": "CHP",
    }
    return mapping.get(tl, t)


def _try_match_generators(existing_generators: Set[str], market_node: str, tech_text: str) -> List[str]:
    mn = _normalize_str(market_node)
    tt = _normalize_str(tech_text)
    if not mn or not tt:
        return []

    exact = f"{mn}_{tt}"
    if exact in existing_generators:
        return [exact]

    variants = {
        f"{mn}_{tt.title()}",
        f"{mn}_{tt.upper()}",
        f"{mn}_{tt.lower()}",
    }
    for v in variants:
        if v in existing_generators:
            return [v]

    mn_lc = mn.lower()
    tt_lc = tt.lower()

    hits = []
    for g in existing_generators:
        glc = g.lower()
        if not glc.startswith(mn_lc + "_"):
            continue
        if tt_lc in glc:
            hits.append(g)

    hits.sort()
    return hits


def _resolve_generator_property_enum(db, SystemNS, prop_candidates: List[str]) -> int:
    child_candidates = ["Generator", "Generators"]
    collection_candidates = ["SystemGenerators", "Generators", "System Generators"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop),
                    ))
                    return enum_id
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue

    raise RuntimeError(
        f"Could not resolve Generator property enum id for candidates={prop_candidates}. "
        f"Last attempt: {last}"
    )


def _load_bidding_zones_from_xlsx(xlsx: Path) -> Set[str]:
    if not xlsx.exists():
        raise FileNotFoundError(f"Bidding zone list not found: {xlsx}")

    xdf = pd.read_excel(xlsx, dtype=str)
    if xdf.empty:
        raise RuntimeError(f"Bidding zone list is empty: {xlsx}")

    cols = list(xdf.columns)
    cols_lc = [str(c).lower() for c in cols]
    preferred = None
    for key in ["bidding", "market_node", "market node", "node", "zone", "bidding zone", "market"]:
        for c, cl in zip(cols, cols_lc):
            if key in cl:
                preferred = c
                break
        if preferred is not None:
            break

    use_col = preferred if preferred is not None else cols[0]
    zones = set(_normalize_str(v) for v in xdf[use_col].tolist())
    zones = {z for z in zones if z}
    if not zones:
        raise RuntimeError(f"Could not read any bidding zones from column '{use_col}' in {xlsx}")
    print(f"[INFO] Read {len(zones)} bidding zones from '{xlsx.name}' (column='{use_col}')")
    return zones


# -----------------------------
# ONLY-CHANGED PART: MTTR values by Technology
# -----------------------------
def _mttr_hours_from_technology(tech: str) -> float:
    """
    Per your instruction:
      - Nuclear -> 168
      - all other technologies -> 24
    """
    t = _normalize_str(tech).lower()
    if "nuclear" in t:
        return 168.0
    return 24.0


# -----------------------------
# MAIN STEP
# -----------------------------
def step_forced_outage_detailed():
    bidding_zones = _load_bidding_zones_from_xlsx(BIDDING_ZONE_XLSX)

    df = pd.read_csv(FORCED_OUTAGE_SRC_CSV, dtype=str)

    required_cols = [
        "MARKET_NODE",
        "Technology",
        "TARGET_YEAR",
        "Weighted Average Forced Outage Rate",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Forced outage source is missing columns: {missing}. Found: {list(df.columns)}")

    df["MARKET_NODE"] = df["MARKET_NODE"].map(_normalize_str)
    df["Technology"] = df["Technology"].map(_normalize_str)
    df["TARGET_YEAR"] = df["TARGET_YEAR"].map(_normalize_str)
    df["Weighted Average Forced Outage Rate"] = pd.to_numeric(
        df["Weighted Average Forced Outage Rate"], errors="coerce"
    ).fillna(0.0)

    before = len(df)
    df = df[df["MARKET_NODE"].isin(bidding_zones)].copy()
    after = len(df)
    print(f"[INFO] Filtered to bidding zones: {after}/{before} rows kept")

    target_years = sorted({int(y) for y in df["TARGET_YEAR"].unique().tolist() if str(y).strip().isdigit()})
    if not target_years:
        raise RuntimeError("No valid TARGET_YEAR values found in Forced_outage_rate_extraction.csv (after filtering).")

    db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        existing_generators = set(_get_objects_safe(db, ClassEnum.Generator))
        if not existing_generators:
            raise RuntimeError("No Generator objects found in the PLEXOS DB (ClassEnum.Generator).")

        gen_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=1
        )
        datafile_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "data", "file"],
            ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
            fallback_id=None
        )

        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"[INFO] Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        forced_outage_enum = _resolve_generator_property_enum(
            db, SystemNS,
            prop_candidates=[
                "Forced Outage Rate",
                "Forced outage rate",
                "ForcedOutageRate",
                "FOR",
            ]
        )

        # Reuse the exact MTTR enum logic/value from Step 4
        mttr_enum = int(ENUM_MTTR)

        print(f"[INFO] Using Generator property enums: Forced Outage Rate={forced_outage_enum}, Mean Time to Repair={mttr_enum} (explicit)")

        # Build forced outage matrix + capture a representative technology per generator for MTTR writing
        value_map: Dict[Tuple[str, int], float] = {}
        matched_generators: Set[str] = set()
        warned_pairs: Set[Tuple[str, str]] = set()

        # NEW (but only for MTTR writing): generator -> mttr hours (from technology rule)
        gen_to_mttr_hours: Dict[str, float] = {}

        for _, r in df.iterrows():
            mn = _normalize_str(r["MARKET_NODE"])
            tech_raw = _normalize_str(r["Technology"])
            y_raw = _normalize_str(r["TARGET_YEAR"])
            for_ratio = float(r["Weighted Average Forced Outage Rate"])

            if not (mn and tech_raw and y_raw and y_raw.isdigit()):
                continue

            y = int(y_raw)

            tech_for_name = _canon_tech_for_generator_name(tech_raw)
            gens = _try_match_generators(existing_generators, mn, tech_for_name)
            if not gens:
                gens = _try_match_generators(existing_generators, mn, tech_raw)

            if not gens:
                key = (mn, tech_raw)
                if key not in warned_pairs:
                    warned_pairs.add(key)
                    print(f"[WARN] No generator match -> ignored: MARKET_NODE={mn!r}, Technology={tech_raw!r}")
                continue

            val = for_ratio * 100.0
            mttr_h = _mttr_hours_from_technology(tech_raw)

            for g in gens:
                matched_generators.add(g)
                k = (g, y)
                value_map[k] = value_map.get(k, 0.0) + float(val)

                # store MTTR rule value for this generator (prefer Nuclear=168 if any row indicates Nuclear)
                if g not in gen_to_mttr_hours:
                    gen_to_mttr_hours[g] = mttr_h
                else:
                    if mttr_h > gen_to_mttr_hours[g]:
                        gen_to_mttr_hours[g] = mttr_h

        if not matched_generators:
            raise RuntimeError(
                "No matches found between Forced_outage_rate_extraction.csv and existing generator names "
                "(after bidding zone filtering)."
            )

        generator_columns = sorted(matched_generators)
        out = pd.DataFrame({"Year": target_years})
        for g in generator_columns:
            out[g] = 0.0

        year_to_idx = {y: i for i, y in enumerate(target_years)}
        for (g, y), v in value_map.items():
            if y in year_to_idx and g in out.columns:
                out.at[year_to_idx[y], g] = float(v)

        out.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"[INFO] Wrote forced outage matrix CSV: {OUTPUT_CSV_PATH}")
        print(f"[INFO] Matched generators (columns created): {len(generator_columns)}")

        # DataFile category + object + filename property row WITH scenario (unchanged)
        try:
            db.AddCategory(ClassEnum.DataFile, str(DATAFILE_CATEGORY_NAME))
        except Exception:
            pass

        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME, category=DATAFILE_CATEGORY_NAME)
        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)

        _add_property_row(
            db, datafile_filename_enum, df_mem_id, 1, 0.0,
            DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
            Scenario=scenario_str
        )

        # Link Forced Outage Rate (Band 4) to generators using the DataFile + Scenario,
        # and write MTTR (Band 4) using the fixed technology rule + Scenario
        linked_for = 0
        linked_mttr = 0

        for g in generator_columns:
            try:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)

                _add_property_row(
                    db, forced_outage_enum, g_mem_id, 1, 0.0,
                    DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                    Scenario=scenario_str
                )
                linked_for += 1

                mttr_h = float(gen_to_mttr_hours.get(g, 24.0))
                _add_property_row(
                    db, mttr_enum, g_mem_id, 1, mttr_h,
                    Scenario=scenario_str
                )
                linked_mttr += 1

            except Exception:
                pass

        print("Done.")
        print(f"Scenario used: {SCENARIO_NAME}")
        print(f"Linked Forced Outage Rate (Enum {forced_outage_enum}) Band 4 to generators: {linked_for}")
        print(f"Wrote MTTR (Enum {mttr_enum}) Band 4 (Scenario) to generators: {linked_mttr}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_forced_outage_detailed()




In [ ]:
import os
from pathlib import Path
from typing import Optional, Dict, Tuple, List, Set
import pandas as pd

# ============================================================
# STEP â€” Must-run Detailed -> Min Stable Factor (SINGLE year-indexed DataFile)
#
# CHANGE (ONLY):
#   - If df becomes empty after filtering to bidding zones,
#     DO NOT raise RuntimeError. Print a warning and exit gracefully.
#
# Everything else unchanged.
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
MUST_RUN_SRC_CSV = OTHER_DATA_DIR / "Must-run_data.csv"

OUTPUT_DIR = DATA_FILES_ROOT / "Generator Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_NAME = "Generator_Must_Run_Detailed.csv"
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV_NAME
PLEXOS_REL_CSV_PATH = r"Data Files\Generator Data\Generator_Must_Run_Detailed.csv"

DATAFILE_OBJECT_NAME = "Generator_Must_Run_Detailed"
DATAFILE_CATEGORY_NAME = "Other_Detailed_Data"

# Scenario
SCENARIO_NAME = "Must_Run_Detailed"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------

CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS = [API_DIR / "PLEXOS_NET.dll", BIN_DIR / "PLEXOS_NET.dll"]


# -----------------------------
# HELPERS (PLEXOS .NET bootstrap)
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime

    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, category: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category=category, description="")


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _add_property_row(
    db,
    enum_id: int,
    mem_id: int,
    band: int,
    value: float,
    DateFrom=None,
    DateTo=None,
    Variable=None,
    DataFile=None,
    Pattern=None,
    Scenario=None,
    Action=None,
    PeriodTypeId=None,
):
    if PeriodTypeId is None:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action,
        )
    else:
        db.AddProperty(
            int(mem_id),
            int(enum_id),
            int(band),
            float(value),
            DateFrom,
            DateTo,
            Variable,
            DataFile,
            Pattern,
            Scenario,
            Action,
            PeriodTypeId,
        )


def _find_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError(
        f"Could not resolve CollectionEnum. required_tokens={required_tokens_lc}, preferred_names={preferred_names}, fallback_id={fallback_id}"
    )


# -----------------------------
# HELPERS (string + matching)
# -----------------------------
def _normalize_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _canon_tech_for_generator_name(tech: str) -> str:
    t = _normalize_str(tech)
    tl = t.lower()
    mapping = {
        "gas": "Natural gas",
        "natural gas": "Natural gas",
        "hard coal": "Hard coal",
        "coal": "Hard coal",
        "lignite": "Lignite",
        "oil": "Oil",
        "fuel oil": "Oil",
        "diesel": "Oil",
        "nuclear": "Nuclear",
        "biomass": "Biomass",
        "waste": "Waste",
        "geothermal": "Geothermal",
        "oil shale": "Oil Shale",
        "hydro": "Hydro",
        "wind": "Wind",
        "solar": "Solar",
    }
    if tl in mapping:
        return mapping[tl]
    return t


def _try_match_generators(existing_generators: Set[str], market_node: str, tech_text: str) -> List[str]:
    mn = _normalize_str(market_node)
    tt = _normalize_str(tech_text)
    if not mn or not tt:
        return []

    exact = f"{mn}_{tt}"
    if exact in existing_generators:
        return [exact]

    mn_lc = mn.lower()
    tt_lc = tt.lower()
    hits = []
    for g in existing_generators:
        glc = g.lower()
        if not glc.startswith(mn_lc + "_"):
            continue
        if tt_lc in glc:
            hits.append(g)
    hits.sort()
    return hits


# -----------------------------
# HELPERS (Bidding zone list)
# -----------------------------
def _load_bidding_zones() -> Set[str]:
    xdf = pd.read_excel(BIDDING_ZONE_XLSX, dtype=str)
    if xdf.empty:
        raise RuntimeError(f"Bidding zone file is empty: {BIDDING_ZONE_XLSX}")

    cols = list(xdf.columns)
    cols_lc = {str(c).strip().lower(): c for c in cols}

    preferred = [
        "bidding zone",
        "bidding_zone",
        "biddingzone",
        "market node",
        "market_node",
        "marketnode",
        "zone",
        "code",
    ]

    chosen_col = None
    for p in preferred:
        if p in cols_lc:
            chosen_col = cols_lc[p]
            break

    if chosen_col is None:
        for c in cols:
            cl = str(c).strip().lower()
            if "bidding" in cl and "zone" in cl:
                chosen_col = c
                break

    if chosen_col is None:
        for c in cols:
            cl = str(c).strip().lower()
            if "market" in cl and "node" in cl:
                chosen_col = c
                break

    if chosen_col is None:
        chosen_col = cols[0]
        print(f"[WARN] Could not find a standard bidding zone column; using first column: {chosen_col!r}")
    else:
        print(f"[INFO] Using bidding zone column: {chosen_col!r}")

    zones = set(_normalize_str(x) for x in xdf[chosen_col].tolist())
    zones = {z for z in zones if z}
    if not zones:
        raise RuntimeError(f"No bidding zones found in column {chosen_col!r} of {BIDDING_ZONE_XLSX}")
    return zones


# -----------------------------
# HELPERS (Enum resolution)
# -----------------------------
def _resolve_datafile_filename_enum(db, SystemNS):
    child_candidates = ["Data File", "DataFile", "Data Files"]
    collection_candidates = ["Data Files", "DataFiles", "SystemDataFiles", "System Data Files", "SystemDataFile", "DataFile"]
    prop_candidates = ["Filename", "File Name", "File", "Path", "File Path", "FilePath"]

    last = None
    for child in child_candidates:
        for col in collection_candidates:
            for prop in prop_candidates:
                try:
                    enum_id = int(
                        db.PropertyName2EnumId(
                            SystemNS.String("System"),
                            SystemNS.String(child),
                            SystemNS.String(col),
                            SystemNS.String(prop),
                        )
                    )
                    return enum_id, col, prop, child
                except Exception as e:
                    last = (child, col, prop, str(e))
                    continue

    raise RuntimeError(
        "Could not resolve Data File filename enum id.\n"
        f"Last attempt: {last}"
    )


def _resolve_generator_property_enum(db, SystemNS, prop_name: str):
    child_candidates = ["Generator", "Generators"]
    collection_candidates = ["SystemGenerators", "Generators", "System Generator", "System Generators"]
    last = None
    for child in child_candidates:
        for col in collection_candidates:
            try:
                enum_id = int(
                    db.PropertyName2EnumId(
                        SystemNS.String("System"),
                        SystemNS.String(child),
                        SystemNS.String(col),
                        SystemNS.String(prop_name),
                    )
                )
                return enum_id, col, prop_name, child
            except Exception as e:
                last = (child, col, prop_name, str(e))
                continue

    raise RuntimeError(
        f"Could not resolve Generator property enum id for property={prop_name!r}.\n"
        f"Last attempt: {last}"
    )


# -----------------------------
# MAIN STEP
# -----------------------------
def step_must_run_min_stable_factor():
    zones = _load_bidding_zones()

    df = pd.read_csv(MUST_RUN_SRC_CSV, dtype=str)
    required_cols = ["MARKET_NODE", "Technology", "TARGET_YEAR", "Weighted Average Must-run Ratio"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Must-run source is missing columns: {missing}. Found: {list(df.columns)}")

    df["MARKET_NODE"] = df["MARKET_NODE"].map(_normalize_str)
    df["Technology"] = df["Technology"].map(_normalize_str)
    df["TARGET_YEAR"] = df["TARGET_YEAR"].map(_normalize_str)
    df["Weighted Average Must-run Ratio"] = pd.to_numeric(df["Weighted Average Must-run Ratio"], errors="coerce").fillna(0.0)

    # Filter to bidding zones
    df = df[df["MARKET_NODE"].isin(zones)].copy()

    # =========================
    # CHANGE (ONLY): do NOT crash if empty after filtering
    # =========================
    if df.empty:
        print("[WARN] After filtering to bidding zones from Bidding_Zone_List.xlsx, there are no rows to import.")
        print("       This can be expected if there is no Must-run data for the current bidding zone list.")
        print("       No CSV/DataFile/Min Stable Factor links were created.")
        return
    # =========================

    target_years = sorted({int(y) for y in df["TARGET_YEAR"].unique().tolist() if str(y).strip().isdigit()})
    if not target_years:
        raise RuntimeError("No valid TARGET_YEAR values found in Must-run_data.csv")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        # Ensure scenario exists
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        existing_generators = set(_get_objects_safe(db, ClassEnum.Generator))
        if not existing_generators:
            raise RuntimeError("No Generator objects found in the PLEXOS DB (ClassEnum.Generator).")

        # Collections
        gen_mem_collection = _find_collection_enum(CollectionEnum, ["system", "generator"], ["SystemGenerators", "Generators"], fallback_id=1)
        datafile_mem_collection = _find_collection_enum(
            CollectionEnum,
            ["system", "data", "file"],
            ["SystemDataFiles", "DataFiles", "SystemDataFile", "Data Files"],
            fallback_id=None,
        )

        # Resolve enum ids
        datafile_filename_enum, df_col_used, df_prop_used, df_child_used = _resolve_datafile_filename_enum(db, SystemNS)
        print(f"[INFO] Resolved DataFile filename EnumId={datafile_filename_enum} using child={df_child_used!r}, collection={df_col_used!r}, property={df_prop_used!r}")

        min_stable_factor_enum, gcol_used, gprop_used, gchild_used = _resolve_generator_property_enum(db, SystemNS, "Min Stable Factor")
        print(f"[INFO] Resolved Generator property 'Min Stable Factor' EnumId={min_stable_factor_enum} using child={gchild_used!r}, collection={gcol_used!r}, property={gprop_used!r}")

        # Build value_map[(generator, year)] = value
        value_map: Dict[Tuple[str, int], float] = {}
        matched_generators: Set[str] = set()
        warned_unmatched: Set[Tuple[str, str]] = set()

        for _, r in df.iterrows():
            mn = _normalize_str(r["MARKET_NODE"])
            tech_raw = _normalize_str(r["Technology"])
            y_raw = _normalize_str(r["TARGET_YEAR"])
            ratio = float(r["Weighted Average Must-run Ratio"])

            if not (mn and tech_raw and y_raw and y_raw.isdigit()):
                continue
            y = int(y_raw)

            tech_for_name = _canon_tech_for_generator_name(tech_raw)
            gens = _try_match_generators(existing_generators, mn, tech_for_name)
            if not gens:
                gens = _try_match_generators(existing_generators, mn, tech_raw)

            if not gens:
                key = (mn, tech_raw)
                if key not in warned_unmatched:
                    print(f"[WARN] Ignored (no generator match): MARKET_NODE={mn!r}, Technology={tech_raw!r}")
                    warned_unmatched.add(key)
                continue

            v = ratio * 100.0  # IMPORTANT: multiply by 100 before PLEXOS
            for g in gens:
                matched_generators.add(g)
                # If multiple rows map to same generator/year, overwrite with last seen
                value_map[(g, y)] = float(v)

        if not matched_generators:
            raise RuntimeError(
                "No matches found between Must-run_data.csv and existing generator names.\n"
                "Matching rule tried:\n"
                " 1) {MARKET_NODE}_{Technology (normalized)}\n"
                " 2) any generator starting with '{MARKET_NODE}_' and containing Technology text"
            )

        # Write ONE matrix CSV with Year column (like your forced outage detailed file)
        generator_columns = sorted(matched_generators)
        out = pd.DataFrame({"Year": target_years})
        for g in generator_columns:
            out[g] = 0.0

        year_to_idx = {y: i for i, y in enumerate(target_years)}
        for (g, y), v in value_map.items():
            if y in year_to_idx and g in out.columns:
                out.at[year_to_idx[y], g] = float(v)

        out.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"[INFO] Wrote must-run matrix CSV: {OUTPUT_CSV_PATH}")
        print(f"[INFO] Matched generators (columns created): {len(generator_columns)}")

        # DataFile category + DataFile object + filename property (Scenario-tagged)
        try:
            db.AddCategory(ClassEnum.DataFile, str(DATAFILE_CATEGORY_NAME))
        except Exception:
            pass

        _ensure_object(db, ClassEnum.DataFile, DATAFILE_OBJECT_NAME, category=DATAFILE_CATEGORY_NAME)

        df_mem_id = _ensure_membership(db, datafile_mem_collection, "System", DATAFILE_OBJECT_NAME)
        _add_property_row(
            db,
            datafile_filename_enum,
            df_mem_id,
            1,
            0.0,
            DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
            Scenario=scenario_str,
        )

        # Link "Min Stable Factor" to the DataFile ONCE per generator (NO DateFrom/DateTo), Scenario-tagged
        linked = 0
        for g in generator_columns:
            try:
                g_mem_id = _ensure_membership(db, gen_mem_collection, "System", g)
                _add_property_row(
                    db,
                    min_stable_factor_enum,
                    g_mem_id,
                    1,
                    0.0,
                    DataFile=SystemNS.String(PLEXOS_REL_CSV_PATH),
                    Scenario=scenario_str,
                )
                linked += 1
            except Exception:
                pass

        print("Done.")
        print(f"Scenario used: {SCENARIO_NAME}")
        print(f"Linked 'Min Stable Factor' to generators: {linked}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_must_run_min_stable_factor()




In [ ]:
import os
from pathlib import Path
from typing import Optional, Set, List, Tuple, Dict, Any
import pandas as pd

# ============================================================
# STEP â€” Reservoir Levels Detailed -> Storage Initial Volume + Storage Target Year + Penalty
#
# CHANGE (ONLY):
#  - If no year-specific Max Volumes can be built from Hydro CSV (after filters),
#    DO NOT raise RuntimeError. Print a warning and exit gracefully.
#
# FIXES vs last version:
#  - Reflection invoke now passes System.Int32 / System.Double (avoids PyInt->Int32 error)
#  - PeriodEnum "Year" resolved dynamically (your build has no PeriodEnum.Year attribute)
#
# Behavior:
#  - Ensures Scenario RES_Level_Detailed
#  - Reads Reservoir Levels ratios (start/end) by MARKET_NODE + TECHNOLOGY
#  - Uses Hydro additional information.csv to compute year-specific Max Volume (GWh)
#  - Writes (Scenario-tagged):
#       * Initial Volume (Enum 20) year-specific (DateFrom=Jan1 target year)
#       * Storage Target Year (Enum 49) year-specific (DateFrom=Jan1 target year),
#         forced to "Target Year" variant by PeriodTypeId = <resolved Year period>
#       * Target Penalty (Enum 51) once per storage (NO DateFrom)
#  - Applies to Head and Tail if they exist
#  - Non-matched bases get default Storage Target Year = 50% MaxVol(year) + penalty
#  - Skips writing Initial Volume for a year if existing Initial Volume already matches
# ============================================================

# -----------------------------
# USER PATHS
# -----------------------------
RES_LEVELS_CSV = OTHER_DATA_DIR / "Reservoir Levels.csv"
HYDRO_CSV = DASHBOARD_RAWDATA_DIR / "Hydro additional information.csv"

# -----------------------------
# Scenario + enums
# -----------------------------
SCENARIO_NAME = "RES_Level_Detailed"

ENUM_INITIAL_VOLUME = 20        # Initial Volume (GWh)
ENUM_STORAGE_TARGET_YEAR = 49   # Storage Target Year (same EnumId as "Target" in your build)
ENUM_TARGET_PENALTY = 51        # Target Penalty

TARGET_PENALTY_VALUE = 1_000_000.0
DEFAULT_TARGET_FRACTION = 0.5

SYSTEM_STORAGE_COLLECTION_ID = 99
EPS = 1e-6

# -----------------------------
# Filters for Hydro CSV
# -----------------------------
FILTER_DATA_VERSION = "ERAA 2025 final"
OMIT_PLANT_TYPES = {"Run of river"}

# Hydro CSV columns
COL_NODE = "MARKET_NODE"
COL_PLANT = "PEMMDB_PLANT_TYPE"
COL_YEAR = "TARGET_YEAR"
COL_STORAGE_TWH = "Storage Capacity [TWh]"
COL_DATA_VERSION = "data_version"

# Reservoir Levels CSV columns
COL_MARKET_NODE = "MARKET_NODE"
COL_TECH = "TECHNOLOGY"
COL_START = "RESERVOIR_LEVEL_START"
COL_END = "RESERVOIR_LEVEL_END"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------

CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS = [API_DIR / "PLEXOS_NET.dll", BIN_DIR / "PLEXOS_NET.dll"]


# -----------------------------
# Bootstrap
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum, PeriodEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, category: str = "", description: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name not in existing:
        _add_object(db, class_enum_value, name, add_to_system=True, category=category, description=description)


# -----------------------------
# Utils
# -----------------------------
def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\u00A0", " ").strip().split())

def _as_float(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _read_bidding_zones_flat(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals

def _get_membership_id_only(db, CollectionEnum, SystemNS, collection_enum_raw, parent_name: str, child_name: str) -> Optional[int]:
    candidates = []
    try:
        if isinstance(collection_enum_raw, int):
            try:
                candidates.append(CollectionEnum(collection_enum_raw))
            except Exception:
                candidates.append(collection_enum_raw)
        else:
            candidates.append(collection_enum_raw)
    except Exception:
        candidates.append(collection_enum_raw)

    if collection_enum_raw not in candidates:
        candidates.append(collection_enum_raw)

    for col_enum in candidates:
        try:
            return int(db.GetMembershipID(col_enum, parent_name, child_name))
        except Exception:
            pass
        try:
            return int(db.GetMembershipID(col_enum, SystemNS.String(parent_name), SystemNS.String(child_name)))
        except Exception:
            pass
    return None


# -----------------------------
# PeriodEnum: resolve "Year" dynamically for your build
# -----------------------------
def _resolve_period_year_int(PeriodEnum) -> int:
    """
    Your build has no PeriodEnum.Year attribute.
    We find something that represents a year period by name.
    """
    preferred = ["Year", "Years", "Yearly", "Annual", "Annually", "CalendarYear", "FiscalYear"]

    for n in preferred:
        if hasattr(PeriodEnum, n):
            try:
                return int(getattr(PeriodEnum, n))
            except Exception:
                try:
                    return int(getattr(PeriodEnum, n).value__)
                except Exception:
                    pass

    candidates = []
    for attr in dir(PeriodEnum):
        if attr.startswith("_"):
            continue
        al = attr.lower()
        if "year" in al or "annual" in al:
            try:
                v = getattr(PeriodEnum, attr)
                try:
                    iv = int(v)
                except Exception:
                    iv = int(v.value__)
                candidates.append((len(al), attr, iv))
            except Exception:
                continue

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved PeriodEnum YEAR as {chosen[1]} -> {chosen[2]}")
        return int(chosen[2])

    raise RuntimeError("Could not resolve a 'Year' PeriodEnum value in your PLEXOS build.")


# -----------------------------
# Reflection AddProperty (12 params) with correct .NET types
# -----------------------------
def _add_property_row_force_period(db, enum_id: int, mem_id: int, band: int, value: float,
                                  DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                                  Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    import System
    from System import Object
    from System.Reflection import BindingFlags

    Int32 = System.Int32
    Double = System.Double

    def as_i32(x):
        return Int32(int(x))

    def as_dbl(x):
        return Double(float(x))

    t = db.GetType()
    m12 = []
    for m in t.GetMethods(BindingFlags.Public | BindingFlags.Instance):
        try:
            if m.Name == "AddProperty" and m.GetParameters().Length == 12:
                m12.append(m)
        except Exception:
            pass

    if not m12:
        db.AddProperty(int(mem_id), int(enum_id), int(band), float(value),
                       DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action, PeriodTypeId)
        return

    chosen = m12[0]

    args = [
        as_i32(mem_id),
        as_i32(enum_id),
        as_i32(band),
        as_dbl(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action,
        None if PeriodTypeId is None else as_i32(PeriodTypeId),
    ]

    chosen.Invoke(db, System.Array[Object](args))


# -----------------------------
# Existing Initial Volume readback (best effort)
# -----------------------------
def _iter_rows_from_datatable(dt) -> List[Any]:
    if dt is None:
        return []
    if hasattr(dt, "Rows"):
        try:
            return list(dt.Rows)
        except Exception:
            pass
    try:
        return list(dt)
    except Exception:
        return []

def _row_get(row, key_candidates: List[str]):
    for k in key_candidates:
        try:
            return row[k]
        except Exception:
            pass
        try:
            if hasattr(row, k):
                return getattr(row, k)
        except Exception:
            pass
    return None

def _build_existing_initial_volume_map(db, mem_id: int) -> Dict[int, float]:
    out: Dict[int, float] = {}
    if not hasattr(db, "GetProperties"):
        return out
    try:
        dt = db.GetProperties(int(mem_id))
    except Exception:
        return out

    for r in _iter_rows_from_datatable(dt):
        enum_val = _row_get(r, ["EnumId", "EnumID", "PropertyEnumId", "PropertyId", "Enum"])
        band_val = _row_get(r, ["BandId", "BandID", "Band", "BandNum"])
        val_val  = _row_get(r, ["Value", "PropertyValue", "Val"])
        df_val   = _row_get(r, ["DateFrom", "FromDate", "StartDate"])

        try:
            if int(enum_val) != int(ENUM_INITIAL_VOLUME):
                continue
        except Exception:
            continue

        try:
            band_i = int(band_val) if band_val is not None else 1
        except Exception:
            band_i = 1
        if band_i != 1:
            continue

        fv = _as_float(val_val)
        if fv is None:
            continue

        y = None
        try:
            if df_val is not None and hasattr(df_val, "Year"):
                y = int(df_val.Year)
        except Exception:
            y = None

        if y is not None:
            out[y] = float(fv)

    return out


# -----------------------------
# Build YEAR-SPECIFIC MaxVol map from Hydro CSV
# -----------------------------
def _build_yearly_maxvol_map_from_hydro_csv(hydro_csv: Path, allowed_nodes: Set[str]) -> Dict[Tuple[str, str], Dict[int, float]]:
    df = pd.read_csv(hydro_csv)

    required = [COL_NODE, COL_PLANT, COL_YEAR, COL_STORAGE_TWH, COL_DATA_VERSION]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Hydro CSV missing required columns: {missing}\nFound: {list(df.columns)}")

    df[COL_NODE] = df[COL_NODE].astype(str).map(_clean_cell)
    df[COL_PLANT] = df[COL_PLANT].astype(str).map(_clean_cell)
    df[COL_DATA_VERSION] = df[COL_DATA_VERSION].astype(str).map(_clean_cell)

    df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce")
    df[COL_STORAGE_TWH] = df[COL_STORAGE_TWH].apply(_as_float)

    df = df[
        (df[COL_DATA_VERSION] == FILTER_DATA_VERSION) &
        (df[COL_NODE].isin(allowed_nodes)) &
        (df[COL_YEAR].notna()) &
        (~df[COL_PLANT].isin(OMIT_PLANT_TYPES))
    ].copy()

    if df.empty:
        return {}

    df_g = (
        df.groupby([COL_NODE, COL_PLANT, COL_YEAR], as_index=False)
          .agg({COL_STORAGE_TWH: "sum"})
          .sort_values([COL_NODE, COL_PLANT, COL_YEAR])
    )

    out: Dict[Tuple[str, str], Dict[int, float]] = {}
    for _, r in df_g.iterrows():
        node = _clean_cell(r[COL_NODE])
        plant = _clean_cell(r[COL_PLANT])
        year = int(r[COL_YEAR])
        twh = _as_float(r[COL_STORAGE_TWH])
        if twh is None:
            continue
        gwh = round(float(twh) * 1000.0, 1)
        out.setdefault((node, plant), {})[year] = float(gwh)

    return out


# -----------------------------
# MAIN STEP
# -----------------------------
def step_reservoir_levels_detailed():
    allowed_nodes = _read_bidding_zones_flat(BIDDING_ZONE_XLSX)
    if not allowed_nodes:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    yearly_maxvol = _build_yearly_maxvol_map_from_hydro_csv(HYDRO_CSV, allowed_nodes)

    # =========================
    # CHANGE (ONLY): do NOT crash if empty after filtering Hydro CSV
    # =========================
    if not yearly_maxvol:
        print("[WARN] No year-specific Max Volumes could be built from Hydro CSV (after filters).")
        print("       This can be expected if Hydro additional information.csv has no rows matching:")
        print(f"         - data_version == {FILTER_DATA_VERSION!r}")
        print(f"         - MARKET_NODE in Bidding_Zone_List.xlsx (count={len(allowed_nodes)})")
        print(f"         - PEMMDB_PLANT_TYPE not in {sorted(OMIT_PLANT_TYPES)!r}")
        print("       No reservoir level properties were written.")
        return
    # =========================

    df = pd.read_csv(RES_LEVELS_CSV)
    required = [COL_MARKET_NODE, COL_TECH, COL_START, COL_END]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Reservoir Levels CSV missing required columns: {missing}\nFound: {list(df.columns)}")

    df[COL_MARKET_NODE] = df[COL_MARKET_NODE].astype(str).map(_clean_cell)
    df[COL_TECH] = df[COL_TECH].astype(str).map(_clean_cell)
    df[COL_START] = df[COL_START].apply(_as_float)
    df[COL_END] = df[COL_END].apply(_as_float)

    df = df[df[COL_MARKET_NODE].isin(allowed_nodes)].copy()
    if df.empty:
        print("[STEP] No rows remain after filtering Reservoir Levels to allowed bidding zones (MARKET_NODE).")
        return

    print("\n[STEP] Preview (first ~10 filtered reservoir rows):")
    try:
        display(df.head(10))
    except Exception:
        print(df.head(10).to_string(index=False))

    ratios_by_base: Dict[Tuple[str, str], Tuple[Optional[float], Optional[float]]] = {}
    for _, r in df.iterrows():
        node = _clean_cell(r[COL_MARKET_NODE])
        tech = _clean_cell(r[COL_TECH])
        if not node or not tech:
            continue
        ratios_by_base[(node, tech)] = (_as_float(r[COL_START]), _as_float(r[COL_END]))

    db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME)
        scenario_str = SystemNS.String(SCENARIO_NAME)

        year_period_int = _resolve_period_year_int(PeriodEnum)
        storages_in_db = set(_get_objects_safe(db, ClassEnum.Storage))

        props_written = 0
        init_skipped_equal = 0
        targetyear_written = 0
        warnings: List[str] = []
        penalty_written_for: Set[str] = set()

        def _write_penalty_once(storage_name: str, mem_id: int):
            nonlocal props_written
            if storage_name in penalty_written_for:
                return
            try:
                _add_property_row_force_period(
                    db,
                    ENUM_TARGET_PENALTY,
                    mem_id,
                    1,
                    float(TARGET_PENALTY_VALUE),
                    DateFrom=None,
                    Scenario=scenario_str,
                    PeriodTypeId=None
                )
                props_written += 1
                penalty_written_for.add(storage_name)
            except Exception as e:
                warnings.append(f"Failed Target Penalty for '{storage_name}': {e}")

        def _apply_yearly(storage_name: str, node: str, plant: str,
                          start_ratio: Optional[float], end_ratio: Optional[float],
                          is_matched: bool):
            nonlocal props_written, init_skipped_equal, targetyear_written

            mem_id = _get_membership_id_only(db, CollectionEnum, SystemNS, SYSTEM_STORAGE_COLLECTION_ID, "System", storage_name)
            if mem_id is None:
                warnings.append(f"Could not read System->Storage membership id for '{storage_name}'; skipping.")
                return

            _write_penalty_once(storage_name, mem_id)

            years_map = yearly_maxvol.get((node, plant), {})
            if not years_map:
                warnings.append(f"No yearly MaxVol mapping for '{node}_{plant}' (needed for '{storage_name}'); skipping.")
                return

            existing_init_by_year = _build_existing_initial_volume_map(db, int(mem_id))

            for y, max_vol_gwh in sorted(years_map.items()):
                dt_from = NetDateTime(int(y), 1, 1, 0, 0, 0)

                if is_matched:
                    if start_ratio is not None:
                        init_vol = float(start_ratio) * float(max_vol_gwh)
                        ex = existing_init_by_year.get(int(y), None)
                        if ex is not None and abs(float(ex) - float(init_vol)) <= EPS:
                            init_skipped_equal += 1
                        else:
                            try:
                                _add_property_row_force_period(
                                    db,
                                    ENUM_INITIAL_VOLUME,
                                    mem_id,
                                    1,
                                    float(init_vol),
                                    DateFrom=dt_from,
                                    Scenario=scenario_str,
                                    PeriodTypeId=None
                                )
                                props_written += 1
                            except Exception as e:
                                warnings.append(f"Failed Initial Volume for '{storage_name}' year={y}: {e}")

                    if end_ratio is not None:
                        tgt = float(end_ratio) * float(max_vol_gwh)
                        try:
                            _add_property_row_force_period(
                                db,
                                ENUM_STORAGE_TARGET_YEAR,
                                mem_id,
                                1,
                                float(tgt),
                                DateFrom=dt_from,
                                Scenario=scenario_str,
                                PeriodTypeId=year_period_int
                            )
                            props_written += 1
                            targetyear_written += 1
                        except Exception as e:
                            warnings.append(f"Failed Storage Target Year for '{storage_name}' year={y}: {e}")

                else:
                    default_tgt = float(DEFAULT_TARGET_FRACTION) * float(max_vol_gwh)
                    try:
                        _add_property_row_force_period(
                            db,
                            ENUM_STORAGE_TARGET_YEAR,
                            mem_id,
                            1,
                            float(default_tgt),
                            DateFrom=dt_from,
                            Scenario=scenario_str,
                            PeriodTypeId=year_period_int
                        )
                        props_written += 1
                        targetyear_written += 1
                    except Exception as e:
                        warnings.append(f"Failed DEFAULT Storage Target Year for '{storage_name}' year={y}: {e}")

        candidate_bases_in_db: Set[Tuple[str, str]] = set()
        for (node, plant) in yearly_maxvol.keys():
            base = f"{node}_{plant}"
            if (base + " Head") in storages_in_db or (base + " Tail") in storages_in_db:
                candidate_bases_in_db.add((node, plant))

        matched_set = set(ratios_by_base.keys())
        matched_attempted = 0

        for (node, plant), (start_ratio, end_ratio) in ratios_by_base.items():
            base = f"{node}_{plant}"
            head = base + " Head"
            tail = base + " Tail"

            if (node, plant) not in yearly_maxvol:
                warnings.append(f"No yearly MaxVol in Hydro CSV for base '{base}'; skipping matched base.")
                continue

            if head not in storages_in_db and tail not in storages_in_db:
                warnings.append(f"Neither Head nor Tail storage found in DB for base '{base}' (from Reservoir Levels); skipping.")
                continue

            if head in storages_in_db:
                _apply_yearly(head, node, plant, start_ratio, end_ratio, is_matched=True)
            if tail in storages_in_db:
                _apply_yearly(tail, node, plant, start_ratio, end_ratio, is_matched=True)

            matched_attempted += 1

        defaults_attempted = 0
        for (node, plant) in sorted(candidate_bases_in_db):
            if (node, plant) in matched_set:
                continue

            base = f"{node}_{plant}"
            head = base + " Head"
            tail = base + " Tail"

            if head in storages_in_db:
                _apply_yearly(head, node, plant, None, None, is_matched=False)
            if tail in storages_in_db:
                _apply_yearly(tail, node, plant, None, None, is_matched=False)

            defaults_attempted += 1

        print("\n[STEP] Done.")
        print(f"  Scenario used:                              {SCENARIO_NAME}")
        print(f"  Matched bases attempted:                    {matched_attempted}")
        print(f"  Default (non-matched) bases attempted:      {defaults_attempted}")
        print(f"  Initial Volume rows skipped (equal):        {init_skipped_equal}")
        print(f"  Storage Target Year rows written:           {targetyear_written}")
        print(f"  Properties written total (incl penalty):    {props_written}")

        if warnings:
            print("\n[STEP] Warnings (first 300):")
            for w in warnings[:300]:
                print("  - " + w)
            if len(warnings) > 300:
                print(f"  ... and {len(warnings) - 300} more")
        else:
            print("\n[STEP] No warnings.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_reservoir_levels_detailed()




#### 14. NTC

In [ ]:
# ============================================================
# STEP â€” NTCs (HVAC + HVDC) -> Lines + ICL Max/Min Flow Data Files
#
# UPDATED (incl your latest additions):
#  A) If a line is included because either end (or both) is in the Bidding Zone list:
#     - Populate BOTH ICL_Max_Flow_{year}.csv and ICL_Min_Flow_{year}.csv using BOTH directions
#       if available (e.g. CH00-FR00 and FR00-CH00),
#     - AND always link BOTH Max Flow and Min Flow properties to the respective CSV files
#       (even when one end is Dummy).
#     - Dummy node creation/coupling rules are still applied for topology (memberships),
#       but time series are treated "as if both ends were in the BZ list".
#
#  B) Forced Outage Rate is rounded to 2 decimals before writing to PLEXOS.
#
#  C) Dummy Market gets properties:
#       Price        (Enum 12) = 7499
#       Max Sales    (Enum 28) = 10000000
#       Max Purchases(Enum 29) = 10000000
#       Units        (Enum 7)  = 1
#
#  D) NEW: For all created lines:
#       - Units (Enum 14) with NO DateFrom = 0
#       - Wheeling Charge (Enum 42) = 0.01  (no DateFrom)
#       - Wheeling Charge Back (Enum 43) = 0.01 (no DateFrom)
# ============================================================

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Optional, Dict, List, Set, Any, Tuple

import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------
NTC_DIR = NTC_DIR_ROOT

OUTPUT_DIR = DATA_FILES_ROOT / "ICL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These MUST appear in the PLEXOS "Data File" column:
PLEXOS_REL_MAX_FMT = r"Data Files\ICL\ICL_Max_Flow_{year}.csv"
PLEXOS_REL_MIN_FMT = r"Data Files\ICL\ICL_Min_Flow_{year}.csv"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# ENUMS you provided
# -----------------------------
ENUM_UNITS = 14
ENUM_MAX_FLOW = 15
ENUM_MIN_FLOW = 16
ENUM_FORCED_OUTAGE_RATE = 58
ENUM_MTTR = 65
MTTR_HOURS = 168

# NEW line properties
ENUM_WHEELING_CHARGE = 42
ENUM_WHEELING_CHARGE_BACK = 43

# Dummy Market requested enums
ENUM_MARKET_PRICE = 12
ENUM_MARKET_MAX_SALES = 28
ENUM_MARKET_MAX_PURCHASES = 29
ENUM_MARKET_UNITS = 7

DUMMY_MARKET_PRICE_VAL = 50.0
DUMMY_MARKET_MAX_SALES_VAL = 10000000.0
DUMMY_MARKET_MAX_PURCHASES_VAL = 0.0
DUMMY_MARKET_UNITS_VAL = 1.0

SCENARIO_NTC = "NTC"
DUMMY = "Dummy"

# -----------------------------
# Helpers
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals: Set[str] = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = "") -> bool:
    existing = set(_get_objects_safe(db, class_enum_value))
    if name in existing:
        return False
    _add_object(db, class_enum_value, name, add_to_system=add_to_system, category=category, description=description)
    return True


def _ensure_category(db, class_enum_value, category_name: str):
    if not category_name:
        return
    try:
        db.AddCategory(class_enum_value, str(category_name))
    except Exception:
        pass


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _get_membership_id_optional(db, collection_enum, a: str, b: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row_positional(
    db,
    enum_id: int,
    mem_id: int,
    band: int,
    value: float,
    DateFrom=None,
    DateTo=None,
    Variable=None,
    DataFile=None,
    Pattern=None,
    Scenario=None,
    Action=None,
):
    # Signature: (MembershipId, EnumId, BandId, Value, DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action)
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _find_collection_enum_optional(CollectionEnum, must_contain_tokens: List[str], preferred: List[str]) -> Optional[Any]:
    for n in preferred:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)
    toks = [t.lower().strip() for t in must_contain_tokens]
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        ln = attr.lower()
        if all(t in ln for t in toks):
            try:
                return getattr(CollectionEnum, attr)
            except Exception:
                pass
    return None


def _parse_year_from_filename(p: Path) -> Optional[int]:
    m = re.search(r"\bTY(\d{4})\b", p.name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _hours_in_year(year: int) -> int:
    leap = (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0))
    return 8784 if leap else 8760


def _build_year_month_day_period(year: int) -> pd.DataFrame:
    import datetime as dt
    start = dt.date(year, 1, 1)
    end = dt.date(year + 1, 1, 1)
    rows = []
    d = start
    while d < end:
        for p in range(1, 25):
            rows.append((year, d.month, d.day, p))
        d += dt.timedelta(days=1)
    return pd.DataFrame(rows, columns=["Year", "Month", "Day", "Period"])


def _coerce_series_to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)


def _safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _get_line_mem_id(db, CollectionEnum, line_name: str) -> int:
    candidates_cols = []
    for nm in ["SystemLines", "Lines"]:
        if hasattr(CollectionEnum, nm):
            candidates_cols.append(getattr(CollectionEnum, nm))

    try:
        candidates_cols.append(CollectionEnum(int(321)))
    except Exception:
        pass

    trial_pairs = [("System", line_name), (line_name, "System")]

    last_err = None
    for col in candidates_cols:
        for a, b in trial_pairs:
            mid = _get_membership_id_optional(db, col, a, b)
            if mid is not None:
                return int(mid)
            try:
                _ensure_membership(db, col, a, b)
                mid2 = _get_membership_id_optional(db, col, a, b)
                if mid2 is not None:
                    return int(mid2)
            except Exception as e:
                last_err = e

    raise RuntimeError(f"Could not resolve Line membership for property writes (line={line_name}). LastErr={last_err}")


def _resolve_line_from_to_collections(CollectionEnum) -> Tuple[Optional[Any], Optional[Any]]:
    from_pref = [
        "LineNodeFrom", "LineFromNode", "LineFromNodes", "LineFrom",
        "LineFromNodeMembership", "LineFromNodesMembership",
        "TransmissionLineFromNode", "TransmissionLineFromNodes",
    ]
    to_pref = [
        "LineNodeTo", "LineToNode", "LineToNodes", "LineTo",
        "LineToNodeMembership", "LineToNodesMembership",
        "TransmissionLineToNode", "TransmissionLineToNodes",
    ]
    line_from = _find_collection_enum_optional(CollectionEnum, ["line", "from", "node"], from_pref)
    line_to   = _find_collection_enum_optional(CollectionEnum, ["line", "to", "node"], to_pref)
    return line_from, line_to


def _couple_line_to_nodes(
    db,
    ClassEnum,
    CollectionEnum,
    existing_nodes: Set[str],
    line_id: str,
    from_node: str,
    to_node: str,
    warnings: List[str],
):
    from_node = (from_node or "").strip() or DUMMY
    to_node = (to_node or "").strip() or DUMMY

    for n in [from_node, to_node]:
        if n not in existing_nodes:
            try:
                _ensure_object(db, ClassEnum.Node, n)
                existing_nodes.add(n)
            except Exception as e:
                warnings.append(f"Node ensure failed '{n}': {e}")

    line_from_col, line_to_col = _resolve_line_from_to_collections(CollectionEnum)

    def _try_membership(col, a, b) -> bool:
        if col is None:
            return False
        try:
            _ensure_membership(db, col, a, b)
            return True
        except Exception:
            pass
        try:
            _ensure_membership(db, col, b, a)
            return True
        except Exception:
            return False

    ok_from = _try_membership(line_from_col, line_id, from_node)
    ok_to   = _try_membership(line_to_col, line_id, to_node)

    node_line_col = _find_collection_enum_optional(CollectionEnum, ["node", "line"], ["NodeLines", "LineNodes"])
    if node_line_col is not None:
        for n in [from_node, to_node]:
            try:
                _ensure_membership(db, node_line_col, n, line_id)
            except Exception:
                try:
                    _ensure_membership(db, node_line_col, line_id, n)
                except Exception as e:
                    warnings.append(f"Node<->Line membership failed '{n}' <-> '{line_id}': {e}")

    if (line_from_col is None or not ok_from) or (line_to_col is None or not ok_to):
        if line_from_col is None:
            warnings.append("Could not resolve Line->FromNode membership collection (token search 'line/from/node').")
        if line_to_col is None:
            warnings.append("Could not resolve Line->ToNode membership collection (token search 'line/to/node').")
        if line_from_col is not None and not ok_from:
            warnings.append(f"Failed FromNode membership for line '{line_id}' -> '{from_node}'.")
        if line_to_col is not None and not ok_to:
            warnings.append(f"Failed ToNode membership for line '{line_id}' -> '{to_node}'.")


def _ensure_dummy_setup_and_market_props(db, ClassEnum, CollectionEnum, SystemNS, warnings: List[str]):
    try:
        _ensure_object(db, ClassEnum.Region, DUMMY)
    except Exception as e:
        warnings.append(f"Dummy Region create failed: {e}")

    try:
        _ensure_object(db, ClassEnum.Node, DUMMY)
    except Exception as e:
        warnings.append(f"Dummy Node create failed: {e}")

    has_market = hasattr(ClassEnum, "Market")
    if has_market:
        try:
            _ensure_object(db, ClassEnum.Market, DUMMY)
        except Exception as e:
            warnings.append(f"Dummy Market create failed: {e}")

    try:
        node_region_col = _find_collection_enum_optional(CollectionEnum, ["node", "region"], ["NodeRegions", "RegionNodes"])
        if node_region_col is not None:
            _ensure_membership(db, node_region_col, DUMMY, DUMMY)
        else:
            warnings.append("Could not resolve Node<->Region membership collection (NodeRegions/RegionNodes).")
    except Exception as e:
        warnings.append(f"Dummy Node->Region membership failed: {e}")

    try:
        node_market_col = _find_collection_enum_optional(CollectionEnum, ["node", "market"], ["NodeMarkets", "MarketNodes"])
        if node_market_col is not None:
            _ensure_membership(db, node_market_col, DUMMY, DUMMY)
    except Exception as e:
        warnings.append(f"Dummy Node->Market membership failed: {e}")

    if has_market:
        try:
            market_col = _find_collection_enum_optional(CollectionEnum, ["system", "market"], ["SystemMarkets", "Markets"])
            if market_col is None:
                market_col = _find_collection_enum_optional(CollectionEnum, ["market"], ["SystemMarkets", "Markets"])

            if market_col is None:
                warnings.append("Could not resolve Market membership collection; Dummy Market properties not written.")
                return

            mem_id = _get_membership_id_optional(db, market_col, "System", DUMMY)
            if mem_id is None:
                try:
                    _ensure_membership(db, market_col, "System", DUMMY)
                except Exception:
                    _ensure_membership(db, market_col, DUMMY, "System")
                mem_id = _get_membership_id_optional(db, market_col, "System", DUMMY)

            if mem_id is None:
                warnings.append("Could not get membership id for Dummy Market (System<->Dummy). Properties not written.")
                return

            _add_property_row_positional(db, ENUM_MARKET_PRICE,         int(mem_id), 1, float(DUMMY_MARKET_PRICE_VAL))
            _add_property_row_positional(db, ENUM_MARKET_MAX_SALES,     int(mem_id), 1, float(DUMMY_MARKET_MAX_SALES_VAL))
            _add_property_row_positional(db, ENUM_MARKET_MAX_PURCHASES, int(mem_id), 1, float(DUMMY_MARKET_MAX_PURCHASES_VAL))
            _add_property_row_positional(db, ENUM_MARKET_UNITS,         int(mem_id), 1, float(DUMMY_MARKET_UNITS_VAL))

        except Exception as e:
            warnings.append(f"Dummy Market property write failed: {e}")


# ============================================================
# MAIN
# ============================================================
def step_ntc_lines_hvac_hvdc():
    allowed = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    files = sorted(NTC_DIR.glob("NTCs Consolidated TY*.xlsx"))
    if not files:
        raise RuntimeError(f"No files found in {NTC_DIR} matching 'NTCs Consolidated TY*.xlsx'")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    warnings: List[str] = []
    per_year: Dict[int, Dict[str, dict]] = {}

    try:
        # Scenario "NTC"
        try:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NTC)
        except Exception:
            pass
        scenario_ntc = SystemNS.String(SCENARIO_NTC)

        _ensure_category(db, ClassEnum.Line, "HVAC")
        _ensure_category(db, ClassEnum.Line, "HVDC")

        _ensure_dummy_setup_and_market_props(db, ClassEnum, CollectionEnum, SystemNS, warnings)

        existing_lines = set(_get_objects_safe(db, ClassEnum.Line))
        existing_nodes = set(_get_objects_safe(db, ClassEnum.Node))

        # Circuits enum (optional because property name can vary)
        circuits_enum = None
        try_candidates = [
            ("System", "Line", "Lines", "Circuits"),
            ("System", "Line", "SystemLines", "Circuits"),
            ("System", "Transmission", "Lines", "Circuits"),
            ("System", "Transmission", "SystemLines", "Circuits"),
        ]
        for a, b, c, d in try_candidates:
            try:
                circuits_enum = int(db.PropertyName2EnumId(SystemNS.String(a), SystemNS.String(b), SystemNS.String(c), SystemNS.String(d)))
                break
            except Exception:
                pass

        # -----------------------------
        # Read Excel and write per-year CSVs (Max/Min)
        # -----------------------------
        for xlsx in files:
            year = _parse_year_from_filename(xlsx)
            if year is None:
                warnings.append(f"Could not parse year from {xlsx.name}")
                continue

            n_hours = _hours_in_year(year)
            base = _build_year_month_day_period(year)
            max_df = base.copy()
            min_df = base.copy()

            per_year.setdefault(year, {})

            for sheet in ["HVAC", "HVDC"]:
                try:
                    sh = pd.read_excel(xlsx, sheet_name=sheet, header=None, engine="openpyxl")
                except Exception as e:
                    warnings.append(f"{xlsx.name}: sheet {sheet} missing/failed: {e}")
                    continue

                # Row 8 (index 7): line IDs
                col_id: Dict[int, str] = {}
                for c in range(sh.shape[1]):
                    lid = _safe_str(sh.iat[7, c])
                    if lid:
                        col_id[c] = lid
                id_to_col = {lid: c for c, lid in col_id.items()}

                used_cols: Set[int] = set()

                for c in sorted(col_id.keys()):
                    if c in used_cols:
                        continue

                    line_id = col_id[c].strip()
                    parts = [p.strip() for p in line_id.split("-")]
                    if len(parts) < 2:
                        warnings.append(f"{xlsx.name}/{sheet}: bad line id '{line_id}' (no '-')")
                        used_cols.add(c)
                        continue

                    from_id, to_id = parts[0], parts[1]
                    from_ok = from_id in allowed
                    to_ok = to_id in allowed
                    if not (from_ok or to_ok):
                        used_cols.add(c)
                        continue

                    # topology nodes (Dummy if needed)
                    eff_from, eff_to = from_id, to_id
                    if from_ok and not to_ok:
                        eff_to = DUMMY
                    elif (not from_ok) and to_ok:
                        eff_from = DUMMY

                    # circuits
                    circuits_val: Optional[int] = None
                    poles_raw = sh.iat[11, c]
                    try:
                        if not pd.isna(poles_raw):
                            circuits_val = int(float(poles_raw))
                    except Exception:
                        circuits_val = None

                    # forced outage (rounded)
                    for_raw = sh.iat[12, c]
                    try:
                        forced_out = float(for_raw) if not pd.isna(for_raw) else 0.0
                    except Exception:
                        forced_out = 0.0
                    if forced_out <= 0.0:
                        forced_out = 0.0 if sheet.upper() == "HVAC" else 6.0
                    forced_out = round(float(forced_out), 2)

                    # time series
                    ts_fwd = _coerce_series_to_float(sh.iloc[16:16 + n_hours, c])

                    rev_id = f"{to_id}-{from_id}"
                    ts_rev: Optional[pd.Series] = None
                    c2: Optional[int] = None
                    if rev_id in id_to_col and id_to_col[rev_id] != c:
                        c2 = id_to_col[rev_id]
                        ts_rev = _coerce_series_to_float(sh.iloc[16:16 + n_hours, c2])

                    if line_id not in max_df.columns:
                        max_df[line_id] = 0.0
                    if line_id not in min_df.columns:
                        min_df[line_id] = 0.0

                    max_df[line_id] = ts_fwd.values
                    if ts_rev is not None:
                        min_df[line_id] = (-1.0 * ts_rev).values
                        used_cols.add(c2)
                    else:
                        min_df[line_id] = 0.0
                        warnings.append(f"{xlsx.name}/{sheet}: reverse '{rev_id}' missing for '{line_id}' -> Min series is 0 but Min file still linked.")

                    per_year[year][line_id] = {
                        "category": sheet.upper(),
                        "from_node": eff_from,
                        "to_node": eff_to,
                        "forced_outage": forced_out,
                        "circuits": circuits_val,
                    }

                    used_cols.add(c)

            max_csv = OUTPUT_DIR / f"ICL_Max_Flow_{year}.csv"
            min_csv = OUTPUT_DIR / f"ICL_Min_Flow_{year}.csv"
            max_df.to_csv(max_csv, index=False)
            min_df.to_csv(min_csv, index=False)
            print(f"[INFO] wrote {max_csv}")
            print(f"[INFO] wrote {min_csv}")

        # -----------------------------
        # Create Lines + write PLEXOS properties + memberships
        # -----------------------------
        mttr_done: Set[str] = set()
        created_lines: Set[str] = set()
        created_lines_count = 0
        prop_writes = 0
        membership_writes = 0

        for year in sorted(per_year.keys()):
            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

            max_path_str = SystemNS.String(PLEXOS_REL_MAX_FMT.format(year=year))
            min_path_str = SystemNS.String(PLEXOS_REL_MIN_FMT.format(year=year))

            for line_id, info in sorted(per_year[year].items()):
                cat = "HVDC" if info["category"] == "HVDC" else "HVAC"

                # create line if needed
                is_new_line = False
                if line_id not in existing_lines:
                    try:
                        _add_object(db, ClassEnum.Line, line_id, add_to_system=True, category=cat, description="")
                        existing_lines.add(line_id)
                        is_new_line = True
                        created_lines.add(line_id)
                        created_lines_count += 1
                    except Exception as e:
                        warnings.append(f"Line create failed '{line_id}': {e}")
                        continue

                # membership for property writes
                try:
                    mem_id = _get_line_mem_id(db, CollectionEnum, line_id)
                except Exception as e:
                    warnings.append(str(e))
                    continue

                # ------------------------------------------------------------
                # NEW: Non-year-specific defaults for ALL CREATED LINES
                #   Units (no DateFrom) = 0
                #   Wheeling Charge/Back = 0.01
                # Only write once per created line.
                # ------------------------------------------------------------
                if is_new_line:
                    try:
                        _add_property_row_positional(db, ENUM_UNITS, mem_id, 1, 0.0)  # no DateFrom
                        prop_writes += 1
                    except Exception as e:
                        warnings.append(f"Base Units(no date) write failed '{line_id}': {e}")

                    try:
                        _add_property_row_positional(db, ENUM_WHEELING_CHARGE, mem_id, 1, 0.01)  # no DateFrom
                        prop_writes += 1
                    except Exception as e:
                        warnings.append(f"Wheeling Charge write failed '{line_id}': {e}")

                    try:
                        _add_property_row_positional(db, ENUM_WHEELING_CHARGE_BACK, mem_id, 1, 0.01)  # no DateFrom
                        prop_writes += 1
                    except Exception as e:
                        warnings.append(f"Wheeling Charge Back write failed '{line_id}': {e}")

                # MTTR once (no DateFrom)
                if line_id not in mttr_done:
                    try:
                        _add_property_row_positional(db, ENUM_MTTR, mem_id, 1, float(MTTR_HOURS))
                        prop_writes += 1
                        mttr_done.add(line_id)
                    except Exception as e:
                        warnings.append(f"MTTR write failed '{line_id}': {e}")

                # YEAR-SPECIFIC properties
                try:
                    # Units year-specific + Scenario NTC
                    _add_property_row_positional(
                        db, ENUM_UNITS, mem_id, 1, 1.0,
                        dt_from, None,
                        None, None, None,
                        scenario_ntc, None
                    )
                    prop_writes += 1
                except Exception as e:
                    warnings.append(f"Units(TY) write failed '{line_id}' {year}: {e}")

                try:
                    _add_property_row_positional(db, ENUM_FORCED_OUTAGE_RATE, mem_id, 1, float(info["forced_outage"]), dt_from)
                    prop_writes += 1
                except Exception as e:
                    warnings.append(f"FOR write failed '{line_id}' {year}: {e}")

                if circuits_enum is not None and info["circuits"] is not None:
                    try:
                        _add_property_row_positional(db, int(circuits_enum), mem_id, 1, float(int(info["circuits"])), dt_from)
                        prop_writes += 1
                    except Exception as e:
                        warnings.append(f"Circuits write failed '{line_id}' {year}: {e}")

                # Flow linking - ALWAYS link both
                try:
                    _add_property_row_positional(db, ENUM_MAX_FLOW, mem_id, 1, 1.0, dt_from, None, None, max_path_str)
                    prop_writes += 1
                except Exception as e:
                    warnings.append(f"Max Flow write failed '{line_id}' {year}: {e}")

                try:
                    _add_property_row_positional(db, ENUM_MIN_FLOW, mem_id, 1, 1.0, dt_from, None, None, min_path_str)
                    prop_writes += 1
                except Exception as e:
                    warnings.append(f"Min Flow write failed '{line_id}' {year}: {e}")

                # Node coupling (Dummy topology if needed)
                try:
                    _couple_line_to_nodes(
                        db=db,
                        ClassEnum=ClassEnum,
                        CollectionEnum=CollectionEnum,
                        existing_nodes=existing_nodes,
                        line_id=line_id,
                        from_node=str(info["from_node"]).strip() or DUMMY,
                        to_node=str(info["to_node"]).strip() or DUMMY,
                        warnings=warnings,
                    )
                    membership_writes += 1
                except Exception as e:
                    warnings.append(f"Line->Node coupling failed '{line_id}': {e}")

        print("\n[STEP] DONE")
        print(f"  Years processed:         {len(per_year)}")
        print(f"  Lines created:           {created_lines_count}")
        print(f"  Property rows written:   {prop_writes}")
        print(f"  Membership ops (approx): {membership_writes}")

        if warnings:
            print("\n[STEP] Warnings (first 200):")
            for w in warnings[:200]:
                print("  - " + w)

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_ntc_lines_hvac_hvdc()




In [ ]:
# ============================================================
# STEP â€” Limits import (Constraints + RHS DataFiles + Region Coefficients)
#
# Reads "Limits" sheet from:
#   NTC_DIR_ROOT / "NTCs Consolidated TY*.xlsx"
#
# Sheet layout:
#   - Starting in column C (index 2): limits time series columns
#   - Row 8 (index 7):  Constraint name  (used as Constraint object name AND CSV column header)
#   - Row 9 (index 8):  Type text (ignored for now; we key off Import/Export in row 8 name)
#   - Row 10 ignored
#   - Rows 11..end: hourly series (8784 rows for leap years; 8760 otherwise)
#
# Behavior:
#   - Creates 1 CSV per year with columns:
#       Year,Month,Day,Period,<ConstraintName1>,<ConstraintName2>,...
#     (same format as ICL_Max_Flow_{year}.csv)
#   - For each constraint where row-8 name contains "Import" or "Export":
#       * Resolve Region token (e.g. CH00) from the constraint name by matching any entry
#         from Bidding_Zone_List.xlsx as a standalone token in the name.
#       * Create Constraint object (if missing)
#       * System<->Constraint membership:
#            - Sense (Enum 1) = -1  (<=)
#            - RHS (resolved dynamically; NOT RHS Day) linked to the year CSV via DataFile column
#              Value written as 1.0 when linking a DataFile (consistent with the existing flow logic)
#       * Region<->Constraint membership (Collection ID = 242):
#            - Imports Coefficient (Enum 4) = 1  if Import
#            - Exports Coefficient (Enum 5) = 1  if Export
#
# IMPORTANT:
#   - If ZERO limit constraints match a Region in the Bidding Zone list, DO NOT FAIL:
#       print a warning and exit gracefully.
# ============================================================

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Optional, Dict, List, Set, Any, Tuple

import pandas as pd

# -----------------------------
# USER PATHS (same base as earlier flow logic)
# -----------------------------
NTC_DIR = NTC_DIR_ROOT

# Write limits CSVs alongside ICL flow CSVs (same "format style")
OUTPUT_DIR_LIMITS = DATA_FILES_ROOT / "ICL"
OUTPUT_DIR_LIMITS.mkdir(parents=True, exist_ok=True)

# Must appear in PLEXOS "Data File" column
PLEXOS_REL_LIMITS_FMT = r"Data Files\ICL\Limits_{year}.csv"

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# ENUMS / IDS (given)
# -----------------------------
ENUM_CONSTRAINT_SENSE = 1               # on System<->Constraint membership
SENSE_LESS_EQUAL = -1.0                 # <=

REGION_CONSTRAINTS_COLLECTION_ID = 243  # Region.Constraints
ENUM_IMPORTS_COEFF = 4                  # on Region<->Constraint membership
ENUM_EXPORTS_COEFF = 5                  # on Region<->Constraint membership


# -----------------------------
# Helpers (standalone, safe to paste)
# -----------------------------
def _read_bidding_zones(xlsx_path: Path) -> Set[str]:
    df = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    vals: Set[str] = set()
    for v in df.values.ravel():
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            vals.add(s)
    return vals


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _add_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)


def _ensure_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = "") -> bool:
    existing = set(_get_objects_safe(db, class_enum_value))
    if name in existing:
        return False
    _add_object(db, class_enum_value, name, add_to_system=add_to_system, category=category, description=description)
    return True


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _ensure_membership_bi(db, collection_enum, a: str, b: str) -> Optional[int]:
    # returns membership id (either direction) if possible
    try:
        _ensure_membership(db, collection_enum, a, b)
        return int(db.GetMembershipID(collection_enum, a, b))
    except Exception:
        pass
    try:
        _ensure_membership(db, collection_enum, b, a)
        return int(db.GetMembershipID(collection_enum, b, a))
    except Exception:
        return None


def _add_property_row_positional(
    db,
    enum_id: int,
    mem_id: int,
    band: int,
    value: float,
    DateFrom=None,
    DateTo=None,
    Variable=None,
    DataFile=None,
    Pattern=None,
    Scenario=None,
    Action=None,
):
    # Signature: (MembershipId, EnumId, BandId, Value, DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action)
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _parse_year_from_filename(p: Path) -> Optional[int]:
    m = re.search(r"\bTY(\d{4})\b", p.name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _hours_in_year(year: int) -> int:
    leap = (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0))
    return 8784 if leap else 8760


def _build_year_month_day_period(year: int) -> pd.DataFrame:
    import datetime as dt
    start = dt.date(year, 1, 1)
    end = dt.date(year + 1, 1, 1)
    rows = []
    d = start
    while d < end:
        for p in range(1, 25):
            rows.append((year, d.month, d.day, p))
        d += dt.timedelta(days=1)
    return pd.DataFrame(rows, columns=["Year", "Month", "Day", "Period"])


def _coerce_series_to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)


def _safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _resolve_constraint_class(ClassEnum) -> Optional[Any]:
    for cand in ["Constraint", "Constraints"]:
        if hasattr(ClassEnum, cand):
            return getattr(ClassEnum, cand)
    # fuzzy
    for attr in dir(ClassEnum):
        if attr.startswith("_"):
            continue
        if "constraint" in attr.lower():
            try:
                return getattr(ClassEnum, attr)
            except Exception:
                pass
    return None


def _resolve_system_constraints_collection(CollectionEnum) -> Optional[Any]:
    for cand in ["SystemConstraints", "Constraints"]:
        if hasattr(CollectionEnum, cand):
            return getattr(CollectionEnum, cand)
    # fallback known from your other script
    try:
        return CollectionEnum(int(759))
    except Exception:
        return int(759)


def _resolve_property_enum_constraint_rhs(db, SystemNS) -> Optional[int]:
    """
    Resolve the EnumId for Constraint RHS (NOT RHS Day) dynamically.
    We try common property names via PropertyName2EnumId.
    """
    # Try (System, Constraint, Constraints, <propname>) patterns
    a = SystemNS.String("System")
    b = SystemNS.String("Constraint")
    c = SystemNS.String("Constraints")

    for prop in ["RHS", "Right Hand Side", "RightHandSide", "Constraint RHS", "RHS (MW)"]:
        try:
            eid = int(db.PropertyName2EnumId(a, b, c, SystemNS.String(prop)))
            return eid
        except Exception:
            pass

    # Some builds may name the class "Constraints" but collection differently; try a few
    for c_name in ["SystemConstraints", "Constraints"]:
        for prop in ["RHS", "Right Hand Side", "RightHandSide"]:
            try:
                eid = int(db.PropertyName2EnumId(a, b, SystemNS.String(c_name), SystemNS.String(prop)))
                return eid
            except Exception:
                pass

    return None


def _match_region_from_constraint_name(constraint_name: str, allowed_regions: Set[str]) -> Optional[str]:
    """
    Find a region token (e.g. CH00) contained in the constraint name as a standalone token.
    Standalone means bounded by non-alphanumeric characters (including underscores).
    """
    s = str(constraint_name or "")
    if not s.strip():
        return None

    # Fast path: exact tokenization on common separators
    tokens = re.split(r"[^A-Za-z0-9]+", s)
    token_set = {t for t in tokens if t}
    for r in allowed_regions:
        if r in token_set:
            return r

    # Boundary match fallback
    for r in allowed_regions:
        pat = rf"(^|[^A-Za-z0-9]){re.escape(r)}([^A-Za-z0-9]|$)"
        if re.search(pat, s):
            return r

    return None


# ============================================================
# MAIN STEP
# ============================================================
def step_limits_import():
    allowed = _read_bidding_zones(BIDDING_ZONE_XLSX)
    if not allowed:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    files = sorted(NTC_DIR.glob("NTCs Consolidated TY*.xlsx"))
    if not files:
        raise RuntimeError(f"No files found in {NTC_DIR} matching 'NTCs Consolidated TY*.xlsx'")

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    warnings: List[str] = []
    total_constraints_created = 0
    total_constraint_props = 0
    total_region_links = 0
    total_region_coeff_props = 0
    total_year_csvs = 0

    try:
        constraint_class = _resolve_constraint_class(ClassEnum)
        if constraint_class is None:
            raise RuntimeError("Could not resolve ClassEnum for Constraint objects (no Constraint/Constraints in ClassEnum).")

        system_constraints_col = _resolve_system_constraints_collection(CollectionEnum)

        # Region.Constraints collection (given id=242)
        try:
            region_constraints_col = CollectionEnum(int(REGION_CONSTRAINTS_COLLECTION_ID))
        except Exception:
            region_constraints_col = int(REGION_CONSTRAINTS_COLLECTION_ID)

        # Resolve RHS enum dynamically
        rhs_enum = _resolve_property_enum_constraint_rhs(db, SystemNS)
        if rhs_enum is None:
            warnings.append("Could not resolve Constraint RHS EnumId (NOT RHS Day). RHS DataFile linking will be skipped.")

        existing_constraints = set(_get_objects_safe(db, constraint_class))
        existing_regions = set(_get_objects_safe(db, ClassEnum.Region))

        # We only build constraints for names containing Import/Export AND whose region token matches allowed list.
        matched_any = False

        for xlsx in files:
            year = _parse_year_from_filename(xlsx)
            if year is None:
                warnings.append(f"Could not parse year from {xlsx.name}")
                continue

            n_hours = _hours_in_year(year)
            base = _build_year_month_day_period(year)
            limits_df = base.copy()

            try:
                sh = pd.read_excel(xlsx, sheet_name="Limits", header=None, engine="openpyxl")
            except Exception as e:
                warnings.append(f"{xlsx.name}: sheet 'Limits' missing/failed: {e}")
                continue

            # Row 8 (index 7): names, starting col C (index 2)
            # Row 9 (index 8): types (not needed for core behavior)
            col_names: Dict[int, str] = {}
            for c in range(2, sh.shape[1]):
                nm = _safe_str(sh.iat[7, c])
                if nm:
                    col_names[c] = nm

            # Build per-year list of (constraint_name, region, import_or_export, series)
            year_items: List[Tuple[str, str, str, pd.Series]] = []

            for c, cname in sorted(col_names.items()):
                cname_str = str(cname).strip()
                cname_lc = cname_str.lower()

                is_import = "import" in cname_lc
                is_export = "export" in cname_lc
                if not (is_import or is_export):
                    continue

                region = _match_region_from_constraint_name(cname_str, allowed)
                if not region:
                    # Not a match -> ignore (but track warning lightly)
                    continue

                # Ensure region exists as Region object (if it doesn't, we can still create constraint but coeff link will fail)
                if region not in existing_regions:
                    warnings.append(f"Region '{region}' from constraint '{cname_str}' not found as a Region object in DB; Region<->Constraint coeff may fail.")

                # Time series rows: row 11.. (index 10..10+n_hours-1), same column c
                ts = _coerce_series_to_float(sh.iloc[10:10 + n_hours, c])
                if len(ts) != n_hours:
                    warnings.append(f"{xlsx.name}/Limits: '{cname_str}' series length {len(ts)} != {n_hours} (year {year}). Using available rows, padding/truncation not applied.")
                    # Align by truncating/padding with zeros to match base length
                    if len(ts) > n_hours:
                        ts = ts.iloc[:n_hours]
                    else:
                        ts = pd.concat([ts, pd.Series([0.0] * (n_hours - len(ts)))], ignore_index=True)

                year_items.append((cname_str, region, "import" if is_import else "export", ts))

            # If no constraints in this year matched, still write nothing and continue
            if not year_items:
                continue

            matched_any = True

            # Add columns to dataframe
            for cname_str, _, _, ts in year_items:
                if cname_str not in limits_df.columns:
                    limits_df[cname_str] = 0.0
                limits_df[cname_str] = ts.values

            # Write year CSV
            out_csv = OUTPUT_DIR_LIMITS / f"Limits_{year}.csv"
            limits_df.to_csv(out_csv, index=False)
            total_year_csvs += 1
            print(f"[INFO] wrote {out_csv}")

            # PLEXOS DataFile path string (relative)
            df_path_str = SystemNS.String(PLEXOS_REL_LIMITS_FMT.format(year=year))

            # Create constraints + properties + region coeffs
            dt_from = NetDateTime(int(year), 1, 1, 0, 0, 0)

            for cname_str, region, kind, _ in year_items:
                # Create constraint if needed
                if cname_str not in existing_constraints:
                    try:
                        _add_object(db, constraint_class, cname_str, add_to_system=True, category="", description="")
                        existing_constraints.add(cname_str)
                        total_constraints_created += 1
                    except Exception as e:
                        warnings.append(f"Constraint create failed '{cname_str}': {e}")
                        continue

                # System<->Constraint membership for Sense + RHS
                try:
                    c_sys_mem = _ensure_membership(db, system_constraints_col, "System", cname_str)
                except Exception as e:
                    warnings.append(f"System<->Constraint membership failed '{cname_str}': {e}")
                    continue

                # Sense <=
                try:
                    _add_property_row_positional(db, ENUM_CONSTRAINT_SENSE, c_sys_mem, 1, float(SENSE_LESS_EQUAL), dt_from)
                    total_constraint_props += 1
                except Exception as e:
                    warnings.append(f"Constraint Sense write failed '{cname_str}' {year}: {e}")

                # RHS DataFile link (NOT RHS Day)
                if rhs_enum is not None:
                    try:
                        # When linking a DataFile, write Value=1.0 (same pattern as the existing flow logic)
                        _add_property_row_positional(db, int(rhs_enum), c_sys_mem, 1, 1.0, dt_from, None, None, df_path_str)
                        total_constraint_props += 1
                    except Exception as e:
                        warnings.append(f"Constraint RHS(DataFile) write failed '{cname_str}' {year}: {e}")

                # Region<->Constraint membership (collection id 242) + coefficient
                try:
                    rc_mem = _ensure_membership_bi(db, region_constraints_col, region, cname_str)
                    if rc_mem is None:
                        warnings.append(f"Region<->Constraint membership failed '{region}' <-> '{cname_str}'")
                    else:
                        total_region_links += 1
                        coeff_enum = ENUM_IMPORTS_COEFF if kind == "import" else ENUM_EXPORTS_COEFF
                        try:
                            _add_property_row_positional(db, int(coeff_enum), int(rc_mem), 1, 1.0, dt_from)
                            total_region_coeff_props += 1
                        except Exception as e:
                            warnings.append(f"Region coeff write failed '{region}' <-> '{cname_str}' {year}: {e}")
                except Exception as e:
                    warnings.append(f"Region<->Constraint handling failed '{region}' <-> '{cname_str}' {year}: {e}")

        # ---------------------------------
        # Exit behavior if no matches at all
        # ---------------------------------
        if not matched_any:
            print("\n[STEP] WARNING: No Limit constraints matched any Region in Bidding_Zone_List.xlsx.")
            print("             No Limits CSVs were written and no Constraint/Region updates were applied.")
            return

        print("\n[STEP] DONE")
        print(f"  Year CSVs written:                 {total_year_csvs}")
        print(f"  Constraints created:               {total_constraints_created}")
        print(f"  Constraint property rows written:  {total_constraint_props}  (Sense + RHS links)")
        print(f"  Region<->Constraint links:         {total_region_links}")
        print(f"  Region coefficient rows written:   {total_region_coeff_props}")

        if warnings:
            print("\n[STEP] Warnings (first 200):")
            for w in warnings[:200]:
                print("  - " + w)

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_limits_import()





#### 14.2 FBMC - Nordics

In [ ]:
# STEP 14.2 - FBMC hub, lines, interfaces, PTDF/RAM files, and year-specific links
RUN_FBMC_NORDICS_14_2 = True

from __future__ import annotations

import calendar
import os
from pathlib import Path
from typing import Any, Optional

import pandas as pd

# -----------------------------
# Inputs
# -----------------------------
API_DIR = globals().get("API_DIR", Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0 API"))
BIN_DIR = globals().get("BIN_DIR", Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0"))

FBMC_HUB_NAME = "FBMC_HUB"
FBMC_SCENARIO_NAME = "FBMC"
FBMC_LINE_CATEGORY = "FBMC"
FBMC_TARGET_YEARS = [2028, 2030, 2033, 2035]
FBMC_INTERFACE_DATE_RANGES = {
    2028: ((2028, 1, 1), (2029, 12, 31)),
    2030: ((2030, 1, 1), (2032, 12, 31)),
    2033: ((2033, 1, 1), (2034, 12, 31)),
    2035: ((2035, 1, 1), None),
}
FBMC_DOMAIN_ASSIGNMENT_SHEET = "Domain Assignment"

FBMC_MAPPING_XLSX = BASE_DIR / "FBMC_Mapping.xlsx"
FBMC_DOMAIN_XLSX = BASE_DIR / "ERAA_2025_FBDomains" / "FB-Domain-NORDIC.xlsx"
FBMC_DATA_FILES_DIR = BASE_DIR / "Database" / "Data Files" / "FBMC"

# Confirmed from 12R02_Tables.xlsx for PLEXOS 12.0 R02.
FBMC_CLASS_IDS = {
    "Region": 21,
    "Node": 24,
    "Line": 26,
    "Interface": 30,
    "Scenario": 85,
}

FBMC_COLLECTION_IDS = {
    "SystemRegions": 214,
    "SystemNodes": 298,
    "NodeRegion": 302,
    "SystemLines": 322,
    "LineNodeFrom": 326,
    "LineNodeTo": 327,
    "SystemInterfaces": 353,
    "InterfaceLines": 356,
}

FBMC_LINE_ENUMS = {
    "Units": 14,
    "Max Flow": 15,
    "Min Flow": 16,
    "Wheeling Charge": 42,
    "Wheeling Charge Back": 43,
}

FBMC_REGION_ENUMS = {
    "Units": 50,
}

FBMC_NODE_ENUMS = {
    "Units": 21,
}

FBMC_INTERFACE_ENUMS = {
    "Units": 3,
    "Max Flow": 5,
    "Limit Penalty": 8,
}

FBMC_INTERFACE_LINE_ENUMS = {
    "Flow Coefficient": 1,
}

CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS = [API_DIR / "PLEXOS_NET.dll", BIN_DIR / "PLEXOS_NET.dll"]


# -----------------------------
# PLEXOS API helpers
# -----------------------------
def _fbmc_bootstrap_plexos_net():
    for directory in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(directory) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(directory))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        dll_name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll_path = base / f"{dll_name}.dll"
            if dll_path.exists():
                return Assembly.LoadFile(str(dll_path))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for dll_path in CORE_DLLS:
        if dll_path.exists():
            Assembly.LoadFile(str(dll_path))
            break
    else:
        raise FileNotFoundError(f"PLEXOS_NET.Core.dll not found in {API_DIR} or {BIN_DIR}")

    for dll_path in NET_DLLS:
        if dll_path.exists():
            Assembly.LoadFile(str(dll_path))
            break
    else:
        raise FileNotFoundError(f"PLEXOS_NET.dll not found in {API_DIR} or {BIN_DIR}")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _fbmc_enum_from_id(SystemNS, enum_type, enum_id: int):
    return SystemNS.Enum.ToObject(enum_type, int(enum_id))


def _fbmc_class_enum(ClassEnum, SystemNS, preferred_name: str, fallback_id: int):
    if hasattr(ClassEnum, preferred_name):
        return getattr(ClassEnum, preferred_name)
    return _fbmc_enum_from_id(SystemNS, ClassEnum, fallback_id)


def _fbmc_collection_enum(CollectionEnum, SystemNS, preferred_names: list[str], fallback_id: int):
    for name in preferred_names:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return _fbmc_enum_from_id(SystemNS, CollectionEnum, fallback_id)


def _fbmc_get_objects_safe(db, class_enum_value) -> set[str]:
    try:
        result = db.GetObjects(class_enum_value)
    except Exception:
        return set()
    if result is None:
        return set()
    try:
        return {str(item).strip() for item in list(result) if str(item).strip()}
    except Exception:
        return set()


def _fbmc_get_objects_in_category_safe(db, class_enum_value, category_name: str) -> set[str]:
    try:
        result = db.GetObjectsInCategory(class_enum_value, str(category_name))
    except Exception:
        return set()
    if result is None:
        return set()
    try:
        return {str(item).strip() for item in list(result) if str(item).strip()}
    except Exception:
        return set()


def _fbmc_ensure_category(db, class_enum_value, category_name: str) -> None:
    if not category_name:
        return
    try:
        if bool(db.CategoryExists(class_enum_value, str(category_name))):
            return
    except Exception:
        pass
    try:
        db.AddCategory(class_enum_value, str(category_name))
    except Exception:
        pass


def _fbmc_categorize_object(db, class_enum_value, name: str, category_name: str) -> bool:
    if not name or not category_name:
        return False
    try:
        return bool(db.CategorizeObject(class_enum_value, str(name), str(category_name)))
    except Exception:
        return False


def _fbmc_ensure_object(
    db,
    class_enum_value,
    name: str,
    *,
    add_to_system: bool = True,
    category: str = "",
    description: str = "",
) -> bool:
    object_name = str(name).strip()
    if not object_name:
        return False
    if object_name in _fbmc_get_objects_safe(db, class_enum_value):
        if category:
            _fbmc_categorize_object(db, class_enum_value, object_name, category)
        return False
    try:
        db.AddObject(object_name, class_enum_value, bool(add_to_system), str(category or ""), str(description or ""))
    except Exception as exc:
        if category:
            db.AddObject(object_name, class_enum_value, bool(add_to_system), "", str(description or ""))
            _fbmc_categorize_object(db, class_enum_value, object_name, category)
            print(f"[WARN] Added '{object_name}' without direct category '{category}' because category write failed: {exc}")
        else:
            raise
    return True


def _fbmc_get_membership_id_optional(db, collection_enum, parent_name: str, child_name: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, str(parent_name), str(child_name)))
    except Exception:
        return None


def _fbmc_ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> Optional[int]:
    parent_name = str(parent_name).strip()
    child_name = str(child_name).strip()
    if not parent_name or not child_name:
        return None

    existing_id = _fbmc_get_membership_id_optional(db, collection_enum, parent_name, child_name)
    if existing_id is not None:
        return existing_id

    try:
        db.AddMembership(collection_enum, parent_name, child_name)
    except Exception:
        pass
    return _fbmc_get_membership_id_optional(db, collection_enum, parent_name, child_name)


def _fbmc_ensure_membership_bi(db, collection_enum, first_name: str, second_name: str) -> Optional[int]:
    membership_id = _fbmc_ensure_membership(db, collection_enum, first_name, second_name)
    if membership_id is not None:
        return membership_id
    return _fbmc_ensure_membership(db, collection_enum, second_name, first_name)


def _fbmc_add_property(
    db,
    enum_id: int,
    membership_id: int,
    value: float,
    *,
    scenario=None,
    data_file=None,
    date_from=None,
    date_to=None,
) -> None:
    db.AddProperty(
        int(membership_id),
        int(enum_id),
        1,
        float(value),
        date_from,
        date_to,
        None,
        data_file,
        None,
        scenario,
        None,
    )


def _fbmc_property_enum(
    db,
    SystemNS,
    parent_name: str,
    child_name: str,
    collection_name: str,
    property_name: str,
    fallback_enum: int,
) -> int:
    try:
        return int(db.PropertyName2EnumId(
            SystemNS.String(parent_name),
            SystemNS.String(child_name),
            SystemNS.String(collection_name),
            SystemNS.String(property_name),
        ))
    except Exception:
        return int(fallback_enum)


def _fbmc_datafile_ref(SystemNS, path: Path):
    try:
        rel = path.resolve().relative_to(DATABASE_DIR.resolve())
    except Exception:
        rel = path
    return SystemNS.String(str(rel).replace("/", "\\"))


def _fbmc_datetime(SystemNS, date_tuple: tuple[int, int, int] | None):
    if date_tuple is None:
        return None
    year, month, day = date_tuple
    return SystemNS.DateTime(int(year), int(month), int(day), 0, 0, 0)


# -----------------------------
# Workbook helpers
# -----------------------------
def _fbmc_clean_text(value: Any) -> str:
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def _fbmc_read_mapping(mapping_xlsx: Path) -> pd.DataFrame:
    if not mapping_xlsx.exists():
        raise FileNotFoundError(f"FBMC mapping file not found: {mapping_xlsx}")
    mapping_df = pd.read_excel(mapping_xlsx, sheet_name=0)
    if mapping_df.shape[1] < 2:
        raise RuntimeError(f"FBMC mapping must have at least two columns: {mapping_xlsx}")

    out = mapping_df.iloc[:, :2].copy()
    out.columns = ["Dataset", "PLEXOS"]
    out["Dataset"] = out["Dataset"].map(_fbmc_clean_text)
    out["PLEXOS"] = out["PLEXOS"].map(_fbmc_clean_text)
    out = out[(out["Dataset"] != "") & (out["PLEXOS"] != "")].copy()
    if out.empty:
        raise RuntimeError(f"No usable Dataset/PLEXOS rows found in {mapping_xlsx}")
    return out


def _fbmc_read_interfaces_and_ptdf(domain_xlsx: Path, sheet_name: str) -> tuple[pd.DataFrame, list[tuple[int, str]]]:
    if not domain_xlsx.exists():
        raise FileNotFoundError(f"FB-domain workbook not found: {domain_xlsx}")

    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name, header=None)
    if raw.shape[0] < 3 or raw.shape[1] < 4:
        raise RuntimeError(f"Sheet '{sheet_name}' does not have the expected PTDF layout")

    interface_rows: list[tuple[int, str]] = []
    for row_idx in range(2, raw.shape[0]):
        interface_name = _fbmc_clean_text(raw.iat[row_idx, 2])
        if not interface_name:
            break
        interface_rows.append((row_idx, interface_name))

    if not interface_rows:
        raise RuntimeError(f"No interface names found in {domain_xlsx}, sheet '{sheet_name}', column C")

    return raw, interface_rows


def _fbmc_safe_filename(stem: str) -> str:
    invalid_chars = '<>:"/\\|?*'
    safe = str(stem).strip()
    for char in invalid_chars:
        safe = safe.replace(char, "_")
    return safe or "unnamed"


def _fbmc_interface_name_for_year(interface_name: str, target_year: int) -> str:
    base_name = str(interface_name).strip()
    year_tag = f"TY{int(target_year)}"
    if base_name.upper().endswith(f" - {year_tag}".upper()):
        return base_name
    return f"{base_name} - {year_tag}"


def _fbmc_numeric_or_zero(value: Any) -> float:
    try:
        if pd.isna(value):
            return 0.0
    except Exception:
        pass
    numeric_value = pd.to_numeric(value, errors="coerce")
    if pd.isna(numeric_value):
        return 0.0
    return float(numeric_value)


def _fbmc_write_ptdf_files(
    raw_ptdf: pd.DataFrame,
    interface_rows: list[tuple[int, str]],
    mapping_df: pd.DataFrame,
    output_dir: Path,
) -> tuple[int, list[str], dict[str, Path]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_to_plexos = dict(zip(mapping_df["Dataset"], mapping_df["PLEXOS"]))

    files_written = 0
    warnings: list[str] = []
    ptdf_file_by_line: dict[str, Path] = {}

    for col_idx in range(3, raw_ptdf.shape[1]):
        dataset_line = _fbmc_clean_text(raw_ptdf.iat[1, col_idx])
        column_has_values = raw_ptdf.iloc[2:, col_idx].notna().any()
        if not dataset_line and not column_has_values:
            break
        if not dataset_line:
            warnings.append(f"Skipped PTDF column {col_idx + 1}: missing dataset line header in row 2")
            continue

        plexos_line = dataset_to_plexos.get(dataset_line)
        if not plexos_line:
            warnings.append(f"Skipped PTDF column '{dataset_line}': no PLEXOS mapping found")
            continue

        rows = []
        for row_idx, interface_name in interface_rows:
            value = _fbmc_numeric_or_zero(raw_ptdf.iat[row_idx, col_idx])
            rows.append({"Name": f"{interface_name}{plexos_line}", "Value": value})

        out_path = output_dir / f"{_fbmc_safe_filename(plexos_line)}.csv"
        pd.DataFrame(rows, columns=["Name", "Value"]).to_csv(out_path, index=False)
        ptdf_file_by_line[plexos_line] = out_path
        files_written += 1

    return files_written, warnings, ptdf_file_by_line


def _fbmc_read_domain_assignment(
    domain_xlsx: Path,
    sheet_name: str,
    target_year: int,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name)
    required_cols = ["Year", "Month", "Day", "Hour"]
    missing = [col for col in required_cols if col not in raw.columns]
    if missing:
        raise RuntimeError(f"Sheet '{sheet_name}' missing required columns: {missing}")

    domain_cols = list(raw.columns[4:40])
    if len(domain_cols) != 36:
        raise RuntimeError(f"Sheet '{sheet_name}' expected 36 domain columns from E:AN, found {len(domain_cols)}")

    year_df = raw[pd.to_numeric(raw["Year"], errors="coerce").eq(int(target_year))].copy()
    if year_df.empty:
        raise RuntimeError(f"No rows for year {target_year} found in sheet '{sheet_name}'")

    for col in ["Year", "Month", "Day", "Hour"]:
        year_df[col] = pd.to_numeric(year_df[col], errors="coerce").astype("Int64")
    year_df["Period"] = (year_df["Hour"] + 1).astype("Int64")

    warnings: list[str] = []
    has_feb29 = year_df["Month"].eq(2).mul(year_df["Day"].eq(29)).any()
    if calendar.isleap(int(target_year)) and not has_feb29:
        feb28 = year_df[year_df["Month"].eq(2) & year_df["Day"].eq(28)].copy()
        if feb28.empty:
            warnings.append(f"Could not add Feb 29 for {target_year}: no Feb 28 rows found")
        else:
            feb29 = feb28.copy()
            feb29["Day"] = 29
            year_df = pd.concat([year_df, feb29], ignore_index=True)
            warnings.append(f"Inserted Feb 29 for {target_year} by copying Feb 28 domain assignments")

    year_df = year_df.sort_values(["Year", "Month", "Day", "Period"]).reset_index(drop=True)
    out = year_df[["Year", "Month", "Day", "Period"] + domain_cols].copy()
    return out, domain_cols, warnings


def _fbmc_read_ram_lookup(domain_xlsx: Path, sheet_name: str) -> dict[str, dict[int, float]]:
    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name)
    if raw.shape[1] < 9:
        raise RuntimeError(f"Sheet '{sheet_name}' does not have the expected RAM layout")

    name_col = raw.columns[1]
    domain_value_cols = list(raw.columns[2:9])
    lookup: dict[str, dict[int, float]] = {}

    for _, row in raw.iterrows():
        interface_name = _fbmc_clean_text(row.get(name_col))
        if not interface_name:
            continue
        values: dict[int, float] = {}
        for col in domain_value_cols:
            try:
                domain_id = int(float(col))
            except Exception:
                continue
            values[domain_id] = _fbmc_numeric_or_zero(row.get(col))
        lookup[interface_name] = values

    if not lookup:
        raise RuntimeError(f"No interface RAM values found in sheet '{sheet_name}'")
    return lookup


def _fbmc_write_ram_files(
    domain_schedule: pd.DataFrame,
    domain_cols: list[str],
    ram_lookup: dict[str, dict[int, float]],
    interface_name_pairs: list[tuple[str, str]],
    output_dir: Path,
) -> tuple[int, list[str], dict[str, Path]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    warnings: list[str] = []
    ram_file_by_interface: dict[str, Path] = {}
    base_cols = ["Year", "Month", "Day", "Period"]

    for raw_interface_name, plexos_interface_name in interface_name_pairs:
        values_by_domain = ram_lookup.get(raw_interface_name)
        if values_by_domain is None:
            warnings.append(f"Skipped RAM file for '{plexos_interface_name}': no matching row in RAM lookup for '{raw_interface_name}'")
            continue

        out_df = domain_schedule[base_cols].copy()
        for output_col, domain_col in enumerate(domain_cols, start=1):
            domain_codes = pd.to_numeric(domain_schedule[domain_col], errors="coerce").astype("Int64")
            out_df[output_col] = domain_codes.map(values_by_domain).fillna(0.0).astype(float)

        out_path = output_dir / f"{_fbmc_safe_filename(plexos_interface_name)}.csv"
        out_df.to_csv(out_path, index=False)
        ram_file_by_interface[plexos_interface_name] = out_path

    return len(ram_file_by_interface), warnings, ram_file_by_interface


# -----------------------------
# Main feature
# -----------------------------
def step_fbmc_14_2() -> None:
    mapping_df = _fbmc_read_mapping(FBMC_MAPPING_XLSX)
    all_mapping_lines = mapping_df["PLEXOS"].dropna().astype(str).str.strip().tolist()
    all_mapping_lines = [line for line in dict.fromkeys(all_mapping_lines) if line]
    hub_lines = [line for line in all_mapping_lines if "-HUB" in line]

    db, ClassEnum, CollectionEnum, SystemNS = _fbmc_bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    region_class = _fbmc_class_enum(ClassEnum, SystemNS, "Region", FBMC_CLASS_IDS["Region"])
    node_class = _fbmc_class_enum(ClassEnum, SystemNS, "Node", FBMC_CLASS_IDS["Node"])
    line_class = _fbmc_class_enum(ClassEnum, SystemNS, "Line", FBMC_CLASS_IDS["Line"])
    interface_class = _fbmc_class_enum(ClassEnum, SystemNS, "Interface", FBMC_CLASS_IDS["Interface"])
    scenario_class = _fbmc_class_enum(ClassEnum, SystemNS, "Scenario", FBMC_CLASS_IDS["Scenario"])

    node_region_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["NodeRegions", "RegionNodes", "Region"],
        FBMC_COLLECTION_IDS["NodeRegion"],
    )
    system_region_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemRegions", "Regions"],
        FBMC_COLLECTION_IDS["SystemRegions"],
    )
    system_node_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemNodes", "Nodes"],
        FBMC_COLLECTION_IDS["SystemNodes"],
    )
    system_line_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemLines", "Lines"],
        FBMC_COLLECTION_IDS["SystemLines"],
    )
    line_node_from_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["LineNodeFrom", "NodeFrom", "LineNodesFrom"],
        FBMC_COLLECTION_IDS["LineNodeFrom"],
    )
    line_node_to_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["LineNodeTo", "NodeTo", "LineNodesTo"],
        FBMC_COLLECTION_IDS["LineNodeTo"],
    )
    system_interface_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemInterfaces", "Interfaces"],
        FBMC_COLLECTION_IDS["SystemInterfaces"],
    )
    interface_line_collection = _fbmc_collection_enum(
        CollectionEnum,
        SystemNS,
        ["InterfaceLines", "Lines"],
        FBMC_COLLECTION_IDS["InterfaceLines"],
    )

    enum_units = _fbmc_property_enum(db, SystemNS, "System", "Line", "Lines", "Units", FBMC_LINE_ENUMS["Units"])
    enum_region_units = _fbmc_property_enum(db, SystemNS, "System", "Region", "Regions", "Units", FBMC_REGION_ENUMS["Units"])
    enum_node_units = _fbmc_property_enum(db, SystemNS, "System", "Node", "Nodes", "Units", FBMC_NODE_ENUMS["Units"])
    enum_max_flow = _fbmc_property_enum(db, SystemNS, "System", "Line", "Lines", "Max Flow", FBMC_LINE_ENUMS["Max Flow"])
    enum_min_flow = _fbmc_property_enum(db, SystemNS, "System", "Line", "Lines", "Min Flow", FBMC_LINE_ENUMS["Min Flow"])
    enum_wheeling_charge = _fbmc_property_enum(db, SystemNS, "System", "Line", "Lines", "Wheeling Charge", FBMC_LINE_ENUMS["Wheeling Charge"])
    enum_wheeling_charge_back = _fbmc_property_enum(db, SystemNS, "System", "Line", "Lines", "Wheeling Charge Back", FBMC_LINE_ENUMS["Wheeling Charge Back"])

    enum_interface_units = _fbmc_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Units", FBMC_INTERFACE_ENUMS["Units"])
    enum_interface_max_flow = _fbmc_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Max Flow", FBMC_INTERFACE_ENUMS["Max Flow"])
    enum_interface_limit_penalty = _fbmc_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Limit Penalty", FBMC_INTERFACE_ENUMS["Limit Penalty"])
    enum_flow_coefficient = _fbmc_property_enum(db, SystemNS, "Interface", "Line", "Lines", "Flow Coefficient", FBMC_INTERFACE_LINE_ENUMS["Flow Coefficient"])

    created_nodes = 0
    created_regions = 0
    created_lines = 0
    created_interfaces = 0
    categorized_hub_lines = 0
    line_property_rows = 0
    hub_object_property_rows = 0
    existing_hvac_hvdc_units_rows = 0
    interface_property_rows = 0
    flow_coefficient_property_rows = 0
    interface_line_memberships = 0
    warnings: list[str] = []

    try:
        _fbmc_ensure_category(db, line_class, FBMC_LINE_CATEGORY)

        if _fbmc_ensure_object(db, region_class, FBMC_HUB_NAME, add_to_system=True):
            created_regions += 1
        if _fbmc_ensure_object(db, node_class, FBMC_HUB_NAME, add_to_system=True):
            created_nodes += 1

        if _fbmc_ensure_membership_bi(db, node_region_collection, FBMC_HUB_NAME, FBMC_HUB_NAME) is None:
            warnings.append(f"Could not create Node/Region membership for {FBMC_HUB_NAME}")

        _fbmc_ensure_object(db, scenario_class, FBMC_SCENARIO_NAME, add_to_system=True)
        fbmc_scenario = SystemNS.String(FBMC_SCENARIO_NAME)

        hub_region_mem_id = _fbmc_ensure_membership(db, system_region_collection, "System", FBMC_HUB_NAME)
        if hub_region_mem_id is None:
            warnings.append(f"Could not resolve System/Region membership for {FBMC_HUB_NAME}; skipped hub region Units")
        else:
            _fbmc_add_property(db, enum_region_units, hub_region_mem_id, 0.0)
            _fbmc_add_property(db, enum_region_units, hub_region_mem_id, 1.0, scenario=fbmc_scenario)
            hub_object_property_rows += 2

        hub_node_mem_id = _fbmc_ensure_membership(db, system_node_collection, "System", FBMC_HUB_NAME)
        if hub_node_mem_id is None:
            warnings.append(f"Could not resolve System/Node membership for {FBMC_HUB_NAME}; skipped hub node Units")
        else:
            _fbmc_add_property(db, enum_node_units, hub_node_mem_id, 0.0)
            _fbmc_add_property(db, enum_node_units, hub_node_mem_id, 1.0, scenario=fbmc_scenario)
            hub_object_property_rows += 2

        existing_hvac_lines = _fbmc_get_objects_in_category_safe(db, line_class, "HVAC")
        existing_hvdc_lines = _fbmc_get_objects_in_category_safe(db, line_class, "HVDC")
        existing_hvac_hvdc_lines = sorted(existing_hvac_lines | existing_hvdc_lines)
        existing_hvac_hvdc_line_mem_ids: dict[str, Any] = {}
        for line_name in existing_hvac_hvdc_lines:
            line_mem_id = _fbmc_ensure_membership(db, system_line_collection, "System", line_name)
            if line_mem_id is None:
                warnings.append(f"Could not resolve System/Line membership for existing HVAC/HVDC line {line_name}; skipped FBMC Units")
                continue
            existing_hvac_hvdc_line_mem_ids[line_name] = line_mem_id
            _fbmc_add_property(db, enum_units, line_mem_id, 0.0, scenario=fbmc_scenario)
            existing_hvac_hvdc_units_rows += 1

        nodes_in_db = _fbmc_get_objects_safe(db, node_class)

        for line_name in hub_lines:
            if _fbmc_ensure_object(db, line_class, line_name, add_to_system=True, category=FBMC_LINE_CATEGORY):
                created_lines += 1
            if _fbmc_categorize_object(db, line_class, line_name, FBMC_LINE_CATEGORY):
                categorized_hub_lines += 1

            line_mem_id = _fbmc_ensure_membership(db, system_line_collection, "System", line_name)
            if line_mem_id is None:
                warnings.append(f"Could not resolve System/Line membership for {line_name}; skipped line properties")
                continue

            source_node = line_name.split("-HUB", 1)[0].strip()
            if source_node and source_node in nodes_in_db:
                if _fbmc_ensure_membership_bi(db, line_node_from_collection, line_name, source_node) is None:
                    warnings.append(f"Could not create Node From membership for {line_name} -> {source_node}")
            elif source_node:
                warnings.append(f"Source node '{source_node}' for line '{line_name}' was not found; Node From not created")

            if _fbmc_ensure_membership_bi(db, line_node_to_collection, line_name, FBMC_HUB_NAME) is None:
                warnings.append(f"Could not create Node To membership for {line_name} -> {FBMC_HUB_NAME}")

            _fbmc_add_property(db, enum_units, line_mem_id, 0.0)
            _fbmc_add_property(db, enum_units, line_mem_id, 1.0, scenario=fbmc_scenario)
            _fbmc_add_property(db, enum_max_flow, line_mem_id, 20000.0)
            _fbmc_add_property(db, enum_min_flow, line_mem_id, -20000.0)
            _fbmc_add_property(db, enum_wheeling_charge, line_mem_id, 0.01)
            _fbmc_add_property(db, enum_wheeling_charge_back, line_mem_id, 0.01)
            line_property_rows += 6

        target_year_summaries: list[dict[str, Any]] = []

        for target_year in FBMC_TARGET_YEARS:
            ptdf_sheet = f"PTDF {target_year}"
            ram_sheet = f"RAM {target_year}"
            interface_category = f"TY{target_year}"
            ptdf_output_dir = FBMC_DATA_FILES_DIR / f"PTDF_{target_year}"
            ram_output_dir = FBMC_DATA_FILES_DIR / f"RAM_{target_year}"

            raw_ptdf, raw_interface_rows = _fbmc_read_interfaces_and_ptdf(FBMC_DOMAIN_XLSX, ptdf_sheet)
            interface_name_pairs = [
                (raw_interface_name, _fbmc_interface_name_for_year(raw_interface_name, target_year))
                for _, raw_interface_name in raw_interface_rows
            ]
            interface_rows = [
                (row_idx, _fbmc_interface_name_for_year(raw_interface_name, target_year))
                for row_idx, raw_interface_name in raw_interface_rows
            ]
            interface_names = [plexos_interface_name for _, plexos_interface_name in interface_name_pairs]
            domain_schedule, domain_cols, domain_warnings = _fbmc_read_domain_assignment(
                FBMC_DOMAIN_XLSX,
                FBMC_DOMAIN_ASSIGNMENT_SHEET,
                target_year,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in domain_warnings])
            ram_lookup = _fbmc_read_ram_lookup(FBMC_DOMAIN_XLSX, ram_sheet)
            _fbmc_ensure_category(db, interface_class, interface_category)

            files_written, file_warnings, ptdf_file_by_line = _fbmc_write_ptdf_files(
                raw_ptdf,
                interface_rows,
                mapping_df,
                ptdf_output_dir,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in file_warnings])

            ram_files_written, ram_warnings, ram_file_by_interface = _fbmc_write_ram_files(
                domain_schedule,
                domain_cols,
                ram_lookup,
                interface_name_pairs,
                ram_output_dir,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in ram_warnings])

            date_from_tuple, date_to_tuple = FBMC_INTERFACE_DATE_RANGES[target_year]
            interface_date_from = _fbmc_datetime(SystemNS, date_from_tuple)
            interface_date_to = _fbmc_datetime(SystemNS, date_to_tuple)
            year_interfaces_created = 0
            year_interface_property_rows = 0
            year_interface_line_memberships = 0
            year_flow_coefficient_property_rows = 0
            year_existing_hvac_hvdc_units_rows = 0
            year_flow_coefficient_lines: set[str] = set()

            for interface_name in interface_names:
                if _fbmc_ensure_object(
                    db,
                    interface_class,
                    interface_name,
                    add_to_system=True,
                    category=interface_category,
                ):
                    created_interfaces += 1
                    year_interfaces_created += 1

                interface_mem_id = _fbmc_ensure_membership(db, system_interface_collection, "System", interface_name)
                if interface_mem_id is None:
                    warnings.append(f"TY{target_year}: Could not resolve System/Interface membership for {interface_name}; skipped interface properties")
                else:
                    _fbmc_add_property(db, enum_interface_units, interface_mem_id, 0.0)
                    _fbmc_add_property(
                        db,
                        enum_interface_units,
                        interface_mem_id,
                        1.0,
                        scenario=fbmc_scenario,
                        date_from=interface_date_from,
                        date_to=interface_date_to,
                    )
                    _fbmc_add_property(db, enum_interface_limit_penalty, interface_mem_id, -1.0)
                    interface_property_rows += 3
                    year_interface_property_rows += 3

                    ram_file_path = ram_file_by_interface.get(interface_name)
                    if ram_file_path is None:
                        warnings.append(f"TY{target_year}: No RAM file available for interface '{interface_name}'; Max Flow DataFile link skipped")
                    else:
                        _fbmc_add_property(
                            db,
                            enum_interface_max_flow,
                            interface_mem_id,
                            1.0,
                            data_file=_fbmc_datafile_ref(SystemNS, ram_file_path),
                            date_from=interface_date_from,
                            date_to=interface_date_to,
                        )
                        interface_property_rows += 1
                        year_interface_property_rows += 1

                for line_name in all_mapping_lines:
                    membership_id = _fbmc_ensure_membership_bi(db, interface_line_collection, interface_name, line_name)
                    if membership_id is None:
                        warnings.append(f"TY{target_year}: Could not create Interface/Line membership: {interface_name} / {line_name}")
                        continue

                    interface_line_memberships += 1
                    year_interface_line_memberships += 1
                    ptdf_file_path = ptdf_file_by_line.get(line_name)
                    if ptdf_file_path is None:
                        warnings.append(f"TY{target_year}: No PTDF file available for line '{line_name}'; Flow Coefficient DataFile link skipped for '{interface_name}'")
                        continue

                    _fbmc_add_property(
                        db,
                        enum_flow_coefficient,
                        membership_id,
                        1.0,
                        data_file=_fbmc_datafile_ref(SystemNS, ptdf_file_path),
                    )
                    flow_coefficient_property_rows += 1
                    year_flow_coefficient_property_rows += 1
                    year_flow_coefficient_lines.add(line_name)

            year_existing_hvac_hvdc_lines = sorted(
                line_name
                for line_name in year_flow_coefficient_lines
                if line_name in existing_hvac_hvdc_line_mem_ids
            )
            for line_name in year_existing_hvac_hvdc_lines:
                _fbmc_add_property(
                    db,
                    enum_units,
                    existing_hvac_hvdc_line_mem_ids[line_name],
                    1.0,
                    scenario=fbmc_scenario,
                    date_from=interface_date_from,
                    date_to=interface_date_to,
                )
                existing_hvac_hvdc_units_rows += 1
                year_existing_hvac_hvdc_units_rows += 1

            target_year_summaries.append({
                "year": target_year,
                "interfaces": len(interface_rows),
                "interfaces_created": year_interfaces_created,
                "interface_property_rows": year_interface_property_rows,
                "interface_line_memberships": year_interface_line_memberships,
                "flow_coefficient_property_rows": year_flow_coefficient_property_rows,
                "existing_hvac_hvdc_units_rows": year_existing_hvac_hvdc_units_rows,
                "existing_hvac_hvdc_ptdf_lines": len(year_existing_hvac_hvdc_lines),
                "ptdf_files": files_written,
                "ptdf_output_dir": ptdf_output_dir,
                "ram_files": ram_files_written,
                "ram_output_dir": ram_output_dir,
            })

        total_ptdf_files_written = sum(item["ptdf_files"] for item in target_year_summaries)
        total_ram_files_written = sum(item["ram_files"] for item in target_year_summaries)

        print("\n[STEP 14.2] FBMC import complete.")
        print(f"  Hub node ensured: {FBMC_HUB_NAME} (created {created_nodes})")
        print(f"  Hub region ensured: {FBMC_HUB_NAME} (created {created_regions})")
        print(f"  FBMC scenario ensured: {FBMC_SCENARIO_NAME}")
        print(f"  HUB lines in mapping: {len(hub_lines)}")
        print(f"  HUB lines created: {created_lines}")
        print(f"  HUB lines categorized as {FBMC_LINE_CATEGORY}: {categorized_hub_lines}")
        print(f"  Line property rows written: {line_property_rows}")
        print(f"  Hub node/region Units rows written: {hub_object_property_rows}")
        print(f"  Existing HVAC/HVDC line FBMC Units rows written: {existing_hvac_hvdc_units_rows}")
        print(f"  Target years processed: {FBMC_TARGET_YEARS}")
        print(f"  Interfaces created: {created_interfaces}")
        print(f"  Interface property rows written: {interface_property_rows}")
        print(f"  Interface/Line memberships touched: {interface_line_memberships}")
        print(f"  Flow Coefficient DataFile rows written: {flow_coefficient_property_rows}")
        print(f"  PTDF files written: {total_ptdf_files_written}")
        print(f"  RAM files written: {total_ram_files_written}")
        for summary in target_year_summaries:
            print(
                f"  TY{summary['year']}: interfaces={summary['interfaces']}, "
                f"created={summary['interfaces_created']}, PTDF files={summary['ptdf_files']}, "
                f"RAM files={summary['ram_files']}, "
                f"existing HVAC/HVDC Units rows={summary['existing_hvac_hvdc_units_rows']}"
            )
            print(f"    PTDF output folder: {summary['ptdf_output_dir']}")
            print(f"    RAM output folder: {summary['ram_output_dir']}")

        if warnings:
            print("\n[WARNINGS]")
            for message in warnings[:50]:
                print(f"  - {message}")
            if len(warnings) > 50:
                print(f"  - ... {len(warnings) - 50} more warnings")

    finally:
        try:
            db.Close()
        except Exception:
            pass


if RUN_FBMC_NORDICS_14_2:
    step_fbmc_14_2()
else:
    print("[SKIP] Step 14.2 FBMC - Nordics skipped because RUN_FBMC_NORDICS_14_2 is False.")


#### 14.3 FBMC - CORE


In [ ]:
# STEP 14.3 - FBMC CORE hub, lines, interfaces, PTDF/RAM files, and year-specific links
RUN_FBMC_CORE_14_3 = False

from __future__ import annotations

import calendar
import os
from pathlib import Path
from typing import Any, Optional

import pandas as pd

# -----------------------------
# Inputs
# -----------------------------
API_DIR = globals().get("API_DIR", Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0 API"))
BIN_DIR = globals().get("BIN_DIR", Path(r"C:\Program Files\Energy Exemplar\PLEXOS 12.0"))

FBMC_CORE_HUB_NAME = "FBMC_CORE_HUB"
FBMC_CORE_SCENARIO_NAME = "FBMC"
FBMC_CORE_LINE_CATEGORY = "FBMC"
FBMC_CORE_TARGET_YEARS = [2028, 2030, 2033, 2035]
FBMC_CORE_INTERFACE_DATE_RANGES = {
    2028: ((2028, 1, 1), (2029, 12, 31)),
    2030: ((2030, 1, 1), (2032, 12, 31)),
    2033: ((2033, 1, 1), (2034, 12, 31)),
    2035: ((2035, 1, 1), None),
}
FBMC_CORE_DOMAIN_ASSIGNMENT_SHEET = "Domain Assignment"

FBMC_CORE_MAPPING_XLSX = BASE_DIR / "FBMC_Mapping_CORE.xlsx"
FBMC_CORE_DOMAIN_XLSX = BASE_DIR / "ERAA_2025_FBDomains" / "FB-Domain-CORE_simplified.xlsx"
FBMC_CORE_DATA_FILES_DIR = BASE_DIR / "Database" / "Data Files" / "FBMC_CORE"

# Confirmed from 12R02_Tables.xlsx for PLEXOS 12.0 R02.
FBMC_CORE_CLASS_IDS = {
    "Region": 21,
    "Node": 24,
    "Line": 26,
    "Interface": 30,
    "Scenario": 85,
    "Timeslice": 83,
}

FBMC_CORE_COLLECTION_IDS = {
    "SystemRegions": 214,
    "SystemNodes": 298,
    "NodeRegion": 302,
    "SystemLines": 322,
    "LineNodeFrom": 326,
    "LineNodeTo": 327,
    "SystemInterfaces": 353,
    "InterfaceLines": 356,
    "SystemTimeslices": 786,
}

FBMC_CORE_LINE_ENUMS = {
    "Units": 14,
    "Max Flow": 15,
    "Min Flow": 16,
    "Wheeling Charge": 42,
    "Wheeling Charge Back": 43,
}

FBMC_CORE_REGION_ENUMS = {
    "Units": 50,
}

FBMC_CORE_NODE_ENUMS = {
    "Units": 21,
}

FBMC_CORE_INTERFACE_ENUMS = {
    "Units": 3,
    "Max Flow": 5,
    "Limit Penalty": 8,
}

FBMC_CORE_INTERFACE_LINE_ENUMS = {
    "Flow Coefficient": 1,
}

FBMC_CORE_TIMESLICE_ENUMS = {
    "Include": 1,
}

FBMC_CORE_TIMESLICES = {
    "winter1": {"name": "Winter1", "pattern": "!M4-9"},
    "summer1": {"name": "Summer1", "pattern": "M4-9"},
}

CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS = [API_DIR / "PLEXOS_NET.dll", BIN_DIR / "PLEXOS_NET.dll"]


# -----------------------------
# PLEXOS API helpers
# -----------------------------
def _fbmc_core_bootstrap_plexos_net():
    for directory in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(directory) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(directory))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        dll_name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll_path = base / f"{dll_name}.dll"
            if dll_path.exists():
                return Assembly.LoadFile(str(dll_path))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for dll_path in CORE_DLLS:
        if dll_path.exists():
            Assembly.LoadFile(str(dll_path))
            break
    else:
        raise FileNotFoundError(f"PLEXOS_NET.Core.dll not found in {API_DIR} or {BIN_DIR}")

    for dll_path in NET_DLLS:
        if dll_path.exists():
            Assembly.LoadFile(str(dll_path))
            break
    else:
        raise FileNotFoundError(f"PLEXOS_NET.dll not found in {API_DIR} or {BIN_DIR}")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _fbmc_core_enum_from_id(SystemNS, enum_type, enum_id: int):
    return SystemNS.Enum.ToObject(enum_type, int(enum_id))


def _fbmc_core_class_enum(ClassEnum, SystemNS, preferred_name: str, fallback_id: int):
    if hasattr(ClassEnum, preferred_name):
        return getattr(ClassEnum, preferred_name)
    return _fbmc_core_enum_from_id(SystemNS, ClassEnum, fallback_id)


def _fbmc_core_collection_enum(CollectionEnum, SystemNS, preferred_names: list[str], fallback_id: int):
    for name in preferred_names:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)
    return _fbmc_core_enum_from_id(SystemNS, CollectionEnum, fallback_id)


def _fbmc_core_get_objects_safe(db, class_enum_value) -> set[str]:
    try:
        result = db.GetObjects(class_enum_value)
    except Exception:
        return set()
    if result is None:
        return set()
    try:
        return {str(item).strip() for item in list(result) if str(item).strip()}
    except Exception:
        return set()


def _fbmc_core_get_objects_in_category_safe(db, class_enum_value, category_name: str) -> set[str]:
    try:
        result = db.GetObjectsInCategory(class_enum_value, str(category_name))
    except Exception:
        return set()
    if result is None:
        return set()
    try:
        return {str(item).strip() for item in list(result) if str(item).strip()}
    except Exception:
        return set()


def _fbmc_core_ensure_category(db, class_enum_value, category_name: str) -> None:
    if not category_name:
        return
    try:
        if bool(db.CategoryExists(class_enum_value, str(category_name))):
            return
    except Exception:
        pass
    try:
        db.AddCategory(class_enum_value, str(category_name))
    except Exception:
        pass


def _fbmc_core_categorize_object(db, class_enum_value, name: str, category_name: str) -> bool:
    if not name or not category_name:
        return False
    try:
        return bool(db.CategorizeObject(class_enum_value, str(name), str(category_name)))
    except Exception:
        return False


def _fbmc_core_ensure_object(
    db,
    class_enum_value,
    name: str,
    *,
    add_to_system: bool = True,
    category: str = "",
    description: str = "",
) -> bool:
    object_name = str(name).strip()
    if not object_name:
        return False
    if object_name in _fbmc_core_get_objects_safe(db, class_enum_value):
        if category:
            _fbmc_core_categorize_object(db, class_enum_value, object_name, category)
        return False
    try:
        db.AddObject(object_name, class_enum_value, bool(add_to_system), str(category or ""), str(description or ""))
    except Exception as exc:
        if category:
            db.AddObject(object_name, class_enum_value, bool(add_to_system), "", str(description or ""))
            _fbmc_core_categorize_object(db, class_enum_value, object_name, category)
            print(f"[WARN] Added '{object_name}' without direct category '{category}' because category write failed: {exc}")
        else:
            raise
    return True


def _fbmc_core_get_membership_id_optional(db, collection_enum, parent_name: str, child_name: str) -> Optional[int]:
    try:
        return int(db.GetMembershipID(collection_enum, str(parent_name), str(child_name)))
    except Exception:
        return None


def _fbmc_core_ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> Optional[int]:
    parent_name = str(parent_name).strip()
    child_name = str(child_name).strip()
    if not parent_name or not child_name:
        return None

    existing_id = _fbmc_core_get_membership_id_optional(db, collection_enum, parent_name, child_name)
    if existing_id is not None:
        return existing_id

    try:
        db.AddMembership(collection_enum, parent_name, child_name)
    except Exception:
        pass
    return _fbmc_core_get_membership_id_optional(db, collection_enum, parent_name, child_name)


def _fbmc_core_ensure_membership_bi(db, collection_enum, first_name: str, second_name: str) -> Optional[int]:
    membership_id = _fbmc_core_ensure_membership(db, collection_enum, first_name, second_name)
    if membership_id is not None:
        return membership_id
    return _fbmc_core_ensure_membership(db, collection_enum, second_name, first_name)


def _fbmc_core_add_property(
    db,
    enum_id: int,
    membership_id: int,
    value: float,
    *,
    scenario=None,
    data_file=None,
    date_from=None,
    date_to=None,
    pattern=None,
) -> None:
    db.AddProperty(
        int(membership_id),
        int(enum_id),
        1,
        float(value),
        date_from,
        date_to,
        None,
        data_file,
        pattern,
        scenario,
        None,
    )


def _fbmc_core_property_enum(
    db,
    SystemNS,
    parent_name: str,
    child_name: str,
    collection_name: str,
    property_name: str,
    fallback_enum: int,
) -> int:
    try:
        return int(db.PropertyName2EnumId(
            SystemNS.String(parent_name),
            SystemNS.String(child_name),
            SystemNS.String(collection_name),
            SystemNS.String(property_name),
        ))
    except Exception:
        return int(fallback_enum)


def _fbmc_core_datafile_ref(SystemNS, path: Path):
    try:
        rel = path.resolve().relative_to(DATABASE_DIR.resolve())
    except Exception:
        rel = path
    return SystemNS.String(str(rel).replace("/", "\\"))


def _fbmc_core_datetime(SystemNS, date_tuple: tuple[int, int, int] | None):
    if date_tuple is None:
        return None
    year, month, day = date_tuple
    return SystemNS.DateTime(int(year), int(month), int(day), 0, 0, 0)


# -----------------------------
# Workbook helpers
# -----------------------------
def _fbmc_core_clean_text(value: Any) -> str:
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def _fbmc_core_read_mapping(mapping_xlsx: Path) -> pd.DataFrame:
    if not mapping_xlsx.exists():
        raise FileNotFoundError(f"FBMC mapping file not found: {mapping_xlsx}")
    mapping_df = pd.read_excel(mapping_xlsx, sheet_name=0)
    if mapping_df.shape[1] < 2:
        raise RuntimeError(f"FBMC mapping must have at least two columns: {mapping_xlsx}")

    out = mapping_df.iloc[:, :2].copy()
    out.columns = ["Dataset", "PLEXOS"]
    out["Dataset"] = out["Dataset"].map(_fbmc_core_clean_text)
    out["PLEXOS"] = out["PLEXOS"].map(_fbmc_core_clean_text)
    out = out[(out["Dataset"] != "") & (out["PLEXOS"] != "")].copy()
    if out.empty:
        raise RuntimeError(f"No usable Dataset/PLEXOS rows found in {mapping_xlsx}")
    return out


def _fbmc_core_read_interfaces_and_ptdf(domain_xlsx: Path, sheet_name: str) -> tuple[pd.DataFrame, list[tuple[int, str, str]]]:
    if not domain_xlsx.exists():
        raise FileNotFoundError(f"FB-domain workbook not found: {domain_xlsx}")

    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name, header=None)
    if raw.shape[0] < 3 or raw.shape[1] < 4:
        raise RuntimeError(f"Sheet '{sheet_name}' does not have the expected PTDF layout")

    interface_rows: list[tuple[int, str, str]] = []
    for row_idx in range(2, raw.shape[0]):
        timeslice_label = _fbmc_core_clean_text(raw.iat[row_idx, 0]).lower()
        cnec_id = _fbmc_core_clean_text(raw.iat[row_idx, 1])
        if not timeslice_label and not cnec_id:
            break
        if timeslice_label not in FBMC_CORE_TIMESLICES:
            continue
        if not cnec_id:
            raise RuntimeError(f"Missing CNEC_ID in {domain_xlsx}, sheet '{sheet_name}', row {row_idx + 1}")
        interface_rows.append((row_idx, timeslice_label, cnec_id))

    if not interface_rows:
        raise RuntimeError(f"No winter1/summer1 CNEC_ID rows found in {domain_xlsx}, sheet '{sheet_name}', columns A:B")

    return raw, interface_rows


def _fbmc_core_safe_filename(stem: str) -> str:
    invalid_chars = '<>:"/\\|?*'
    safe = str(stem).strip()
    for char in invalid_chars:
        safe = safe.replace(char, "_")
    return safe or "unnamed"


def _fbmc_core_interface_name_for_year(interface_name: str, target_year: int) -> str:
    base_name = str(interface_name).strip()
    year_tag = f"TY{int(target_year)}"
    if base_name.upper().endswith(f" - {year_tag}".upper()):
        return base_name
    return f"{base_name} - {year_tag}"


def _fbmc_core_numeric_or_zero(value: Any) -> float:
    try:
        if pd.isna(value):
            return 0.0
    except Exception:
        pass
    numeric_value = pd.to_numeric(value, errors="coerce")
    if pd.isna(numeric_value):
        return 0.0
    return float(numeric_value)


def _fbmc_core_write_ptdf_files(
    raw_ptdf: pd.DataFrame,
    interface_rows: list[tuple[int, str, str]],
    mapping_df: pd.DataFrame,
    output_dir: Path,
) -> tuple[int, list[str], dict[str, dict[str, Path]], dict[str, set[str]]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_to_plexos = dict(zip(mapping_df["Dataset"], mapping_df["PLEXOS"]))

    files_written = 0
    warnings: list[str] = []
    ptdf_file_by_line_and_timeslice: dict[str, dict[str, Path]] = {}
    timeslices_by_interface: dict[str, set[str]] = {}

    rows_by_timeslice: dict[str, list[tuple[int, str]]] = {}
    for row_idx, timeslice_label, interface_name in interface_rows:
        rows_by_timeslice.setdefault(timeslice_label, []).append((row_idx, interface_name))
        timeslices_by_interface.setdefault(interface_name, set()).add(timeslice_label)

    for col_idx in range(3, raw_ptdf.shape[1]):
        dataset_line = _fbmc_core_clean_text(raw_ptdf.iat[1, col_idx])
        column_has_values = raw_ptdf.iloc[2:, col_idx].notna().any()
        if not dataset_line and not column_has_values:
            break
        if not dataset_line:
            warnings.append(f"Skipped PTDF column {col_idx + 1}: missing dataset line header in row 2")
            continue

        plexos_line = dataset_to_plexos.get(dataset_line)
        if not plexos_line:
            warnings.append(f"Skipped PTDF column '{dataset_line}': no PLEXOS mapping found")
            continue

        for timeslice_label, timeslice_rows in rows_by_timeslice.items():
            rows = []
            for row_idx, interface_name in timeslice_rows:
                value = _fbmc_core_numeric_or_zero(raw_ptdf.iat[row_idx, col_idx])
                rows.append({"Name": f"{interface_name}{plexos_line}", "Value": value})

            if not rows:
                continue

            timeslice_name = FBMC_CORE_TIMESLICES[timeslice_label]["name"]
            label_output_dir = output_dir / timeslice_name
            label_output_dir.mkdir(parents=True, exist_ok=True)
            out_path = label_output_dir / f"{_fbmc_core_safe_filename(plexos_line)}.csv"
            pd.DataFrame(rows, columns=["Name", "Value"]).to_csv(out_path, index=False)
            ptdf_file_by_line_and_timeslice.setdefault(plexos_line, {})[timeslice_label] = out_path
            files_written += 1

    return files_written, warnings, ptdf_file_by_line_and_timeslice, timeslices_by_interface


def _fbmc_core_read_domain_assignment(
    domain_xlsx: Path,
    sheet_name: str,
    target_year: int,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name)
    required_cols = ["Year", "Month", "Day", "Hour"]
    missing = [col for col in required_cols if col not in raw.columns]
    if missing:
        raise RuntimeError(f"Sheet '{sheet_name}' missing required columns: {missing}")

    domain_cols = list(raw.columns[4:40])
    if len(domain_cols) != 36:
        raise RuntimeError(f"Sheet '{sheet_name}' expected 36 domain columns from E:AN, found {len(domain_cols)}")

    year_df = raw[pd.to_numeric(raw["Year"], errors="coerce").eq(int(target_year))].copy()
    if year_df.empty:
        raise RuntimeError(f"No rows for year {target_year} found in sheet '{sheet_name}'")

    for col in ["Year", "Month", "Day", "Hour"]:
        year_df[col] = pd.to_numeric(year_df[col], errors="coerce").astype("Int64")
    year_df["Period"] = (year_df["Hour"] + 1).astype("Int64")

    warnings: list[str] = []
    has_feb29 = year_df["Month"].eq(2).mul(year_df["Day"].eq(29)).any()
    if calendar.isleap(int(target_year)) and not has_feb29:
        feb28 = year_df[year_df["Month"].eq(2) & year_df["Day"].eq(28)].copy()
        if feb28.empty:
            warnings.append(f"Could not add Feb 29 for {target_year}: no Feb 28 rows found")
        else:
            feb29 = feb28.copy()
            feb29["Day"] = 29
            year_df = pd.concat([year_df, feb29], ignore_index=True)
            warnings.append(f"Inserted Feb 29 for {target_year} by copying Feb 28 domain assignments")

    year_df = year_df.sort_values(["Year", "Month", "Day", "Period"]).reset_index(drop=True)
    out = year_df[["Year", "Month", "Day", "Period"] + domain_cols].copy()
    return out, domain_cols, warnings


def _fbmc_core_read_ram_lookup(domain_xlsx: Path, sheet_name: str) -> dict[str, dict[str, float]]:
    raw = pd.read_excel(domain_xlsx, sheet_name=sheet_name)
    if raw.shape[1] < 2:
        raise RuntimeError(f"Sheet '{sheet_name}' does not have the expected CORE RAM layout")

    name_col = raw.columns[0]
    domain_value_cols = list(raw.columns[1:])
    lookup: dict[str, dict[str, float]] = {}

    for _, row in raw.iterrows():
        interface_key = _fbmc_core_clean_text(row.get(name_col))
        if not interface_key:
            continue
        values: dict[str, float] = {}
        for col in domain_value_cols:
            domain_label = _fbmc_core_clean_text(col).lower()
            if not domain_label:
                continue
            values[domain_label] = _fbmc_core_numeric_or_zero(row.get(col))
        lookup[interface_key] = values

    if not lookup:
        raise RuntimeError(f"No interface RAM values found in sheet '{sheet_name}'")
    return lookup


def _fbmc_core_write_ram_files(
    domain_schedule: pd.DataFrame,
    domain_cols: list[str],
    ram_lookup: dict[str, dict[str, float]],
    interface_name_pairs: list[tuple[str, str]],
    output_dir: Path,
) -> tuple[int, list[str], dict[str, Path]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    warnings: list[str] = []
    ram_file_by_interface: dict[str, Path] = {}
    base_cols = ["Year", "Month", "Day", "Period"]

    for raw_interface_key, plexos_interface_name in interface_name_pairs:
        values_by_domain = ram_lookup.get(raw_interface_key)
        if values_by_domain is None:
            warnings.append(f"Skipped RAM file for '{plexos_interface_name}': no matching row in RAM lookup for '{raw_interface_key}'")
            continue

        out_df = domain_schedule[base_cols].copy()
        for output_col, domain_col in enumerate(domain_cols, start=1):
            domain_labels = domain_schedule[domain_col].apply(lambda value: _fbmc_core_clean_text(value).lower())
            out_df[output_col] = domain_labels.map(values_by_domain).fillna(0.0).astype(float)

        out_path = output_dir / f"{_fbmc_core_safe_filename(plexos_interface_name)}.csv"
        out_df.to_csv(out_path, index=False)
        ram_file_by_interface[plexos_interface_name] = out_path

    return len(ram_file_by_interface), warnings, ram_file_by_interface


# -----------------------------
# Main feature
# -----------------------------
def step_fbmc_core_14_3() -> None:
    mapping_df = _fbmc_core_read_mapping(FBMC_CORE_MAPPING_XLSX)
    all_mapping_lines = mapping_df["PLEXOS"].dropna().astype(str).str.strip().tolist()
    all_mapping_lines = [line for line in dict.fromkeys(all_mapping_lines) if line]
    hub_lines = [line for line in all_mapping_lines if "-HUB" in line]

    db, ClassEnum, CollectionEnum, SystemNS = _fbmc_core_bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    region_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Region", FBMC_CORE_CLASS_IDS["Region"])
    node_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Node", FBMC_CORE_CLASS_IDS["Node"])
    line_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Line", FBMC_CORE_CLASS_IDS["Line"])
    interface_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Interface", FBMC_CORE_CLASS_IDS["Interface"])
    scenario_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Scenario", FBMC_CORE_CLASS_IDS["Scenario"])
    timeslice_class = _fbmc_core_class_enum(ClassEnum, SystemNS, "Timeslice", FBMC_CORE_CLASS_IDS["Timeslice"])

    node_region_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["NodeRegions", "RegionNodes", "Region"],
        FBMC_CORE_COLLECTION_IDS["NodeRegion"],
    )
    system_region_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemRegions", "Regions"],
        FBMC_CORE_COLLECTION_IDS["SystemRegions"],
    )
    system_node_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemNodes", "Nodes"],
        FBMC_CORE_COLLECTION_IDS["SystemNodes"],
    )
    system_line_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemLines", "Lines"],
        FBMC_CORE_COLLECTION_IDS["SystemLines"],
    )
    line_node_from_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["LineNodeFrom", "NodeFrom", "LineNodesFrom"],
        FBMC_CORE_COLLECTION_IDS["LineNodeFrom"],
    )
    line_node_to_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["LineNodeTo", "NodeTo", "LineNodesTo"],
        FBMC_CORE_COLLECTION_IDS["LineNodeTo"],
    )
    system_interface_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemInterfaces", "Interfaces"],
        FBMC_CORE_COLLECTION_IDS["SystemInterfaces"],
    )
    interface_line_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["InterfaceLines", "Lines"],
        FBMC_CORE_COLLECTION_IDS["InterfaceLines"],
    )
    system_timeslice_collection = _fbmc_core_collection_enum(
        CollectionEnum,
        SystemNS,
        ["SystemTimeslices", "SystemTimeslice", "Timeslices"],
        FBMC_CORE_COLLECTION_IDS["SystemTimeslices"],
    )

    enum_units = _fbmc_core_property_enum(db, SystemNS, "System", "Line", "Lines", "Units", FBMC_CORE_LINE_ENUMS["Units"])
    enum_region_units = _fbmc_core_property_enum(db, SystemNS, "System", "Region", "Regions", "Units", FBMC_CORE_REGION_ENUMS["Units"])
    enum_node_units = _fbmc_core_property_enum(db, SystemNS, "System", "Node", "Nodes", "Units", FBMC_CORE_NODE_ENUMS["Units"])
    enum_max_flow = _fbmc_core_property_enum(db, SystemNS, "System", "Line", "Lines", "Max Flow", FBMC_CORE_LINE_ENUMS["Max Flow"])
    enum_min_flow = _fbmc_core_property_enum(db, SystemNS, "System", "Line", "Lines", "Min Flow", FBMC_CORE_LINE_ENUMS["Min Flow"])
    enum_wheeling_charge = _fbmc_core_property_enum(db, SystemNS, "System", "Line", "Lines", "Wheeling Charge", FBMC_CORE_LINE_ENUMS["Wheeling Charge"])
    enum_wheeling_charge_back = _fbmc_core_property_enum(db, SystemNS, "System", "Line", "Lines", "Wheeling Charge Back", FBMC_CORE_LINE_ENUMS["Wheeling Charge Back"])

    enum_interface_units = _fbmc_core_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Units", FBMC_CORE_INTERFACE_ENUMS["Units"])
    enum_interface_max_flow = _fbmc_core_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Max Flow", FBMC_CORE_INTERFACE_ENUMS["Max Flow"])
    enum_interface_limit_penalty = _fbmc_core_property_enum(db, SystemNS, "System", "Interface", "Interfaces", "Limit Penalty", FBMC_CORE_INTERFACE_ENUMS["Limit Penalty"])
    enum_flow_coefficient = _fbmc_core_property_enum(db, SystemNS, "Interface", "Line", "Lines", "Flow Coefficient", FBMC_CORE_INTERFACE_LINE_ENUMS["Flow Coefficient"])
    enum_timeslice_include = _fbmc_core_property_enum(db, SystemNS, "System", "Timeslice", "Timeslices", "Include", FBMC_CORE_TIMESLICE_ENUMS["Include"])

    created_nodes = 0
    created_regions = 0
    created_lines = 0
    created_interfaces = 0
    categorized_hub_lines = 0
    line_property_rows = 0
    hub_object_property_rows = 0
    existing_hvac_hvdc_units_rows = 0
    interface_property_rows = 0
    flow_coefficient_property_rows = 0
    interface_line_memberships = 0
    timeslices_created = 0
    timeslice_property_rows = 0
    warnings: list[str] = []

    try:
        _fbmc_core_ensure_category(db, line_class, FBMC_CORE_LINE_CATEGORY)

        if _fbmc_core_ensure_object(db, region_class, FBMC_CORE_HUB_NAME, add_to_system=True):
            created_regions += 1
        if _fbmc_core_ensure_object(db, node_class, FBMC_CORE_HUB_NAME, add_to_system=True):
            created_nodes += 1

        if _fbmc_core_ensure_membership_bi(db, node_region_collection, FBMC_CORE_HUB_NAME, FBMC_CORE_HUB_NAME) is None:
            warnings.append(f"Could not create Node/Region membership for {FBMC_CORE_HUB_NAME}")

        _fbmc_core_ensure_object(db, scenario_class, FBMC_CORE_SCENARIO_NAME, add_to_system=True)
        fbmc_scenario = SystemNS.String(FBMC_CORE_SCENARIO_NAME)

        for timeslice_config in FBMC_CORE_TIMESLICES.values():
            timeslice_name = timeslice_config["name"]
            if _fbmc_core_ensure_object(db, timeslice_class, timeslice_name, add_to_system=True):
                timeslices_created += 1
            timeslice_mem_id = _fbmc_core_ensure_membership(db, system_timeslice_collection, "System", timeslice_name)
            if timeslice_mem_id is None:
                warnings.append(f"Could not resolve System/Timeslice membership for {timeslice_name}; skipped timeslice Include")
                continue
            _fbmc_core_add_property(
                db,
                enum_timeslice_include,
                timeslice_mem_id,
                -1.0,
                scenario=fbmc_scenario,
                pattern=SystemNS.String(timeslice_config["pattern"]),
            )
            timeslice_property_rows += 1

        hub_region_mem_id = _fbmc_core_ensure_membership(db, system_region_collection, "System", FBMC_CORE_HUB_NAME)
        if hub_region_mem_id is None:
            warnings.append(f"Could not resolve System/Region membership for {FBMC_CORE_HUB_NAME}; skipped hub region Units")
        else:
            _fbmc_core_add_property(db, enum_region_units, hub_region_mem_id, 0.0)
            _fbmc_core_add_property(db, enum_region_units, hub_region_mem_id, 1.0, scenario=fbmc_scenario)
            hub_object_property_rows += 2

        hub_node_mem_id = _fbmc_core_ensure_membership(db, system_node_collection, "System", FBMC_CORE_HUB_NAME)
        if hub_node_mem_id is None:
            warnings.append(f"Could not resolve System/Node membership for {FBMC_CORE_HUB_NAME}; skipped hub node Units")
        else:
            _fbmc_core_add_property(db, enum_node_units, hub_node_mem_id, 0.0)
            _fbmc_core_add_property(db, enum_node_units, hub_node_mem_id, 1.0, scenario=fbmc_scenario)
            hub_object_property_rows += 2

        existing_hvac_lines = _fbmc_core_get_objects_in_category_safe(db, line_class, "HVAC")
        existing_hvdc_lines = _fbmc_core_get_objects_in_category_safe(db, line_class, "HVDC")
        existing_hvac_hvdc_lines = sorted(existing_hvac_lines | existing_hvdc_lines)
        existing_hvac_hvdc_line_mem_ids: dict[str, Any] = {}
        for line_name in existing_hvac_hvdc_lines:
            line_mem_id = _fbmc_core_ensure_membership(db, system_line_collection, "System", line_name)
            if line_mem_id is None:
                warnings.append(f"Could not resolve System/Line membership for existing HVAC/HVDC line {line_name}; skipped FBMC Units")
                continue
            existing_hvac_hvdc_line_mem_ids[line_name] = line_mem_id
            _fbmc_core_add_property(db, enum_units, line_mem_id, 0.0, scenario=fbmc_scenario)
            existing_hvac_hvdc_units_rows += 1

        nodes_in_db = _fbmc_core_get_objects_safe(db, node_class)

        for line_name in hub_lines:
            if _fbmc_core_ensure_object(db, line_class, line_name, add_to_system=True, category=FBMC_CORE_LINE_CATEGORY):
                created_lines += 1
            if _fbmc_core_categorize_object(db, line_class, line_name, FBMC_CORE_LINE_CATEGORY):
                categorized_hub_lines += 1

            line_mem_id = _fbmc_core_ensure_membership(db, system_line_collection, "System", line_name)
            if line_mem_id is None:
                warnings.append(f"Could not resolve System/Line membership for {line_name}; skipped line properties")
                continue

            source_node = line_name.split("-HUB", 1)[0].strip()
            if source_node and source_node in nodes_in_db:
                if _fbmc_core_ensure_membership_bi(db, line_node_from_collection, line_name, source_node) is None:
                    warnings.append(f"Could not create Node From membership for {line_name} -> {source_node}")
            elif source_node:
                warnings.append(f"Source node '{source_node}' for line '{line_name}' was not found; Node From not created")

            if _fbmc_core_ensure_membership_bi(db, line_node_to_collection, line_name, FBMC_CORE_HUB_NAME) is None:
                warnings.append(f"Could not create Node To membership for {line_name} -> {FBMC_CORE_HUB_NAME}")

            _fbmc_core_add_property(db, enum_units, line_mem_id, 0.0)
            _fbmc_core_add_property(db, enum_units, line_mem_id, 1.0, scenario=fbmc_scenario)
            _fbmc_core_add_property(db, enum_max_flow, line_mem_id, 50000.0)
            _fbmc_core_add_property(db, enum_min_flow, line_mem_id, -50000.0)
            _fbmc_core_add_property(db, enum_wheeling_charge, line_mem_id, 0.01)
            _fbmc_core_add_property(db, enum_wheeling_charge_back, line_mem_id, 0.01)
            line_property_rows += 6

        target_year_summaries: list[dict[str, Any]] = []

        for target_year in FBMC_CORE_TARGET_YEARS:
            ptdf_sheet = f"PTDF {target_year}"
            ram_sheet = f"RAM {target_year}"
            interface_category = f"TY{target_year}"
            ptdf_output_dir = FBMC_CORE_DATA_FILES_DIR / f"PTDF_{target_year}"
            ram_output_dir = FBMC_CORE_DATA_FILES_DIR / f"RAM_{target_year}"

            raw_ptdf, raw_interface_rows = _fbmc_core_read_interfaces_and_ptdf(FBMC_CORE_DOMAIN_XLSX, ptdf_sheet)
            raw_interface_names = [cnec_id for _, _, cnec_id in raw_interface_rows]
            interface_name_pairs = [
                (cnec_id, _fbmc_core_interface_name_for_year(cnec_id, target_year))
                for cnec_id in dict.fromkeys(raw_interface_names)
            ]
            tagged_interface_by_raw = dict(interface_name_pairs)
            interface_rows = [
                (row_idx, timeslice_label, tagged_interface_by_raw[cnec_id])
                for row_idx, timeslice_label, cnec_id in raw_interface_rows
            ]
            interface_names = [plexos_interface_name for _, plexos_interface_name in interface_name_pairs]
            domain_schedule, domain_cols, domain_warnings = _fbmc_core_read_domain_assignment(
                FBMC_CORE_DOMAIN_XLSX,
                FBMC_CORE_DOMAIN_ASSIGNMENT_SHEET,
                target_year,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in domain_warnings])
            ram_lookup = _fbmc_core_read_ram_lookup(FBMC_CORE_DOMAIN_XLSX, ram_sheet)
            _fbmc_core_ensure_category(db, interface_class, interface_category)

            files_written, file_warnings, ptdf_file_by_line_and_timeslice, timeslices_by_interface = _fbmc_core_write_ptdf_files(
                raw_ptdf,
                interface_rows,
                mapping_df,
                ptdf_output_dir,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in file_warnings])

            ram_files_written, ram_warnings, ram_file_by_interface = _fbmc_core_write_ram_files(
                domain_schedule,
                domain_cols,
                ram_lookup,
                interface_name_pairs,
                ram_output_dir,
            )
            warnings.extend([f"TY{target_year}: {message}" for message in ram_warnings])

            date_from_tuple, date_to_tuple = FBMC_CORE_INTERFACE_DATE_RANGES[target_year]
            interface_date_from = _fbmc_core_datetime(SystemNS, date_from_tuple)
            interface_date_to = _fbmc_core_datetime(SystemNS, date_to_tuple)
            year_interfaces_created = 0
            year_interface_property_rows = 0
            year_interface_line_memberships = 0
            year_flow_coefficient_property_rows = 0
            year_existing_hvac_hvdc_units_rows = 0
            year_flow_coefficient_lines: set[str] = set()

            for interface_name in interface_names:
                if _fbmc_core_ensure_object(
                    db,
                    interface_class,
                    interface_name,
                    add_to_system=True,
                    category=interface_category,
                ):
                    created_interfaces += 1
                    year_interfaces_created += 1

                interface_mem_id = _fbmc_core_ensure_membership(db, system_interface_collection, "System", interface_name)
                if interface_mem_id is None:
                    warnings.append(f"TY{target_year}: Could not resolve System/Interface membership for {interface_name}; skipped interface properties")
                else:
                    _fbmc_core_add_property(db, enum_interface_units, interface_mem_id, 0.0)
                    _fbmc_core_add_property(
                        db,
                        enum_interface_units,
                        interface_mem_id,
                        1.0,
                        scenario=fbmc_scenario,
                        date_from=interface_date_from,
                        date_to=interface_date_to,
                    )
                    _fbmc_core_add_property(db, enum_interface_limit_penalty, interface_mem_id, -1.0)
                    interface_property_rows += 3
                    year_interface_property_rows += 3

                    ram_file_path = ram_file_by_interface.get(interface_name)
                    if ram_file_path is None:
                        warnings.append(f"TY{target_year}: No RAM file available for interface '{interface_name}'; Max Flow DataFile link skipped")
                    else:
                        _fbmc_core_add_property(
                            db,
                            enum_interface_max_flow,
                            interface_mem_id,
                            1.0,
                            data_file=_fbmc_core_datafile_ref(SystemNS, ram_file_path),
                            date_from=interface_date_from,
                            date_to=interface_date_to,
                        )
                        interface_property_rows += 1
                        year_interface_property_rows += 1

                for line_name in all_mapping_lines:
                    membership_id = _fbmc_core_ensure_membership_bi(db, interface_line_collection, interface_name, line_name)
                    if membership_id is None:
                        warnings.append(f"TY{target_year}: Could not create Interface/Line membership: {interface_name} / {line_name}")
                        continue

                    interface_line_memberships += 1
                    year_interface_line_memberships += 1
                    interface_timeslices = sorted(timeslices_by_interface.get(interface_name, set()))
                    if not interface_timeslices:
                        warnings.append(f"TY{target_year}: No winter1/summer1 PTDF rows available for interface '{interface_name}'; Flow Coefficient DataFile link skipped for line '{line_name}'")
                        continue

                    line_timeslice_files = ptdf_file_by_line_and_timeslice.get(line_name, {})
                    added_flow_coefficient = False
                    for timeslice_label in interface_timeslices:
                        ptdf_file_path = line_timeslice_files.get(timeslice_label)
                        if ptdf_file_path is None:
                            warnings.append(f"TY{target_year}: No {FBMC_CORE_TIMESLICES[timeslice_label]['name']} PTDF file available for line '{line_name}'; Flow Coefficient DataFile link skipped for '{interface_name}'")
                            continue

                        _fbmc_core_add_property(
                            db,
                            enum_flow_coefficient,
                            membership_id,
                            1.0,
                            data_file=_fbmc_core_datafile_ref(SystemNS, ptdf_file_path),
                            pattern=SystemNS.String(FBMC_CORE_TIMESLICES[timeslice_label]["name"]),
                        )
                        flow_coefficient_property_rows += 1
                        year_flow_coefficient_property_rows += 1
                        added_flow_coefficient = True

                    if added_flow_coefficient:
                        year_flow_coefficient_lines.add(line_name)

            year_existing_hvac_hvdc_lines = sorted(
                line_name
                for line_name in year_flow_coefficient_lines
                if line_name in existing_hvac_hvdc_line_mem_ids
            )
            for line_name in year_existing_hvac_hvdc_lines:
                _fbmc_core_add_property(
                    db,
                    enum_units,
                    existing_hvac_hvdc_line_mem_ids[line_name],
                    1.0,
                    scenario=fbmc_scenario,
                    date_from=interface_date_from,
                    date_to=interface_date_to,
                )
                existing_hvac_hvdc_units_rows += 1
                year_existing_hvac_hvdc_units_rows += 1

            target_year_summaries.append({
                "year": target_year,
                "interfaces": len(interface_rows),
                "interfaces_created": year_interfaces_created,
                "interface_property_rows": year_interface_property_rows,
                "interface_line_memberships": year_interface_line_memberships,
                "flow_coefficient_property_rows": year_flow_coefficient_property_rows,
                "existing_hvac_hvdc_units_rows": year_existing_hvac_hvdc_units_rows,
                "existing_hvac_hvdc_ptdf_lines": len(year_existing_hvac_hvdc_lines),
                "ptdf_files": files_written,
                "ptdf_output_dir": ptdf_output_dir,
                "ram_files": ram_files_written,
                "ram_output_dir": ram_output_dir,
            })

        total_ptdf_files_written = sum(item["ptdf_files"] for item in target_year_summaries)
        total_ram_files_written = sum(item["ram_files"] for item in target_year_summaries)

        print("\n[STEP 14.3] FBMC CORE import complete.")
        print(f"  Hub node ensured: {FBMC_CORE_HUB_NAME} (created {created_nodes})")
        print(f"  Hub region ensured: {FBMC_CORE_HUB_NAME} (created {created_regions})")
        print(f"  FBMC scenario ensured: {FBMC_CORE_SCENARIO_NAME}")
        print(f"  HUB lines in mapping: {len(hub_lines)}")
        print(f"  HUB lines created: {created_lines}")
        print(f"  HUB lines categorized as {FBMC_CORE_LINE_CATEGORY}: {categorized_hub_lines}")
        print(f"  Line property rows written: {line_property_rows}")
        print(f"  Hub node/region Units rows written: {hub_object_property_rows}")
        print(f"  Timeslices created: {timeslices_created}")
        print(f"  Timeslice Include rows written: {timeslice_property_rows}")
        print(f"  Existing HVAC/HVDC line FBMC Units rows written: {existing_hvac_hvdc_units_rows}")
        print(f"  Target years processed: {FBMC_CORE_TARGET_YEARS}")
        print(f"  Interfaces created: {created_interfaces}")
        print(f"  Interface property rows written: {interface_property_rows}")
        print(f"  Interface/Line memberships touched: {interface_line_memberships}")
        print(f"  Flow Coefficient DataFile rows written: {flow_coefficient_property_rows}")
        print(f"  PTDF files written: {total_ptdf_files_written}")
        print(f"  RAM files written: {total_ram_files_written}")
        for summary in target_year_summaries:
            print(
                f"  TY{summary['year']}: interfaces={summary['interfaces']}, "
                f"created={summary['interfaces_created']}, PTDF files={summary['ptdf_files']}, "
                f"RAM files={summary['ram_files']}, "
                f"existing HVAC/HVDC Units rows={summary['existing_hvac_hvdc_units_rows']}"
            )
            print(f"    PTDF output folder: {summary['ptdf_output_dir']}")
            print(f"    RAM output folder: {summary['ram_output_dir']}")

        if warnings:
            print("\n[WARNINGS]")
            for message in warnings[:50]:
                print(f"  - {message}")
            if len(warnings) > 50:
                print(f"  - ... {len(warnings) - 50} more warnings")

    finally:
        try:
            db.Close()
        except Exception:
            pass


if RUN_FBMC_CORE_14_3:
    step_fbmc_core_14_3()
else:
    print("[SKIP] Step 14.3 FBMC - CORE skipped because RUN_FBMC_CORE_14_3 is False.")


#### 15. EVA Candidates and Retirements

In [ ]:
# ============================================================================
# STEP — EVA Candidates (PATCH)
#  CHANGES in THIS VERSION (in addition to your prior EVA_Weights change):
#   A) Global "EVA Weight" -> Sample Weight (enum 6) bands 1..36
#      are written with Scenario = "EVA_Weights" (auto-created if missing)
#   B) NEW: For ALL created Generators and Batteries:
#      - Write "Max Units Built" as a CUMULATIVE sum of "Max Units Built in Year"
#        across target years, written year-specific (DateFrom=YYYY-01-01):
#          * Generator Max Units Built enum = 260
#          * Battery   Max Units Built enum = 113
#        Rule: value(2028)=pot(2028)
#              value(2030)=pot(2028)+pot(2030)
#              value(2033)=pot(2028)+pot(2030)+pot(2033)
#              value(2035)=... etc
#      - Uses the same "Potential" sheet values that were previously written to:
#          * Generators enum 264 (Max Units Built in Year)
#          * Batteries  enum 121 (Max Units Potential / per-year)
#        If a year's per-year value is missing, it's treated as 0 for cumulative.
#
#   C) FIX: EVA generators with Gas / Hydrogen in their name/technology now get:
#      - Generator↔Fuel membership to Gas or Hydrogen
#      - Generator↔Start Fuel membership to Gas or Hydrogen
#      - Offtake at Start written on that actual Start Fuel membership
# ============================================================================
from __future__ import annotations
import os
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple, Set
import pandas as pd

# -----------------------------
# USER CONFIG / PATHS
# -----------------------------

COUNTRY_XLSX = ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Country Specific.xlsx"
DEFAULT_XLSX = ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Default.xlsx"

SCENARIO_NAME: Optional[str] = None
UNITS_ZERO_SCENARIO_NAME = "EVA_Candidates"

EVA_WEIGHTS_SCENARIO_NAME = "EVA_Weights"

EVA_GENERATOR_CATEGORY_NAME = "EVA"

# -----------------------------
# Enums / IDs
# -----------------------------
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1
BATTERY_COLLECTION_ID = 81

# NEW: fallback for System->Fuel collection
FALLBACK_SYSTEM_FUELS_COLLECTION_ID = 40

ENUM_HEAT_RATE = 59
ENUM_MIN_STABLE_FACTOR = 56
ENUM_MIN_UP = 86
ENUM_MIN_DOWN = 88

ENUM_FOR = 230
ENUM_MAINT_RATE = 234
ENUM_MTTR = 244

ENUM_VOM = 64
ENUM_OFFTAKE_AT_START = 1
OFFTAKE_START_COLLECTION_ID = 19  # fallback for Generator<->Start Fuel membership collection

ENUM_EXPANSION_OPTIM = 24
ENUM_UNITS_GEN = 52
ENUM_MAX_CAPACITY = 53

ENUM_GEN_FOM = 67
ENUM_GEN_BUILDCOST = 249
ENUM_GEN_WACC = 258
ENUM_GEN_ECON_LIFE = 260
ENUM_GEN_MAX_UNITS = 265  # "Max Units Built in Year" (per-year)

# NEW: cumulative "Max Units Built"
ENUM_GEN_MAX_UNITS_CUM = 261

# Build Non-anticipativity enums
ENUM_BUILD_NONANTICIP_GEN = 287
ENUM_BUILD_NONANTICIP_BAT = 139
BUILD_NONANTICIP_VALUE = -1

ENUM_BAT_UNITS = 25
ENUM_BAT_CAPACITY = 26
ENUM_BAT_MAX_POWER = 28
ENUM_BAT_INITIAL_SOC = 32
ENUM_BAT_CHARGE_EFF = 33
ENUM_BAT_DISCHARGE_EFF = 34
ENUM_BAT_EXPANSION_OPT = 15
ENUM_BAT_VOM = 35
ENUM_BAT_FOM = 38
ENUM_BAT_BUILDCOST = 119
ENUM_BAT_WACC = 120
ENUM_BAT_ECON_LIFE = 121
ENUM_BAT_MAX_UNITS_POT = 123  # per-year potential (used as "Max Units Built in Year" for batteries)

# NEW: cumulative "Max Units Built" for batteries
ENUM_BAT_MAX_UNITS_CUM = 115

ENUM_OFFER_QTY = 199
ENUM_OFFER_PRICE = 200
ENUM_CONSTRAINT_OPERATING_HOURS_COEFF = 6
ENUM_CONSTRAINT_SENSE = 1
ENUM_CONSTRAINT_RHS_DAY = 16
SENSE_LESS_EQUAL = -1.0

GENERATOR_NODES_STAR_COLLECTION_ID = 13
BATTERY_NODES_STAR_COLLECTION_ID = 86

# EVA Weight global object settings
EVA_WEIGHT_OBJECT_NAME = "EVA Weight"
GLOBAL_CLASS_ID_84 = 84
GLOBAL_COLLECTION_ID_788 = 788
ENUM_SAMPLE_WEIGHT = 6  # bands 1..36

# -----------------------------
# Thermal defaults
# -----------------------------
THERMAL_DEFAULTS = {
    "Gas CCGT": {
        "HeatRate": 6.0, "MinStableFactor": 40.0, "MinUp": 2, "MinDown": 2,
        "FOR_b1": 5.0, "Maint_b2": 15.0, "MTTR_b1_b2": 24.0, "VOM": 2.324, "OfftakeStart": 7.6,
    },
    "Hydrogen CCGT": {
        "HeatRate": 6.0, "MinStableFactor": 40.0, "MinUp": 2, "MinDown": 2,
        "FOR_b1": 5.0, "Maint_b2": 15.0, "MTTR_b1_b2": 24.0, "VOM": 2.324, "OfftakeStart": 7.6,
    },
    "Gas OCGT": {
        "HeatRate": 8.57, "MinStableFactor": 40.0, "MinUp": 1, "MinDown": 1,
        "FOR_b1": 5.0, "Maint_b2": 15.0, "MTTR_b1_b2": 24.0, "VOM": 2.5, "OfftakeStart": 0.2,
    },
    "Hydrogen OCGT": {
        "HeatRate": 8.57, "MinStableFactor": 40.0, "MinUp": 1, "MinDown": 1,
        "FOR_b1": 5.0, "Maint_b2": 15.0, "MTTR_b1_b2": 24.0, "VOM": 2.5, "OfftakeStart": 0.2,
    },
}

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# Helpers
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum, PeriodEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime

def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []

def _ensure_category(db, class_enum_value, category: str):
    if not category:
        return
    try:
        db.AddCategory(class_enum_value, str(category))
    except Exception:
        pass

def _ensure_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name in existing:
        return
    db.AddObject(str(name), class_enum_value, bool(add_to_system), str(category or ""), str(description or ""))

def _add_object_raw(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    db.AddObject(name, class_enum_value, bool(add_to_system), category, description)

def _add_object_safe_category(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    if not category:
        _add_object_raw(db, class_enum_value, name, add_to_system=add_to_system, category="", description=description)
        return
    try:
        _add_object_raw(db, class_enum_value, name, add_to_system=add_to_system, category=category, description=description)
    except Exception as e:
        msg = str(e)
        if "Unable to locate the category" in msg or "category titled" in msg:
            _add_object_raw(db, class_enum_value, name, add_to_system=add_to_system, category="", description=description)
        else:
            raise

def _is_int_like(x: float) -> bool:
    return abs(x - round(x)) < 1e-12

def _fmt_num(x) -> float:
    if x is None:
        return 0.0
    fx = float(x)
    if _is_int_like(fx):
        return float(int(round(fx)))
    return float(round(fx, 2))

def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None, PeriodTypeId=None):
    v = _fmt_num(value)
    if PeriodTypeId is None:
        db.AddProperty(int(mem_id), int(enum_id), int(band), float(v),
                       DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action)
    else:
        db.AddProperty(int(mem_id), int(enum_id), int(band), float(v),
                       DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action, PeriodTypeId)

def _resolve_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)
    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][2]
    if fallback_id is not None:
        return int(fallback_id)
    raise RuntimeError("Could not resolve CollectionEnum")

def _resolve_collection_enum_optional(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    try:
        return _resolve_collection_enum(CollectionEnum, required_tokens_lc, preferred_names, fallback_id=fallback_id)
    except Exception:
        return None

def _resolve_class_enum_by_tokens(ClassEnum, required_tokens_lc: List[str], preferred_names: List[str]) -> Optional[int]:
    for n in preferred_names:
        if hasattr(ClassEnum, n):
            return getattr(ClassEnum, n)
    candidates = []
    for attr in dir(ClassEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(ClassEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            candidates.append((len(name_lc), attr, val))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][2]
    return None

def _sanitize_name(s: str) -> str:
    s = str(s).strip().replace("/", "_").replace("\\", "_").replace(":", "_")
    for ch in "()[]{}":
        s = s.replace(ch, "")
    s = s.replace(" ", "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s

def _sheet_to_df(xlsx: Path, sheet: str) -> pd.DataFrame:
    if not xlsx.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(xlsx, sheet_name=sheet, header=0)
    except Exception:
        return pd.DataFrame()

def _find_col(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        for c in df.columns:
            if str(c).strip().lower() == str(n).strip().lower():
                return c
    return None

def _first_float(x) -> Optional[float]:
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x).replace(",", "."))
        except Exception:
            return None

def _val_from(country_df: pd.DataFrame, default_df: pd.DataFrame,
              node: str, pem: str, ref: str, year: int, year_cols: List[str]) -> Optional[float]:
    def pick(df: pd.DataFrame, node_req: bool, ref_req: bool) -> Optional[float]:
        if df is None or df.empty:
            return None
        node_col = _find_col(df, ["Node","MARKET_NODE","NODE"])
        pem_col  = _find_col(df, ["PEMMDB Technology","PEMMDB_Technology","PEMMDBTechnology"])
        ref_col  = _find_col(df, ["Reference Technology","Reference","Reference_Technology"])
        if pem_col is None:
            return None

        mask = pd.Series([True]*len(df), index=df.index)

        if node_req and node_col is not None and str(node).strip():
            mask = mask & (df[node_col].astype(str).str.strip() == str(node).strip())

        mask = mask & (df[pem_col].astype(str).str.strip() == str(pem).strip())

        if ref_req and ref_col is not None and str(ref).strip():
            mask = mask & (df[ref_col].astype(str).str.strip() == str(ref).strip())

        sub = df[mask]
        if sub.empty:
            return None

        ycol = str(year)
        if ycol not in df.columns:
            for yc in year_cols:
                if str(yc) in df.columns:
                    ycol = str(yc)
                    break
        if ycol not in df.columns:
            return None

        return _first_float(sub.iloc[0][ycol])

    v = pick(country_df, node_req=True, ref_req=True)
    if v is not None: return v
    v = pick(country_df, node_req=True, ref_req=False)
    if v is not None: return v

    v = pick(default_df, node_req=True, ref_req=True)
    if v is not None: return v
    v = pick(default_df, node_req=True, ref_req=False)
    if v is not None: return v

    v = pick(country_df, node_req=False, ref_req=False)
    if v is not None: return v
    v = pick(default_df, node_req=False, ref_req=False)
    return v

def _ensure_membership_robust(db, CollectionEnum, SystemNS, collection_raw, parent: str, child: str) -> Tuple[Optional[int], Optional[str]]:
    col_candidates = []
    try:
        if isinstance(collection_raw, int):
            try:
                col_candidates.append(CollectionEnum(collection_raw))
            except Exception:
                col_candidates.append(collection_raw)
        else:
            col_candidates.append(collection_raw)
    except Exception:
        col_candidates.append(collection_raw)

    def _try(col_enum, a, b) -> Optional[int]:
        try:
            return int(db.GetMembershipID(col_enum, a, b))
        except Exception:
            try:
                db.AddMembership(col_enum, a, b)
            except Exception:
                return None
        try:
            return int(db.GetMembershipID(col_enum, a, b))
        except Exception:
            return None

    last_err = None
    for col_enum in col_candidates:
        for (a, b) in [(parent, child), (child, parent)]:
            for wrap in [0, 1]:
                aa = SystemNS.String(a) if wrap else a
                bb = SystemNS.String(b) if wrap else b
                mid = _try(col_enum, aa, bb)
                if mid is not None:
                    return mid, None
                last_err = f"failed col={collection_raw} ({col_enum}), a={parent}, b={child}"
    return None, last_err

# NEW: infer fuel object name from generator naming/technology
def _infer_fuel_name(*parts: str) -> Optional[str]:
    txt = " ".join(str(p or "") for p in parts).strip().lower()
    if "hydrogen" in txt:
        return "Hydrogen"
    if "gas" in txt:
        return "Gas"
    return None

# -----------------------------
# MAIN STEP
# -----------------------------
def step_eva_candidates():
    # Bidding zones
    bz = pd.read_excel(BIDDING_ZONE_XLSX, sheet_name=0, header=None)
    allowed_bzs = {str(v).strip() for v in bz.values.ravel() if not pd.isna(v) and str(v).strip()}
    if not allowed_bzs:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    sheets = [
        "FOM","CAPEX","WACC","Hurdle Premium","Economic Lifetime","Potential",
        "VOM","DSR Activation Price","Activation Limit"
    ]
    country = {s: _sheet_to_df(COUNTRY_XLSX, s) for s in sheets}
    default = {s: _sheet_to_df(DEFAULT_XLSX, s) for s in sheets}

    # Discover candidates from FOM
    candidates: List[Dict[str,str]] = []
    for origin, df in (("country", country.get("FOM", pd.DataFrame())), ("default", default.get("FOM", pd.DataFrame()))):
        if df is None or df.empty:
            continue
        node_col = _find_col(df, ["Node","MARKET_NODE","NODE"])
        pem_col  = _find_col(df, ["PEMMDB Technology","PEMMDB_Technology","PEMMDBTechnology"])
        ref_col  = _find_col(df, ["Reference Technology","Reference","Reference_Technology"])
        if pem_col is None:
            continue

        for _, r in df.iterrows():
            node = str(r[node_col]).strip() if node_col else ""
            pem  = str(r[pem_col]).strip()
            ref  = str(r[ref_col]).strip() if ref_col else ""

            if origin == "default" and ref.strip().lower() == "retirement":
                continue

            if pem:
                candidates.append({"Node": node, "PEM": pem, "Ref": ref, "Origin": origin})

    # Year columns
    year_cols = []
    for df in (country.get("FOM", pd.DataFrame()), default.get("FOM", pd.DataFrame())):
        if df is not None and not df.empty:
            year_cols = [str(c) for c in df.columns if str(c).strip().isdigit()]
            if year_cols:
                break
    if not year_cols:
        year_cols = ["2028","2030","2033","2035"]
    years = [int(y) for y in year_cols if str(y).isdigit()]
    years_sorted = sorted(years)

    # Connect DB
    db, ClassEnum, CollectionEnum, PeriodEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)

    ENUM_HEAT_RATE = _resolve_semantic_enum("System", "Generator", "Generators", "Heat Rate", 59)
    ENUM_MIN_STABLE_FACTOR = _resolve_semantic_enum("System", "Generator", "Generators", "Min Stable Factor", 56)
    ENUM_MIN_UP = _resolve_semantic_enum("System", "Generator", "Generators", "Min Up Time", 86)
    ENUM_MIN_DOWN = _resolve_semantic_enum("System", "Generator", "Generators", "Min Down Time", 88)
    ENUM_FOR = _resolve_semantic_enum("System", "Generator", "Generators", "Forced Outage Rate", 230)
    ENUM_MTTR = _resolve_semantic_enum("System", "Generator", "Generators", "Mean Time to Repair", 244)
    ENUM_VOM = _resolve_semantic_enum("System", "Generator", "Generators", "VO&M Charge", 64)
    ENUM_EXPANSION_OPTIM = _resolve_semantic_enum("System", "Generator", "Generators", "Expansion Optimality", 24)
    ENUM_GEN_FOM = _resolve_semantic_enum("System", "Generator", "Generators", "FO&M Charge", 67)
    ENUM_GEN_WACC = _resolve_semantic_enum("System", "Generator", "Generators", "WACC", 258)
    ENUM_GEN_ECON_LIFE = _resolve_semantic_enum("System", "Generator", "Generators", "Economic Life", 260)
    ENUM_GEN_MAX_UNITS = _resolve_semantic_enum("System", "Generator", "Generators", "Max Units Built in Year", 265)
    ENUM_GEN_MAX_UNITS_CUM = _resolve_semantic_enum("System", "Generator", "Generators", "Max Units Built", 261)
    ENUM_BUILD_NONANTICIP_GEN = _resolve_semantic_enum("System", "Generator", "Generators", "Build Non-anticipativity", 287)
    ENUM_OFFER_QTY = _resolve_semantic_enum("System", "Generator", "Generators", "Offer Quantity", 199)
    ENUM_OFFER_PRICE = _resolve_semantic_enum("System", "Generator", "Generators", "Offer Price", 200)

    try:
        # Ensure scenarios exist
        _ensure_object(db, ClassEnum.Scenario, UNITS_ZERO_SCENARIO_NAME, add_to_system=True)
        units_scenario_str = SystemNS.String(UNITS_ZERO_SCENARIO_NAME)

        _ensure_object(db, ClassEnum.Scenario, EVA_WEIGHTS_SCENARIO_NAME, add_to_system=True)
        eva_weights_scenario_str = SystemNS.String(EVA_WEIGHTS_SCENARIO_NAME)

        scenario_str = None
        if SCENARIO_NAME:
            _ensure_object(db, ClassEnum.Scenario, SCENARIO_NAME, add_to_system=True)
            scenario_str = SystemNS.String(SCENARIO_NAME)

        # Ensure Generator category EVA exists
        _ensure_category(db, ClassEnum.Generator, EVA_GENERATOR_CATEGORY_NAME)

        # Collections
        gen_mem_collection = _resolve_collection_enum(
            CollectionEnum, ["system","generator"],
            ["SystemGenerators","Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID
        )
        bat_mem_collection = _resolve_collection_enum(
            CollectionEnum, ["system","battery"],
            ["SystemBatteries","Batteries","SystemBattery"],
            fallback_id=BATTERY_COLLECTION_ID
        )

        # NEW: Fuel / Start Fuel related collections
        fuel_mem_collection = _resolve_collection_enum_optional(
            CollectionEnum, ["system","fuel"],
            ["SystemFuels","Fuels"],
            fallback_id=FALLBACK_SYSTEM_FUELS_COLLECTION_ID
        )

        gen_fuel_collection = _resolve_collection_enum_optional(
            CollectionEnum, ["generator","fuel"],
            ["GeneratorFuels","FuelGenerators"]
        )

        gen_startfuel_collection = _resolve_collection_enum_optional(
            CollectionEnum, ["generator","start","fuel"],
            ["GeneratorStartFuels","StartFuelGenerators"],
            fallback_id=OFFTAKE_START_COLLECTION_ID
        )

        # NEW: StartFuel class (best effort)
        startfuel_class = _resolve_class_enum_by_tokens(
            ClassEnum,
            ["start","fuel"],
            ["StartFuel", "StartFuels"]
        )

        # Country origin uses normal Node<->X membership if available
        gen_node_collection = None
        for nm in ["NodeGenerators","GeneratorNodes","PowerStationNodes"]:
            if hasattr(CollectionEnum, nm):
                gen_node_collection = getattr(CollectionEnum, nm)
                break

        bat_node_collection = None
        for nm in ["NodeBatteries","BatteryNodes","StorageNodes","NodeStorages","NodeStorage"]:
            if hasattr(CollectionEnum, nm):
                bat_node_collection = getattr(CollectionEnum, nm)
                break

        # Constraint class + collections
        constraint_class = _resolve_class_enum_by_tokens(ClassEnum, ["constraint"], ["Constraint","Constraints"])
        system_constraint_collection = None
        constraint_generator_collection = None
        if constraint_class is not None:
            try:
                system_constraint_collection = _resolve_collection_enum(
                    CollectionEnum, ["system","constraint"], ["SystemConstraints","Constraints"], fallback_id=759
                )
            except Exception:
                system_constraint_collection = int(759)
            try:
                constraint_generator_collection = _resolve_collection_enum(
                    CollectionEnum, ["constraint","generator"], ["ConstraintGenerators","GeneratorConstraints"], fallback_id=32
                )
            except Exception:
                constraint_generator_collection = int(32)

        # Global class id 84 (for EVA Weight)
        global_class = _resolve_class_enum_by_tokens(ClassEnum, ["global"], ["Global","Globals"])
        if global_class is None:
            global_class = int(GLOBAL_CLASS_ID_84)

        # Caches
        nodes_in_db = set(_get_objects_safe(db, ClassEnum.Node))
        gens_in_db = set(_get_objects_safe(db, ClassEnum.Generator))
        try:
            bats_in_db = set(_get_objects_safe(db, ClassEnum.Battery))
        except Exception:
            bats_in_db = set()

        # NEW: Fuel caches
        try:
            fuels_in_db = set(_get_objects_safe(db, ClassEnum.Fuel))
        except Exception:
            fuels_in_db = set()

        try:
            startfuels_in_db = set(_get_objects_safe(db, startfuel_class)) if startfuel_class is not None else set()
        except Exception:
            startfuels_in_db = set()

        def get_econ(sheet: str, node: str, pem: str, ref: str, year: int) -> Optional[float]:
            return _val_from(
                country.get(sheet, pd.DataFrame()),
                default.get(sheet, pd.DataFrame()),
                node=node, pem=pem, ref=ref, year=year, year_cols=year_cols
            )

        def _default_hurdle_expansion_only(pem: str, year: int) -> Optional[float]:
            df = default.get("Hurdle Premium", pd.DataFrame())
            if df is None or df.empty:
                return None
            pem_col = _find_col(df, ["PEMMDB Technology","PEMMDB_Technology","PEMMDBTechnology"])
            ref_col = _find_col(df, ["Reference Technology","Reference","Reference_Technology"])
            if pem_col is None or ref_col is None:
                return None
            sub = df[
                (df[pem_col].astype(str).str.strip() == str(pem).strip()) &
                (df[ref_col].astype(str).str.strip().str.lower() == "expansion")
            ]
            if sub.empty:
                return None
            ycol = str(year)
            if ycol not in df.columns:
                for yc in year_cols:
                    if str(yc) in df.columns:
                        ycol = str(yc)
                        break
            if ycol not in df.columns:
                return None
            return _first_float(sub.iloc[0][ycol])

        def get_hurdle_premium(node: str, pem: str, ref: str, year: int, origin: str) -> Optional[float]:
            if origin == "default":
                return _val_from(
                    pd.DataFrame(),
                    default.get("Hurdle Premium", pd.DataFrame()),
                    node=node, pem=pem, ref=ref, year=year, year_cols=year_cols
                )

            h_country = _val_from(
                country.get("Hurdle Premium", pd.DataFrame()),
                pd.DataFrame(),
                node=node, pem=pem, ref=ref, year=year, year_cols=year_cols
            )
            if h_country is not None:
                return h_country
            return _default_hurdle_expansion_only(pem=pem, year=year)

        def get_wacc_with_hurdle(node: str, pem: str, ref: str, year: int, origin: str) -> Optional[float]:
            w = get_econ("WACC", node, pem, ref, year)
            h = get_hurdle_premium(node, pem, ref, year, origin)
            if w is None and h is None:
                return None
            w = float(w or 0.0)
            h = float(h or 0.0)
            return (w + h) * 100.0

        def apply_generator_economics(mem_id: int, node: str, pem: str, ref: str, origin: str):
            _add_property_row(db, ENUM_EXPANSION_OPTIM, mem_id, 1, 0.0)
            _add_property_row(db, ENUM_BUILD_NONANTICIP_GEN, mem_id, 1, BUILD_NONANTICIP_VALUE)

            for y in years_sorted:
                dt = NetDateTime(int(y), 1, 1, 0, 0, 0)

                v_fom = get_econ("FOM", node, pem, ref, y)
                if v_fom is not None:
                    _add_property_row(db, ENUM_GEN_FOM, mem_id, 1, v_fom, DateFrom=dt)

                v_capex = get_econ("CAPEX", node, pem, ref, y)
                if v_capex is not None:
                    _add_property_row(db, ENUM_GEN_BUILDCOST, mem_id, 1, v_capex, DateFrom=dt)

                v_wacc = get_wacc_with_hurdle(node, pem, ref, y, origin)
                if v_wacc is not None:
                    _add_property_row(db, ENUM_GEN_WACC, mem_id, 1, v_wacc, DateFrom=dt)

                v_life = get_econ("Economic Lifetime", node, pem, ref, y)
                if v_life is not None:
                    _add_property_row(db, ENUM_GEN_ECON_LIFE, mem_id, 1, v_life, DateFrom=dt)

                v_pot = get_econ("Potential", node, pem, ref, y)
                if v_pot is not None:
                    _add_property_row(db, ENUM_GEN_MAX_UNITS, mem_id, 1, v_pot, DateFrom=dt)

        # NEW: write cumulative Max Units Built (260/113) from per-year Potential values
        def apply_cumulative_max_units_built(mem_id: int, node: str, pem: str, ref: str, origin: str,
                                            per_year_enum: int, cum_enum: int):
            running = 0.0
            for y in years_sorted:
                dt = NetDateTime(int(y), 1, 1, 0, 0, 0)
                v = get_econ("Potential", node, pem, ref, y)
                fv = _first_float(v)
                if fv is None:
                    fv = 0.0
                running += float(fv)
                _add_property_row(db, cum_enum, mem_id, 1, running, DateFrom=dt)

        # NEW: ensure actual Fuel / Start Fuel memberships for Gas or Hydrogen
        def ensure_generator_fuel_memberships(obj_name: str, pem: str, ref: str) -> Optional[str]:
            fuel_name = _infer_fuel_name(obj_name, pem, ref)
            if not fuel_name:
                return None

            # Ensure Fuel object exists
            try:
                if fuel_name not in fuels_in_db:
                    _ensure_object(db, ClassEnum.Fuel, fuel_name, add_to_system=True)
                    fuels_in_db.add(fuel_name)

                    if fuel_mem_collection is not None:
                        _ensure_membership_robust(db, CollectionEnum, SystemNS, fuel_mem_collection, "System", fuel_name)
            except Exception:
                pass

            # Ensure Start Fuel object exists
            try:
                if startfuel_class is not None and fuel_name not in startfuels_in_db:
                    _ensure_object(db, startfuel_class, fuel_name, add_to_system=True)
                    startfuels_in_db.add(fuel_name)
            except Exception:
                pass

            # Create Generator<->Fuel membership
            if gen_fuel_collection is not None:
                _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_fuel_collection, obj_name, fuel_name)

            # Create Generator<->Start Fuel membership
            if gen_startfuel_collection is not None:
                _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_startfuel_collection, obj_name, fuel_name)

            return fuel_name

        # --- Create Global EVA Weight + Sample Weight bands ---
        _ensure_object(db, global_class, EVA_WEIGHT_OBJECT_NAME, add_to_system=True, category="", description="")
        eva_weight_mem, err = _ensure_membership_robust(db, CollectionEnum, SystemNS, GLOBAL_COLLECTION_ID_788, "System", EVA_WEIGHT_OBJECT_NAME)
        if eva_weight_mem is None:
            print(f"[WARN] Could not ensure System->Global membership for '{EVA_WEIGHT_OBJECT_NAME}' in collection 788: {err}")
        else:
            special = {23: 0.33, 27: 0.33, 35: 0.34}
            for band in range(1, 37):
                val = special.get(band, 0.0)
                _add_property_row(db, ENUM_SAMPLE_WEIGHT, eva_weight_mem, band, val, Scenario=eva_weights_scenario_str)

        created_g = 0
        created_b = 0
        constraints_created = 0

        uniq = {(c["Node"], c["PEM"], c["Ref"], c["Origin"]) for c in candidates}
        print(f"[STEP] Unique candidate rows (after default Retirement skip): {len(uniq)}")

        for node, pem, ref, origin in sorted(uniq):
            node = str(node).strip()
            pem = str(pem).strip()
            ref = str(ref).strip()

            if origin == "country":
                if (not node) or (node not in allowed_bzs):
                    continue

            is_battery = (ref == "Battery Storage_6")
            is_dsr = ("DSR" in pem) or ("DSR" in ref)

            if is_battery:
                obj_name = _sanitize_name(f"{node}_{ref}")
            elif is_dsr:
                obj_name = _sanitize_name(f"{node}_{ref}")
            else:
                obj_name = _sanitize_name(f"{node}_{pem}")

            # ---------------- Batteries ----------------
            if is_battery:
                if obj_name not in bats_in_db:
                    _add_object_safe_category(db, ClassEnum.Battery, obj_name, add_to_system=True, category="", description="")
                    bats_in_db.add(obj_name)
                    created_b += 1

                mem_id, _ = _ensure_membership_robust(db, CollectionEnum, SystemNS, bat_mem_collection, "System", obj_name)
                if mem_id is None:
                    continue

                _add_property_row(db, ENUM_BAT_UNITS, mem_id, 1, 0.0, Scenario=units_scenario_str)
                _add_property_row(db, ENUM_BUILD_NONANTICIP_BAT, mem_id, 1, BUILD_NONANTICIP_VALUE)

                _add_property_row(db, ENUM_BAT_CAPACITY, mem_id, 1, 6.0)
                _add_property_row(db, ENUM_BAT_MAX_POWER, mem_id, 1, 1.0)
                _add_property_row(db, ENUM_BAT_INITIAL_SOC, mem_id, 1, 50.0)
                _add_property_row(db, ENUM_BAT_CHARGE_EFF, mem_id, 1, 98.0)
                _add_property_row(db, ENUM_BAT_DISCHARGE_EFF, mem_id, 1, 98.0)
                _add_property_row(db, ENUM_BAT_EXPANSION_OPT, mem_id, 1, 0.0)

                if origin == "country":
                    if bat_node_collection is not None and node in nodes_in_db:
                        _ensure_membership_robust(db, CollectionEnum, SystemNS, bat_node_collection, node, obj_name)
                else:
                    for bzname in sorted(allowed_bzs):
                        _ensure_membership_robust(db, CollectionEnum, SystemNS, BATTERY_NODES_STAR_COLLECTION_ID, bzname, obj_name)

                for y in years_sorted:
                    dt = NetDateTime(int(y), 1, 1, 0, 0, 0)

                    v_vom = get_econ("VOM", node, pem, ref, y)
                    if v_vom is not None:
                        _add_property_row(db, ENUM_BAT_VOM, mem_id, 1, v_vom, DateFrom=dt)

                    v_fom = get_econ("FOM", node, pem, ref, y)
                    if v_fom is not None:
                        _add_property_row(db, ENUM_BAT_FOM, mem_id, 1, v_fom, DateFrom=dt)

                    v_capex = get_econ("CAPEX", node, pem, ref, y)
                    if v_capex is not None:
                        _add_property_row(db, ENUM_BAT_BUILDCOST, mem_id, 1, v_capex, DateFrom=dt)

                    v_wacc = get_wacc_with_hurdle(node, pem, ref, y, origin)
                    if v_wacc is not None:
                        _add_property_row(db, ENUM_BAT_WACC, mem_id, 1, v_wacc, DateFrom=dt)

                    v_life = get_econ("Economic Lifetime", node, pem, ref, y)
                    if v_life is not None:
                        _add_property_row(db, ENUM_BAT_ECON_LIFE, mem_id, 1, v_life, DateFrom=dt)

                    v_pot = get_econ("Potential", node, pem, ref, y)
                    if v_pot is not None:
                        _add_property_row(db, ENUM_BAT_MAX_UNITS_POT, mem_id, 1, v_pot, DateFrom=dt)

                # NEW: cumulative Max Units Built (enum 113) for batteries
                apply_cumulative_max_units_built(
                    mem_id, node, pem, ref, origin,
                    per_year_enum=ENUM_BAT_MAX_UNITS_POT,
                    cum_enum=ENUM_BAT_MAX_UNITS_CUM
                )

                continue

            # ---------------- Generators ----------------
            if obj_name not in gens_in_db:
                _add_object_safe_category(
                    db, ClassEnum.Generator, obj_name,
                    add_to_system=True, category=EVA_GENERATOR_CATEGORY_NAME, description=""
                )
                gens_in_db.add(obj_name)
                created_g += 1

            mem_id, _ = _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_mem_collection, "System", obj_name)
            if mem_id is None:
                continue

            try:
                _add_property_row(db, ENUM_UNITS_GEN, mem_id, 1, 0.0, Scenario=units_scenario_str)
            except BaseException:
                # Retry with explicit System->Generator membership and semantic Units enum
                mem_id_fallback, _ = _ensure_membership_robust(db, CollectionEnum, SystemNS, FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID, "System", obj_name)
                if mem_id_fallback is not None:
                    mem_id = mem_id_fallback
                units_enum_retry = _resolve_semantic_enum("System", "Generator", "Generators", "Units", ENUM_UNITS_GEN)
                _add_property_row(db, units_enum_retry, mem_id, 1, 0.0, Scenario=units_scenario_str)

            apply_generator_economics(mem_id, node, pem, ref, origin)
            try:
                _add_property_row(db, ENUM_MAX_CAPACITY, mem_id, 1, 1.0)
            except BaseException:
                maxcap_enum_retry = _resolve_semantic_enum("System", "Generator", "Generators", "Max Capacity", ENUM_MAX_CAPACITY)
                _add_property_row(db, maxcap_enum_retry, mem_id, 1, 1.0)

            # NEW: cumulative Max Units Built (enum 260) for generators
            apply_cumulative_max_units_built(
                mem_id, node, pem, ref, origin,
                per_year_enum=ENUM_GEN_MAX_UNITS,
                cum_enum=ENUM_GEN_MAX_UNITS_CUM
            )

            if is_dsr:
                _add_property_row(db, ENUM_OFFER_QTY, mem_id, 1, 1.0)

                for y in years_sorted:
                    dt = NetDateTime(int(y), 1, 1, 0, 0, 0)
                    price = get_econ("DSR Activation Price", node, pem, ref, y)
                    if price is not None:
                        _add_property_row(db, ENUM_OFFER_PRICE, mem_id, 1, price, DateFrom=dt)

                limit_val = None
                for y in years_sorted:
                    v = get_econ("Activation Limit", node, pem, ref, y)
                    fv = _first_float(v)
                    if fv is not None and abs(fv - 24.0) > 1e-9:
                        limit_val = fv
                        break

                if limit_val is not None and constraint_class is not None:
                    cname = f"{obj_name}_Daily_Activation"
                    existing_constraints = set(_get_objects_safe(db, constraint_class))
                    if cname not in existing_constraints:
                        _add_object_safe_category(db, constraint_class, cname, add_to_system=True, category="", description="")
                        constraints_created += 1

                    if system_constraint_collection is not None:
                        c_sys_mem, _ = _ensure_membership_robust(db, CollectionEnum, SystemNS, system_constraint_collection, "System", cname)
                        if c_sys_mem is not None:
                            _add_property_row(db, ENUM_CONSTRAINT_RHS_DAY, c_sys_mem, 1, limit_val, PeriodTypeId=PeriodEnum.Day)
                            _add_property_row(db, ENUM_CONSTRAINT_SENSE, c_sys_mem, 1, SENSE_LESS_EQUAL)

                            if constraint_generator_collection is not None:
                                ok, _ = _ensure_membership_robust(db, CollectionEnum, SystemNS, constraint_generator_collection, cname, obj_name)
                                if ok is not None:
                                    try:
                                        cg_mem = int(db.GetMembershipID(constraint_generator_collection, cname, obj_name))
                                    except Exception:
                                        cg_mem = int(db.GetMembershipID(constraint_generator_collection, obj_name, cname))
                                    _add_property_row(db, ENUM_CONSTRAINT_OPERATING_HOURS_COEFF, cg_mem, 1, 1.0)

            if not is_dsr:
                match_key = None
                for k in THERMAL_DEFAULTS.keys():
                    if k.lower() in pem.lower() or k.lower() in ref.lower():
                        match_key = k
                        break
                if match_key:
                    vals = THERMAL_DEFAULTS[match_key]
                    _add_property_row(db, ENUM_HEAT_RATE, mem_id, 1, vals["HeatRate"])
                    _add_property_row(db, ENUM_MIN_STABLE_FACTOR, mem_id, 1, vals["MinStableFactor"])
                    _add_property_row(db, ENUM_MIN_UP, mem_id, 1, vals["MinUp"])
                    _add_property_row(db, ENUM_MIN_DOWN, mem_id, 1, vals["MinDown"])

                    _add_property_row(db, ENUM_FOR, mem_id, 1, vals["FOR_b1"])
                    _add_property_row(db, ENUM_MAINT_RATE, mem_id, 2, vals["Maint_b2"])
                    _add_property_row(db, ENUM_MTTR, mem_id, 1, vals["MTTR_b1_b2"])
                    _add_property_row(db, ENUM_MTTR, mem_id, 2, vals["MTTR_b1_b2"])

                    _add_property_row(db, ENUM_VOM, mem_id, 1, vals["VOM"])

                    # NEW: ensure actual Gas/Hydrogen Fuel + Start Fuel memberships,
                    # then write Offtake at Start on that actual start-fuel membership
                    assigned_fuel = ensure_generator_fuel_memberships(obj_name, pem, ref)
                    if assigned_fuel and gen_startfuel_collection is not None:
                        try:
                            start_mem_id, _ = _ensure_membership_robust(
                                db, CollectionEnum, SystemNS, gen_startfuel_collection, obj_name, assigned_fuel
                            )
                            if start_mem_id is not None:
                                _add_property_row(db, ENUM_OFFTAKE_AT_START, start_mem_id, 1, vals["OfftakeStart"])
                        except Exception:
                            pass

            if origin == "country":
                if gen_node_collection is not None and node in nodes_in_db:
                    _ensure_membership_robust(db, CollectionEnum, SystemNS, gen_node_collection, node, obj_name)
            else:
                for bzname in sorted(allowed_bzs):
                    _ensure_membership_robust(db, CollectionEnum, SystemNS, GENERATOR_NODES_STAR_COLLECTION_ID, bzname, obj_name)

        print("\n[STEP] Done.")
        print(f"  Created Generators: {created_g}")
        print(f"  Created Batteries:  {created_b}")
        print(f"  Constraints created: {constraints_created}")
        print(f"  Global '{EVA_WEIGHT_OBJECT_NAME}' Sample Weights written (bands 1..36) with Scenario '{EVA_WEIGHTS_SCENARIO_NAME}'.")
        print("  NEW: Cumulative 'Max Units Built' written for all created generators (enum 260) and batteries (enum 113).")
        print("  FIX: Gas/Hydrogen EVA generators now get Fuel and Start Fuel memberships, and Offtake at Start is written on the actual Start Fuel membership.")

    finally:
        try:
            db.Close()
        except Exception:
            pass

# Run
step_eva_candidates()

In [ ]:
# STEP 15 — Add Generator Unit Commitment Aggregation for EVA candidate generators only
from __future__ import annotations

import os
from pathlib import Path
from typing import Optional, List, Dict

import pandas as pd

COUNTRY_XLSX = ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Country Specific.xlsx"
DEFAULT_XLSX = ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Default.xlsx"

ENUM_GEN_UNIT_COMMITMENT_AGGREGATION = 20
GEN_UNIT_COMMITMENT_AGGREGATION_VALUE = -1.0
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1

CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]


def _bootstrap_plexos_net_for_uc_aggregation():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _sheet_to_df_uc_aggregation(xlsx: Path, sheet: str) -> pd.DataFrame:
    if not xlsx.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(xlsx, sheet_name=sheet, header=0)
    except Exception:
        return pd.DataFrame()


def _find_col_uc_aggregation(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for name in names:
        for col in df.columns:
            if str(col).strip().lower() == str(name).strip().lower():
                return col
    return None


def _sanitize_name_uc_aggregation(value: str) -> str:
    value = str(value).strip().replace("/", "_").replace("\\", "_").replace(":", "_")
    for char in "()[]{}":
        value = value.replace(char, "")
    value = value.replace(" ", "_")
    while "__" in value:
        value = value.replace("__", "_")
    return value


def _get_objects_safe_uc_aggregation(db, class_enum):
    try:
        result = db.GetObjects(class_enum)
    except Exception:
        return []
    if result is None:
        return []
    try:
        return list(result)
    except Exception:
        return []


def _resolve_collection_enum_uc_aggregation(CollectionEnum, required_tokens_lc, preferred_names, fallback_id: Optional[int] = None):
    for name in preferred_names:
        if hasattr(CollectionEnum, name):
            return getattr(CollectionEnum, name)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            value = getattr(CollectionEnum, attr)
        except Exception:
            continue
        attr_lc = str(attr).lower()
        if all(token in attr_lc for token in required_tokens_lc):
            score = 0 if attr_lc.startswith("system") else 5
            candidates.append((score + len(attr_lc), attr, value))

    if candidates:
        candidates.sort(key=lambda item: item[0])
        return candidates[0][2]

    if fallback_id is not None:
        return int(fallback_id)

    raise RuntimeError("Could not resolve CollectionEnum")


def _build_step_15_candidate_generator_names() -> set[str]:
    bidding_zones = pd.read_excel(BIDDING_ZONE_XLSX, sheet_name=0, header=None)
    allowed_bzs = {str(value).strip() for value in bidding_zones.values.ravel() if not pd.isna(value) and str(value).strip()}

    candidate_names: set[str] = set()
    for origin, xlsx in (("country", COUNTRY_XLSX), ("default", DEFAULT_XLSX)):
        df = _sheet_to_df_uc_aggregation(xlsx, "FOM")
        if df.empty:
            continue

        node_col = _find_col_uc_aggregation(df, ["Node", "MARKET_NODE", "NODE"])
        pem_col = _find_col_uc_aggregation(df, ["PEMMDB Technology", "PEMMDB_Technology", "PEMMDBTechnology"])
        ref_col = _find_col_uc_aggregation(df, ["Reference Technology", "Reference", "Reference_Technology"])
        if pem_col is None:
            continue

        for _, row in df.iterrows():
            node = str(row[node_col]).strip() if node_col else ""
            pem = str(row[pem_col]).strip()
            ref = str(row[ref_col]).strip() if ref_col else ""

            if origin == "default" and ref.strip().lower() == "retirement":
                continue
            if origin == "country" and ((not node) or (node not in allowed_bzs)):
                continue
            if not pem:
                continue
            if ref == "Battery Storage_6":
                continue

            if "DSR" in pem or "DSR" in ref:
                candidate_names.add(_sanitize_name_uc_aggregation(f"{node}_{ref}"))
            else:
                candidate_names.add(_sanitize_name_uc_aggregation(f"{node}_{pem}"))

    return candidate_names


def step_15_add_generator_unit_commitment_aggregation():
    target_generator_names = _build_step_15_candidate_generator_names()

    db, ClassEnum, CollectionEnum, SystemNS = _bootstrap_plexos_net_for_uc_aggregation()
    db.Connection(str(XML_PATH))

    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)

    enum_uc_aggregation = _resolve_semantic_enum(
        "System",
        "Generator",
        "Generators",
        "Generator Unit Commitment Aggregation",
        ENUM_GEN_UNIT_COMMITMENT_AGGREGATION,
    )

    try:
        gen_mem_collection = _resolve_collection_enum_uc_aggregation(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID,
        )

        generators_in_db = {str(name).strip() for name in _get_objects_safe_uc_aggregation(db, ClassEnum.Generator) if str(name).strip()}
        existing_target_generators = sorted(target_generator_names & generators_in_db)

        wrote_rows = 0
        skipped_not_in_db = len(target_generator_names - generators_in_db)
        skipped_no_membership = 0

        for generator_name in existing_target_generators:
            try:
                mem_id = int(db.GetMembershipID(gen_mem_collection, "System", generator_name))
            except Exception:
                skipped_no_membership += 1
                continue

            db.AddProperty(
                int(mem_id),
                int(enum_uc_aggregation),
                1,
                float(GEN_UNIT_COMMITMENT_AGGREGATION_VALUE),
                None,
                None,
                None,
                None,
                None,
                None,
                None,
            )
            wrote_rows += 1

        print("\n[STEP] Generator Unit Commitment Aggregation added for Step 15 EVA candidate generators.")
        print(f"  Property: Generator Unit Commitment Aggregation (enum {enum_uc_aggregation})")
        print(f"  Value: {GEN_UNIT_COMMITMENT_AGGREGATION_VALUE} (Yes)")
        print(f"  Candidate generator names found in workbook: {len(target_generator_names)}")
        print(f"  Candidate generators found in DB: {len(existing_target_generators)}")
        print(f"  Rows written: {wrote_rows}")
        print(f"  Candidate generator names not in DB: {skipped_not_in_db}")
        print(f"  Candidate generators without System membership: {skipped_no_membership}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


step_15_add_generator_unit_commitment_aggregation()

In [ ]:

# ============================================================================
# STEP — Retirement Generators (FUEL-TOKEN GROUPING) — FINAL UPDATE
#
# Adds:
#   - Expansion Optimality (enum 24) = 0 (Linear), single value (no DateFrom)
#
# Still:
#   - Round ALL property inputs to 2 decimals.
#   - FOM (enum 66) year-specific (DateFrom=YYYY-01-01)
#   - WACC (enum 257) single value (no DateFrom), token-average across years
#   - Economic Life (enum 259) = 10, single value
#   - Max Units Retired (enum 261) = 1, single value
#   - Retire Non-anticipativity (enum 287) = -1, single value
#   - All properties Scenario="EVA_Retirement"
#
# NOTE: Generators with "New" in their name (EVA new-build candidates from
#       Step 15) are explicitly excluded from EVA_Retirement data, regardless
#       of category. Only existing generators receive this data.
# ============================================================================

from __future__ import annotations
import os
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import pandas as pd

# -----------------------------
# USER PATHS
# -----------------------------

DEFAULT_XLSX = Path(
    str(ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Default.xlsx")
)

RETIREMENT_SCENARIO_NAME = "EVA_Retirement"

# Exclude generators in category "EVA" (best-effort)
EXCLUDE_CATEGORY_NAME = "EVA"
EXCLUDE_EVA_CATEGORY = True

# Also exclude generators whose name contains this token (EVA new-build candidates)
EXCLUDE_NEW_NAME_TOKEN = "New"

# Target years (columns in Default sheets)
TARGET_YEARS = [2028, 2030, 2033, 2035]

# -----------------------------
# ENUMS
# -----------------------------
ENUM_GEN_FOM = 67
ENUM_GEN_WACC = 258
ENUM_ECON_LIFE = 260
ENUM_MAX_UNITS_RETIRED = 262
ENUM_RETIRE_NONANTICIP = 288
ENUM_EXPANSION_OPT = 24

RETIRE_NONANTICIP_VALUE = -1.0
ECON_LIFE_VALUE = 10.0
MAX_UNITS_RETIRED_VALUE = 1.0
EXPANSION_OPT_LINEAR_VALUE = 0.0  # Linear

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# COLLECTION ENUM FALLBACK IDS
# -----------------------------
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1
FALLBACK_SYSTEM_FUELS_COLLECTION_ID = 40

# -----------------------------
# FUEL TOKEN RULES
# -----------------------------
FUEL_TOKENS = [
    "Light Oil",
    "Heavy Oil",
    "Hard coal",
    "Oil Shale",
    "Natural Gas",
    "Gas",
    "Lignite",
    "Uranium",
    "Hydrogen",
    "Biomass",
    "Waste",
    "Geothermal",
    "Coal",
    "Oil",
]

def _extract_fuel_token(pemmdb_tech: str) -> Optional[str]:
    s = str(pemmdb_tech or "").strip().lower()
    if not s:
        return None
    for tok in FUEL_TOKENS:
        if tok.lower() in s:
            return tok
    return None

def _round2(x: Optional[float]) -> Optional[float]:
    if x is None:
        return None
    try:
        return float(round(float(x), 2))
    except Exception:
        return None


# -----------------------------
# HELPERS
# -----------------------------
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    from System import DateTime as NetDateTime
    return Database(), ClassEnum, CollectionEnum, SystemNS, NetDateTime


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _ensure_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name in existing:
        return
    db.AddObject(str(name), class_enum_value, bool(add_to_system), str(category or ""), str(description or ""))


def _find_col(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        for c in df.columns:
            if str(c).strip().lower() == str(n).strip().lower():
                return c
    return None


def _sheet_to_df(xlsx: Path, sheet: str) -> pd.DataFrame:
    if not xlsx.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(xlsx, sheet_name=sheet, header=0)
    except Exception:
        return pd.DataFrame()


def _resolve_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)

    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0
            score += 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        chosen = candidates[0]
        print(f"[INFO] Resolved CollectionEnum via fuzzy match: {chosen[1]} -> {int(chosen[2]) if str(chosen[2]).isdigit() else chosen[2]}")
        return chosen[2]

    if fallback_id is not None:
        print(f"[WARN] Could not resolve CollectionEnum by name; using fallback collection id={fallback_id}.")
        return int(fallback_id)

    raise RuntimeError("Could not resolve CollectionEnum")


def _ensure_membership(db, collection_enum, parent_name: str, child_name: str) -> int:
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _membership_exists(db, collection_enum, a: str, b: str) -> bool:
    try:
        _ = int(db.GetMembershipID(collection_enum, a, b))
        return True
    except Exception:
        pass
    try:
        _ = int(db.GetMembershipID(collection_enum, b, a))
        return True
    except Exception:
        return False


def _add_property_row(db, enum_id: int, mem_id: int, band: int, value: float,
                      DateFrom=None, DateTo=None, Variable=None, DataFile=None,
                      Pattern=None, Scenario=None, Action=None):
    db.AddProperty(
        int(mem_id),
        int(enum_id),
        int(band),
        float(value),
        DateFrom,
        DateTo,
        Variable,
        DataFile,
        Pattern,
        Scenario,
        Action
    )


def _best_effort_get_generator_category(db, ClassEnum, gen_name: str) -> Optional[str]:
    for meth in ["GetCategory", "GetObjectCategory", "Category", "GetCategoryName"]:
        try:
            fn = getattr(db, meth)
        except Exception:
            continue
        try:
            cat = fn(ClassEnum.Generator, gen_name)
            if cat is None:
                return None
            return str(cat)
        except Exception:
            try:
                cat = fn(gen_name, ClassEnum.Generator)
                if cat is None:
                    return None
                return str(cat)
            except Exception:
                continue
    return None


# -----------------------------
# DATA PREP: token averages (Retirement-only)
# -----------------------------
def _build_retirement_fuel_token_averages(default_xlsx: Path, years: List[int]) -> Tuple[
    Dict[str, Dict[int, float]],  # token_fom_year
    Dict[str, Dict[int, float]],  # token_waccpct_year
    Dict[str, float],             # token_waccpct_single
]:
    fom_df = _sheet_to_df(default_xlsx, "FOM")
    wacc_df = _sheet_to_df(default_xlsx, "WACC")
    hurd_df = _sheet_to_df(default_xlsx, "Hurdle Premium")

    if fom_df.empty:
        raise RuntimeError("Default.xlsx sheet 'FOM' could not be read or is empty.")

    pem_col = _find_col(fom_df, ["PEMMDB Technology", "PEMMDB_Technology", "PEMMDBTechnology"])
    ref_col = _find_col(fom_df, ["Reference Technology", "Reference", "Reference_Technology"])
    if pem_col is None or ref_col is None:
        raise RuntimeError(f"Could not find required columns in FOM sheet. Found columns: {list(fom_df.columns)}")

    is_ret = fom_df[ref_col].astype(str).str.strip().str.lower() == "retirement"
    ret_fom = fom_df[is_ret].copy()
    ret_fom[pem_col] = ret_fom[pem_col].astype(str).str.strip()
    ret_fom = ret_fom[ret_fom[pem_col].astype(str).str.strip() != ""].copy()
    if ret_fom.empty:
        raise RuntimeError("No rows with Reference Technology == 'Retirement' found in Default.xlsx FOM sheet.")

    ret_fom["__FuelToken"] = ret_fom[pem_col].map(_extract_fuel_token)
    before = len(ret_fom)
    ret_fom = ret_fom[ret_fom["__FuelToken"].notna()].copy()
    dropped = before - len(ret_fom)
    if dropped > 0:
        print(f"[WARN] Dropped {dropped} Retirement FOM rows because no fuel token could be extracted from PEMMDB Technology.")

    if ret_fom.empty:
        raise RuntimeError("After fuel-token extraction, no Retirement rows remain. Update FUEL_TOKENS list to match your naming.")

    year_cols_present = [str(y) for y in years if str(y) in ret_fom.columns]
    if not year_cols_present:
        raise RuntimeError(f"None of target year columns {years} were found in FOM sheet. Found: {list(ret_fom.columns)}")

    for yc in year_cols_present:
        ret_fom[yc] = pd.to_numeric(ret_fom[yc], errors="coerce")

    token_fom_year: Dict[str, Dict[int, float]] = {}
    for tok, sub in ret_fom.groupby("__FuelToken"):
        tok = str(tok).strip()
        if not tok:
            continue
        token_fom_year[tok] = {}
        for y in years:
            yc = str(y)
            if yc not in sub.columns:
                continue
            vals = sub[yc].dropna()
            if len(vals) == 0:
                continue
            token_fom_year[tok][y] = float(vals.mean())

    def _token_year_avg_from_sheet(df: pd.DataFrame, years: List[int]) -> Dict[str, Dict[int, float]]:
        if df is None or df.empty:
            return {}
        pemc = _find_col(df, ["PEMMDB Technology", "PEMMDB_Technology", "PEMMDBTechnology"])
        refc = _find_col(df, ["Reference Technology", "Reference", "Reference_Technology"])
        if pemc is None:
            return {}

        dfx = df.copy()
        if refc is not None:
            dfx = dfx[dfx[refc].astype(str).str.strip().str.lower() == "retirement"].copy()
        if dfx.empty:
            return {}

        dfx[pemc] = dfx[pemc].astype(str).str.strip()
        dfx = dfx[dfx[pemc].astype(str).str.strip() != ""].copy()
        dfx["__FuelToken"] = dfx[pemc].map(_extract_fuel_token)
        dfx = dfx[dfx["__FuelToken"].notna()].copy()
        if dfx.empty:
            return {}

        for y in years:
            yc = str(y)
            if yc in dfx.columns:
                dfx[yc] = pd.to_numeric(dfx[yc], errors="coerce")

        out: Dict[str, Dict[int, float]] = {}
        for tok, sub in dfx.groupby("__FuelToken"):
            tok = str(tok).strip()
            if not tok:
                continue
            out[tok] = {}
            for y in years:
                yc = str(y)
                if yc not in sub.columns:
                    continue
                vals = sub[yc].dropna()
                if len(vals) == 0:
                    continue
                out[tok][y] = float(vals.mean())
        return out

    token_wacc = _token_year_avg_from_sheet(wacc_df, years)
    token_hurd = _token_year_avg_from_sheet(hurd_df, years)

    token_waccpct_year: Dict[str, Dict[int, float]] = {}
    all_tokens = sorted(set(token_fom_year.keys()) | set(token_wacc.keys()) | set(token_hurd.keys()))
    for tok in all_tokens:
        for y in years:
            w = token_wacc.get(tok, {}).get(y, None)
            h = token_hurd.get(tok, {}).get(y, None)
            if w is None and h is None:
                continue
            w = float(w or 0.0)
            h = float(h or 0.0)
            token_waccpct_year.setdefault(tok, {})[y] = (w + h) * 100.0

    token_waccpct_single: Dict[str, float] = {}
    for tok, yd in token_waccpct_year.items():
        vals = [float(v) for v in yd.values() if v is not None]
        if vals:
            token_waccpct_single[tok] = float(sum(vals) / len(vals))

    return token_fom_year, token_waccpct_year, token_waccpct_single


# -----------------------------
# MAIN
# -----------------------------
def step_apply_retirement_properties_final():
    token_fom_year, _, token_waccpct_single = _build_retirement_fuel_token_averages(DEFAULT_XLSX, TARGET_YEARS)

    tokens = sorted(set(token_fom_year.keys()) | set(token_waccpct_single.keys()))
    print(f"[STEP] Fuel tokens with Retirement averages: {len(tokens)}")
    print("          Tokens:", tokens)

    db, ClassEnum, CollectionEnum, SystemNS, NetDateTime = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))
    def _resolve_semantic_enum(parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
        try:
            return int(db.PropertyName2EnumId(
                SystemNS.String(str(parent_class_name)),
                SystemNS.String(str(child_class_name)),
                SystemNS.String(str(collection_name)),
                SystemNS.String(str(property_name)),
            ))
        except Exception:
            return int(fallback_enum)
    
    ENUM_GEN_FOM = _resolve_semantic_enum("System", "Generator", "Generators", "FO&M Charge", 67)
    ENUM_GEN_WACC = _resolve_semantic_enum("System", "Generator", "Generators", "WACC", 258)
    ENUM_ECON_LIFE = _resolve_semantic_enum("System", "Generator", "Generators", "Economic Life", 260)
    ENUM_MAX_UNITS_RETIRED = _resolve_semantic_enum("System", "Generator", "Generators", "Max Units Retired", 262)
    ENUM_RETIRE_NONANTICIP = _resolve_semantic_enum("System", "Generator", "Generators", "Retire Non-anticipativity", 288)
    ENUM_EXPANSION_OPT = _resolve_semantic_enum("System", "Generator", "Generators", "Expansion Optimality", 24)
    

    try:
        _ensure_object(db, ClassEnum.Scenario, RETIREMENT_SCENARIO_NAME, add_to_system=True)
        retirement_scenario_str = SystemNS.String(RETIREMENT_SCENARIO_NAME)

        gen_mem_collection = _resolve_collection_enum(
            CollectionEnum, ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID
        )
        _ = _resolve_collection_enum(
            CollectionEnum, ["system", "fuel"],
            ["SystemFuels", "Fuels"],
            fallback_id=FALLBACK_SYSTEM_FUELS_COLLECTION_ID
        )

        # Try Generator↔Fuel membership
        gen_fuel_collection = None
        if hasattr(CollectionEnum, "GeneratorFuels"):
            gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels")
        elif hasattr(CollectionEnum, "FuelGenerators"):
            gen_fuel_collection = getattr(CollectionEnum, "FuelGenerators")

        fuels_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Fuel)))
        fuels_lc = [(str(f), str(f).strip().lower()) for f in fuels_in_db]

        token_to_db_fuels: Dict[str, List[str]] = {}
        for tok in tokens:
            tlc = tok.lower()
            matches = [fname for (fname, flc) in fuels_lc if tlc in flc]
            if matches:
                token_to_db_fuels[tok] = sorted(set(matches))
            else:
                print(f"[WARN] Fuel token '{tok}' matched 0 DB fuel objects by name substring.")

        gens_in_db = sorted(set(_get_objects_safe(db, ClassEnum.Generator)))
        all_relevant_db_fuels = sorted({f for fuels in token_to_db_fuels.values() for f in fuels})

        updated_gens = 0
        wrote_rows = 0
        skipped_eva = 0
        skipped_new = 0
        matched_by_membership = 0
        matched_by_name = 0

        exclude_new_lc = EXCLUDE_NEW_NAME_TOKEN.lower()

        for g in gens_in_db:
            gname = str(g).strip()
            if not gname:
                continue

            # Skip EVA new-build generators (those with "New" in the name)
            if exclude_new_lc in gname.lower():
                skipped_new += 1
                continue

            if EXCLUDE_EVA_CATEGORY:
                cat = _best_effort_get_generator_category(db, ClassEnum, gname)
                if cat is not None and str(cat).strip().lower() == EXCLUDE_CATEGORY_NAME.lower():
                    skipped_eva += 1
                    continue

            matched_tokens: List[str] = []

            if gen_fuel_collection is not None and all_relevant_db_fuels:
                gen_matched_fuels = []
                for fuel in all_relevant_db_fuels:
                    if _membership_exists(db, gen_fuel_collection, gname, fuel):
                        gen_matched_fuels.append(fuel)

                if gen_matched_fuels:
                    matched_by_membership += 1
                    gen_fuels_lc = [f.lower() for f in gen_matched_fuels]
                    for tok in tokens:
                        tlc = tok.lower()
                        if any(tlc in flc for flc in gen_fuels_lc):
                            matched_tokens.append(tok)
            else:
                glc = gname.lower()
                for tok in tokens:
                    if tok.lower() in glc:
                        matched_tokens.append(tok)
                if matched_tokens:
                    matched_by_name += 1

            if not matched_tokens:
                continue

            matched_tokens = sorted(set(matched_tokens))
            chosen_tok = matched_tokens[0]
            if len(matched_tokens) > 1:
                print(f"[WARN] Generator '{gname}' matches multiple fuel tokens {matched_tokens}; using '{chosen_tok}'.")

            # membership System->Generator
            try:
                gen_mem_id = int(db.GetMembershipID(gen_mem_collection, "System", gname))
            except Exception:
                try:
                    gen_mem_id = _ensure_membership(db, gen_mem_collection, "System", gname)
                except Exception:
                    continue

            any_written = False

            # ---------------- Year-specific FOM ----------------
            for y in TARGET_YEARS:
                fom_val = token_fom_year.get(chosen_tok, {}).get(y, None)
                if fom_val is None:
                    continue
                dt = NetDateTime(int(y), 1, 1, 0, 0, 0)
                v = _round2(fom_val)
                if v is None:
                    continue
                _add_property_row(
                    db, ENUM_GEN_FOM, gen_mem_id, 1, float(v),
                    DateFrom=dt, Scenario=retirement_scenario_str
                )
                wrote_rows += 1
                any_written = True

            # ---------------- Non-year-specific properties ----------------
            # Expansion Optimality = Linear (0)
            _add_property_row(
                db, ENUM_EXPANSION_OPT, gen_mem_id, 1, float(_round2(EXPANSION_OPT_LINEAR_VALUE) or 0.0),
                DateFrom=None, Scenario=retirement_scenario_str
            )
            wrote_rows += 1
            any_written = True

            # WACC single value: avg across years for token
            wacc_single = token_waccpct_single.get(chosen_tok, None)
            if wacc_single is not None:
                v = _round2(wacc_single)
                if v is not None:
                    _add_property_row(
                        db, ENUM_GEN_WACC, gen_mem_id, 1, float(v),
                        DateFrom=None, Scenario=retirement_scenario_str
                    )
                    wrote_rows += 1
                    any_written = True

            # Econ Life = 10
            _add_property_row(
                db, ENUM_ECON_LIFE, gen_mem_id, 1, float(_round2(ECON_LIFE_VALUE) or 10.0),
                DateFrom=None, Scenario=retirement_scenario_str
            )
            wrote_rows += 1
            any_written = True

            # Max Units Retired = 1
            _add_property_row(
                db, ENUM_MAX_UNITS_RETIRED, gen_mem_id, 1, float(_round2(MAX_UNITS_RETIRED_VALUE) or 1.0),
                DateFrom=None, Scenario=retirement_scenario_str
            )
            wrote_rows += 1
            any_written = True

            # Retire Non-anticipativity = -1
            _add_property_row(
                db, ENUM_RETIRE_NONANTICIP, gen_mem_id, 1, float(_round2(RETIRE_NONANTICIP_VALUE) or RETIRE_NONANTICIP_VALUE),
                DateFrom=None, Scenario=retirement_scenario_str
            )
            wrote_rows += 1
            any_written = True

            if any_written:
                updated_gens += 1

        print("\n[STEP] Done.")
        print(f"  Scenario used: {RETIREMENT_SCENARIO_NAME}")
        print(f"  Fuel tokens computed: {len(tokens)} -> {tokens}")
        print(f"  Generators updated: {updated_gens}")
        print(f"  Property rows written: {wrote_rows}")
        print(f"  Generators skipped (name contains '{EXCLUDE_NEW_NAME_TOKEN}'): {skipped_new}")
        if EXCLUDE_EVA_CATEGORY:
            print(f"  Generators skipped due to category '{EXCLUDE_CATEGORY_NAME}': {skipped_eva}")
        print("  Matching method counts (route used):")
        print(f"    - Matched via Generator↔Fuel memberships: {matched_by_membership}")
        print(f"    - Matched via Generator name substring (fallback): {matched_by_name}")
        print("  Written properties (Scenario=EVA_Retirement):")
        print(f"    - FOM (enum {ENUM_GEN_FOM}) year-specific (DateFrom=YYYY-01-01)")
        print(f"    - Expansion Optimality (enum {ENUM_EXPANSION_OPT}) = 0 (Linear), single value (no DateFrom)")
        print(f"    - WACC (enum {ENUM_GEN_WACC}) single value (no DateFrom), token-average across years")
        print(f"    - Economic Life (enum {ENUM_ECON_LIFE}) = {ECON_LIFE_VALUE} (no DateFrom)")
        print(f"    - Max Units Retired (enum {ENUM_MAX_UNITS_RETIRED}) = {MAX_UNITS_RETIRED_VALUE} (no DateFrom)")
        print(f"    - Retire Non-anticipativity (enum {ENUM_RETIRE_NONANTICIP}) = {RETIRE_NONANTICIP_VALUE} (no DateFrom)")
        print("  All numeric inputs rounded to 2 decimals.")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_apply_retirement_properties_final()


#### 16. Model Setup

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import List, Optional, Tuple, Dict

# ============================================================
# STEP â€” Link Scenarios to Models (Memberships)
#
# Fix vs previous:
#   - Your PLEXOS API requires CollectionEnum (typed enum), NOT raw int.
#     We convert collection id 797 -> CollectionEnum value using:
#         System.Enum.ToObject(CollectionEnum, 797)
#
# Goal:
#   Assign specific scenario sets per Model category:
#     - Adequacy
#     - Adequacy Light - S3
#     - Adequacy Lighter - S1
#     - EVA
#     - Post-EVA
#
# IMPORTANT UPDATE:
#   Fallback category inference from Model NAME now also accepts optional
#   suffixes after the scenario token, e.g.:
#     - "Adequacy - TY2028 - 36 WS - S1_LP"           -> Adequacy
#     - "Adequacy - TY2028 - 3 WS - S3_LP"            -> Adequacy Light - S3
#     - "Adequacy - TY2028 - 1 WS - S1_LP"            -> Adequacy Lighter - S1
#     - "Adequacy - TY2028 - 1 WS - S1 - Post_EVA"    -> Post-EVA
#
# Your confirmed IDs:
#   Scenario class id: 85
#   Model class id: 87
#   Model->Scenario set membership collection id: 797
# ============================================================

# -----------------------------
# USER SETTINGS
# -----------------------------
EXCLUDED_SCENARIOS: List[str] = []  # kept but not used to exclude (your sets include previously excluded)
DRY_RUN = False  # True => do not create memberships, only report

# -----------------------------
# UPDATED: Category -> Scenarios mapping
# -----------------------------
SCENARIOS_BY_MODEL_CATEGORY: Dict[str, List[str]] = {
    "Adequacy": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "Adequacy Light - S3": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "Adequacy Lighter - S1": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "EVA": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "EVA_Candidates",
        "EVA_Retirement",
        "EVA_Weights",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Optimized_Maintenance",
        "RES_Level_Detailed",
    ],
    "Post-EVA": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "EVA_Cap_Adjuster",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "Post-EVA Full": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "EVA_Cap_Adjuster",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "NTC": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "NTC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
    "FBMC": [
        "36_WS",
        "Battery_Adjusted_Demand",
        "DSR_Detailed",
        "Forced_Outage",
        "Forced_Outage_Detailed",
        "Hydro_Constraints",
        "Hydro_Inflows",
        "Min_Up_Down_Time",
        "Must_Run_Detailed",
        "FBMC",
        "Planned_Maintenance",
        "RES_Level_Detailed",
    ],
}

MODEL_CATEGORIES_TO_INCLUDE = set(SCENARIOS_BY_MODEL_CATEGORY.keys())

# -----------------------------
# Your confirmed IDs
# -----------------------------
SCENARIO_CLASS_ID = 85
MODEL_CLASS_ID = 87
MODEL_SCENARIO_MEMBERSHIP_COLLECTION_ID = 798

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# Name-pattern category inference (used in FALLBACK)
# -----------------------------
# Accept optional suffixes after the S-token, such as:
#   S1_LP
#   S1-Test
#   S1 Something
# while still keeping the WS bucket as the deciding factor.
RE_ADEQUACY_WS = re.compile(
    r"^\s*Adequacy\s*-\s*TY(\d{4})\s*-\s*(\d+)\s*WS\s*-\s*S(\d+)(?:[\s_-].*)?\s*$",
    re.IGNORECASE,
)
RE_ADEQUACY_POST = re.compile(
    r"^\s*Adequacy\s*-\s*TY(\d{4})\s*-\s*(\d+)\s*WS\s*-\s*S(\d+)(?:[\s_-].*)?\s*-\s*Post[_ -]?EVA(?:[\s_-].*)?\s*$",
    re.IGNORECASE,
)
RE_EVA = re.compile(
    r"^\s*EVA[_-]TY(\d{4})[_-](\d+)\s*WS[_-]S(\d+)(?:[_\-\s].*)?\s*$",
    re.IGNORECASE,
)  # handles EVA_TY2028_3WS_S1 etc., with optional suffixes


def _infer_model_category_from_name(model_name: str) -> Optional[str]:
    n = (model_name or "").strip()

    # Post-EVA explicitly
    if "Post_EVA" in n or "Post-EVA" in n or "Post EVA" in n:
        mpost = RE_ADEQUACY_POST.match(n)
        if mpost:
            return "Post-EVA"
        # If someone renames slightly but still contains marker, treat as Post-EVA
        return "Post-EVA"

    # EVA pattern
    if n.upper().startswith("EVA"):
        if RE_EVA.match(n):
            return "EVA"
        # Still treat as EVA if it starts with EVA_ (safer for your naming convention)
        if n.upper().startswith("EVA_") or n.upper().startswith("EVA-"):
            return "EVA"

    if "FBMC" in n.upper():
        return "FBMC"

    # Adequacy patterns (36 WS / 3 WS / 1 WS), now also with optional suffixes like _LP
    m = RE_ADEQUACY_WS.match(n)
    if m:
        ws = int(m.group(2))
        if ws == 36:
            return "Adequacy"
        if ws == 3:
            return "Adequacy Light - S3"
        if ws == 1:
            return "Adequacy Lighter - S1"

    return None


# ============================================================
# Helpers
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found in API/BIN dirs.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found in API/BIN dirs.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _get_objects_safe(db, class_enum_or_id) -> List[str]:
    try:
        res = db.GetObjects(class_enum_or_id)
    except Exception:
        return []
    if res is None:
        return []
    try:
        out = []
        for x in list(res):
            s = str(x).strip()
            if s:
                out.append(s)
        return out
    except Exception:
        return []


def _try_get_objects_by_category(db, class_enum_or_id, category: str) -> Optional[List[str]]:
    # Some builds expose GetObjects(class, category)
    try:
        res = db.GetObjects(class_enum_or_id, category)
        if res is None:
            return []
        out = []
        for x in list(res):
            s = str(x).strip()
            if s:
                out.append(s)
        return out
    except Exception:
        return None


def _try_get_object_category(db, class_id: int, obj_name: str) -> Optional[str]:
    candidates = [
        "GetObjectCategory",
        "GetCategory",
        "GetObjectFolder",
        "GetObjectGroup",
    ]
    for m in candidates:
        if hasattr(db, m):
            try:
                fn = getattr(db, m)
                v = fn(int(class_id), str(obj_name))
                if v is None:
                    continue
                s = str(v).strip()
                if s:
                    return s
            except Exception:
                pass
    return None


def _collection_enum_from_id(SystemNS, CollectionEnum, collection_id: int):
    return SystemNS.Enum.ToObject(CollectionEnum, int(collection_id))


def _get_membership_id_any_direction(db, col_enum_value, a: str, b: str) -> Tuple[Optional[int], Optional[str]]:
    try:
        mid = int(db.GetMembershipID(col_enum_value, str(a), str(b)))
        return mid, "A->B"
    except Exception:
        pass
    try:
        mid = int(db.GetMembershipID(col_enum_value, str(b), str(a)))
        return mid, "B->A"
    except Exception:
        return None, None


def _ensure_membership_either_direction(
    db,
    col_enum_value,
    a: str,
    b: str,
    *,
    debug_first_error: List[str],
) -> Tuple[bool, Optional[int], Optional[str]]:
    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return False, mid, direction

    if DRY_RUN:
        return True, None, None

    try:
        db.AddMembership(col_enum_value, str(a), str(b))
    except Exception as e:
        if not debug_first_error:
            debug_first_error.append(f"AddMembership(typedEnum, '{a}', '{b}') failed: {e}")

    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return True, mid, direction

    try:
        db.AddMembership(col_enum_value, str(b), str(a))
    except Exception as e:
        if not debug_first_error:
            debug_first_error.append(f"AddMembership(typedEnum, '{b}', '{a}') failed: {e}")

    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return True, mid, direction

    return False, None, None


def _remove_membership_either_direction(
    db,
    col_enum_value,
    a: str,
    b: str,
    *,
    debug_first_error: List[str],
) -> bool:
    removed = False
    for parent, child in [(a, b), (b, a)]:
        try:
            mid = int(db.GetMembershipID(col_enum_value, str(parent), str(child)))
        except Exception:
            mid = None
        if mid is None:
            continue
        if DRY_RUN:
            removed = True
            continue
        try:
            db.RemoveMembership(col_enum_value, str(parent), str(child))
            removed = True
        except Exception as e:
            if not debug_first_error:
                debug_first_error.append(f"RemoveMembership(typedEnum, '{parent}', '{child}') failed: {e}")
    return removed


# ============================================================
# MAIN
# ============================================================
def link_scenarios_to_models():
    db, ClassEnum, CollectionEnum, SystemNS = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    scenario_class = getattr(ClassEnum, "Scenario", SCENARIO_CLASS_ID)
    model_class = getattr(ClassEnum, "Model", MODEL_CLASS_ID)

    col_797 = _collection_enum_from_id(SystemNS, CollectionEnum, MODEL_SCENARIO_MEMBERSHIP_COLLECTION_ID)

    try:
        all_scenarios = _get_objects_safe(db, scenario_class)
        if not all_scenarios:
            raise RuntimeError("No Scenario objects found in DB.")

        scenarios_in_db_lc: Dict[str, str] = {s.strip().lower(): s for s in all_scenarios if s.strip()}

        print("\n" + "=" * 100)
        print("STEP: Link Scenarios -> Models (collection 797)")
        print("=" * 100)
        print(f"[INFO] Total scenarios in DB:           {len(all_scenarios)}")
        print(f"[INFO] Excluded scenarios (editable):   {len(EXCLUDED_SCENARIOS)} -> {EXCLUDED_SCENARIOS}")
        print(f"[INFO] Model categories to include:     {sorted(MODEL_CATEGORIES_TO_INCLUDE)}")
        print(f"[INFO] DRY_RUN:                         {DRY_RUN}")

        models_in_scope: List[str] = []
        category_method_used: str = ""

        model_to_category: Dict[str, str] = {}

        got_any = False
        for cat in MODEL_CATEGORIES_TO_INCLUDE:
            res = _try_get_objects_by_category(db, model_class, cat)
            if res is not None:
                got_any = True
                for m in res:
                    models_in_scope.append(m)
                    if m.strip():
                        model_to_category[m] = cat

        if got_any:
            seen = set()
            deduped = []
            for m in models_in_scope:
                key = m.lower()
                if key in seen:
                    continue
                seen.add(key)
                deduped.append(m)
            models_in_scope = deduped
            category_method_used = "GetObjects(class, category) overload"

        if not models_in_scope:
            all_models = _get_objects_safe(db, model_class)
            if not all_models:
                raise RuntimeError("No Model objects found in DB.")

            kept = []
            for m in all_models:
                cat = _try_get_object_category(db, MODEL_CLASS_ID, m)
                if cat is not None and cat.strip() in MODEL_CATEGORIES_TO_INCLUDE:
                    kept.append(m)
                    model_to_category[m] = cat.strip()

            if kept:
                models_in_scope = kept
                category_method_used = "Per-object category lookup"
            else:
                models_in_scope = all_models
                category_method_used = "FALLBACK (could not filter by category on this build)"

        print(f"[INFO] Models selected:                 {len(models_in_scope)}")
        print(f"[INFO] Category filter method:         {category_method_used}")
        if "FALLBACK" in category_method_used:
            print("[WARN] Could not read Model categories via API methods.")
            print("       Will infer Model category from NAME patterns instead.")

        created = 0
        already = 0
        failed = 0
        skipped_models = 0
        removed_ntc_from_fbmc = 0
        direction_counts: Dict[str, int] = {"A->B": 0, "B->A": 0, "None": 0}

        first_error: List[str] = []
        fail_msgs: List[str] = []
        max_fail_print = 200

        missing_scenarios_by_cat: Dict[str, List[str]] = {k: [] for k in MODEL_CATEGORIES_TO_INCLUDE}

        inferred_counts: Dict[str, int] = {k: 0 for k in MODEL_CATEGORIES_TO_INCLUDE}
        unknown_name_examples: List[str] = []
        max_unknown_examples = 30

        for model_name in models_in_scope:
            model_cat = model_to_category.get(model_name)

            if model_cat is None or model_cat not in SCENARIOS_BY_MODEL_CATEGORY:
                inferred = _infer_model_category_from_name(model_name)
                if inferred is not None:
                    model_cat = inferred
                    inferred_counts[model_cat] = inferred_counts.get(model_cat, 0) + 1

            if model_cat is None or model_cat not in SCENARIOS_BY_MODEL_CATEGORY:
                skipped_models += 1
                if len(unknown_name_examples) < max_unknown_examples:
                    unknown_name_examples.append(model_name)
                continue

            desired_scenarios = SCENARIOS_BY_MODEL_CATEGORY.get(model_cat, [])

            if model_cat == "FBMC":
                ntc_real_name = scenarios_in_db_lc.get("ntc")
                if ntc_real_name is not None:
                    if _remove_membership_either_direction(
                        db,
                        col_797,
                        model_name,
                        ntc_real_name,
                        debug_first_error=first_error,
                    ):
                        removed_ntc_from_fbmc += 1

            scenarios_to_link: List[str] = []
            for s in desired_scenarios:
                key = str(s).strip().lower()
                real_name = scenarios_in_db_lc.get(key)
                if real_name is None:
                    if s not in missing_scenarios_by_cat[model_cat]:
                        missing_scenarios_by_cat[model_cat].append(s)
                    continue
                scenarios_to_link.append(real_name)

            for scen_name in scenarios_to_link:
                mid, _dir = _get_membership_id_any_direction(db, col_797, model_name, scen_name)
                if mid is not None:
                    already += 1
                    direction_counts[_dir or "None"] += 1
                    continue

                created_now, mid2, direction = _ensure_membership_either_direction(
                    db, col_797, model_name, scen_name, debug_first_error=first_error
                )

                if DRY_RUN:
                    created += 1
                    direction_counts["None"] += 1
                    continue

                if mid2 is not None:
                    if created_now:
                        created += 1
                    else:
                        already += 1
                    direction_counts[direction or "None"] += 1
                else:
                    failed += 1
                    if len(fail_msgs) < max_fail_print:
                        fail_msgs.append(
                            f"Failed membership: Model='{model_name}' <-> Scenario='{scen_name}' (collection 797; tried both directions)"
                        )

        print("\n[RESULT]")
        print(f"  Memberships already present: {already}")
        print(f"  Memberships created:         {created}")
        print(f"  Memberships failed:          {failed}")
        print(f"  NTC memberships removed from FBMC models: {removed_ntc_from_fbmc}")
        print(f"  Models skipped (unknown category): {skipped_models}")
        print(f"  Direction used (in DB):       {direction_counts}")

        if "FALLBACK" in category_method_used:
            print("\n[INFO] Category inference (from Model names) counts:")
            for k in sorted(inferred_counts.keys()):
                print(f"  - {k}: {inferred_counts.get(k, 0)}")

        if unknown_name_examples:
            print("\n[WARN] Example Models with names that did NOT match any known pattern (first {}):".format(len(unknown_name_examples)))
            for m in unknown_name_examples:
                print(f"  - {m}")

        any_missing = any(v for v in missing_scenarios_by_cat.values())
        if any_missing:
            print("\n[WARN] Some requested scenarios were NOT found in the DB (by category):")
            for cat in sorted(missing_scenarios_by_cat.keys()):
                miss = missing_scenarios_by_cat[cat]
                if miss:
                    print(f"  - {cat}: {miss}")

        if first_error:
            print("\n[DEBUG] First AddMembership exception encountered:")
            print("  " + first_error[0])

        if fail_msgs:
            print("\n[FAILURES] (first {}):".format(len(fail_msgs)))
            for msg in fail_msgs:
                print("  - " + msg)

        print("\n[DONE]")

    finally:
        try:
            db.Close()
        except Exception:
            pass


if __name__ == "__main__":
    link_scenarios_to_models()

#### 17. Cloud Sync

In [ ]:
#%% Sync with PLEXOS Cloud (push changeset) + CAPTURE changeset id (with interactive commit message)
from eecloud.cloudsdk import CloudSDK
from eecloud.models import *
import re
from pathlib import Path
import pandas as pd

# -----------------------------
# INPUTS
# -----------------------------
dotnet_exe_cli_path = CLI_PATH
ACTIVE_STUDY_ID = require_study_id()
# Keep existing SDK init
pxc = CloudSDK()

# -----------------------------
# Build a default commit message including Bidding Zones (if available)
# -----------------------------
def read_bidding_zones(xlsx_path: Path) -> list:
    try:
        # Read first sheet, first column (assumes header in first row and data from row 2)
        df = pd.read_excel(xlsx_path, engine="openpyxl")  # openpyxl tends to be reliable for .xlsx
        if df.shape[1] >= 1:
            col = df.iloc[:, 0].dropna().astype(str).tolist()
            # If the first row is a header name rather than a zone, pandas will treat it as header already,
            # so col will be the data rows. This should match "row 2 and beyond".
            return [c.strip() for c in col if c.strip()]
    except Exception as e:
        print(f"Warning: could not read bidding zones from {xlsx_path}: {e}")
    return []

zones = read_bidding_zones(BIDDING_ZONE_XLSX)
zones_str = ", ".join(zones) if zones else ""
base_msg = "Base Model Sync"
default_commit = base_msg + (f". Bidding Zones: {zones_str}" if zones_str else "")

# -----------------------------
# Prompt the user for a commit message (GUI pop-up if possible, else console)
# -----------------------------
def ask_commit_message(default: str) -> str:
    # Try GUI prompt (tkinter). If that fails (no display, running headless), fall back to console input.
    try:
        import tkinter as tk
        from tkinter import simpledialog
        root = tk.Tk()
        root.withdraw()  # hide main window
        # show dialog with the default pre-filled
        result = simpledialog.askstring("Commit message", "Edit commit message for this changeset:", initialvalue=default)
        root.destroy()
        if result is None or str(result).strip() == "":
            # User cancelled or empty -> use default
            return default
        return str(result).strip()
    except Exception as e:
        # No GUI available: fallback to console input
        try:
            print("GUI not available, falling back to console input.")
            print("Press Enter to accept the default commit message shown below.")
            print("Default commit message:")
            print("  " + default)
            user_in = input("Enter commit message (or press Enter to accept default):\n> ")
            if user_in is None:
                return default
            user_in = user_in.strip()
            return user_in if user_in else default
        except Exception:
            # Last fallback: return default
            return default

commit_message = ask_commit_message(default_commit)
print("Using commit message:", commit_message)

# -----------------------------
# Push changeset and capture id (same as before)
# -----------------------------
command_responses = pxc.study.push_changeset(
    study_id=ACTIVE_STUDY_ID,
    commit_message=commit_message,
    print_message=True
)

last_command_response: CommandResponse[Contracts_PushChangesetResponse] = pxc.solution.get_final_response(command_responses)

# This variable will be used by the enqueue builder step later
PUSHED_CHANGESET_ID = None

GUID_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")

if last_command_response is not None:
    print("Push status:", last_command_response.Status)

if last_command_response is not None and last_command_response.Status == "Success":
    data: Contracts_PushChangesetResponse = last_command_response.EventData
    print(data)

    # Try common attribute names used across SDK versions
    for attr in ("ChangesetId", "ChangeSetId", "Id", "NewChangesetId", "CreatedChangesetId"):
        if hasattr(data, attr):
            v = getattr(data, attr)
            if v:
                PUSHED_CHANGESET_ID = str(v)
                break

    # Fallback: try extracting a GUID from the string representation
    if not PUSHED_CHANGESET_ID:
        m = GUID_RE.search(str(data))
        if m:
            PUSHED_CHANGESET_ID = m.group(0)

print("PUSHED_CHANGESET_ID:", PUSHED_CHANGESET_ID)




#### 18. Create EVA Enqueue JSON For Latest Changeset

In [ ]:
#%% Build Enqueue JSON for pushed changeset (else latest) â€” model name from generated JSON
import json
import subprocess
import re
import time
from pathlib import Path

# -----------------------------
# INPUTS
# -----------------------------
SIMULATION_ID = "28f7d687-aa38-4e28-8a13-580f7223f217"
OUTPUT_DIRECTORY = str(IMPORTER_DIR)
dotnet_exe_cli_path = CLI_PATH

Path(OUTPUT_DIRECTORY).mkdir(parents=True, exist_ok=True)

GUID_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")

def run_cli(cmd: list[str]) -> str:
    cp = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return ((cp.stdout or "") + "\n" + (cp.stderr or "")).strip()

def extract_guids(text: str) -> list[str]:
    return GUID_RE.findall(text or "")

def get_latest_changeset_id_cli(study_id: str) -> str:
    variants = [
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "--studyId", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "--study-id", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "-s", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", study_id],
    ]
    last_out = None
    for cmd in variants:
        out = run_cli(cmd)
        last_out = out
        gids = extract_guids(out)

        # Choose the last GUID that is not the study_id
        for g in reversed(gids):
            if g != study_id:
                return g

    raise RuntimeError(f"Could not parse latest changeset id from CLI output:\n{last_out}")

def make_filename_safe(name: str) -> str:
    bad = r'<>:"/\|?*'
    for ch in bad:
        name = name.replace(ch, "_")
    return name.strip().replace(" ", "_")

# -----------------------------
# Determine which changeset to use
# -----------------------------
changeset_id = None

if "PUSHED_CHANGESET_ID" in globals() and globals().get("PUSHED_CHANGESET_ID"):
    changeset_id = globals()["PUSHED_CHANGESET_ID"]

if not changeset_id:
    changeset_id = get_latest_changeset_id_cli(ACTIVE_STUDY_ID)

print("Using changeset id:", changeset_id)

# -----------------------------
# Build temp enqueue using CLI (no modelName!)
# -----------------------------
TEMP_FILE_STEM = "TEMP_ENQUEUE"
TEMP_JSON = f"{TEMP_FILE_STEM}.json"
TEMP_JSONL = f"{TEMP_FILE_STEM}.jsonl"

cmd = [
    dotnet_exe_cli_path,
    "simulation",
    "build-request-from-previous",
    "--outputDirectory", OUTPUT_DIRECTORY,
    "--simulationId", SIMULATION_ID,
    "--studyId", ACTIVE_STUDY_ID,
    "--changesetId", changeset_id,
    "--file", TEMP_JSON,
    "--overwrite"
]

print("Running CLI command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

temp_path = Path(OUTPUT_DIRECTORY) / TEMP_JSON
if not temp_path.exists():
    temp_path = Path(OUTPUT_DIRECTORY) / TEMP_JSONL
    if not temp_path.exists():
        raise RuntimeError(f"Temporary enqueue file not found: {TEMP_JSON} or {TEMP_JSONL}")

# -----------------------------
# Read model name from generated JSON and rename
# -----------------------------
with temp_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

models = payload.get("Models", [])
if not models:
    raise RuntimeError("No 'Models' field found in generated enqueue JSON.")

real_model_name = models[0]
safe_name = make_filename_safe(real_model_name)

final_path = Path(OUTPUT_DIRECTORY) / f"Enqueue_{safe_name}.json"

if final_path.exists():
    final_path.unlink()

temp_path.rename(final_path)

print("\nâœ“ Final enqueue file created:")
print(final_path)
print("âœ“ Models:", models)
print("âœ“ Changeset used:", changeset_id)

# -----------------------------
# 10-second pause before next cell
# -----------------------------
print("\nWaiting 10 seconds before continuing...")
for i in range(10, 0, -1):
    print(f"{i}...", end=" ", flush=True)
    time.sleep(1)

print("\nâœ“ Continuing to next step.")




#### 19. Enqueue EVA Simulation and Download Results

In [ ]:
#%% Stand-alone: Enqueue from JSON (CLI), wait (CLI polling), download outputs (CLI)
from __future__ import annotations

from pathlib import Path
import subprocess
import json
import time
import re
from typing import Iterable, Optional

from eecloud.cloudsdk import CloudSDK

# -----------------------------
# INPUTS
# -----------------------------

# (A) OPTIONAL MANUAL OVERRIDE:
#     - Leave as None to auto-detect the newest Enqueue_*.json
#     - Or set to a specific file path string
ENQUEUE_JSON_MANUAL: Optional[str] = None
# Example:
# ENQUEUE_JSON_MANUAL = str(IMPORTER_DIR / "Enqueue_EVA_TY28-35_3WS_S1_Fitted_Light.json")

# This is where your FIRST script writes the enqueue file
OUTPUT_DIRECTORY = str(IMPORTER_DIR)

RESULTS_DIR = IMPORTER_DIR / "Result_Downloads"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Special EVA output folder (requested)
EVA_RESULTS_DIR = RESULTS_DIR / "EVA"
EVA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

dotnet_exe_cli_path = CLI_PATH
POLL_SECONDS = 10  # how often to poll status

# Needed for `pxc solution latest-id ...`
ACTIVE_STUDY_ID: Optional[str] = require_study_id()

# -----------------------------
# EVA model name trigger (requested)
# -----------------------------
EVA_TRIGGER_MODEL_NAME = "EVA_TY28-35_3WS_S1_Fitted_Light"

# -----------------------------
# SDK init
# -----------------------------
pxc = CloudSDK(CLI_PATH)

# -----------------------------
# Helpers
# -----------------------------
def run_cli(cmd: list[str]) -> tuple[int, str, str]:
    cp = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return cp.returncode, (cp.stdout or "").strip(), (cp.stderr or "").strip()

def run_cli_json(cmd: list[str]):
    code, out, err = run_cli(cmd)
    if code != 0:
        raise RuntimeError(f"CLI failed (code {code}).\nSTDOUT:\n{out}\n\nSTDERR:\n{err}")

    text = (out if out else err).strip()
    try:
        return json.loads(text)
    except Exception:
        first_obj, last_obj = text.find("{"), text.rfind("}")
        first_arr, last_arr = text.find("["), text.rfind("]")
        if first_arr != -1 and last_arr != -1 and last_arr > first_arr:
            return json.loads(text[first_arr:last_arr+1])
        if first_obj != -1 and last_obj != -1 and last_obj > first_obj:
            return json.loads(text[first_obj:last_obj+1])
        raise RuntimeError(f"CLI output not parseable JSON.\nOutput:\n{text[:2000]}")

def extract_sim_and_exec_from_enqueue_response(resp) -> tuple[Optional[str], Optional[str]]:
    if isinstance(resp, list) and resp and isinstance(resp[0], dict):
        return resp[0].get("Id"), resp[0].get("ExecutionId")
    if isinstance(resp, dict):
        ed = resp.get("EventData", resp)
        if isinstance(ed, dict):
            if "Id" in ed:
                return ed.get("Id"), ed.get("ExecutionId")
            ss = ed.get("SimulationStarted") or ed.get("simulationStarted")
            if isinstance(ss, list) and ss and isinstance(ss[0], dict):
                s0 = ss[0]
                sim_id = s0.get("Id")
                exe_id = s0.get("ExecutionId")
                if isinstance(sim_id, dict) and "Value" in sim_id:
                    sim_id = sim_id["Value"]
                if isinstance(exe_id, dict) and "Value" in exe_id:
                    exe_id = exe_id["Value"]
                return sim_id, exe_id
    return None, None

def make_filename_safe(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_. \-]+', "_", (name or "")).strip().replace(" ", "_")

def _normalize_status(s: Optional[str]) -> str:
    return (s or "").strip()

def _is_terminal_status(status: str) -> bool:
    return status in {"CompletedSuccess", "CompletedFailed", "Cancelled", "Canceled", "Failed"}

def _coerce_records_from_cli_json(data) -> list[dict]:
    if isinstance(data, list):
        return [r for r in data if isinstance(r, dict)]
    if isinstance(data, dict):
        for k in ("SimulationRecords", "simulationRecords", "Records", "records", "Items", "items", "Value", "value"):
            v = data.get(k)
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
        if any(k in data for k in ("Status", "status")) and any(k in data for k in ("Id", "SimulationId", "simulationId")):
            return [data]
    return []

def get_simulation_status_cli(simulation_id: str) -> dict:
    variants = [
        [dotnet_exe_cli_path, "simulation", "list", "--simulationId", simulation_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "list-simulations", "--simulationId", simulation_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "get", "--simulationId", simulation_id, "--format", "JSON"],
    ]
    last_err = None
    for cmd in variants:
        try:
            data = run_cli_json(cmd)
            recs = _coerce_records_from_cli_json(data)
            if recs:
                return recs[0]
            last_err = f"Unexpected JSON shape: {type(data)} {str(data)[:200]}"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Could not fetch simulation status.\nLast error:\n{last_err}")

def wait_for_simulation_cli(simulation_id: str, poll_seconds: int = 10) -> dict:
    last = None
    while True:
        rec = get_simulation_status_cli(simulation_id)
        status = _normalize_status(rec.get("Status") or rec.get("status"))
        if status != last:
            print(f"[simulation {simulation_id}] Status: {status}")
            last = status
        if _is_terminal_status(status):
            return rec
        time.sleep(poll_seconds)

def list_simulations_by_execution_cli(execution_id: str) -> list[dict]:
    variants = [
        [dotnet_exe_cli_path, "simulation", "list", "--executionId", execution_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "list-simulations", "--executionId", execution_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "get", "--executionId", execution_id, "--format", "JSON"],
    ]
    last_err = None
    for cmd in variants:
        try:
            data = run_cli_json(cmd)
            recs = _coerce_records_from_cli_json(data)
            if recs:
                return recs
            last_err = f"Unexpected JSON shape: {type(data)} {str(data)[:200]}"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Could not list simulations by executionId.\nLast error:\n{last_err}")

def _looks_like_stitch_record(rec: dict) -> bool:
    name = (rec.get("Name") or rec.get("name") or rec.get("ModelName") or rec.get("modelName") or "")
    sim_type = (rec.get("Type") or rec.get("type") or rec.get("JobType") or rec.get("jobType") or "")
    desc = (rec.get("Description") or rec.get("description") or "")
    hay = f"{name} {sim_type} {desc}".lower()
    return ("stitch" in hay) or ("solution stitching" in hay) or ("solution-stitch" in hay) or ("parquet" in hay)

def get_parquet_simulation_id_from_execution_cli(execution_id: str) -> Optional[str]:
    try:
        recs = list_simulations_by_execution_cli(execution_id)
    except Exception:
        return None
    for r in recs:
        if _looks_like_stitch_record(r):
            sid = r.get("Id") or r.get("SimulationId") or r.get("simulationId")
            if sid:
                return str(sid)
    return None

def wait_for_execution_cli(execution_id: str, parent_simulation_id: str, poll_seconds: int = 10) -> dict:
    print(f"\n[execution {execution_id}] Waiting for ALL simulations in execution to finish (split-safe).")
    last_status_by_id: dict[str, str] = {}
    last_count = None

    while True:
        try:
            recs = list_simulations_by_execution_cli(execution_id)
        except Exception as e:
            print(f"[WARN] Could not list by executionId ({execution_id}). Falling back to parent-only wait.\n  Reason: {e}")
            return wait_for_simulation_cli(parent_simulation_id, poll_seconds=poll_seconds)

        if last_count != len(recs):
            print(f"[execution {execution_id}] Simulations discovered: {len(recs)}")
            last_count = len(recs)

        all_terminal = True
        stitch_found = False
        stitch_terminal = False

        for r in recs:
            sid = str(r.get("Id") or r.get("SimulationId") or r.get("simulationId") or "").strip()
            status = _normalize_status(r.get("Status") or r.get("status"))
            if sid:
                prev = last_status_by_id.get(sid)
                if status and status != prev:
                    label = r.get("Name") or r.get("ModelName") or r.get("Type") or ""
                    label = f" ({label})" if label else ""
                    print(f"[execution {execution_id}] [simulation {sid}]{label} Status: {status}")
                    last_status_by_id[sid] = status

            if not _is_terminal_status(status):
                all_terminal = False

            if _looks_like_stitch_record(r):
                stitch_found = True
                stitch_terminal = stitch_terminal or _is_terminal_status(status)

        ready = (all_terminal and stitch_terminal) if stitch_found else all_terminal
        if ready:
            print(f"[execution {execution_id}] âœ“ Execution is ready (all terminal; stitch terminal if present).")
            return get_simulation_status_cli(parent_simulation_id)

        time.sleep(poll_seconds)

# -----------------------------
# Auto-detect enqueue JSON
# -----------------------------
def _detect_enqueue_json_path() -> Path:
    # 1) Manual override wins
    if ENQUEUE_JSON_MANUAL:
        p = Path(ENQUEUE_JSON_MANUAL)
        if not p.exists():
            raise FileNotFoundError(f"Manual ENQUEUE_JSON_MANUAL not found: {p}")
        return p

    # 2) If the first script ran in same notebook/session, reuse its output if available
    #    (your first script creates a variable named 'final_path')
    for key in ("final_path", "FINAL_ENQUEUE_JSON_PATH", "ENQUEUE_JSON_PATH", "LAST_ENQUEUE_JSON"):
        try:
            v = globals().get(key)
        except Exception:
            v = None
        if v:
            p = Path(str(v))
            if p.exists() and p.suffix.lower() == ".json" and p.name.lower().startswith("enqueue_"):
                return p

    # 3) Otherwise, pick newest Enqueue_*.json in OUTPUT_DIRECTORY
    out_dir = Path(OUTPUT_DIRECTORY)
    if not out_dir.exists():
        raise FileNotFoundError(f"OUTPUT_DIRECTORY does not exist: {out_dir}")

    candidates = sorted(
        out_dir.glob("Enqueue_*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No Enqueue_*.json found in OUTPUT_DIRECTORY: {out_dir}")

    return candidates[0]

# -----------------------------
# Enqueue JSON: extract EXACT model name from Models[0]
# -----------------------------
def extract_model_name_from_enqueue_json(enqueue_path: Path) -> Optional[str]:
    try:
        j = json.loads(enqueue_path.read_text(encoding="utf-8"))
    except Exception:
        return None

    if isinstance(j, dict):
        models = j.get("Models") or j.get("models")
        if isinstance(models, list) and models and isinstance(models[0], str) and models[0].strip():
            return models[0].strip()

    return None

# -----------------------------
# Solution type detection + downloads
# -----------------------------
def solution_files_list_types_cli(solution_id: str) -> list[str]:
    data = run_cli_json([dotnet_exe_cli_path, "solution", "files", "list-types", "--solutionId", str(solution_id), "--format", "JSON"])
    if isinstance(data, list):
        return [str(x) for x in data]
    if isinstance(data, dict):
        for k in ("Types", "types", "Value", "value", "Items", "items"):
            v = data.get(k)
            if isinstance(v, list):
                return [str(x) for x in v]
    raise RuntimeError(f"Unexpected list-types JSON: {type(data)} {str(data)[:200]}")

def solution_latest_id_cli(study_id: str, model_name: str) -> str:
    data = run_cli_json([dotnet_exe_cli_path, "solution", "latest-id", "--studyId", study_id, "--model", model_name, "--format", "JSON"])
    if isinstance(data, str) and data.strip():
        return data.strip().strip('"')
    if isinstance(data, dict):
        for k in ("Id", "id", "Value", "value", "SolutionId", "solutionId"):
            v = data.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
    if isinstance(data, list) and data and isinstance(data[0], dict):
        for k in ("Id", "id", "Value", "value", "SolutionId", "solutionId"):
            v = data[0].get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
    raise RuntimeError(f"Could not parse latest-id JSON: {type(data)} {str(data)[:200]}")

def download_solution_outputs_cli(solution_id: str, out_dir: Path, type_name: str) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        dotnet_exe_cli_path,
        "solution",
        "files",
        "download",
        "--type", str(type_name),
        "--solutionId", str(solution_id),
        "--outputDirectory", str(out_dir),
    ]
    print(" ".join(cmd))
    code, out, err = run_cli(cmd)
    if code != 0:
        raise RuntimeError(f"Download failed for solution {solution_id} type {type_name}.\nSTDOUT:\n{out}\n\nSTDERR:\n{err}")

# --- PARQUET ONLY (existing) ---
def pick_parquet_type(available: Iterable[str]) -> Optional[str]:
    avail = {str(x) for x in available}
    for t in ("StitchedParquet", "Parquet", "StitchedHybrid"):
        if t in avail:
            return t
    return None

def download_parquet_only_for_solution(solution_id: str, out_dir: Path) -> None:
    types = solution_files_list_types_cli(solution_id)
    parquet_t = pick_parquet_type(types)

    if not parquet_t:
        raise RuntimeError(f"No Parquet type found for solution {solution_id}. Available: {types}")

    print(f"[INFO] Solution {solution_id} available types: {types}")
    print(f"[INFO] Downloading '{parquet_t}' (Parquet only) ...")
    download_solution_outputs_cli(solution_id, out_dir, parquet_t)

# --- CSVOUTPUT ONLY (requested EVA behavior) ---
def pick_csvoutput_type(available: Iterable[str]) -> Optional[str]:
    avail = {str(x) for x in available}
    return "CSVOutput" if "CSVOutput" in avail else None

def download_csvoutput_only_for_solution(solution_id: str, out_dir: Path) -> None:
    types = solution_files_list_types_cli(solution_id)
    csv_t = pick_csvoutput_type(types)

    if not csv_t:
        raise RuntimeError(f"No CSVOutput type found for solution {solution_id}. Available: {types}")

    print(f"[INFO] Solution {solution_id} available types: {types}")
    print(f"[INFO] Downloading '{csv_t}' (CSVOutput only) ...")
    download_solution_outputs_cli(solution_id, out_dir, csv_t)

def download_parquet_only_with_latest_fallback(solution_id: str, out_dir: Path, *, study_id: Optional[str], enqueue_model_name: Optional[str]) -> None:
    def try_solution(sid: str):
        download_parquet_only_for_solution(str(sid), out_dir)

    # Try the provided id
    try:
        try_solution(solution_id)
        return
    except Exception as e1:
        if not (study_id and enqueue_model_name):
            raise RuntimeError(
                "Initial download failed and cannot try latest-id fallback (missing STUDY_ID or enqueue model name).\n"
                f"Original error: {e1}"
            )

        latest = solution_latest_id_cli(study_id, enqueue_model_name)
        print("\n" + "!" * 80)
        print("[WARN] Initial solution id was not downloadable. Falling back to solution latest-id for enqueue model:")
        print(f"       model='{enqueue_model_name}' -> latest solutionId={latest}")
        print("!" * 80 + "\n")

        try_solution(latest)

def download_csvoutput_only_with_latest_fallback(solution_id: str, out_dir: Path, *, study_id: Optional[str], enqueue_model_name: Optional[str]) -> None:
    def try_solution(sid: str):
        download_csvoutput_only_for_solution(str(sid), out_dir)

    # Try the provided id
    try:
        try_solution(solution_id)
        return
    except Exception as e1:
        if not (study_id and enqueue_model_name):
            raise RuntimeError(
                "Initial download failed and cannot try latest-id fallback (missing STUDY_ID or enqueue model name).\n"
                f"Original error: {e1}"
            )

        latest = solution_latest_id_cli(study_id, enqueue_model_name)
        print("\n" + "!" * 80)
        print("[WARN] Initial solution id was not downloadable. Falling back to solution latest-id for enqueue model:")
        print(f"       model='{enqueue_model_name}' -> latest solutionId={latest}")
        print("!" * 80 + "\n")

        try_solution(latest)

# -----------------------------
# SDK solution id discovery
# -----------------------------
def get_solution_ids_via_sdk(sim_id: str):
    responses = pxc.simulation.list_simulations(simulation_id=sim_id, print_message=False)
    final = pxc.solution.get_final_response(responses)
    if final is None or getattr(final, "Status", None) != "Success":
        raise RuntimeError(f"SDK list_simulations failed. Final={final}")
    data = getattr(final, "EventData", None)
    records = getattr(data, "SimulationRecords", None) or []
    if not records:
        raise RuntimeError("SDK returned no SimulationRecords for this simulation id.")
    sim_obj = records[0]
    model_identifiers = getattr(sim_obj, "ModelIdentifiers", []) or []
    if not model_identifiers:
        raise RuntimeError("No ModelIdentifiers found on simulation object; cannot determine solution ids.")
    return model_identifiers

def download_outputs_for_simulation(sim_id: str, base_dir: Path, *, enqueue_model_name: Optional[str], use_csvoutput_only: bool) -> None:
    model_identifiers = get_solution_ids_via_sdk(sim_id)
    base_dir.mkdir(parents=True, exist_ok=True)

    for mi in model_identifiers:
        model_name = getattr(mi, "Name", None) or getattr(mi, "ModelName", None) or "Model"
        solution_id = getattr(mi, "Id", None)
        if solution_id is None:
            raise RuntimeError("ModelIdentifier missing Id (solution id).")

        safe_model = make_filename_safe(model_name)
        out_dir = base_dir / safe_model

        print(f"\nDownloading outputs for model '{model_name}' (solution {solution_id}) -> {out_dir}")
        if use_csvoutput_only:
            download_csvoutput_only_with_latest_fallback(
                str(solution_id),
                out_dir,
                study_id=ACTIVE_STUDY_ID,
                enqueue_model_name=enqueue_model_name,   # IMPORTANT: exact enqueue Models[0]
            )
        else:
            download_parquet_only_with_latest_fallback(
                str(solution_id),
                out_dir,
                study_id=ACTIVE_STUDY_ID,
                enqueue_model_name=enqueue_model_name,   # IMPORTANT: exact enqueue Models[0]
            )

# -----------------------------
# MAIN
# -----------------------------
ENQUEUE_JSON = _detect_enqueue_json_path()
print(f"[INFO] Using Enqueue JSON: {ENQUEUE_JSON}")

if not ENQUEUE_JSON.exists():
    raise FileNotFoundError(f"Enqueue JSON not found: {ENQUEUE_JSON}")

enqueue_model_name = extract_model_name_from_enqueue_json(ENQUEUE_JSON)
print(f"[INFO] Model name extracted from Enqueue JSON: {enqueue_model_name!r}")

# Requested behavior switch:
use_csvoutput_only = (enqueue_model_name == EVA_TRIGGER_MODEL_NAME)
if use_csvoutput_only:
    print(f"[INFO] EVA trigger matched ({EVA_TRIGGER_MODEL_NAME!r}). Will download CSVOutput only and store under: {EVA_RESULTS_DIR}")

enqueue_cmd = [
    dotnet_exe_cli_path,
    "simulation",
    "enqueue",
    "--file", str(ENQUEUE_JSON),
    "--format", "JSON"
]
print("Running CLI enqueue:")
print(" ".join(enqueue_cmd))

enqueue_resp = run_cli_json(enqueue_cmd)
simulation_id, execution_id = extract_sim_and_exec_from_enqueue_response(enqueue_resp)

if not simulation_id:
    raise RuntimeError(f"Could not extract SimulationId from enqueue response:\n{json.dumps(enqueue_resp, indent=2)[:2000]}")

print(f"\nâœ“ Enqueued. SimulationId={simulation_id} ExecutionId={execution_id}")

if execution_id:
    final_rec = wait_for_execution_cli(execution_id, parent_simulation_id=simulation_id, poll_seconds=POLL_SECONDS)
else:
    final_rec = wait_for_simulation_cli(simulation_id, poll_seconds=POLL_SECONDS)

final_status = _normalize_status(final_rec.get("Status") or final_rec.get("status"))
print(f"\nâœ“ Simulation finished with status: {final_status}")

if final_status != "CompletedSuccess":
    print("Not completed successfully; skipping downloads.")
else:
    parent_base_dir = (EVA_RESULTS_DIR / str(simulation_id)) if use_csvoutput_only else (RESULTS_DIR / str(simulation_id))
    try:
        download_outputs_for_simulation(
            simulation_id,
            parent_base_dir,
            enqueue_model_name=enqueue_model_name,
            use_csvoutput_only=use_csvoutput_only
        )
        print(f"\nâœ“ Downloads complete under:\n{parent_base_dir}")
    except Exception as e_parent:
        print("\n" + "!" * 90)
        print("[WARN] Parent download failed. Will try Parquet Solution (stitch) simulation instead.")
        print("Parent download error:")
        print(str(e_parent))
        print("!" * 90 + "\n")

        parquet_sim_id = get_parquet_simulation_id_from_execution_cli(execution_id) if execution_id else None
        if not parquet_sim_id:
            raise RuntimeError(
                "Parent download failed, and could not locate a Parquet/stitch simulation in this execution for fallback.\n"
                f"ExecutionId={execution_id}\nOriginal error:\n{e_parent}"
            )

        parquet_base_dir = ((EVA_RESULTS_DIR / f"{simulation_id}__PARQUET_FALLBACK__{parquet_sim_id}") if use_csvoutput_only
                            else (RESULTS_DIR / f"{simulation_id}__PARQUET_FALLBACK__{parquet_sim_id}"))
        print(f"[INFO] Found Parquet/stitch SimulationId={parquet_sim_id}. Downloading fallback outputs -> {parquet_base_dir}")

        download_outputs_for_simulation(
            parquet_sim_id,
            parquet_base_dir,
            enqueue_model_name=enqueue_model_name,
            use_csvoutput_only=use_csvoutput_only
        )
        print(f"\nâœ“ Fallback downloads complete under:\n{parquet_base_dir}")





#### 20. EVA Results Manipulation and Capacity Scaler Creation (Including EVA Scenario Add)

In [ ]:
# ============================================================================
# EVA Capacity Scalers (retirement variables) + EVA Investments Units (EVA gens)
# + Default Battery Units (shared DB battery object, zonal CSV headers)
#
# GUARANTEES:
#  - EVA/Units generator logic: unchanged from your now-working version
#  - Retirement variable logic: preserved for non-EVA generators
#  - Outage Rating (enum 239) band = 3 for retirement rows
#  - Scenario name: X_EVA_Cap_Adjuster
#  - No extra duplicate "=" rows for retirement properties (only the × row is added)
#  - Default battery CSV columns remain zonal, e.g. Default_Battery_Storage_6_DE00
#  - Battery Units property is linked to ONE shared DB battery object:
#       Default_Battery_Storage_6
#  - Battery Units property uses:
#       enum = 25, collection id = 81
#  - FIX: raw int collection ids are converted to CollectionEnum(...) for Python.NET 3
#  - FIX: Generators in category "EVA" do NOT get retirement variable rows
#  - NEW FIX: Any generator whose name contains "New" also does NOT get retirement
#              variable rows, even if category lookup misses it
# ============================================================================

from __future__ import annotations

import os
import re
import csv
import zipfile
from pathlib import Path
from typing import Optional, Dict, List, Tuple, Set

import pandas as pd

COUNTRY_XLSX = Path(
    str(ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Country Specific.xlsx")
)
DEFAULT_XLSX = Path(
    str(ECONOMIC_TECHNICAL_INVESTMENT_DIR / "Economic and technical investment parameters_Default.xlsx")
)

CAP_ADJUSTER_SCENARIO_NAME = "X_EVA_Cap_Adjuster"
VARIABLE_CATEGORY_NAME = "EVA Capacity Scalers"
EVA_GENERATOR_CATEGORY_NAME = "EVA"

EVA_RESULTS_ROOT = Path(str(IMPORTER_DIR / r"Result_Downloads\EVA"))

OUTPUT_DIR = Path(str(DATA_FILES_ROOT / "Generator Data"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RET_SCALER_CSV_PATH = OUTPUT_DIR / "EVA_Capacity_Scaler.csv"
PLEXOS_REL_RET_SCALER = r"Data Files\Generator Data\EVA_Capacity_Scaler.csv"

EVA_INVEST_CSV_PATH = OUTPUT_DIR / "EVA_Investments.csv"
PLEXOS_REL_EVA_INVEST = r"Data Files\Generator Data\EVA_Investments.csv"

EVA_INVEST_BATTERY_CSV_PATH = OUTPUT_DIR / "EVA_Investment_Battery.csv"
PLEXOS_REL_EVA_INVEST_BATTERY = r"Data Files\Generator Data\EVA_Investment_Battery.csv"

DF_MAXCAP = r"Data Files\Generator Data\Generator_Max_Capacities.csv"
DF_STARTCOST = r"Data Files\Generator Data\Generator_Start_Cost.csv"
DF_RAMPUP = r"Data Files\Generator Data\Generator_Max_Ramp_Up.csv"
DF_RAMPDOWN = r"Data Files\Generator Data\Generator_Max_Ramp_Down.csv"
DF_OUTAGE = r"Data Files\Generator Data\Generator_Planned_Maintenance.csv"
DF_OFFTAKE = r"Data Files\Generator Data\Generator_Offtake_at_Start.csv"

ENUM_UNITS = 51
ENUM_OFFTAKE_AT_START = 1
ENUM_BATTERY_UNITS = 25
BATTERY_COLLECTION_ID = 81
SHARED_BATTERY_DB_OBJECT_NAME = "Default_Battery_Storage_6"

GEN_ENUMS_AND_DATAFILES: List[Tuple[str, int, str, int]] = [
    ("Max Capacity", 52, DF_MAXCAP, 1),
    ("Start Cost", 71, DF_STARTCOST, 1),
    ("Max Ramp Up", 104, DF_RAMPUP, 1),
    ("Max Ramp Down", 108, DF_RAMPDOWN, 1),
    ("Outage Rating", 239, DF_OUTAGE, 3),
]

ACTION_CANDIDATES = ["×"]
FUEL_TOKENS = ["Gas", "Hard coal", "Heavy Oil", "Light Oil", "Lignite", "Oil Shale", "Natural Gas", "Coal", "Oil"]
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS = [API_DIR / "PLEXOS_NET.dll", BIN_DIR / "PLEXOS_NET.dll"]
FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID = 1
START_FUEL_COLLECTION_IDS_TO_TRY = [8, 393]


def _latest_csvoutput_zip(root: Path) -> Path:
    zips = list(root.rglob("CSVOutput.zip"))
    if not zips:
        raise FileNotFoundError(f"No CSVOutput.zip found under: {root}")
    zips.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return zips[0]


def _detect_delimiter_from_text(sample: str) -> str:
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"]).delimiter
    except Exception:
        return ","


def _read_zip_csv(zf: zipfile.ZipFile, member: str, dtype=str) -> pd.DataFrame:
    with zf.open(member, "r") as f:
        raw = f.read()
    text = raw.decode("utf-8", errors="ignore")
    delim = _detect_delimiter_from_text(text[:4096])
    from io import StringIO
    return pd.read_csv(StringIO(text), sep=delim, dtype=dtype, engine="python")


def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _pick_datetime_and_value_columns(df: pd.DataFrame) -> Tuple[str, str]:
    cols = [str(c).strip() for c in df.columns]
    dt_candidates = [c for c in cols if c.strip().lower() in ("datetime", "date_time", "time", "timestamp")]
    dt_col = dt_candidates[0] if dt_candidates else cols[0]
    val_candidates = [c for c in cols if c != dt_col]
    if not val_candidates:
        raise RuntimeError(f"Units CSV has only one column; cannot find value column. Columns={cols}")
    return dt_col, val_candidates[0]


def _norm_key(name: str) -> str:
    s = str(name or "").strip().lower()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _sanitize_name(s: str) -> str:
    s = str(s).strip()
    s = s.replace("/", "_").replace("\\", "_").replace(":", "_")
    for ch in "()[]{}":
        s = s.replace(ch, "")
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ", "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s


def _load_bidding_zones(bz_xlsx: Path) -> List[str]:
    bz = pd.read_excel(bz_xlsx, sheet_name=0, header=None)
    vals = [str(v).strip() for v in bz.values.ravel() if not pd.isna(v) and str(v).strip()]
    return sorted(set(vals))


def _endswith_any_bz(name: str, bidding_zones: List[str]) -> bool:
    s = str(name or "").strip()
    for bz in bidding_zones:
        if s.endswith(f"_{bz}") or s == bz:
            return True
    return False


def _find_member_case_insensitive(zf: zipfile.ZipFile, preferred_member: str) -> Optional[str]:
    if preferred_member in zf.namelist():
        return preferred_member
    preferred_lc = preferred_member.lower()
    for n in zf.namelist():
        if n.lower() == preferred_lc:
            return n
    suffix_lc = preferred_member.split("/")[-1].lower()
    candidates = [n for n in zf.namelist() if n.lower().endswith(suffix_lc)]
    return candidates[0] if candidates else None


def _sheet_to_df(xlsx: Path, sheet: str) -> pd.DataFrame:
    if not xlsx.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(xlsx, sheet_name=sheet, header=0)
    except Exception:
        return pd.DataFrame()


def _find_col(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        for c in df.columns:
            if str(c).strip().lower() == str(n).strip().lower():
                return c
    return None


def _build_step_like_generator_names(country_xlsx: Path, default_xlsx: Path, bz_xlsx: Path) -> List[str]:
    bz = pd.read_excel(bz_xlsx, sheet_name=0, header=None)
    allowed_bzs = {str(v).strip() for v in bz.values.ravel() if not pd.isna(v) and str(v).strip()}
    if not allowed_bzs:
        raise RuntimeError("No bidding zones found in Bidding_Zone_List.xlsx")

    fom_country = _sheet_to_df(country_xlsx, "FOM")
    fom_default = _sheet_to_df(default_xlsx, "FOM")
    candidates: List[Dict[str, str]] = []
    for origin, df in (("country", fom_country), ("default", fom_default)):
        if df is None or df.empty:
            continue
        node_col = _find_col(df, ["Node", "MARKET_NODE", "NODE"])
        pem_col = _find_col(df, ["PEMMDB Technology", "PEMMDB_Technology", "PEMMDBTechnology"])
        ref_col = _find_col(df, ["Reference Technology", "Reference", "Reference_Technology"])
        if pem_col is None:
            continue
        for _, r in df.iterrows():
            node = str(r[node_col]).strip() if node_col else ""
            pem = str(r[pem_col]).strip()
            ref = str(r[ref_col]).strip() if ref_col else ""
            if origin == "default" and ref.strip().lower() == "retirement":
                continue
            if pem:
                candidates.append({"Node": node, "PEM": pem, "Ref": ref, "Origin": origin})
    uniq = {(c["Node"], c["PEM"], c["Ref"], c["Origin"]) for c in candidates}
    names: List[str] = []
    for node, pem, ref, origin in sorted(uniq):
        if origin == "country" and ((not str(node).strip()) or (str(node).strip() not in allowed_bzs)):
            continue
        is_battery = str(ref).strip() == "Battery Storage_6"
        is_dsr = ("DSR" in str(pem)) or ("DSR" in str(ref))
        if is_battery:
            continue
        obj_name = _sanitize_name(f"{node}_{ref}" if is_dsr else f"{node}_{pem}")
        if obj_name:
            names.append(obj_name)
    return sorted(set(names))


def _extract_fuel_token(pemmdb_tech: str) -> Optional[str]:
    s = str(pemmdb_tech or "").strip().lower()
    if not s:
        return None
    for tok in FUEL_TOKENS:
        if tok.lower() in s:
            return tok
    return None


def _build_retirement_tokens_from_default(default_xlsx: Path) -> List[str]:
    fom_df = _sheet_to_df(default_xlsx, "FOM")
    if fom_df.empty:
        raise RuntimeError("Default.xlsx sheet 'FOM' could not be read or is empty.")
    pem_col = _find_col(fom_df, ["PEMMDB Technology", "PEMMDB_Technology", "PEMMDBTechnology"])
    ref_col = _find_col(fom_df, ["Reference Technology", "Reference", "Reference_Technology"])
    if pem_col is None or ref_col is None:
        raise RuntimeError(f"Could not find required columns in FOM sheet. Found columns: {list(fom_df.columns)}")
    is_ret = fom_df[ref_col].astype(str).str.strip().str.lower() == "retirement"
    ret_fom = fom_df[is_ret].copy()
    ret_fom[pem_col] = ret_fom[pem_col].astype(str).str.strip()
    ret_fom = ret_fom[ret_fom[pem_col].astype(str).str.strip() != ""].copy()
    if ret_fom.empty:
        raise RuntimeError("No rows with Reference Technology == 'Retirement' found in Default.xlsx FOM sheet.")
    ret_fom["__FuelToken"] = ret_fom[pem_col].map(_extract_fuel_token)
    ret_fom = ret_fom[ret_fom["__FuelToken"].notna()].copy()
    tokens = sorted(set(str(t).strip() for t in ret_fom["__FuelToken"].tolist() if str(t).strip()))
    if not tokens:
        raise RuntimeError("After fuel-token extraction, no Retirement tokens remain.")
    return tokens


def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass
    import clr
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler
    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None
    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)
    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found.")
    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found.")
    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _get_objects_safe(db, class_enum):
    try:
        res = db.GetObjects(class_enum)
    except Exception:
        return []
    if res is None:
        return []
    try:
        return list(res)
    except Exception:
        return []


def _ensure_object(db, class_enum_value, name: str, *, add_to_system: bool = True, category: str = "", description: str = ""):
    existing = set(_get_objects_safe(db, class_enum_value))
    if name in existing:
        return
    db.AddObject(str(name), class_enum_value, bool(add_to_system), str(category or ""), str(description or ""))


def _resolve_collection_enum(CollectionEnum, required_tokens_lc: List[str], preferred_names: List[str], fallback_id: Optional[int] = None):
    for n in preferred_names:
        if hasattr(CollectionEnum, n):
            return getattr(CollectionEnum, n)
    candidates = []
    for attr in dir(CollectionEnum):
        if attr.startswith("_"):
            continue
        try:
            val = getattr(CollectionEnum, attr)
        except Exception:
            continue
        name_lc = str(attr).lower()
        if all(tok in name_lc for tok in required_tokens_lc):
            score = 0 if name_lc.startswith("system") else 5
            score += len(name_lc)
            candidates.append((score, attr, val))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        print(f"[INFO] Resolved CollectionEnum: {candidates[0][1]}")
        return candidates[0][2]
    if fallback_id is not None:
        print(f"[WARN] Using fallback collection id={fallback_id} for {required_tokens_lc}")
        return int(fallback_id)
    raise RuntimeError(f"Could not resolve CollectionEnum for tokens={required_tokens_lc}")


def _coerce_collection_enum(CollectionEnum, collection_enum):
    try:
        _ = collection_enum.ToString
        return collection_enum
    except Exception:
        pass
    try:
        return CollectionEnum(int(collection_enum))
    except Exception:
        return collection_enum


def _ensure_membership(db, CollectionEnum, collection_enum, parent_name: str, child_name: str) -> int:
    collection_enum = _coerce_collection_enum(CollectionEnum, collection_enum)
    try:
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))
    except Exception:
        try:
            db.AddMembership(collection_enum, parent_name, child_name)
        except Exception:
            pass
        return int(db.GetMembershipID(collection_enum, parent_name, child_name))


def _membership_exists(db, CollectionEnum, collection_enum, a: str, b: str) -> bool:
    collection_enum = _coerce_collection_enum(CollectionEnum, collection_enum)
    try:
        _ = int(db.GetMembershipID(collection_enum, a, b))
        return True
    except Exception:
        pass
    try:
        _ = int(db.GetMembershipID(collection_enum, b, a))
        return True
    except Exception:
        return False


def _resolve_start_fuel_collection(CollectionEnum):
    for cand in ["GeneratorStartFuels", "GeneratorStartFuel", "StartFuels", "StartFuelGenerators"]:
        if hasattr(CollectionEnum, cand):
            return getattr(CollectionEnum, cand), f"name:{cand}"
    for cid in START_FUEL_COLLECTION_IDS_TO_TRY:
        try:
            return CollectionEnum(int(cid)), f"id:{cid}"
        except Exception:
            continue
    return None, None


def _resolve_variable_profile_enum(db, SystemNS):
    prop_candidates = ["Profile", "Profiles", "Data File", "DataFile", "Filename"]
    collection_candidates = ["Variables", "SystemVariables", "Variable"]
    last = None
    for col in collection_candidates:
        for prop in prop_candidates:
            try:
                enum_id = int(db.PropertyName2EnumId(SystemNS.String("System"), SystemNS.String("Variable"), SystemNS.String(col), SystemNS.String(prop)))
                return enum_id, col, prop
            except Exception as e:
                last = (col, prop, str(e))
    raise RuntimeError(f"Could not resolve Variable Profile enum id. Last attempt: {last}")


def _resolve_semantic_enum(db_obj, system_ns, parent_class_name: str, child_class_name: str, collection_name: str, property_name: str, fallback_enum: int) -> int:
    try:
        return int(db_obj.PropertyName2EnumId(system_ns.String(parent_class_name), system_ns.String(child_class_name), system_ns.String(collection_name), system_ns.String(property_name)))
    except Exception as e:
        print(f"[WARN] Could not resolve enum for {property_name!r}; using fallback {fallback_enum}. Error: {e}")
        return int(fallback_enum)


def _add_property_row(db, mem_id: int, enum_id: int, band: int, value: float, DateFrom=None, DateTo=None, Variable=None, DataFile=None, Pattern=None, Scenario=None, Action=None):
    db.AddProperty(int(mem_id), int(enum_id), int(band), float(value), DateFrom, DateTo, Variable, DataFile, Pattern, Scenario, Action)


def _add_property_row_with_action_fallback(db, SystemNS, *, mem_id: int, enum_id: int, band: int, value: float, Variable=None, DataFile=None, Scenario=None) -> str:
    last_err = None
    for act in ACTION_CANDIDATES:
        try:
            _add_property_row(db, mem_id=mem_id, enum_id=enum_id, band=band, value=value, Variable=Variable, DataFile=DataFile, Scenario=Scenario, Action=SystemNS.String(act))
            return act
        except Exception as e:
            last_err = e
    try:
        _add_property_row(db, mem_id=mem_id, enum_id=enum_id, band=band, value=value, Variable=Variable, DataFile=DataFile, Scenario=Scenario, Action=None)
        return "<default>"
    except Exception as e:
        last_err = e
    raise RuntimeError(f"Could not add property row with any Action candidate. Last error: {last_err}")


def _try_get_category(db, class_enum_value, obj_name: str) -> Optional[str]:
    trials = [
        lambda: db.GetCategory(class_enum_value, obj_name),
        lambda: db.GetObjectCategory(class_enum_value, obj_name),
        lambda: db.GetCategory(obj_name, class_enum_value),
        lambda: db.GetObjectCategory(obj_name, class_enum_value),
    ]
    for fn in trials:
        try:
            res = fn()
            if res is None:
                continue
            s = str(res).strip()
            if s:
                return s
        except Exception:
            pass
    return None


def _build_generator_category_map(db, ClassEnum) -> Dict[str, str]:
    gens = sorted(set(str(x).strip() for x in _get_objects_safe(db, ClassEnum.Generator) if str(x).strip()))
    out: Dict[str, str] = {}
    for g in gens:
        cat = _try_get_category(db, ClassEnum.Generator, g)
        if cat:
            out[g] = str(cat).strip()
    return out


def _is_eva_like_generator(gen_name: str, generator_category_map: Dict[str, str]) -> bool:
    """
    Treat as EVA investment generator if:
      - category is EVA, OR
      - generator name contains 'New'
    """
    g = str(gen_name or "").strip()
    cat = str(generator_category_map.get(g, "")).strip().lower()
    if cat == EVA_GENERATOR_CATEGORY_NAME.lower():
        return True
    if "new" in g.lower():
        return True
    return False


def _select_retirement_generators(db, ClassEnum, CollectionEnum, generator_category_map: Optional[Dict[str, str]] = None) -> List[str]:
    tokens = _build_retirement_tokens_from_default(DEFAULT_XLSX)
    print(f"[INFO] STEP-derived Retirement fuel tokens: {tokens}")
    gens_in_db = sorted(set(str(x).strip() for x in _get_objects_safe(db, ClassEnum.Generator) if str(x).strip()))
    gen_fuel_collection = getattr(CollectionEnum, "GeneratorFuels") if hasattr(CollectionEnum, "GeneratorFuels") else getattr(CollectionEnum, "FuelGenerators") if hasattr(CollectionEnum, "FuelGenerators") else None
    fuels_in_db = sorted(set(str(x).strip() for x in _get_objects_safe(db, ClassEnum.Fuel) if str(x).strip()))
    retirement: List[str] = []
    if gen_fuel_collection is not None:
        fuels_lc = [(f, f.lower()) for f in fuels_in_db]
        token_to_db_fuels: Dict[str, List[str]] = {}
        for tok in tokens:
            tlc = tok.lower()
            matches = [fname for (fname, flc) in fuels_lc if tlc in flc]
            if matches:
                token_to_db_fuels[tok] = sorted(set(matches))
        all_relevant_db_fuels = sorted({f for fuels in token_to_db_fuels.values() for f in fuels})
        for g in gens_in_db:
            matched = False
            for fuel in all_relevant_db_fuels:
                if _membership_exists(db, CollectionEnum, gen_fuel_collection, g, fuel):
                    if any(tok.lower() in fuel.lower() for tok in tokens):
                        matched = True
                        break
            if matched:
                retirement.append(g)
    else:
        for g in gens_in_db:
            if any(tok.lower() in g.lower() for tok in tokens):
                retirement.append(g)

    if generator_category_map:
        retirement = [
            g for g in retirement
            if not _is_eva_like_generator(g, generator_category_map)
        ]

    retirement = sorted(set(retirement))
    print(f"[INFO] Retirement generators selected (non-EVA / non-New only): {len(retirement)}")
    return retirement


def step_eva_scalers_and_units():
    zip_path = _latest_csvoutput_zip(EVA_RESULTS_ROOT)
    print(f"[INFO] Using CSVOutput.zip: {zip_path}")
    allowed_bzs = _load_bidding_zones(BIDDING_ZONE_XLSX)
    print(f"[INFO] Bidding zones loaded: {len(allowed_bzs)}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        candidates = [n for n in zf.namelist() if n.lower().endswith("id2name.csv")]
        if not candidates:
            raise FileNotFoundError("id2name.csv not found inside CSVOutput.zip")
        id2name = _normalize_cols(_read_zip_csv(zf, candidates[0], dtype=str))
        cols_lc = {c.lower(): c for c in id2name.columns}
        for req in ("class", "id", "name"):
            if req not in cols_lc:
                raise RuntimeError(f"id2name.csv missing required column '{req}'. Found: {list(id2name.columns)}")
        c_class, c_id, c_name = cols_lc["class"], cols_lc["id"], cols_lc["name"]

        sub_gen = id2name[id2name[c_class].astype(str).str.strip().str.lower() == "generator"].copy()
        gen_map_exact: Dict[str, str] = {}
        gen_map_norm: Dict[str, str] = {}
        default_names_from_zip: List[str] = []
        for _, r in sub_gen.iterrows():
            gid, gnm = str(r[c_id]).strip(), str(r[c_name]).strip()
            if gid and gnm and gid.isdigit():
                gen_map_exact[gnm] = gid
                nk = _norm_key(gnm)
                if nk and nk not in gen_map_norm:
                    gen_map_norm[nk] = gid
                if gnm.startswith("Default_"):
                    default_names_from_zip.append(gnm)
        default_names_from_zip = sorted(set(default_names_from_zip))
        print(f"[INFO] Default_* generators found in ZIP id2name.csv: {len(default_names_from_zip)}")

        sub_bat = id2name[id2name[c_class].astype(str).str.strip().str.lower() == "battery"].copy()
        battery_map_exact: Dict[str, str] = {}
        battery_map_norm: Dict[str, str] = {}
        default_battery_names_from_zip: List[str] = []
        for _, r in sub_bat.iterrows():
            bid, bnm = str(r[c_id]).strip(), str(r[c_name]).strip()
            if bid and bnm and bid.isdigit():
                battery_map_exact[bnm] = bid
                nk = _norm_key(bnm)
                if nk and nk not in battery_map_norm:
                    battery_map_norm[nk] = bid
                if bnm.startswith("Default_") and _endswith_any_bz(bnm, allowed_bzs):
                    default_battery_names_from_zip.append(bnm)
        default_battery_names_from_zip = sorted(set(default_battery_names_from_zip))
        print(f"[INFO] Default_* batteries found in ZIP id2name.csv (matching BZ suffix): {len(default_battery_names_from_zip)}")

    db, ClassEnum, CollectionEnum, SystemNS = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    try:
        generator_category_map = _build_generator_category_map(db, ClassEnum)
        eva_category_gens = sorted(
            g for g, cat in generator_category_map.items()
            if str(cat).strip().lower() == EVA_GENERATOR_CATEGORY_NAME.lower()
        )
        new_name_gens = sorted(
            g for g in (str(x).strip() for x in _get_objects_safe(db, ClassEnum.Generator) if str(x).strip())
            if "new" in g.lower()
        )
        print(f"[INFO] Generators detected in category '{EVA_GENERATOR_CATEGORY_NAME}': {len(eva_category_gens)}")
        print(f"[INFO] Generators detected by name containing 'New': {len(new_name_gens)}")

        retirement_gens = _select_retirement_generators(
            db, ClassEnum, CollectionEnum, generator_category_map=generator_category_map
        )

        gens_in_db = set(str(x).strip() for x in _get_objects_safe(db, ClassEnum.Generator) if str(x).strip())
        fuels_in_db = set(str(x).strip() for x in _get_objects_safe(db, ClassEnum.Fuel) if str(x).strip())
        batteries_in_db: Set[str] = set(_get_objects_safe(db, ClassEnum.Battery)) if hasattr(ClassEnum, "Battery") else set()

        eva_investment_names = _build_step_like_generator_names(COUNTRY_XLSX, DEFAULT_XLSX, BIDDING_ZONE_XLSX)
        print(f"[INFO] STEP-like EVA generator names (generators only): {len(eva_investment_names)}")
        eva_investment_names_in_db = [n for n in eva_investment_names if n in gens_in_db]
        print(f"[INFO] STEP-like EVA generator names that exist in DB: {len(eva_investment_names_in_db)}")

        default_names_in_db = [n for n in default_names_from_zip if n in gens_in_db]
        if default_names_in_db:
            print(f"[INFO] Default_* generators (exact ZIP names) that exist in DB: {len(default_names_in_db)}")

        eva_investment_names_in_db_all = sorted(set(eva_investment_names_in_db + default_names_in_db))

        shared_battery_exists_in_db = SHARED_BATTERY_DB_OBJECT_NAME in batteries_in_db
        print(f"[INFO] Shared battery object '{SHARED_BATTERY_DB_OBJECT_NAME}' exists in DB: {shared_battery_exists_in_db}")

        with zipfile.ZipFile(zip_path, "r") as zf:
            ret_series: Dict[str, pd.Series] = {}
            for g in retirement_gens:
                gid = gen_map_exact.get(g) or gen_map_norm.get(_norm_key(g), "")
                if not gid:
                    continue
                member = _find_member_case_insensitive(zf, f"Year/LT Generator({gid}).Units.csv")
                if not member:
                    continue
                dfu = _normalize_cols(_read_zip_csv(zf, member, dtype=str))
                dt_col, val_col = _pick_datetime_and_value_columns(dfu)
                dt = dfu[dt_col].astype(str).str.strip()
                vals = pd.to_numeric(dfu[val_col], errors="coerce").fillna(0.0)
                col = f"{g} EVA"
                ret_series[col] = pd.Series(vals.values, index=dt.values, name=col).groupby(level=0).sum()

            if not ret_series:
                raise RuntimeError("Could not build retirement series; EVA_Capacity_Scaler.csv would be empty.")

            ret_df = pd.concat(ret_series.values(), axis=1).reset_index().rename(columns={"index": "DATETIME"})
            ret_df = ret_df.sort_values("DATETIME")
            ret_df.to_csv(RET_SCALER_CSV_PATH, index=False)
            print(f"[INFO] Created CSV: {RET_SCALER_CSV_PATH} (cols={len(ret_df.columns)-1})")

            master_index = pd.Index(ret_df["DATETIME"].astype(str).tolist(), name="DATETIME")

            eva_cols_data: Dict[str, pd.Series] = {}
            missing_id2name: List[str] = []
            missing_units_csv: List[str] = []

            for g in default_names_from_zip:
                gid = gen_map_exact.get(g)
                if not gid:
                    missing_id2name.append(g)
                    eva_cols_data[g] = pd.Series([0.0] * len(master_index), index=master_index, name=g)
                    continue
                member = _find_member_case_insensitive(zf, f"Year/LT Generator({gid}).Units.csv")
                if not member:
                    missing_units_csv.append(g)
                    eva_cols_data[g] = pd.Series([0.0] * len(master_index), index=master_index, name=g)
                    continue
                dfu = _normalize_cols(_read_zip_csv(zf, member, dtype=str))
                dt_col, val_col = _pick_datetime_and_value_columns(dfu)
                s = pd.Series(pd.to_numeric(dfu[val_col], errors="coerce").fillna(0.0).values, index=dfu[dt_col].astype(str).str.strip().values, name=g).groupby(level=0).sum()
                eva_cols_data[g] = s.reindex(master_index, fill_value=0.0)

            for g in eva_investment_names_in_db_all:
                if str(g).startswith("Default_") or g in eva_cols_data:
                    continue
                gid = gen_map_exact.get(g) or gen_map_norm.get(_norm_key(g), "")
                if not gid:
                    missing_id2name.append(g)
                    eva_cols_data[g] = pd.Series([0.0] * len(master_index), index=master_index, name=g)
                    continue
                member = _find_member_case_insensitive(zf, f"Year/LT Generator({gid}).Units.csv")
                if not member:
                    missing_units_csv.append(g)
                    eva_cols_data[g] = pd.Series([0.0] * len(master_index), index=master_index, name=g)
                    continue
                dfu = _normalize_cols(_read_zip_csv(zf, member, dtype=str))
                dt_col, val_col = _pick_datetime_and_value_columns(dfu)
                s = pd.Series(pd.to_numeric(dfu[val_col], errors="coerce").fillna(0.0).values, index=dfu[dt_col].astype(str).str.strip().values, name=g).groupby(level=0).sum()
                eva_cols_data[g] = s.reindex(master_index, fill_value=0.0)

            pd.DataFrame(eva_cols_data, index=master_index).reset_index().to_csv(EVA_INVEST_CSV_PATH, index=False)
            print(f"[INFO] Created CSV: {EVA_INVEST_CSV_PATH} (cols={len(eva_cols_data)})")

            if missing_id2name:
                print(f"[WARN] EVA investments gens missing in id2name.csv (filled with zeros): {len(missing_id2name)}")
            if missing_units_csv:
                print(f"[WARN] EVA investments gens missing Units CSV in ZIP (filled with zeros): {len(missing_units_csv)}")

            battery_cols_data: Dict[str, pd.Series] = {}
            missing_battery_id2name: List[str] = []
            missing_battery_units_csv: List[str] = []
            for b in default_battery_names_from_zip:
                bid = battery_map_exact.get(b) or battery_map_norm.get(_norm_key(b), "")
                if not bid:
                    missing_battery_id2name.append(b)
                    battery_cols_data[b] = pd.Series([0.0] * len(master_index), index=master_index, name=b)
                    continue
                member = _find_member_case_insensitive(zf, f"Year/LT Battery({bid}).Units.csv")
                if not member:
                    missing_battery_units_csv.append(b)
                    battery_cols_data[b] = pd.Series([0.0] * len(master_index), index=master_index, name=b)
                    continue
                dfu = _normalize_cols(_read_zip_csv(zf, member, dtype=str))
                dt_col, val_col = _pick_datetime_and_value_columns(dfu)
                s = pd.Series(pd.to_numeric(dfu[val_col], errors="coerce").fillna(0.0).values, index=dfu[dt_col].astype(str).str.strip().values, name=b).groupby(level=0).sum()
                battery_cols_data[b] = s.reindex(master_index, fill_value=0.0)

            pd.DataFrame(battery_cols_data, index=master_index).reset_index().to_csv(EVA_INVEST_BATTERY_CSV_PATH, index=False)
            print(f"[INFO] Created CSV: {EVA_INVEST_BATTERY_CSV_PATH} (cols={len(battery_cols_data)})")

        cap_scen = SystemNS.String(CAP_ADJUSTER_SCENARIO_NAME)
        _ensure_object(db, ClassEnum.Scenario, CAP_ADJUSTER_SCENARIO_NAME, add_to_system=True)

        try:
            db.AddCategory(ClassEnum.Variable, str(VARIABLE_CATEGORY_NAME))
        except Exception:
            pass

        gen_mem_collection = _resolve_collection_enum(
            CollectionEnum,
            ["system", "generator"],
            ["SystemGenerators", "Generators"],
            fallback_id=FALLBACK_SYSTEM_GENERATORS_COLLECTION_ID
        )
        var_mem_collection = getattr(CollectionEnum, "SystemVariables") if hasattr(CollectionEnum, "SystemVariables") else getattr(CollectionEnum, "Variables")
        start_fuel_collection, start_fuel_resolved_by = _resolve_start_fuel_collection(CollectionEnum)
        if start_fuel_collection is None:
            raise RuntimeError("Could not resolve Start Fuel collection (ids 8/392 + common names).")
        print(f"[INFO] Resolved Start Fuel collection via {start_fuel_resolved_by}")

        profile_enum, profile_col, profile_prop = _resolve_variable_profile_enum(db, SystemNS)
        print(f"[INFO] Resolved Variable profile: enum={profile_enum} via collection={profile_col!r} prop={profile_prop!r}")

        resolved_gen_enums_and_datafiles: List[Tuple[str, int, str, int]] = [
            ("Max Capacity", _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Max Capacity", 52), DF_MAXCAP, 1),
            ("Start Cost", _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Start Cost", 71), DF_STARTCOST, 1),
            ("Max Ramp Up", _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Max Ramp Up", 104), DF_RAMPUP, 1),
            ("Max Ramp Down", _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Max Ramp Down", 108), DF_RAMPDOWN, 1),
            ("Outage Rating", _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Outage Rating", 239), DF_OUTAGE, 3),
        ]
        for prop_name, enum_id, _, _ in resolved_gen_enums_and_datafiles:
            print(f"[INFO] Resolved Generator property enum: {prop_name} -> {enum_id}")

        resolved_generator_units_enum = _resolve_semantic_enum(db, SystemNS, "System", "Generator", "Generators", "Units", ENUM_UNITS)
        print(f"[INFO] Resolved Generator Units enum: {resolved_generator_units_enum}")

        vars_in_db = set(_get_objects_safe(db, ClassEnum.Variable))
        csv_cols = list(pd.read_csv(RET_SCALER_CSV_PATH, nrows=1).columns)
        var_names = [c for c in csv_cols if str(c).strip().lower() != "datetime"]

        created_vars = 0
        profile_rows = 0
        gen_rows_added = 0
        offtake_rows_added = 0
        eva_units_rows_added = 0
        eva_battery_units_rows_added = 0
        skipped_eva_variable_targets = 0
        skipped_new_variable_targets = 0
        action_used: Optional[str] = None

        for vcol in var_names:
            vname = str(vcol).strip()
            if not vname or not vname.endswith(" EVA"):
                continue

            gen_name = vname[:-4].strip()
            if gen_name not in gens_in_db:
                continue

            is_name_new = "new" in gen_name.lower()
            is_eva_like = _is_eva_like_generator(gen_name, generator_category_map)

            if is_eva_like:
                if is_name_new:
                    skipped_new_variable_targets += 1
                else:
                    skipped_eva_variable_targets += 1
                continue

            if vname not in vars_in_db:
                db.AddObject(str(vname), ClassEnum.Variable, True, VARIABLE_CATEGORY_NAME, "")
                vars_in_db.add(vname)
                created_vars += 1

            var_mem_id = _ensure_membership(db, CollectionEnum, var_mem_collection, "System", vname)
            _add_property_row(
                db,
                mem_id=var_mem_id,
                enum_id=profile_enum,
                band=1,
                value=0.0,
                DataFile=SystemNS.String(PLEXOS_REL_RET_SCALER),
                Scenario=cap_scen
            )
            profile_rows += 1

            gen_mem_id = _ensure_membership(db, CollectionEnum, gen_mem_collection, "System", gen_name)

            for _, enum_id, df_link, band in resolved_gen_enums_and_datafiles:
                used = _add_property_row_with_action_fallback(
                    db, SystemNS,
                    mem_id=gen_mem_id,
                    enum_id=enum_id,
                    band=band,
                    value=0.0,
                    Variable=SystemNS.String(vname),
                    DataFile=SystemNS.String(df_link),
                    Scenario=cap_scen
                )
                if action_used is None:
                    action_used = used
                gen_rows_added += 1

            for f in fuels_in_db:
                if not _membership_exists(db, CollectionEnum, start_fuel_collection, gen_name, f):
                    continue
                sf_mem_id = _ensure_membership(db, CollectionEnum, start_fuel_collection, gen_name, f)
                used = _add_property_row_with_action_fallback(
                    db, SystemNS,
                    mem_id=sf_mem_id,
                    enum_id=ENUM_OFFTAKE_AT_START,
                    band=1,
                    value=0.0,
                    Variable=SystemNS.String(vname),
                    DataFile=SystemNS.String(DF_OFFTAKE),
                    Scenario=cap_scen
                )
                if action_used is None:
                    action_used = used
                offtake_rows_added += 1
                break

        for g in eva_investment_names_in_db_all:
            mem_id = _ensure_membership(db, CollectionEnum, gen_mem_collection, "System", g)
            _add_property_row(
                db,
                mem_id=mem_id,
                enum_id=resolved_generator_units_enum,
                band=1,
                value=0.0,
                DataFile=SystemNS.String(PLEXOS_REL_EVA_INVEST),
                Scenario=cap_scen
            )
            eva_units_rows_added += 1

        if hasattr(ClassEnum, "Battery") and shared_battery_exists_in_db and default_battery_names_from_zip:
            battery_mem_collection = CollectionEnum(BATTERY_COLLECTION_ID)
            mem_id = _ensure_membership(db, CollectionEnum, battery_mem_collection, "System", SHARED_BATTERY_DB_OBJECT_NAME)
            _add_property_row(
                db,
                mem_id=mem_id,
                enum_id=ENUM_BATTERY_UNITS,
                band=1,
                value=0.0,
                DataFile=SystemNS.String(PLEXOS_REL_EVA_INVEST_BATTERY),
                Scenario=cap_scen
            )
            eva_battery_units_rows_added += 1
        elif default_battery_names_from_zip and not shared_battery_exists_in_db:
            print(f"[WARN] Battery CSV was created, but shared DB battery object '{SHARED_BATTERY_DB_OBJECT_NAME}' was not found. No battery Units row added.")

        print("\n[DONE]")
        print(f"  Scenario used:                        {CAP_ADJUSTER_SCENARIO_NAME}")
        print(f"  CSV created (retirement):             {RET_SCALER_CSV_PATH}")
        print(f"  CSV created (EVA Units):              {EVA_INVEST_CSV_PATH}")
        print(f"  CSV created (EVA Battery Units):      {EVA_INVEST_BATTERY_CSV_PATH}")
        print(f"  Default_* generator headers from ZIP: {len(default_names_from_zip)}")
        print(f"  Default_* battery headers from ZIP:   {len(default_battery_names_from_zip)}")
        print(f"  Variables created:                    {created_vars}")
        print(f"  Variable profile rows:                {profile_rows}")
        print(f"  NEW gen rows added:                   {gen_rows_added}")
        print(f"  NEW offtake rows added:               {offtake_rows_added}")
        print(f"  NEW EVA Units rows added:             {eva_units_rows_added}")
        print(f"  NEW EVA Battery Units rows added:     {eva_battery_units_rows_added}")
        print(f"  EVA generators skipped for variable multiplication: {skipped_eva_variable_targets}")
        print(f"  'New' generators skipped for variable multiplication: {skipped_new_variable_targets}")
        if action_used is not None:
            print(f"  Action token used:                    {action_used!r}")

    finally:
        try:
            db.Close()
        except Exception:
            pass


# ---- RUN ----
step_eva_scalers_and_units()

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import List, Optional, Tuple, Dict

# ============================================================
# STEP â€” Add single Scenario to Post-EVA / Post-EVA Full Models
#
# Purpose:
#   Add the Scenario object:
#       X_EVA_Cap_Adjuster
#
#   ...only to Models whose names match:
#       - Post-EVA:
#           Adequacy - TY2028 - 1 WS - S1 - Post_EVA
#           Adequacy - TY2028 - 1 WS - S1 - Post_EVA_LP
#       - Post-EVA Full:
#           Adequacy - TY2028 - 36 WS - S1 - Post_EVA
#           Adequacy - TY2028 - 36 WS - S1 - Post_EVA_LP
#
# Notes:
#   - Uses typed CollectionEnum conversion for collection 797
#   - Checks both membership directions
#   - Adds only missing memberships
#   - Stand-alone script
# ============================================================

# -----------------------------
# USER SETTINGS
# -----------------------------
SCENARIO_TO_ADD = "X_EVA_Cap_Adjuster"
DRY_RUN = False  # True => report only, do not create memberships

# -----------------------------
# Confirmed IDs
# -----------------------------
SCENARIO_CLASS_ID = 85
MODEL_CLASS_ID = 87
MODEL_SCENARIO_MEMBERSHIP_COLLECTION_ID = 798

# -----------------------------
# PLEXOS API dll folders
# -----------------------------
CORE_DLLS = [API_DIR / "PLEXOS_NET.Core.dll", BIN_DIR / "PLEXOS_NET.Core.dll"]
NET_DLLS  = [API_DIR / "PLEXOS_NET.dll",      BIN_DIR / "PLEXOS_NET.dll"]

# -----------------------------
# Model name patterns
# -----------------------------
# Matches e.g.:
#   Adequacy - TY2028 - 1 WS - S1 - Post_EVA
#   Adequacy - TY2035 - 36 WS - S1 - Post_EVA
#   Adequacy - TY2028 - 36 WS - S1 - Post_EVA_LP
#   Adequacy - TY2028 - 1 WS - S1 - Post_EVA_LP
RE_ADEQUACY_POST = re.compile(
    r"^\s*Adequacy\s*-\s*TY(\d{4})\s*-\s*(\d+)\s*WS\s*-\s*S(\d+)\s*-\s*Post[_\-\s]?EVA(?:[_\-\s].*)?\s*$",
    re.IGNORECASE,
)


def _infer_target_group_from_model_name(model_name: str) -> Optional[str]:
    """
    Returns:
        'Post-EVA'       for 1 WS Post_EVA models
        'Post-EVA Full'  for 36 WS Post_EVA models
        None             otherwise
    """
    n = (model_name or "").strip()
    m = RE_ADEQUACY_POST.match(n)
    if not m:
        return None

    ws = int(m.group(2))
    if ws == 1:
        return "Post-EVA"
    if ws == 36:
        return "Post-EVA Full"
    return None


# ============================================================
# Helpers
# ============================================================
def _bootstrap_plexos_net():
    for d in [API_DIR, BIN_DIR]:
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(d))
        except Exception:
            pass

    import clr  # noqa
    from System.Reflection import Assembly
    from System import AppDomain, ResolveEventHandler

    def _resolver(sender, args):
        name = args.Name.split(",")[0]
        for base in [API_DIR, BIN_DIR]:
            dll = base / f"{name}.dll"
            if dll.exists():
                return Assembly.LoadFile(str(dll))
        return None

    AppDomain.CurrentDomain.AssemblyResolve += ResolveEventHandler(_resolver)

    for p in CORE_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.Core.dll not found in API/BIN dirs.")

    for p in NET_DLLS:
        if p.exists():
            Assembly.LoadFile(str(p))
            break
    else:
        raise FileNotFoundError("PLEXOS_NET.dll not found in API/BIN dirs.")

    from PLEXOS_NET import Database
    from PLEXOS_NET.Enums import ClassEnum, CollectionEnum
    import System as SystemNS
    return Database(), ClassEnum, CollectionEnum, SystemNS


def _get_objects_safe(db, class_enum_or_id) -> List[str]:
    try:
        res = db.GetObjects(class_enum_or_id)
    except Exception:
        return []
    if res is None:
        return []
    try:
        out = []
        for x in list(res):
            s = str(x).strip()
            if s:
                out.append(s)
        return out
    except Exception:
        return []


def _collection_enum_from_id(SystemNS, CollectionEnum, collection_id: int):
    return SystemNS.Enum.ToObject(CollectionEnum, int(collection_id))


def _get_membership_id_any_direction(db, col_enum_value, a: str, b: str) -> Tuple[Optional[int], Optional[str]]:
    try:
        mid = int(db.GetMembershipID(col_enum_value, str(a), str(b)))
        return mid, "A->B"
    except Exception:
        pass

    try:
        mid = int(db.GetMembershipID(col_enum_value, str(b), str(a)))
        return mid, "B->A"
    except Exception:
        return None, None


def _ensure_membership_either_direction(
    db,
    col_enum_value,
    a: str,
    b: str,
    *,
    debug_first_error: List[str],
) -> Tuple[bool, Optional[int], Optional[str]]:
    # already exists?
    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return False, mid, direction

    if DRY_RUN:
        return True, None, None

    # try AddMembership(a, b)
    try:
        db.AddMembership(col_enum_value, str(a), str(b))
    except Exception as e:
        if not debug_first_error:
            debug_first_error.append(f"AddMembership(typedEnum, '{a}', '{b}') failed: {e}")

    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return True, mid, direction

    # try AddMembership(b, a)
    try:
        db.AddMembership(col_enum_value, str(b), str(a))
    except Exception as e:
        if not debug_first_error:
            debug_first_error.append(f"AddMembership(typedEnum, '{b}', '{a}') failed: {e}")

    mid, direction = _get_membership_id_any_direction(db, col_enum_value, a, b)
    if mid is not None:
        return True, mid, direction

    return False, None, None


# ============================================================
# MAIN
# ============================================================
def add_x_eva_cap_adjuster_to_post_eva_models():
    db, ClassEnum, CollectionEnum, SystemNS = _bootstrap_plexos_net()
    db.Connection(str(XML_PATH))

    scenario_class = getattr(ClassEnum, "Scenario", SCENARIO_CLASS_ID)
    model_class = getattr(ClassEnum, "Model", MODEL_CLASS_ID)
    col_797 = _collection_enum_from_id(SystemNS, CollectionEnum, MODEL_SCENARIO_MEMBERSHIP_COLLECTION_ID)

    try:
        all_scenarios = _get_objects_safe(db, scenario_class)
        if not all_scenarios:
            raise RuntimeError("No Scenario objects found in DB.")

        all_models = _get_objects_safe(db, model_class)
        if not all_models:
            raise RuntimeError("No Model objects found in DB.")

        # case-insensitive scenario lookup
        scenarios_in_db_lc: Dict[str, str] = {s.strip().lower(): s for s in all_scenarios if s.strip()}
        scenario_real_name = scenarios_in_db_lc.get(SCENARIO_TO_ADD.strip().lower())
        if scenario_real_name is None:
            raise RuntimeError(f"Scenario '{SCENARIO_TO_ADD}' was not found in the DB.")

        # select only matching models
        selected_models: List[str] = []
        selected_by_group: Dict[str, List[str]] = {"Post-EVA": [], "Post-EVA Full": []}

        for m in all_models:
            grp = _infer_target_group_from_model_name(m)
            if grp in ("Post-EVA", "Post-EVA Full"):
                selected_models.append(m)
                selected_by_group[grp].append(m)

        print("\n" + "=" * 100)
        print("STEP: Add X_EVA_Cap_Adjuster to Post-EVA / Post-EVA Full Models")
        print("=" * 100)
        print(f"[INFO] XML path:                         {XML_PATH}")
        print(f"[INFO] Scenario to add:                  {scenario_real_name}")
        print(f"[INFO] Total scenarios in DB:            {len(all_scenarios)}")
        print(f"[INFO] Total models in DB:               {len(all_models)}")
        print(f"[INFO] Matching Post-EVA models:         {len(selected_by_group['Post-EVA'])}")
        print(f"[INFO] Matching Post-EVA Full models:    {len(selected_by_group['Post-EVA Full'])}")
        print(f"[INFO] Total target models:              {len(selected_models)}")
        print(f"[INFO] DRY_RUN:                          {DRY_RUN}")

        if not selected_models:
            print("\n[WARN] No matching Post-EVA / Post-EVA Full model names were found.")
            print("[DONE]")
            return

        created = 0
        already = 0
        failed = 0
        direction_counts: Dict[str, int] = {"A->B": 0, "B->A": 0, "None": 0}

        first_error: List[str] = []
        fail_msgs: List[str] = []
        max_fail_print = 200

        for model_name in selected_models:
            mid, direction = _get_membership_id_any_direction(db, col_797, model_name, scenario_real_name)
            if mid is not None:
                already += 1
                direction_counts[direction or "None"] += 1
                continue

            created_now, mid2, direction2 = _ensure_membership_either_direction(
                db,
                col_797,
                model_name,
                scenario_real_name,
                debug_first_error=first_error,
            )

            if DRY_RUN:
                created += 1
                direction_counts["None"] += 1
                continue

            if mid2 is not None:
                if created_now:
                    created += 1
                else:
                    already += 1
                direction_counts[direction2 or "None"] += 1
            else:
                failed += 1
                if len(fail_msgs) < max_fail_print:
                    fail_msgs.append(
                        f"Failed membership: Model='{model_name}' <-> Scenario='{scenario_real_name}'"
                    )

        print("\n[RESULT]")
        print(f"  Memberships already present: {already}")
        print(f"  Memberships created:         {created}")
        print(f"  Memberships failed:          {failed}")
        print(f"  Direction used (in DB):      {direction_counts}")

        print("\n[INFO] Matched Post-EVA models:")
        for m in selected_by_group["Post-EVA"]:
            print(f"  - {m}")

        print("\n[INFO] Matched Post-EVA Full models:")
        for m in selected_by_group["Post-EVA Full"]:
            print(f"  - {m}")

        if first_error:
            print("\n[DEBUG] First AddMembership exception encountered:")
            print("  " + first_error[0])

        if fail_msgs:
            print("\n[FAILURES] (first {}):".format(len(fail_msgs)))
            for msg in fail_msgs:
                print("  - " + msg)

        print("\n[DONE]")

    finally:
        try:
            db.Close()
        except Exception:
            pass


if __name__ == "__main__":
    add_x_eva_cap_adjuster_to_post_eva_models()

####  21. Post-EVA Cloud Sync

In [ ]:
#%% Sync with PLEXOS Cloud (push changeset) + CAPTURE changeset id (with interactive commit message)
from eecloud.cloudsdk import CloudSDK
from eecloud.models import *
import re
from pathlib import Path
import pandas as pd

# -----------------------------
# INPUTS
# -----------------------------
dotnet_exe_cli_path = CLI_PATH
ACTIVE_STUDY_ID = require_study_id()
# Keep existing SDK init
pxc = CloudSDK()

# -----------------------------
# Build a default commit message including Bidding Zones (if available)
# -----------------------------
def read_bidding_zones(xlsx_path: Path) -> list:
    try:
        # Read first sheet, first column (assumes header in first row and data from row 2)
        df = pd.read_excel(xlsx_path, engine="openpyxl")  # openpyxl tends to be reliable for .xlsx
        if df.shape[1] >= 1:
            col = df.iloc[:, 0].dropna().astype(str).tolist()
            # If the first row is a header name rather than a zone, pandas will treat it as header already,
            # so col will be the data rows. This should match "row 2 and beyond".
            return [c.strip() for c in col if c.strip()]
    except Exception as e:
        print(f"Warning: could not read bidding zones from {xlsx_path}: {e}")
    return []

zones = read_bidding_zones(BIDDING_ZONE_XLSX)
zones_str = ", ".join(zones) if zones else ""
base_msg = "Post-EVA Cloud Sync"
default_commit = base_msg + (f". Bidding Zones: {zones_str}" if zones_str else "")

# -----------------------------
# Prompt the user for a commit message (GUI pop-up if possible, else console)
# -----------------------------
def ask_commit_message(default: str) -> str:
    # Try GUI prompt (tkinter). If that fails (no display, running headless), fall back to console input.
    try:
        import tkinter as tk
        from tkinter import simpledialog
        root = tk.Tk()
        root.withdraw()  # hide main window
        # show dialog with the default pre-filled
        result = simpledialog.askstring("Commit message", "Edit commit message for this changeset:", initialvalue=default)
        root.destroy()
        if result is None or str(result).strip() == "":
            # User cancelled or empty -> use default
            return default
        return str(result).strip()
    except Exception as e:
        # No GUI available: fallback to console input
        try:
            print("GUI not available, falling back to console input.")
            print("Press Enter to accept the default commit message shown below.")
            print("Default commit message:")
            print("  " + default)
            user_in = input("Enter commit message (or press Enter to accept default):\n> ")
            if user_in is None:
                return default
            user_in = user_in.strip()
            return user_in if user_in else default
        except Exception:
            # Last fallback: return default
            return default

commit_message = ask_commit_message(default_commit)
print("Using commit message:", commit_message)

# -----------------------------
# Push changeset and capture id (same as before)
# -----------------------------
command_responses = pxc.study.push_changeset(
    study_id=ACTIVE_STUDY_ID,
    commit_message=commit_message,
    print_message=True
)

last_command_response: CommandResponse[Contracts_PushChangesetResponse] = pxc.solution.get_final_response(command_responses)

# This variable will be used by the enqueue builder step later
PUSHED_CHANGESET_ID = None

GUID_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")

if last_command_response is not None:
    print("Push status:", last_command_response.Status)

if last_command_response is not None and last_command_response.Status == "Success":
    data: Contracts_PushChangesetResponse = last_command_response.EventData
    print(data)

    # Try common attribute names used across SDK versions
    for attr in ("ChangesetId", "ChangeSetId", "Id", "NewChangesetId", "CreatedChangesetId"):
        if hasattr(data, attr):
            v = getattr(data, attr)
            if v:
                PUSHED_CHANGESET_ID = str(v)
                break

    # Fallback: try extracting a GUID from the string representation
    if not PUSHED_CHANGESET_ID:
        m = GUID_RE.search(str(data))
        if m:
            PUSHED_CHANGESET_ID = m.group(0)

print("PUSHED_CHANGESET_ID:", PUSHED_CHANGESET_ID)




####  22. Create Adequacy Enqueue

In [ ]:
#%% Build Enqueue JSON for pushed changeset (else latest) â€” model name from generated JSON
import json
import subprocess
import re
import time
from pathlib import Path

# -----------------------------
# INPUTS
# -----------------------------
SIMULATION_ID = "b779a677-4412-4eff-b6d2-9f63190dad0d"
OUTPUT_DIRECTORY = str(IMPORTER_DIR)
dotnet_exe_cli_path = CLI_PATH

Path(OUTPUT_DIRECTORY).mkdir(parents=True, exist_ok=True)

GUID_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")

def run_cli(cmd: list[str]) -> str:
    cp = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return ((cp.stdout or "") + "\n" + (cp.stderr or "")).strip()

def extract_guids(text: str) -> list[str]:
    return GUID_RE.findall(text or "")

def get_latest_changeset_id_cli(study_id: str) -> str:
    variants = [
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "--studyId", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "--study-id", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", "-s", study_id],
        [dotnet_exe_cli_path, "study", "changeset", "get-latest", study_id],
    ]
    last_out = None
    for cmd in variants:
        out = run_cli(cmd)
        last_out = out
        gids = extract_guids(out)

        # Choose the last GUID that is not the study_id
        for g in reversed(gids):
            if g != study_id:
                return g

    raise RuntimeError(f"Could not parse latest changeset id from CLI output:\n{last_out}")

def make_filename_safe(name: str) -> str:
    bad = r'<>:"/\|?*'
    for ch in bad:
        name = name.replace(ch, "_")
    return name.strip().replace(" ", "_")

# -----------------------------
# Determine which changeset to use
# -----------------------------
changeset_id = None

if "PUSHED_CHANGESET_ID" in globals() and globals().get("PUSHED_CHANGESET_ID"):
    changeset_id = globals()["PUSHED_CHANGESET_ID"]

if not changeset_id:
    changeset_id = get_latest_changeset_id_cli(ACTIVE_STUDY_ID)

print("Using changeset id:", changeset_id)

# -----------------------------
# Build temp enqueue using CLI (no modelName!)
# -----------------------------
TEMP_FILE_STEM = "TEMP_ENQUEUE"
TEMP_JSON = f"{TEMP_FILE_STEM}.json"
TEMP_JSONL = f"{TEMP_FILE_STEM}.jsonl"

cmd = [
    dotnet_exe_cli_path,
    "simulation",
    "build-request-from-previous",
    "--outputDirectory", OUTPUT_DIRECTORY,
    "--simulationId", SIMULATION_ID,
    "--studyId", ACTIVE_STUDY_ID,
    "--changesetId", changeset_id,
    "--file", TEMP_JSON,
    "--overwrite"
]

print("Running CLI command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

temp_path = Path(OUTPUT_DIRECTORY) / TEMP_JSON
if not temp_path.exists():
    temp_path = Path(OUTPUT_DIRECTORY) / TEMP_JSONL
    if not temp_path.exists():
        raise RuntimeError(f"Temporary enqueue file not found: {TEMP_JSON} or {TEMP_JSONL}")

# -----------------------------
# Read model name from generated JSON and rename
# -----------------------------
with temp_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

models = payload.get("Models", [])
if not models:
    raise RuntimeError("No 'Models' field found in generated enqueue JSON.")

real_model_name = models[0]
safe_name = make_filename_safe(real_model_name)

final_path = Path(OUTPUT_DIRECTORY) / f"Enqueue_{safe_name}.json"

if final_path.exists():
    final_path.unlink()

temp_path.rename(final_path)

print("\nâœ“ Final enqueue file created:")
print(final_path)
print("âœ“ Models:", models)
print("âœ“ Changeset used:", changeset_id)

# -----------------------------
# 10-second pause before next cell
# -----------------------------
print("\nWaiting 10 seconds before continuing...")
for i in range(10, 0, -1):
    print(f"{i}...", end=" ", flush=True)
    time.sleep(1)

print("\nâœ“ Continuing to next step.")




####  23. Enqueue Adequacy and Download Results

In [ ]:
#%% Stand-alone: Enqueue from JSON (CLI), wait (CLI polling), download outputs (CLI)
from __future__ import annotations

from pathlib import Path
import subprocess
import json
import time
import re
from typing import Iterable, Optional

from eecloud.cloudsdk import CloudSDK

# -----------------------------
# INPUTS
# -----------------------------

# (A) OPTIONAL MANUAL OVERRIDE:
#     - Leave as None to auto-detect the newest Enqueue_*.json
#     - Or set to a specific file path string
ENQUEUE_JSON_MANUAL: Optional[str] = None
# Example:
# ENQUEUE_JSON_MANUAL = str(IMPORTER_DIR / "Enqueue_Adequacy_-_TY2035_-_36_WS_-_S1_-_Post_EVA.json")

# This is where your FIRST script writes the enqueue file
OUTPUT_DIRECTORY = str(IMPORTER_DIR)

RESULTS_DIR = IMPORTER_DIR / "Result_Downloads"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

dotnet_exe_cli_path = CLI_PATH
POLL_SECONDS = 10  # how often to poll status

# Needed for `pxc solution latest-id ...`
ACTIVE_STUDY_ID: Optional[str] = require_study_id()

# -----------------------------
# SDK init
# -----------------------------
pxc = CloudSDK(CLI_PATH)

# -----------------------------
# Helpers
# -----------------------------
def run_cli(cmd: list[str]) -> tuple[int, str, str]:
    cp = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return cp.returncode, (cp.stdout or "").strip(), (cp.stderr or "").strip()

def run_cli_json(cmd: list[str]):
    code, out, err = run_cli(cmd)
    if code != 0:
        raise RuntimeError(f"CLI failed (code {code}).\nSTDOUT:\n{out}\n\nSTDERR:\n{err}")

    text = (out if out else err).strip()
    try:
        return json.loads(text)
    except Exception:
        first_obj, last_obj = text.find("{"), text.rfind("}")
        first_arr, last_arr = text.find("["), text.rfind("]")
        if first_arr != -1 and last_arr != -1 and last_arr > first_arr:
            return json.loads(text[first_arr:last_arr+1])
        if first_obj != -1 and last_obj != -1 and last_obj > first_obj:
            return json.loads(text[first_obj:last_obj+1])
        raise RuntimeError(f"CLI output not parseable JSON.\nOutput:\n{text[:2000]}")

def extract_sim_and_exec_from_enqueue_response(resp) -> tuple[Optional[str], Optional[str]]:
    if isinstance(resp, list) and resp and isinstance(resp[0], dict):
        return resp[0].get("Id"), resp[0].get("ExecutionId")
    if isinstance(resp, dict):
        ed = resp.get("EventData", resp)
        if isinstance(ed, dict):
            if "Id" in ed:
                return ed.get("Id"), ed.get("ExecutionId")
            ss = ed.get("SimulationStarted") or ed.get("simulationStarted")
            if isinstance(ss, list) and ss and isinstance(ss[0], dict):
                s0 = ss[0]
                sim_id = s0.get("Id")
                exe_id = s0.get("ExecutionId")
                if isinstance(sim_id, dict) and "Value" in sim_id:
                    sim_id = sim_id["Value"]
                if isinstance(exe_id, dict) and "Value" in exe_id:
                    exe_id = exe_id["Value"]
                return sim_id, exe_id
    return None, None

def make_filename_safe(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_. \-]+', "_", (name or "")).strip().replace(" ", "_")

def _normalize_status(s: Optional[str]) -> str:
    return (s or "").strip()

def _is_terminal_status(status: str) -> bool:
    return status in {"CompletedSuccess", "CompletedFailed", "Cancelled", "Canceled", "Failed"}

def _coerce_records_from_cli_json(data) -> list[dict]:
    if isinstance(data, list):
        return [r for r in data if isinstance(r, dict)]
    if isinstance(data, dict):
        for k in ("SimulationRecords", "simulationRecords", "Records", "records", "Items", "items", "Value", "value"):
            v = data.get(k)
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
        if any(k in data for k in ("Status", "status")) and any(k in data for k in ("Id", "SimulationId", "simulationId")):
            return [data]
    return []

def get_simulation_status_cli(simulation_id: str) -> dict:
    variants = [
        [dotnet_exe_cli_path, "simulation", "list", "--simulationId", simulation_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "list-simulations", "--simulationId", simulation_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "get", "--simulationId", simulation_id, "--format", "JSON"],
    ]
    last_err = None
    for cmd in variants:
        try:
            data = run_cli_json(cmd)
            recs = _coerce_records_from_cli_json(data)
            if recs:
                return recs[0]
            last_err = f"Unexpected JSON shape: {type(data)} {str(data)[:200]}"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Could not fetch simulation status.\nLast error:\n{last_err}")

def wait_for_simulation_cli(simulation_id: str, poll_seconds: int = 10) -> dict:
    last = None
    while True:
        rec = get_simulation_status_cli(simulation_id)
        status = _normalize_status(rec.get("Status") or rec.get("status"))
        if status != last:
            print(f"[simulation {simulation_id}] Status: {status}")
            last = status
        if _is_terminal_status(status):
            return rec
        time.sleep(poll_seconds)

def list_simulations_by_execution_cli(execution_id: str) -> list[dict]:
    variants = [
        [dotnet_exe_cli_path, "simulation", "list", "--executionId", execution_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "list-simulations", "--executionId", execution_id, "--format", "JSON"],
        [dotnet_exe_cli_path, "simulation", "get", "--executionId", execution_id, "--format", "JSON"],
    ]
    last_err = None
    for cmd in variants:
        try:
            data = run_cli_json(cmd)
            recs = _coerce_records_from_cli_json(data)
            if recs:
                return recs
            last_err = f"Unexpected JSON shape: {type(data)} {str(data)[:200]}"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Could not list simulations by executionId.\nLast error:\n{last_err}")

def _looks_like_stitch_record(rec: dict) -> bool:
    name = (rec.get("Name") or rec.get("name") or rec.get("ModelName") or rec.get("modelName") or "")
    sim_type = (rec.get("Type") or rec.get("type") or rec.get("JobType") or rec.get("jobType") or "")
    desc = (rec.get("Description") or rec.get("description") or "")
    hay = f"{name} {sim_type} {desc}".lower()
    return ("stitch" in hay) or ("solution stitching" in hay) or ("solution-stitch" in hay) or ("parquet" in hay)

def get_parquet_simulation_id_from_execution_cli(execution_id: str) -> Optional[str]:
    try:
        recs = list_simulations_by_execution_cli(execution_id)
    except Exception:
        return None
    for r in recs:
        if _looks_like_stitch_record(r):
            sid = r.get("Id") or r.get("SimulationId") or r.get("simulationId")
            if sid:
                return str(sid)
    return None

def wait_for_execution_cli(execution_id: str, parent_simulation_id: str, poll_seconds: int = 10) -> dict:
    print(f"\n[execution {execution_id}] Waiting for ALL simulations in execution to finish (split-safe).")
    last_status_by_id: dict[str, str] = {}
    last_count = None

    while True:
        try:
            recs = list_simulations_by_execution_cli(execution_id)
        except Exception as e:
            print(f"[WARN] Could not list by executionId ({execution_id}). Falling back to parent-only wait.\n  Reason: {e}")
            return wait_for_simulation_cli(parent_simulation_id, poll_seconds=poll_seconds)

        if last_count != len(recs):
            print(f"[execution {execution_id}] Simulations discovered: {len(recs)}")
            last_count = len(recs)

        all_terminal = True
        stitch_found = False
        stitch_terminal = False

        for r in recs:
            sid = str(r.get("Id") or r.get("SimulationId") or r.get("simulationId") or "").strip()
            status = _normalize_status(r.get("Status") or r.get("status"))
            if sid:
                prev = last_status_by_id.get(sid)
                if status and status != prev:
                    label = r.get("Name") or r.get("ModelName") or r.get("Type") or ""
                    label = f" ({label})" if label else ""
                    print(f"[execution {execution_id}] [simulation {sid}]{label} Status: {status}")
                    last_status_by_id[sid] = status

            if not _is_terminal_status(status):
                all_terminal = False

            if _looks_like_stitch_record(r):
                stitch_found = True
                stitch_terminal = stitch_terminal or _is_terminal_status(status)

        ready = (all_terminal and stitch_terminal) if stitch_found else all_terminal
        if ready:
            print(f"[execution {execution_id}] âœ“ Execution is ready (all terminal; stitch terminal if present).")
            return get_simulation_status_cli(parent_simulation_id)

        time.sleep(poll_seconds)

# -----------------------------
# Auto-detect enqueue JSON
# -----------------------------
def _detect_enqueue_json_path() -> Path:
    # 1) Manual override wins
    if ENQUEUE_JSON_MANUAL:
        p = Path(ENQUEUE_JSON_MANUAL)
        if not p.exists():
            raise FileNotFoundError(f"Manual ENQUEUE_JSON_MANUAL not found: {p}")
        return p

    # 2) If the first script ran in same notebook/session, reuse its output if available
    #    (your first script creates a variable named 'final_path')
    for key in ("final_path", "FINAL_ENQUEUE_JSON_PATH", "ENQUEUE_JSON_PATH", "LAST_ENQUEUE_JSON"):
        try:
            v = globals().get(key)
        except Exception:
            v = None
        if v:
            p = Path(str(v))
            if p.exists() and p.suffix.lower() == ".json" and p.name.lower().startswith("enqueue_"):
                return p

    # 3) Otherwise, pick newest Enqueue_*.json in OUTPUT_DIRECTORY
    out_dir = Path(OUTPUT_DIRECTORY)
    if not out_dir.exists():
        raise FileNotFoundError(f"OUTPUT_DIRECTORY does not exist: {out_dir}")

    candidates = sorted(
        out_dir.glob("Enqueue_*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No Enqueue_*.json found in OUTPUT_DIRECTORY: {out_dir}")

    return candidates[0]

# -----------------------------
# Enqueue JSON: extract EXACT model name from Models[0]
# -----------------------------
def extract_model_name_from_enqueue_json(enqueue_path: Path) -> Optional[str]:
    try:
        j = json.loads(enqueue_path.read_text(encoding="utf-8"))
    except Exception:
        return None

    if isinstance(j, dict):
        models = j.get("Models") or j.get("models")
        if isinstance(models, list) and models and isinstance(models[0], str) and models[0].strip():
            return models[0].strip()

    return None

# -----------------------------
# Solution type detection + downloads
# -----------------------------
def solution_files_list_types_cli(solution_id: str) -> list[str]:
    data = run_cli_json([dotnet_exe_cli_path, "solution", "files", "list-types", "--solutionId", str(solution_id), "--format", "JSON"])
    if isinstance(data, list):
        return [str(x) for x in data]
    if isinstance(data, dict):
        for k in ("Types", "types", "Value", "value", "Items", "items"):
            v = data.get(k)
            if isinstance(v, list):
                return [str(x) for x in v]
    raise RuntimeError(f"Unexpected list-types JSON: {type(data)} {str(data)[:200]}")

def solution_latest_id_cli(study_id: str, model_name: str) -> str:
    data = run_cli_json([dotnet_exe_cli_path, "solution", "latest-id", "--studyId", study_id, "--model", model_name, "--format", "JSON"])
    if isinstance(data, str) and data.strip():
        return data.strip().strip('"')
    if isinstance(data, dict):
        for k in ("Id", "id", "Value", "value", "SolutionId", "solutionId"):
            v = data.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
    if isinstance(data, list) and data and isinstance(data[0], dict):
        for k in ("Id", "id", "Value", "value", "SolutionId", "solutionId"):
            v = data[0].get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
    raise RuntimeError(f"Could not parse latest-id JSON: {type(data)} {str(data)[:200]}")

def download_solution_outputs_cli(solution_id: str, out_dir: Path, type_name: str) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        dotnet_exe_cli_path,
        "solution",
        "files",
        "download",
        "--type", str(type_name),
        "--solutionId", str(solution_id),
        "--outputDirectory", str(out_dir),
    ]
    print(" ".join(cmd))
    code, out, err = run_cli(cmd)
    if code != 0:
        raise RuntimeError(f"Download failed for solution {solution_id} type {type_name}.\nSTDOUT:\n{out}\n\nSTDERR:\n{err}")

# --- PARQUET ONLY (requested) ---
def pick_parquet_type(available: Iterable[str]) -> Optional[str]:
    avail = {str(x) for x in available}
    for t in ("StitchedParquet", "Parquet", "StitchedHybrid"):
        if t in avail:
            return t
    return None

def download_parquet_only_for_solution(solution_id: str, out_dir: Path) -> None:
    types = solution_files_list_types_cli(solution_id)
    parquet_t = pick_parquet_type(types)

    if not parquet_t:
        raise RuntimeError(f"No Parquet type found for solution {solution_id}. Available: {types}")

    print(f"[INFO] Solution {solution_id} available types: {types}")
    print(f"[INFO] Downloading '{parquet_t}' (Parquet only) ...")
    download_solution_outputs_cli(solution_id, out_dir, parquet_t)

def download_parquet_only_with_latest_fallback(solution_id: str, out_dir: Path, *, study_id: Optional[str], enqueue_model_name: Optional[str]) -> None:
    def try_solution(sid: str):
        download_parquet_only_for_solution(str(sid), out_dir)

    # Try the provided id
    try:
        try_solution(solution_id)
        return
    except Exception as e1:
        if not (study_id and enqueue_model_name):
            raise RuntimeError(
                "Initial download failed and cannot try latest-id fallback (missing STUDY_ID or enqueue model name).\n"
                f"Original error: {e1}"
            )

        latest = solution_latest_id_cli(study_id, enqueue_model_name)
        print("\n" + "!" * 80)
        print("[WARN] Initial solution id was not downloadable. Falling back to solution latest-id for enqueue model:")
        print(f"       model='{enqueue_model_name}' -> latest solutionId={latest}")
        print("!" * 80 + "\n")

        try_solution(latest)

# -----------------------------
# SDK solution id discovery
# -----------------------------
def get_solution_ids_via_sdk(sim_id: str):
    responses = pxc.simulation.list_simulations(simulation_id=sim_id, print_message=False)
    final = pxc.solution.get_final_response(responses)
    if final is None or getattr(final, "Status", None) != "Success":
        raise RuntimeError(f"SDK list_simulations failed. Final={final}")
    data = getattr(final, "EventData", None)
    records = getattr(data, "SimulationRecords", None) or []
    if not records:
        raise RuntimeError("SDK returned no SimulationRecords for this simulation id.")
    sim_obj = records[0]
    model_identifiers = getattr(sim_obj, "ModelIdentifiers", []) or []
    if not model_identifiers:
        raise RuntimeError("No ModelIdentifiers found on simulation object; cannot determine solution ids.")
    return model_identifiers

def download_outputs_for_simulation(sim_id: str, base_dir: Path, *, enqueue_model_name: Optional[str]) -> None:
    model_identifiers = get_solution_ids_via_sdk(sim_id)
    base_dir.mkdir(parents=True, exist_ok=True)

    for mi in model_identifiers:
        model_name = getattr(mi, "Name", None) or getattr(mi, "ModelName", None) or "Model"
        solution_id = getattr(mi, "Id", None)
        if solution_id is None:
            raise RuntimeError("ModelIdentifier missing Id (solution id).")

        safe_model = make_filename_safe(model_name)
        out_dir = base_dir / safe_model

        print(f"\nDownloading outputs for model '{model_name}' (solution {solution_id}) -> {out_dir}")
        download_parquet_only_with_latest_fallback(
            str(solution_id),
            out_dir,
            study_id=ACTIVE_STUDY_ID,
            enqueue_model_name=enqueue_model_name,   # IMPORTANT: exact enqueue Models[0]
        )

# -----------------------------
# MAIN
# -----------------------------
ENQUEUE_JSON = _detect_enqueue_json_path()
print(f"[INFO] Using Enqueue JSON: {ENQUEUE_JSON}")

if not ENQUEUE_JSON.exists():
    raise FileNotFoundError(f"Enqueue JSON not found: {ENQUEUE_JSON}")

enqueue_model_name = extract_model_name_from_enqueue_json(ENQUEUE_JSON)
print(f"[INFO] Model name extracted from Enqueue JSON: {enqueue_model_name!r}")

enqueue_cmd = [
    dotnet_exe_cli_path,
    "simulation",
    "enqueue",
    "--file", str(ENQUEUE_JSON),
    "--format", "JSON"
]
print("Running CLI enqueue:")
print(" ".join(enqueue_cmd))

enqueue_resp = run_cli_json(enqueue_cmd)
simulation_id, execution_id = extract_sim_and_exec_from_enqueue_response(enqueue_resp)

if not simulation_id:
    raise RuntimeError(f"Could not extract SimulationId from enqueue response:\n{json.dumps(enqueue_resp, indent=2)[:2000]}")

print(f"\nâœ“ Enqueued. SimulationId={simulation_id} ExecutionId={execution_id}")

if execution_id:
    final_rec = wait_for_execution_cli(execution_id, parent_simulation_id=simulation_id, poll_seconds=POLL_SECONDS)
else:
    final_rec = wait_for_simulation_cli(simulation_id, poll_seconds=POLL_SECONDS)

final_status = _normalize_status(final_rec.get("Status") or final_rec.get("status"))
print(f"\nâœ“ Simulation finished with status: {final_status}")

if final_status != "CompletedSuccess":
    print("Not completed successfully; skipping downloads.")
else:
    parent_base_dir = RESULTS_DIR / str(simulation_id)
    try:
        download_outputs_for_simulation(simulation_id, parent_base_dir, enqueue_model_name=enqueue_model_name)
        print(f"\nâœ“ Downloads complete under:\n{parent_base_dir}")
    except Exception as e_parent:
        print("\n" + "!" * 90)
        print("[WARN] Parent download failed. Will try Parquet Solution (stitch) simulation instead.")
        print("Parent download error:")
        print(str(e_parent))
        print("!" * 90 + "\n")

        parquet_sim_id = get_parquet_simulation_id_from_execution_cli(execution_id) if execution_id else None
        if not parquet_sim_id:
            raise RuntimeError(
                "Parent download failed, and could not locate a Parquet/stitch simulation in this execution for fallback.\n"
                f"ExecutionId={execution_id}\nOriginal error:\n{e_parent}"
            )

        parquet_base_dir = RESULTS_DIR / f"{simulation_id}__PARQUET_FALLBACK__{parquet_sim_id}"
        print(f"[INFO] Found Parquet/stitch SimulationId={parquet_sim_id}. Downloading fallback outputs -> {parquet_base_dir}")

        download_outputs_for_simulation(parquet_sim_id, parquet_base_dir, enqueue_model_name=enqueue_model_name)
        print(f"\nâœ“ Fallback downloads complete under:\n{parquet_base_dir}")





#### 24. Sumarize Adequacy Results

In [ ]:
#%% STEP â€” Build WS ENS Distribution (Regional Unserved Energy) from downloaded solution (PARQUET ONLY)
from __future__ import annotations

from pathlib import Path
import re

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Jupyter inline display
from IPython.display import display, Image


# --------------------------------------------------------------------------------------
# USER INPUT
# --------------------------------------------------------------------------------------
# OPTION A (recommended for running in sequence):
#   Set SOLUTION_PATH = None and this script will auto-detect the newest downloaded solution
#   from the variables produced by your first script (parquet_base_dir / parent_base_dir / RESULTS_DIR + simulation_id).
SOLUTION_PATH: Path | None = None

# OPTION B (manual override):
# SOLUTION_PATH = IMPORTER_DIR / r"Result_Downloads\ddcb79f0-c8e1-4b36-bb1c-3f71affdac9a"

OUT_BASENAME = "WS_ENS_Distribution"

# Convert units (typical: MWh -> GWh). Set to 1.0 to keep native.
UNIT_DIVISOR = 1000.0
UNIT_LABEL = "GWh"

# How many parquet "data" files to include (stitch can be sharded)
MAX_DATA_FILES = 50


# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------
GUID_RE = re.compile(r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")

def _find_guid_root(p: Path) -> Path:
    for anc in [p] + list(p.parents):
        if GUID_RE.match(anc.name):
            return anc
    return p

def _duck_cols(con: duckdb.DuckDBPyConnection, fp: Path) -> list[str]:
    q = f"SELECT * FROM read_parquet('{fp.as_posix()}') LIMIT 0"
    return list(con.sql(q).df().columns)

def _has_cols(con: duckdb.DuckDBPyConnection, fp: Path, cols_required: list[str]) -> bool:
    try:
        cols = set(_duck_cols(con, fp))
        return set(cols_required).issubset(cols)
    except Exception:
        return False

def _pick_first_existing(cols: list[str], candidates: list[str]) -> str | None:
    s = set(cols)
    for c in candidates:
        if c in s:
            return c
    return None

def _coalesce_expr(colnames: list[str], candidates: list[str]) -> str | None:
    existing = [c for c in candidates if c in set(colnames)]
    if not existing:
        return None
    if len(existing) == 1:
        return existing[0]
    return "COALESCE(" + ", ".join(existing) + ")"

def _sample_sort_key(s: str) -> int:
    """
    Extract integer from strings like:
    '1', 'Sample 1', 'WS 12', etc.
    """
    s = str(s)
    match = re.search(r"\d+", s)
    if match:
        return int(match.group())
    return 10**9

def _write_outputs_and_plot(df_wide: pd.DataFrame, out_dir: Path) -> tuple[Path, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f"{OUT_BASENAME}.csv"
    png_path = out_dir / f"{OUT_BASENAME}.png"

    df_wide.to_csv(csv_path, index=False)

    # Plot stacked bars
    plot_df = df_wide.copy()
    if "Sample Name" not in plot_df.columns:
        raise ValueError("Expected 'Sample Name' column in final wide table.")

    sample_vals = plot_df["Sample Name"].astype(str).tolist()
    region_cols = [c for c in plot_df.columns if c != "Sample Name"]
    if not region_cols:
        raise ValueError("No Region columns found to plot.")

    x = np.arange(len(sample_vals))
    bottom = np.zeros(len(sample_vals), dtype=float)

    fig, ax = plt.subplots(figsize=(14, 7))
    for col in region_cols:
        y = plot_df[col].astype(float).values
        ax.bar(x, y, bottom=bottom, label=col)
        bottom = bottom + y

    ax.set_xticks(x)
    ax.set_xticklabels(sample_vals, rotation=90)
    ax.set_xlabel("Sample Name")
    ax.set_ylabel(f"Unserved Energy [{UNIT_LABEL}]")
    ax.set_title("WS Unserved Energy Distribution by Region (stacked)")
    ax.legend(ncols=3, frameon=False, fontsize=9)
    ax.margins(x=0.01)
    fig.tight_layout()

    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    display(Image(filename=str(png_path)))
    return csv_path, png_path


# --------------------------------------------------------------------------------------
# Autodetect: find newest downloaded solution folder from prior script outputs
# --------------------------------------------------------------------------------------
def _autodetect_solution_path_from_previous_script() -> Path:
    """
    Tries to locate the correct parquet solution folder produced by the enqueue/download script.
    Priority:
      1) parquet_base_dir (fallback stitch sim output)
      2) parent_base_dir  (normal output)
      3) RESULTS_DIR / simulation_id
    Then finds the newest FullKeyInfo.parquet under that base and returns its GUID root folder.
    """
    g = globals()

    candidates: list[Path] = []
    for varname in ("parquet_base_dir", "parent_base_dir"):
        v = g.get(varname)
        if v:
            try:
                p = Path(v)
                if p.exists():
                    candidates.append(p)
            except Exception:
                pass

    if not candidates:
        v_results = g.get("RESULTS_DIR")
        v_sim = g.get("simulation_id")
        if v_results and v_sim:
            try:
                p = Path(v_results) / str(v_sim)
                if p.exists():
                    candidates.append(p)
            except Exception:
                pass

    if not candidates:
        raise RuntimeError(
            "Auto-detect failed.\n"
            "Run the enqueue/download script in the SAME notebook/session first (so it defines parquet_base_dir/parent_base_dir or RESULTS_DIR+simulation_id),\n"
            "or set SOLUTION_PATH manually."
        )

    base = candidates[0]
    fk = sorted(base.rglob("FullKeyInfo.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not fk:
        raise RuntimeError(
            "Auto-detect found a base folder, but could not find any FullKeyInfo.parquet under it:\n"
            f"  base={base}\n"
            "This summary script is PARQUET-only; ensure you downloaded Parquet/StitchedParquet."
        )

    newest_fk = fk[0]
    sol_dir = _find_guid_root(newest_fk.parent)

    print("[INFO] Auto-detected solution folder from previous script:")
    print(f"       base:       {base}")
    print(f"       FullKeyInfo: {newest_fk}")
    print(f"       solution:    {sol_dir}")
    return sol_dir


# --------------------------------------------------------------------------------------
# Autodetect data parquet(s) under a solution folder
# --------------------------------------------------------------------------------------
def find_data_parquets(con: duckdb.DuckDBPyConnection, solution_dir: Path) -> list[Path]:
    all_pq = list(solution_dir.rglob("*.parquet"))
    candidates = []
    for fp in all_pq:
        if fp.name.lower() == "fullkeyinfo.parquet":
            continue

        path_lc = fp.as_posix().lower()
        score = 0
        if "/data/" in path_lc or "\\data\\" in str(fp).lower():
            score += 5
        if "datafileid=0" in path_lc:
            score += 5
        if "_data_" in fp.name.lower():
            score += 3

        ok = (
            _has_cols(con, fp, ["SeriesId", "Value"]) or
            _has_cols(con, fp, ["seriesId", "Value"]) or
            _has_cols(con, fp, ["SeriesId", "value"]) or
            _has_cols(con, fp, ["seriesId", "value"])
        )
        if ok:
            try:
                size = fp.stat().st_size
                mtime = fp.stat().st_mtime
            except Exception:
                size, mtime = 0, 0
            candidates.append((score, size, mtime, fp))

    candidates.sort(key=lambda t: (t[0], t[1], t[2]), reverse=True)
    return [t[3] for t in candidates[:MAX_DATA_FILES]]


# --------------------------------------------------------------------------------------
# PARQUET handler
# --------------------------------------------------------------------------------------
def build_ws_ens_from_parquet(solution_dir: Path, out_dir: Path) -> tuple[pd.DataFrame, Path, Path]:
    fk_candidates = sorted(solution_dir.rglob("FullKeyInfo.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not fk_candidates:
        raise FileNotFoundError(f"No FullKeyInfo.parquet found under: {solution_dir}")

    with duckdb.connect() as con:
        keys_fp = None
        for fp in fk_candidates:
            if _has_cols(con, fp, ["SeriesId"]) or _has_cols(con, fp, ["seriesId"]):
                keys_fp = fp
                break
        if keys_fp is None:
            raise FileNotFoundError("Found FullKeyInfo.parquet files, but none readable (missing SeriesId).")

        data_files = find_data_parquets(con, solution_dir)
        if not data_files:
            some = sorted([p.as_posix() for p in solution_dir.rglob("*.parquet")])[:60]
            raise FileNotFoundError(
                "Could not find ANY parquet files with (SeriesId, Value) under:\n"
                f"  {solution_dir}\n\n"
                "First ~60 parquet paths I did find:\n" + "\n".join(some)
            )

        kcols = _duck_cols(con, keys_fp)

        prop_expr   = _coalesce_expr(kcols, ["PropertyName", "propertyName"])
        series_expr = _coalesce_expr(kcols, ["SeriesId", "seriesId"])
        phase_expr  = _coalesce_expr(kcols, ["PhaseName", "phaseName"])

        if not prop_expr or not series_expr:
            raise RuntimeError(f"Keys parquet missing SeriesId/PropertyName: {keys_fp}")

        parent_class_name = _pick_first_existing(kcols, ["ParentClassName", "parentClassName"])
        child_class_name  = _pick_first_existing(kcols, ["ChildClassName", "childClassName"])
        parent_obj_name   = _pick_first_existing(kcols, ["ParentObjectName", "parentObjectName"])
        child_obj_name    = _pick_first_existing(kcols, ["ChildObjectName", "childObjectName"])

        sample_col = _pick_first_existing(kcols, ["SampleName", "sampleName", "Sample", "sample", "SampleId", "sampleId"])
        sample_expr = sample_col if sample_col else "'1'"

        region_expr = None
        if parent_class_name and parent_obj_name:
            region_expr = f"CASE WHEN lower({parent_class_name})='region' THEN {parent_obj_name} ELSE NULL END"
        if child_class_name and child_obj_name:
            child_case = f"CASE WHEN lower({child_class_name})='region' THEN {child_obj_name} ELSE NULL END"
            region_expr = child_case if region_expr is None else f"COALESCE({region_expr}, {child_case})"
        if parent_obj_name:
            region_expr = parent_obj_name if region_expr is None else f"COALESCE({region_expr}, {parent_obj_name})"
        if child_obj_name:
            region_expr = child_obj_name if region_expr is None else f"COALESCE({region_expr}, {child_obj_name})"
        if region_expr is None:
            raise RuntimeError(f"Could not determine Region name columns in keys parquet: {keys_fp}")

        data_list_sql = ", ".join([f"'{p.as_posix()}'" for p in data_files])

        where_phase = ""
        if phase_expr:
            where_phase = f" AND lower(k.PhaseName)='st' "

        query = f"""
            WITH
            d AS (
                SELECT SeriesId, Value
                FROM read_parquet([{data_list_sql}])
            ),
            k AS (
                SELECT
                    {series_expr} AS SeriesId,
                    {sample_expr} AS SampleName,
                    {region_expr} AS RegionName,
                    {prop_expr} AS PropertyName
                    {"," + phase_expr + " AS PhaseName" if phase_expr else ""}
                FROM read_parquet('{keys_fp.as_posix()}')
            )
            SELECT
                CAST(k.SampleName AS VARCHAR) AS SampleName,
                CAST(k.RegionName AS VARCHAR) AS Region,
                SUM(d.Value) AS TotalUnserved
            FROM d
            JOIN k USING (SeriesId)
            WHERE lower(k.PropertyName) IN (
                    'unserved energy','unserved_energy','unservedenergy',
                    'ens','expected not served','energy not served'
                  )
              AND k.RegionName IS NOT NULL
              AND lower(CAST(k.RegionName AS VARCHAR)) <> 'dummy'
              {where_phase}
            GROUP BY 1, 2
            ORDER BY 1, 2
        """

        df_long = con.sql(query).df()

        if df_long.empty:
            props = con.sql(
                f"SELECT DISTINCT lower({prop_expr}) AS prop FROM read_parquet('{keys_fp.as_posix()}') ORDER BY 1 LIMIT 200"
            ).df()
            raise RuntimeError(
                "No rows matched the Unserved Energy property in this parquet.\n"
                "First properties found (lowercased):\n"
                + props.to_string(index=False)
            )

    df_long["TotalUnserved"] = df_long["TotalUnserved"].astype(float) / float(UNIT_DIVISOR)

    df_long["SampleSort"] = df_long["SampleName"].map(_sample_sort_key)
    df_long = df_long.sort_values(["SampleSort", "Region"]).drop(columns=["SampleSort"])

    df_wide = df_long.pivot_table(
        index="SampleName",
        columns="Region",
        values="TotalUnserved",
        aggfunc="sum",
        fill_value=0.0
    ).reset_index().rename(columns={"SampleName": "Sample Name"})

    df_wide["_SampleSort"] = df_wide["Sample Name"].map(_sample_sort_key)
    df_wide = df_wide.sort_values("_SampleSort").drop(columns=["_SampleSort"]).reset_index(drop=True)

    csv_path, png_path = _write_outputs_and_plot(df_wide, out_dir)
    return df_wide, csv_path, png_path


# --------------------------------------------------------------------------------------
# MAIN (PARQUET ONLY)
# --------------------------------------------------------------------------------------
if SOLUTION_PATH is None:
    SOLUTION_PATH = _autodetect_solution_path_from_previous_script()

if not SOLUTION_PATH.is_dir():
    raise ValueError(f"SOLUTION_PATH must be a directory containing parquet outputs. Got: {SOLUTION_PATH}")

out_dir = SOLUTION_PATH
df_wide, csv_path, png_path = build_ws_ens_from_parquet(SOLUTION_PATH, out_dir)

print("\nFinal WS ENS Distribution (first 20 rows):")
print(df_wide.head(37).to_string(index=False))

print(f"\nâœ“ Saved CSV: {csv_path}")
print(f"âœ“ Saved PNG: {png_path}")


